# Reconstrução de cenas com NeRFacto e Splatfacto

**Código experimental do Trabalho de Conclusão de Curso (TCC).**

Este notebook implementa o preparo dos dados, o treinamento, a seleção de checkpoints com o conjunto de validação e a avaliação dos modelos **NeRFacto-Big** e **Splatfacto**. A análise utiliza **PSNR, SSIM, LPIPS** e detecções do **YOLOv8x** nos cenários **nuScenes (scene-0655)**, **estacionamento** e **Uber**.

### Ambiente e execução

- **Ambiente:** Google Colab com GPU NVIDIA A100, acesso à internet e espaço suficiente para datasets, dependências e checkpoints.
- **Entradas:** `TCC_DATA_ROOT` indica a raiz dos dados do nuScenes; `TCC_H_VIDEO` e `TCC_I_VIDEO` indicam os arquivos de vídeo de estacionamento e Uber, respectivamente. Configure essas variáveis no ambiente de execução antes de iniciar as células correspondentes.
- **Saídas:** arquivos intermediários são produzidos no ambiente de execução; `TCC_EXPORT_DIR` pode ser configurada para exportar os pacotes finais.
- **Ordem:** execute **0 → SETUP → 1**, depois **A–G** (nuScenes), **H** (estacionamento) e **I** (Uber), conforme os cenários desejados.
- **Divisão dos dados:** 95 imagens de treinamento, 10 de validação e 10 de teste em cada cenário.


In [ ]:
# ============================================================
# CÉLULA 0 — CONFIGURAÇÃO DO AMBIENTE EXPERIMENTAL
#
# Host: Python do Google Colab.
# Ambiente dos experimentos: Python 3.12.13 em /content/tcc_env.
# O ambiente isolado e as dependências são instalados pela célula SETUP.
# Esta célula prepara as variáveis e verifica os recursos disponíveis.
# ============================================================

import importlib.metadata as metadata
import json
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path


# ============================================================
# 1. VERSÕES DO AMBIENTE EXPERIMENTAL (NÃO DO HOST COLAB)
# ============================================================

VERSIONS = {
    "python": "3.12.13",             # Python interno do tcc_env.
    "torch": "2.11.0+cu128",         # Instalação PRÓPRIA do tcc_env.
    "cuda": "12.8",                 # CUDA associada ao PyTorch.
    "numpy_tcc": "1.26.4",
    "pandas_tcc": "2.2.2",
    "nerfstudio": "1.1.5",
    "ultralytics": "8.4.132",
    "opencv": "4.11.0.86",           # Versão pip, não cv2.__version__.
    "openpyxl": "3.1.5",
    "gsplat": "1.4.0",
    "nerfacc": "0.5.2",
    "setuptools": "81.0.0",
}

# O SETUP deve criar um ambiente realmente isolado. NÃO utilizar
# python -m venv --system-site-packages no host Python 3.13.
TCC_ENV_ISOLATED = True
TCC_USE_SYSTEM_SITE_PACKAGES = False
TCC_PYTHON_INSTALL_VERSION = VERSIONS["python"]
TCC_TORCH_INDEX_URL = "https://download.pytorch.org/whl/cu128"

TCNN_SOURCE = (
    "git+https://github.com/NVlabs/tiny-cuda-nn/"
    "#subdirectory=bindings/torch"
)
TCNN_CUDA_ARCHITECTURES = "80"  # NVIDIA A100 = sm_80.
REQUIRE_TCNN = True
EXPECTED_GPU_NAME_CONTAINS = "A100"


# ============================================================
# 2. ALIASES ESPERADOS PELAS CÉLULAS EXISTENTES
# ============================================================

PYTHON_VERSION_ESPERADA = VERSIONS["python"]
PYTORCH_VERSION_ESPERADA = VERSIONS["torch"]
CUDA_VERSION_ESPERADA = VERSIONS["cuda"]
NUMPY_TCC_ESPERADO = VERSIONS["numpy_tcc"]
PANDAS_TCC_ESPERADO = VERSIONS["pandas_tcc"]
NERFSTUDIO_ESPERADO = VERSIONS["nerfstudio"]
ULTRALYTICS_ESPERADO = VERSIONS["ultralytics"]
OPENCV_TCC_ESPERADO = VERSIONS["opencv"]
OPENPYXL_ESPERADO = VERSIONS["openpyxl"]
GSPLAT_ESPERADO = VERSIONS["gsplat"]
NERFACC_ESPERADO = VERSIONS["nerfacc"]


# ============================================================
# 3. CAMINHOS DO PIPELINE — entrada configurável, saídas locais
# ============================================================

CONTENT_ROOT = Path("/content")
# Diretório raiz do dataset fornecido durante a execução.
TCC_ROOT = Path(os.environ.get("TCC_DATA_ROOT", "/content/tcc_data")).expanduser()
TCC_ENV = CONTENT_ROOT / "tcc_env"
TCC_ENV_BIN = TCC_ENV / "bin"
TCC_PYTHON = TCC_ENV_BIN / "python"
NS_TRAIN = TCC_ENV_BIN / "ns-train"
NS_EVAL = TCC_ENV_BIN / "ns-eval"
NS_RENDER = TCC_ENV_BIN / "ns-render"
OUTPUT_DATASET = CONTENT_ROOT / "nuscenes_nerfstudio_all_cams"
OUTPUT_DIR = CONTENT_ROOT / "outputs_nuscenes"
LOG_DIR = CONTENT_ROOT / "logs_nuscenes"
RESULTS_DIR = CONTENT_ROOT / "results_nuscenes"
SETUP_INFO = TCC_ENV / "tcc_environment.json"
CONSTRAINTS_FILE = CONTENT_ROOT / "tcc_constraints.txt"


# ============================================================
# 4. AMBIENTE DOS SUBPROCESSOS TCC (SEM AFETAR O KERNEL)
# ============================================================

def build_tcc_subprocess_env():
    """Ambiente próprio para comandos em /content/tcc_env.

    O SETUP criará o venv com Python 3.12; os subprocessos NÃO
    herdarão bibliotecas Python 3.13 do host via PYTHONPATH.
    """
    env = os.environ.copy()

    # Remover possíveis caminhos Python herdados do kernel.
    for key in ("PYTHONPATH", "PYTHONHOME", "PYTHONUSERBASE"):
        env.pop(key, None)

    env["PATH"] = f"{TCC_ENV_BIN}:" + env.get("PATH", "")
    env["VIRTUAL_ENV"] = str(TCC_ENV)
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONNOUSERSITE"] = "1"

    # Configuração de alocação de memória para os subprocessos.
    env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    env["TCNN_CUDA_ARCHITECTURES"] = TCNN_CUDA_ARCHITECTURES

    # SOMENTE para checkpoints de origem confiável criados pelo TCC.
    # Nunca utilize este ambiente para carregar checkpoints desconhecidos.
    env["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
    return env


TCC_SUBPROCESS_ENV = build_tcc_subprocess_env()


# ============================================================
# 5. DIAGNÓSTICO DO HOST (PYTHON 3.13 É ACEITO)
# ============================================================

def package_version(package_name):
    """Consulta metadados sem importar bibliotecas científicas."""
    try:
        return metadata.version(package_name)
    except metadata.PackageNotFoundError:
        return "não instalado"


def optional_command(args, *, timeout=45):
    """Captura um comando de diagnóstico sem modificar a VM."""
    try:
        p = subprocess.run(
            [str(a) for a in args],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            timeout=timeout,
            check=False,
        )
        return {"returncode": p.returncode, "output": p.stdout.strip()}
    except (FileNotFoundError, subprocess.TimeoutExpired, OSError) as exc:
        return {"returncode": None, "output": str(exc)}


PYTHON_ATUAL = ".".join(str(v) for v in sys.version_info[:3])
PYTHON_SERIE = ".".join(str(v) for v in sys.version_info[:2])
COLAB_PYTHON_VERSION = PYTHON_ATUAL
COLAB_PYTHON_EXECUTABLE = sys.executable

print("=" * 100)
print("CÉLULA 0 — HOST COLAB E AMBIENTE EXPERIMENTAL ISOLADO")
print("=" * 100)
print("Python do HOST:", PYTHON_ATUAL)
print("Executável HOST:", sys.executable)
print("Python exigido no TCC:", VERSIONS["python"])
print("PyTorch exigido no TCC:", VERSIONS["torch"])
print("CUDA exigida no TCC:", VERSIONS["cuda"])

if PYTHON_ATUAL != VERSIONS["python"]:
    print("HOST e TCC usam versões distintas de Python: PREVISTO.")
    print("O SETUP deve criar Python 3.12.13 isolado; não herdar pacotes do host.")
else:
    print("HOST e TCC têm a mesma versão de Python; venv isolado continua obrigatório.")

BASE_PACKAGE_SNAPSHOT = {
    "numpy": package_version("numpy"),
    "pandas": package_version("pandas"),
    "torch": package_version("torch"),
    "torchvision": package_version("torchvision"),
    "opencv-python": package_version("opencv-python"),
    "opencv-python-headless": package_version("opencv-python-headless"),
}

print("\n" + "=" * 100)
print("PACOTES DO HOST (REGISTRO; NÃO SÃO PINS DO TCC)")
print("=" * 100)
for name, version in BASE_PACKAGE_SNAPSHOT.items():
    print(f"{name}: {version}")


# ============================================================
# 6. DIAGNÓSTICOS OPCIONAIS DE GPU, NVCC E TORCH DO HOST
# ============================================================

# A GPU da VM é compartilhada pelo host e pelo tcc_env, mas as
# bibliotecas Python e o PyTorch do tcc_env serão instalados à parte.
SMI_PROBE = optional_command([
    "nvidia-smi",
    "--query-gpu=name,driver_version,memory.total",
    "--format=csv,noheader",
])

GPU_RUNTIME_INFO = {
    "nvidia_smi_returncode": SMI_PROBE["returncode"],
    "nvidia_smi_output": SMI_PROBE["output"],
    "name": None,
    "driver": None,
    "memory_total": None,
}

if SMI_PROBE["returncode"] == 0 and SMI_PROBE["output"]:
    first_line = SMI_PROBE["output"].splitlines()[0]
    parts = [p.strip() for p in first_line.split(",", 2)]
    if len(parts) == 3:
        GPU_RUNTIME_INFO.update({
            "name": parts[0],
            "driver": parts[1],
            "memory_total": parts[2],
        })

NVCC_PROBE = optional_command(["nvcc", "--version"])

# Diagnóstico OPCIONAL: o PyTorch do host não precisa ser importável
# para o venv isolado funcionar. Importamos só em um subprocesso.
TORCH_PROBE_CODE = r'''
import json
import torch
available = bool(torch.cuda.is_available())
print("TCC_HOST_TORCH_JSON=" + json.dumps({
    "torch": str(torch.__version__),
    "cuda": str(torch.version.cuda) if torch.version.cuda else None,
    "cuda_available": available,
    "gpu": torch.cuda.get_device_name(0) if available else None,
    "capability": list(torch.cuda.get_device_capability(0)) if available else None,
}))
'''

host_torch_probe = optional_command(
    [sys.executable, "-c", TORCH_PROBE_CODE], timeout=120
)
TORCH_RUNTIME_INFO = {
    "torch": None,
    "cuda": None,
    "cuda_available": False,
    "gpu": None,
    "capability": None,
    "probe_error": None,
}

if host_torch_probe["returncode"] == 0:
    sentinel = "TCC_HOST_TORCH_JSON="
    lines = [
        line[len(sentinel):]
        for line in host_torch_probe["output"].splitlines()
        if line.startswith(sentinel)
    ]
    if lines:
        try:
            TORCH_RUNTIME_INFO.update(json.loads(lines[-1]))
        except json.JSONDecodeError:
            TORCH_RUNTIME_INFO["probe_error"] = "JSON inválido no diagnóstico do host."
    else:
        TORCH_RUNTIME_INFO["probe_error"] = "Diagnóstico do host não retornou JSON."
else:
    TORCH_RUNTIME_INFO["probe_error"] = host_torch_probe["output"][-1500:]

print("\n" + "=" * 100)
print("DIAGNÓSTICO DA GPU — HOST")
print("=" * 100)
print("nvidia-smi:", SMI_PROBE["output"] or "indisponível")
print("nvcc:", NVCC_PROBE["output"].splitlines()[-1]
      if NVCC_PROBE["returncode"] == 0 and NVCC_PROBE["output"]
      else "indisponível — verificar no SETUP")
print("PyTorch HOST:", TORCH_RUNTIME_INFO["torch"] or "indisponível")
print("CUDA PyTorch HOST:", TORCH_RUNTIME_INFO["cuda"] or "indisponível")
print("CUDA disponível HOST:", TORCH_RUNTIME_INFO["cuda_available"])
print("GPU HOST:", TORCH_RUNTIME_INFO["gpu"] or "não detectada pelo PyTorch")
if TORCH_RUNTIME_INFO["probe_error"]:
    print("Aviso diagnóstico PyTorch HOST:", TORCH_RUNTIME_INFO["probe_error"])

# A célula 0 não considera versões do PyTorch do host como válidas
# para o ambiente experimental; isso será confirmado pela Célula 1.
if (
    BASE_PACKAGE_SNAPSHOT["torch"] != VERSIONS["torch"]
    or TORCH_RUNTIME_INFO["cuda"] != VERSIONS["cuda"]
):
    print("AVISO: stack do HOST difere da configuração experimental.")
    print("Isso não impede o TCC: o SETUP instalará o stack fixo no tcc_env.")
else:
    print("Stack HOST coincide com os pins TCC (apenas diagnóstico).")

GPU_DETECTED_NAME = GPU_RUNTIME_INFO["name"] or TORCH_RUNTIME_INFO["gpu"]
if not GPU_DETECTED_NAME:
    print("AVISO: GPU NVIDIA não detectada agora; selecione GPU antes do SETUP.")
elif EXPECTED_GPU_NAME_CONTAINS not in GPU_DETECTED_NAME.upper():
    print("AVISO: GPU diferente de A100:", GPU_DETECTED_NAME)
    print("Treinamentos não serão diretamente comparáveis em tempo/VRAM.")
else:
    print("GPU A100 detectada: OK")


# ============================================================
# 7. SNAPSHOT DE PROVENIÊNCIA DO AMBIENTE HOST
# ============================================================

COLAB_RUNTIME_SNAPSHOT = {
    "host_python": PYTHON_ATUAL,
    "host_python_executable": sys.executable,
    "host_packages": dict(BASE_PACKAGE_SNAPSHOT),
    "host_torch_probe": dict(TORCH_RUNTIME_INFO),
    "host_gpu": dict(GPU_RUNTIME_INFO),
    "host_nvcc_available": NVCC_PROBE["returncode"] == 0,
    "tcc_python_expected": VERSIONS["python"],
    "tcc_torch_expected": VERSIONS["torch"],
    "tcc_cuda_expected": VERSIONS["cuda"],
    "tcc_pinned_packages": dict(VERSIONS),
    "tcc_isolated_env_required": TCC_ENV_ISOLATED,
    "tcc_system_site_packages_allowed": TCC_USE_SYSTEM_SITE_PACKAGES,
}


# ============================================================
# 8. INTEGRIDADE PRÉVIA E RESUMO DOS CAMINHOS
# ============================================================

print("\n" + "=" * 100)
print("CAMINHOS E PRÓXIMO PASSO")
print("=" * 100)
for label, path in (
    ("Dados nuScenes", TCC_ROOT),
    ("Ambiente TCC", TCC_ENV),
    ("Python isolado", TCC_PYTHON),
    ("ns-train", NS_TRAIN),
    ("ns-eval", NS_EVAL),
    ("ns-render", NS_RENDER),
    ("Dataset", OUTPUT_DATASET),
    ("Treinamentos", OUTPUT_DIR),
    ("Logs", LOG_DIR),
    ("Resultados", RESULTS_DIR),
):
    print(f"{label}: {path}")

if TCC_PYTHON.exists():
    existing_python = optional_command([str(TCC_PYTHON), "--version"], timeout=15)
    print("tcc_env existente:", existing_python["output"])
    if VERSIONS["python"] not in existing_python["output"]:
        print("AVISO: o SETUP deve recriar o tcc_env com Python", VERSIONS["python"])
else:
    print("tcc_env pronto: NÃO — o SETUP precisa criá-lo.")

print("\n" + "=" * 100)
print("CÉLULA 0 FINALIZADA COM SUCESSO")
print("=" * 100)
print("Python do Colab não foi alterado.")
print("Nenhuma dependência foi instalada ou atualizada.")
print("Nenhuma biblioteca científica foi importada no kernel principal.")
print("Execute a célula SETUP para instalar as dependências experimentais.")
print("Depois execute a célula 1 para validar o ambiente.")


In [ ]:
# ============================================================
# CÉLULA SETUP — DEPENDÊNCIAS DO AMBIENTE EXPERIMENTAL
#
# Executar após a célula 0, em uma sessão Google Colab.
# Instala Python 3.12.13 e as dependências científicas no tcc_env.
# Mantém o Python do kernel isolado do ambiente de treinamento.
# Usa versões compatíveis de OpenCV para Nerfstudio e YOLO.
#
# Requisitos da VM: NVIDIA A100 80 GB, acesso à internet e espaco livre.
# Se nao existir, instala CUDA Toolkit 12.8 AO LADO do CUDA 13.x do Colab,
# SEM instalar/trocar o driver NVIDIA e SEM alterar o Python do host.
#
# Saídas:
#   /content/tcc_uv_bootstrap/             uv via PyPI no Python do host
#   /content/tcc_env/                     Python 3.12.13 isolado
#   /content/tcc_env/tcc_environment.json  manifesto reprodutível
#   /content/tcc_env/tcc_pip_freeze.txt    dependências efetivas
#   /content/tcc_constraints.txt          pins experimentais
#   /content/logs_nuscenes/setup_tcc_isolado_v4.log
# ============================================================

from collections import deque
from datetime import datetime, timezone
from pathlib import Path
import importlib.metadata as metadata
import json
import os
import re
import shutil
import subprocess
import sys
import time

# -----------------------------------------------------------------
# 1. Contrato com a Célula 0
# -----------------------------------------------------------------
OBRIGATORIAS = (
    "VERSIONS", "TCC_ENV", "TCC_ENV_BIN", "TCC_PYTHON",
    "NS_TRAIN", "NS_EVAL", "NS_RENDER", "SETUP_INFO",
    "CONSTRAINTS_FILE", "BASE_PACKAGE_SNAPSHOT",
    "TCNN_SOURCE", "TCNN_CUDA_ARCHITECTURES", "REQUIRE_TCNN",
    "build_tcc_subprocess_env", "LOG_DIR",
)
faltando = [nome for nome in OBRIGATORIAS if nome not in globals()]
if faltando:
    raise RuntimeError("Execute a célula 0 antes do SETUP. Ausentes: "
                       + ", ".join(faltando))

for chave, valor in {
    "python": "3.12.13", "torch": "2.11.0+cu128", "cuda": "12.8",
    "numpy_tcc": "1.26.4", "pandas_tcc": "2.2.2",
    "nerfstudio": "1.1.5", "ultralytics": "8.4.132",
    "opencv": "4.11.0.86", "openpyxl": "3.1.5",
    "gsplat": "1.4.0", "nerfacc": "0.5.2",
    "setuptools": "81.0.0",
}.items():
    if VERSIONS.get(chave) != valor:
        raise RuntimeError(f"Pin {chave} incompatível com a V4: "
                           f"{VERSIONS.get(chave)!r} != {valor!r}")

if Path(TCC_ENV).resolve() != Path("/content/tcc_env"):
    raise RuntimeError("Proteção: TCC_ENV deve ser exatamente /content/tcc_env.")
if sys.version_info[:2] != (3, 13):
    print("AVISO: host diferente de Python 3.13; a validação do TCC continua rígida.")
if globals().get("TCC_USE_SYSTEM_SITE_PACKAGES", False):
    raise RuntimeError("CÉLULA 0 inadequada: desative system-site-packages.")

# Para reconstrução manual de um tcc_env inconsistente.
TCC_FORCE_REBUILD = False
EXPECTED_TORCHVISION = "0.26.0+cu128"
# Toolkit exigido para compilar tiny-cuda-nn e extensoes do gsplat.
# O nvcc do HOST Colab pode ser 13.x; NAO usa-lo com torch+cu128.
CUDA_TOOLKIT_DIR = Path("/usr/local/cuda-12.8")
CUDA_NVCC = CUDA_TOOLKIT_DIR / "bin" / "nvcc"
CUDA_APT_PACKAGE = "cuda-toolkit-12-8"  # toolkit sem driver NVIDIA
CUDA_REPO_KEYRING = Path("/content/cuda-keyring_1.1-1_all.deb")
# Pin EXATO exigido pelo metadata do Nerfstudio 1.1.5.
# Diferente de VERSIONS["opencv"], que identifica opencv-python para YOLO.
EXPECTED_OPENCV_HEADLESS = "4.10.0.84"
PYTORCH_CUDA_INDEX = "https://download.pytorch.org/whl/cu128"
UV_BIN_DIR = Path("/content/tcc_uv_bin")
UV_EXE = UV_BIN_DIR / "uv"
# uv é instalado pelo PyPI num bootstrap venv SEPARADO do TCC e do kernel.
# Não depende de astral.sh/uv/install.sh, que devolveu HTTP 403 nesta VM.
UV_BOOTSTRAP_DIR = Path("/content/tcc_uv_bootstrap")
UV_BOOTSTRAP_PYTHON = UV_BOOTSTRAP_DIR / "bin" / "python"
UV_BOOTSTRAP_EXECUTABLE = UV_BOOTSTRAP_DIR / "bin" / "uv"
UV_PYTHON_DIR = Path("/content/tcc_python_managed")
UV_CACHE_DIR = Path("/content/tcc_uv_cache")
SETUP_LOG_PATH = Path(LOG_DIR) / "setup_tcc_isolado_v4.log"
PIP_FREEZE_PATH = Path(TCC_ENV) / "tcc_pip_freeze.txt"
Path(LOG_DIR).mkdir(parents=True, exist_ok=True)

# Um log por execução; não mexe nos logs/outputs dos treinamentos.
with SETUP_LOG_PATH.open("a", encoding="utf-8") as f:
    f.write("\n" + "=" * 100 + "\n")
    f.write(f"SETUP V4 iniciado: {datetime.now(timezone.utc).isoformat()}\n")


def comando(args, titulo, *, env=None, timeout=None, mostrar=True, check=True):
    """Stream para o notebook e para log; erro traz últimas linhas."""
    args = [str(a) for a in args]
    print("\n" + "=" * 100, flush=True)
    print(titulo, flush=True)
    print("=" * 100, flush=True)
    print("$", " ".join(args), flush=True)
    ultimas = deque(maxlen=100)
    inicio = time.perf_counter()
    with SETUP_LOG_PATH.open("a", encoding="utf-8") as log:
        log.write(f"\n### {titulo}\n$ {' '.join(args)}\n")
        log.flush()
        processo = subprocess.Popen(
            args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, env=env,
        )
        try:
            assert processo.stdout is not None
            for linha in processo.stdout:
                log.write(linha)
                log.flush()
                ultimas.append(linha.rstrip("\n"))
                if mostrar:
                    print(linha, end="", flush=True)
                if timeout is not None and time.perf_counter() - inicio > timeout:
                    processo.kill()
                    processo.wait()
                    raise TimeoutError(f"{titulo} excedeu {timeout}s; veja {SETUP_LOG_PATH}")
            retorno = processo.wait()
        except BaseException:
            if processo.poll() is None:
                processo.kill()
                processo.wait()
            raise
        duracao = time.perf_counter() - inicio
        log.write(f"[fim] retorno={retorno}, duração={duracao:.1f}s\n")
    print(f"Resultado: {retorno} | {duracao:.1f}s", flush=True)
    if check and retorno != 0:
        raise RuntimeError(
            f"Falha em: {titulo} (código {retorno}).\n"
            f"Log: {SETUP_LOG_PATH}\nÚltimas linhas:\n" + "\n".join(ultimas)
        )
    return retorno, "\n".join(ultimas)


def executar_curto(args, *, env=None, timeout=120, check=True):
    resultado = subprocess.run(
        [str(a) for a in args], env=env, capture_output=True,
        text=True, timeout=timeout,
    )
    if check and resultado.returncode:
        raise RuntimeError(
            f"Comando falhou: {' '.join(str(a) for a in args)}\n"
            f"{(resultado.stdout + resultado.stderr)[-7000:]}"
        )
    return resultado



def validar_nvcc_cuda_128():
    """Evita misturar PyTorch cu128 com nvcc CUDA 13.x."""
    if not CUDA_NVCC.is_file():
        return False
    resultado = executar_curto([CUDA_NVCC, "--version"], timeout=60)
    if not re.search(r"release\s+12\.8\b", resultado.stdout):
        raise RuntimeError(
            "Compilador inesperado em " + str(CUDA_NVCC) + "\n" + resultado.stdout
        )
    return True


def cuda_128_apto():
    """Consulta o APT sem confundir ausencia de candidato com instalacao."""
    p = executar_curto(["apt-cache", "policy", CUDA_APT_PACKAGE],
                      timeout=60, check=False)
    m = re.search(r"^\s*Candidate:\s*(\S+)", p.stdout, re.MULTILINE)
    return bool(m and m.group(1) not in ("(none)", "none"))


def garantir_cuda_toolkit_128():
    """Instala APENAS CUDA Toolkit 12.8 se faltar, nunca o driver.

    APT usa o repositorio NVIDIA oficial apenas se o pacote nao existir
    nos repositorios ja configurados. Nao remove/reinstala CUDA 13.x.
    """
    print("\n" + "=" * 100, flush=True)
    print("CUDA TOOLKIT 12.8 PARA PYTORCH cu128 / TCNN", flush=True)
    print("=" * 100, flush=True)
    print("nvcc selecionado:", CUDA_NVCC, flush=True)

    if validar_nvcc_cuda_128():
        print("CUDA Toolkit 12.8 ja esta instalado: REUTILIZADO.", flush=True)
        return str(CUDA_NVCC)

    if os.geteuid() != 0:
        raise RuntimeError(
            "Instalacao do CUDA Toolkit 12.8 exige permissao root no Colab."
        )
    if os.uname().machine != "x86_64":
        raise RuntimeError("Bootstrap APT CUDA 12.8 disponivel somente para x86_64.")

    os_release = {}
    for linha in Path("/etc/os-release").read_text(encoding="utf-8").splitlines():
        if "=" in linha:
            chave, valor = linha.split("=", 1)
            os_release[chave] = valor.strip().strip('"')
    distro = os_release.get("ID", "")
    versao = os_release.get("VERSION_ID", "")
    suportadas = {"20.04": "ubuntu2004", "22.04": "ubuntu2204",
                  "24.04": "ubuntu2404"}
    if distro != "ubuntu" or versao not in suportadas:
        raise RuntimeError(
            "Bootstrap CUDA 12.8 somente para Ubuntu 20.04/22.04/24.04 x86_64; "
            f"runtime atual: {distro} {versao}. "
            "Informe o resultado de /etc/os-release para ajustar o instalador."
        )
    print("Sistema:", distro, versao)

    # Na VM atual o nvcc 13.3 esta no PATH; instalar cuda-toolkit-12-8
    # separadamente permite preservar o host e o ambiente Python isolado.
    comando(["apt-get", "update", "-qq"],
            "CUDA 12.8: ATUALIZAR REPOSITORIOS EXISTENTES",
            timeout=1200, mostrar=False)

    if not cuda_128_apto():
        # Procedimento do NVIDIA CUDA Installation Guide 12.8, Ubuntu repo.
        from urllib.request import urlopen
        url = (
            "https://developer.download.nvidia.com/compute/cuda/repos/"
            + suportadas[versao] + "/x86_64/cuda-keyring_1.1-1_all.deb"
        )
        print("Pacote indisponivel no APT atual; adicionando repositorio NVIDIA:")
        print(url)
        try:
            with urlopen(url, timeout=90) as resposta:
                conteudo = resposta.read()
        except Exception as exc:
            raise RuntimeError(
                "Nao foi possivel baixar o cuda-keyring oficial. "
                "Confirme o acesso do Colab a developer.download.nvidia.com."
            ) from exc
        if len(conteudo) < 1000:
            raise RuntimeError("Download do cuda-keyring parece incompleto.")
        CUDA_REPO_KEYRING.write_bytes(conteudo)
        comando(["dpkg", "-i", CUDA_REPO_KEYRING],
                "CUDA 12.8: ADICIONAR CHAVE/REPOSITORIO OFICIAL NVIDIA",
                timeout=300)
        comando(["apt-get", "update", "-qq"],
                "CUDA 12.8: ATUALIZAR APT COM REPOSITORIO NVIDIA",
                timeout=1200, mostrar=False)

    if not cuda_128_apto():
        raise RuntimeError(
            "cuda-toolkit-12-8 segue sem versao candidata apos atualizar APT. "
            f"Consulte {SETUP_LOG_PATH}; nao usar cuda-12-8 "
            "(o metapacote cuda-12-8 inclui driver)."
        )

    ambiente_apt = os.environ.copy()
    ambiente_apt["DEBIAN_FRONTEND"] = "noninteractive"
    comando(["apt-get", "install", "-y", "--no-install-recommends",
             CUDA_APT_PACKAGE],
            "CUDA 12.8: INSTALAR TOOLKIT SEM DRIVER NVIDIA",
            env=ambiente_apt, timeout=3600, mostrar=False)

    if not validar_nvcc_cuda_128():
        raise RuntimeError(
            "Instalacao APT concluiu sem nvcc 12.8 em " + str(CUDA_NVCC)
        )
    print("CUDA Toolkit 12.8 instalado e nvcc validado:", CUDA_NVCC)
    return str(CUDA_NVCC)


def ambiente_tcc():
    env = build_tcc_subprocess_env()
    # Prevenir contaminação pelo kernel Python 3.13 ou venv antigo.
    for chave in ("PYTHONPATH", "PYTHONHOME", "PYTHONUSERBASE", "PIP_TARGET"):
        env.pop(chave, None)
    env["PATH"] = (
        str(TCC_ENV_BIN) + ":" + str(CUDA_TOOLKIT_DIR / "bin") + ":"
        + os.environ.get("PATH", "")
    )
    env["VIRTUAL_ENV"] = str(TCC_ENV)
    env["PYTHONNOUSERSITE"] = "1"
    env["TCNN_CUDA_ARCHITECTURES"] = str(TCNN_CUDA_ARCHITECTURES)
    env["TORCH_CUDA_ARCH_LIST"] = "8.0"
    env["MAX_JOBS"] = "4"
    env["CUDA_HOME"] = str(CUDA_TOOLKIT_DIR)
    env["CUDA_PATH"] = str(CUDA_TOOLKIT_DIR)
    env["CUDACXX"] = str(CUDA_NVCC)
    return env


def pip_tcc(*argumentos, titulo, timeout=None, mostrar=True):
    return comando(
        [TCC_PYTHON, "-m", "pip", "install", "--disable-pip-version-check",
         "--prefer-binary", "--no-cache-dir", *argumentos],
        titulo, env=ambiente_tcc(), timeout=timeout, mostrar=mostrar,
    )


# -----------------------------------------------------------------
# 2. Diagnóstico host (não define as versões instaladas no tcc_env)
# -----------------------------------------------------------------
print("=" * 100)
print("SETUP V4 (OPEN CV FIX + RETOMADA): COLAB 3.13 -> TCC 3.12.13")
print("=" * 100)
print("Host:", sys.version.split()[0], sys.executable)
print("TCC Python alvo:", VERSIONS["python"])
print("TCC PyTorch alvo:", VERSIONS["torch"])
print("Torchvision alvo:", EXPECTED_TORCHVISION)
print("GPU exigida: NVIDIA A100 / sm_80")
print("LOG:", SETUP_LOG_PATH)
espaco_livre_gib = shutil.disk_usage("/content").free / (1024 ** 3)
print(f"Disco livre em /content: {espaco_livre_gib:.1f} GiB")
if espaco_livre_gib < 18:
    raise RuntimeError(
        "Espaço insuficiente para Python isolado, PyTorch CUDA e build TCNN. "
        "Preserve os checkpoints no Drive e libere ao menos 18 GiB "
        "sem remover outputs de treinamentos não copiados."
    )

smi = executar_curto(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
     "--format=csv,noheader,nounits"], timeout=40,
).stdout.strip()
if "A100" not in smi.upper():
    raise RuntimeError(f"Esta V4 exige NVIDIA A100; nvidia-smi: {smi}")
print("GPU:", smi)

# O host possui nvcc 13.x, mas o tcc_env usa torch+cu128.
# Instalar automaticamente o Toolkit 12.8 sem tocar no driver Colab.
nvcc_host = shutil.which("nvcc")
if nvcc_host:
    nvcc_host_saida = executar_curto([nvcc_host, "--version"]).stdout
    print("nvcc do host (registro apenas):",
          nvcc_host_saida.strip().splitlines()[-1])
nvcc = garantir_cuda_toolkit_128()
nvcc_saida = executar_curto([nvcc, "--version"]).stdout
print("nvcc utilizado pelo TCC:", nvcc_saida.strip().splitlines()[-1])

# Verifica que a CÉLULA 0 registrou corretamente o host sem import científico.
for pacote, versao in BASE_PACKAGE_SNAPSHOT.items():
    try:
        atual = metadata.version(pacote)
    except metadata.PackageNotFoundError:
        atual = "não instalado"
    if atual != versao:
        raise RuntimeError(f"Host alterado depois da CÉLULA 0: "
                           f"{pacote} {versao!r} -> {atual!r}")

# -----------------------------------------------------------------
# 3. Constraints: impedem pip de alterar a configuração crítica
# -----------------------------------------------------------------
CONSTRAINTS_FILE.parent.mkdir(parents=True, exist_ok=True)
CONSTRAINTS_FILE.write_text(
    "\n".join([
        "# V4 / Python 3.12.13 isolado / CUDA 12.8",
        "torch==2.11.0+cu128",
        f"torchvision=={EXPECTED_TORCHVISION}",
        f"numpy=={VERSIONS['numpy_tcc']}",
        f"pandas=={VERSIONS['pandas_tcc']}",
        f"nerfstudio=={VERSIONS['nerfstudio']}",
        f"ultralytics=={VERSIONS['ultralytics']}",
        f"opencv-python=={VERSIONS['opencv']}",
        f"opencv-python-headless=={EXPECTED_OPENCV_HEADLESS}",
        f"openpyxl=={VERSIONS['openpyxl']}",
        f"gsplat=={VERSIONS['gsplat']}",
        f"nerfacc=={VERSIONS['nerfacc']}",
        f"setuptools=={VERSIONS['setuptools']}",
    ]) + "\n", encoding="utf-8",
)
print("Constraints:", CONSTRAINTS_FILE)

# -----------------------------------------------------------------
# 4. Validação funcional em subprocesso Python do TCC
# -----------------------------------------------------------------
# Código sem f-string externa: evita erro de chaves não escapadas.
VALIDACAO_TCC = r'''
import importlib.metadata as md
import json
import sys
from pathlib import Path

import numpy
import pandas
import cv2
import torch
import torchvision
import ultralytics
import nerfstudio
import gsplat
import nerfacc

expected = json.loads(sys.argv[1])
root = Path(sys.argv[2]).resolve()
require_tcnn = (sys.argv[3] == "1")

versao_python = ".".join(str(x) for x in sys.version_info[:3])
assert versao_python == expected["python"], (versao_python, expected["python"])
assert sys.prefix == str(root), (sys.prefix, root)
assert not sys.base_prefix == sys.prefix, "Não é um venv isolado"
assert numpy.__version__ == expected["numpy_tcc"]
assert pandas.__version__ == expected["pandas_tcc"]
for pacote, versao in (
    ("nerfstudio", expected["nerfstudio"]),
    ("ultralytics", expected["ultralytics"]),
    ("opencv-python", expected["opencv"]),
    ("opencv-python-headless", expected["opencv_headless"]),
    ("openpyxl", expected["openpyxl"]),
    ("gsplat", expected["gsplat"]),
    ("nerfacc", expected["nerfacc"]),
    ("setuptools", expected["setuptools"]),
):
    assert md.version(pacote) == versao, (pacote, md.version(pacote), versao)
for nome, modulo in (
    ("numpy", numpy), ("pandas", pandas),
    ("cv2", cv2), ("torch", torch),
    ("torchvision", torchvision),
    ("ultralytics", ultralytics),
    ("nerfstudio", nerfstudio),
    ("gsplat", gsplat), ("nerfacc", nerfacc),
):
    caminho = Path(modulo.__file__).resolve()
    assert caminho.is_relative_to(root), (nome, str(caminho), str(root))
assert torch.__version__ == expected["torch"], torch.__version__
assert torchvision.__version__ == expected["torchvision"], torchvision.__version__
assert torch.version.cuda == expected["cuda"], torch.version.cuda
assert torch.cuda.is_available(), "CUDA indisponível no tcc_env"
gpu = torch.cuda.get_device_name(0)
assert "A100" in gpu.upper(), gpu
cap = torch.cuda.get_device_capability(0)
assert cap == (8, 0), cap
memoria = torch.cuda.get_device_properties(0).total_memory
assert memoria >= 70 * 1024 ** 3, memoria
x = torch.tensor([1., 2., 3.], device="cuda")
assert (x * 2).cpu().tolist() == [2., 4., 6.]
from gsplat.rendering import rasterization
from nerfacc import OccGridEstimator
# Executa de fato um micro-render CUDA do gsplat, inclusive o primeiro JIT.
# Isso detecta incompatibilidades PyTorch / nvcc antes de B/D.
g_means = torch.tensor([[0., 0., 2.]], device="cuda")
g_quats = torch.tensor([[1., 0., 0., 0.]], device="cuda")
g_scales = torch.full((1, 3), 0.35, device="cuda")
g_opacities = torch.tensor([0.9], device="cuda")
g_colors = torch.tensor([[1., 0., 0.]], device="cuda")
g_view = torch.eye(4, device="cuda")[None]
g_Ks = torch.tensor([[[16., 0., 16.], [0., 16., 16.], [0., 0., 1.]]], device="cuda")
g_rgb, g_alpha, _ = rasterization(
    g_means, g_quats, g_scales, g_opacities, g_colors,
    g_view, g_Ks, 32, 32,
    packed=False, rasterize_mode="antialiased",
)
torch.cuda.synchronize()
assert g_rgb.is_cuda and tuple(g_rgb.shape) == (1, 32, 32, 3)
assert g_alpha.is_cuda and float(g_alpha.max()) > 0
print("GSplat micro-render CUDA / antialiased: OK")
if require_tcnn:
    import tinycudann as tcnn
    assert Path(tcnn.__file__).resolve().is_relative_to(root), tcnn.__file__
    entrada = torch.rand((32, 3), device="cuda", dtype=torch.float32)
    rede = tcnn.Encoding(
        n_input_dims=3,
        encoding_config={
            "otype": "HashGrid", "n_levels": 2,
            "n_features_per_level": 2, "log2_hashmap_size": 12,
            "base_resolution": 8, "per_level_scale": 2.0,
        },
    )
    saida = rede(entrada)
    torch.cuda.synchronize()
    assert saida.is_cuda and saida.shape[0] == 32
print("Python:", versao_python, sys.executable)
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
print("Torchvision:", torchvision.__version__)
print("NumPy/Pandas:", numpy.__version__, pandas.__version__)
print("OpenCV distributions (GUI/headless) / cv2:",
      md.version("opencv-python"), md.version("opencv-python-headless"),
      cv2.__version__)
# Coexistência de wheels compartilha cv2; valida funções efetivamente usadas.
assert cv2.__version__.startswith(("4.10.", "4.11.")), cv2.__version__
assert hasattr(cv2, "imread") and hasattr(cv2, "resize")
print("Nerfstudio/gsplat/nerfacc:",
      md.version("nerfstudio"), md.version("gsplat"), md.version("nerfacc"))
print("Ultralytics:", ultralytics.__version__)
print("GPU:", gpu, "CC:", cap, "VRAM GiB:", round(memoria / 1024**3, 2))
print("TCNN HashGrid CUDA:", "OK" if require_tcnn else "não solicitado")
print("VALIDACAO_TCC_OK")
'''

ESPERADO_TCC = {
    **VERSIONS,
    "torchvision": EXPECTED_TORCHVISION,
    "opencv_headless": EXPECTED_OPENCV_HEADLESS,
}


def validar_stack(*, silencioso=False):
    if not Path(TCC_PYTHON).is_file():
        return False
    cfg = Path(TCC_ENV) / "pyvenv.cfg"
    if not cfg.is_file():
        return False
    conf = cfg.read_text(encoding="utf-8").lower()
    if "include-system-site-packages = false" not in conf:
        return False
    args = [TCC_PYTHON, "-c", VALIDACAO_TCC,
            json.dumps(ESPERADO_TCC), TCC_ENV, "1" if REQUIRE_TCNN else "0"]
    resultado = executar_curto(args, env=ambiente_tcc(),
                              timeout=900, check=False)
    if not silencioso:
        print(resultado.stdout)
        if resultado.returncode:
            print(resultado.stderr[-8000:])
    return resultado.returncode == 0 and "VALIDACAO_TCC_OK" in resultado.stdout


def validar_nerfstudio_cli():
    env = ambiente_tcc()
    for args in (
        [NS_TRAIN, "--help"],
        [NS_TRAIN, "nerfacto", "--help"],
        [NS_TRAIN, "splatfacto", "--help"],
        [NS_EVAL, "--help"],
        [NS_RENDER, "--help"],
    ):
        resultado = executar_curto(args, env=env, timeout=240, check=True)
        if "--help" in args and not resultado.stdout.strip():
            raise RuntimeError(f"Ajuda Nerfstudio vazia: {args}")
        print("CLI OK:", " ".join(str(x) for x in args[:2]))


# -----------------------------------------------------------------
# 5. Reutilização segura: requer manifesto e teste funcional
# -----------------------------------------------------------------
SETUP_NECESSARIO = True
SETUP_RETOMAR_PARCIAL = False


def ambiente_parcial_compatível():
    """Permite continuar após erro do resolver sem baixar PyTorch outra vez.

    Retoma APENAS se a ABI, o isolamento, o PyTorch CUDA, a GPU e a
    configuração NumPy/Pandas já instalados são exatamente os esperados.
    Não confunde um venv antigo com um venv parcialmente instalado.
    """
    py = Path(TCC_PYTHON)
    cfg = Path(TCC_ENV) / "pyvenv.cfg"
    if not py.is_file() or not cfg.is_file():
        return False
    if "include-system-site-packages = false" not in cfg.read_text(
        encoding="utf-8"
    ).lower():
        return False
    code = r'''
import sys, torch, torchvision, numpy, pandas
from pathlib import Path
root = Path(sys.argv[1]).resolve()
assert Path(sys.prefix).resolve() == root, (sys.prefix, root)
assert sys.base_prefix != sys.prefix
assert '.'.join(map(str,sys.version_info[:3])) == '3.12.13'
assert torch.__version__ == '2.11.0+cu128', torch.__version__
assert torchvision.__version__ == '0.26.0+cu128', torchvision.__version__
assert torch.version.cuda == '12.8'
assert torch.cuda.is_available()
assert torch.cuda.get_device_capability(0) == (8, 0)
assert numpy.__version__ == '1.26.4', numpy.__version__
assert pandas.__version__ == '2.2.2', pandas.__version__
for module in (torch, torchvision, numpy, pandas):
    assert Path(module.__file__).resolve().is_relative_to(root), module.__file__
print('TCC_PARCIAL_COMPATIVEL')
'''
    try:
        resultado = executar_curto(
            [TCC_PYTHON, "-c", code, TCC_ENV],
            env=ambiente_tcc(), timeout=240, check=False,
        )
        if resultado.returncode == 0 and "TCC_PARCIAL_COMPATIVEL" in resultado.stdout:
            return True
        print("tcc_env parcial não reutilizável:",
              (resultado.stdout + resultado.stderr)[-2500:])
    except (OSError, subprocess.SubprocessError) as exc:
        print("Falha ao inspecionar o tcc_env parcial:", repr(exc))
    return False


if Path(TCC_ENV).exists() and not TCC_FORCE_REBUILD:
    print("\nAmbiente tcc_env existente encontrado; validando...")
    if Path(SETUP_INFO).is_file():
        try:
            info_anterior = json.loads(Path(SETUP_INFO).read_text(encoding="utf-8"))
            identidade_ok = (
                info_anterior.get("environment_mode") == "isolated_python_312"
                and info_anterior.get("tcc_use_system_site_packages") is False
                and all(info_anterior.get(k) == v for k, v in {
                    "python": VERSIONS["python"],
                    "torch": VERSIONS["torch"],
                    "cuda": VERSIONS["cuda"],
                    "numpy": VERSIONS["numpy_tcc"],
                    "pandas": VERSIONS["pandas_tcc"],
                    "nerfstudio": VERSIONS["nerfstudio"],
                    "gsplat": VERSIONS["gsplat"],
                    "nerfacc": VERSIONS["nerfacc"],
                    "opencv-python": VERSIONS["opencv"],
                    "opencv-python-headless": EXPECTED_OPENCV_HEADLESS,
                }.items())
            )
            if identidade_ok and validar_stack():
                validar_nerfstudio_cli()
                SETUP_NECESSARIO = False
                print("tcc_env existente: REUTILIZADO, SEM REINSTALAÇÃO.")
        except Exception as exc:
            print("Ambiente anterior não pôde ser reutilizado:", repr(exc))
    # Um SETUP interrompido antes do TCNN ainda não tem manifesto.
    # Retomar SEM apagar Python 3.12, Torch CUDA 12.8 e NumPy/Pandas.
    if SETUP_NECESSARIO and ambiente_parcial_compatível():
        SETUP_RETOMAR_PARCIAL = True
        print("\nRETOMADA SEGURA: Python 3.12, Torch, CUDA, NumPy e Pandas "
              "já instalados. Não baixar PyTorch novamente.")

if SETUP_NECESSARIO:
    # -----------------------------------------------------------------
    # 6. Dependências de sistema (sem pip no Python do notebook)
    # -----------------------------------------------------------------
    print("\nCriando ambiente isolado a partir da CÉLULA 0...")
    # Após uma falha no bootstrap de uv, não é necessário repetir o APT.
    ferramentas_apt = ("git", "ninja", "colmap", "ffmpeg", "curl", "gcc", "g++")
    if all(shutil.which(nome) for nome in ferramentas_apt):
        print("Ferramentas APT instaladas anteriormente: REUTILIZADAS.")
    else:
        comando(["apt-get", "update", "-qq"], "APT: atualizar índices", timeout=900)
        comando(
            ["apt-get", "install", "-y", "build-essential", "git", "ninja-build",
             "colmap", "ffmpeg", "libgl1", "libglib2.0-0", "curl", "ca-certificates"],
            "APT: ferramentas de compilação e processamento", timeout=1800,
        )

    # -----------------------------------------------------------------
    # 7. Bootstrap de uv via PyPI — NÃO usa astral.sh (HTTP 403)
    # -----------------------------------------------------------------
    # Cria um pequeno venv host Python 3.13 sem pip; o pip do Colab
    # instala uv DENTRO desse venv usando --python (documentado pelo pip).
    # Depois copia somente o lançador para o caminho esperado neste SETUP.
    # O kernel, site-packages globais e /content/tcc_env NÃO são alterados.
    UV_BIN_DIR.mkdir(parents=True, exist_ok=True)
    if not UV_EXE.is_file():
        if not UV_BOOTSTRAP_PYTHON.is_file():
            comando(
                [sys.executable, "-m", "venv", "--without-pip",
                 str(UV_BOOTSTRAP_DIR)],
                "CRIAR BOOTSTRAP VENV PARA uv (PYTHON HOST; SEM pip)",
                timeout=240,
            )
        if not UV_BOOTSTRAP_PYTHON.is_file():
            raise RuntimeError(
                "Não foi possível criar o bootstrap venv do uv. "
                "Confirme se o módulo venv está disponível no Python do Colab."
            )
        if not UV_BOOTSTRAP_EXECUTABLE.is_file():
            comando(
                [sys.executable, "-m", "pip", "--python",
                 str(UV_BOOTSTRAP_DIR), "install",
                 "--disable-pip-version-check", "--no-cache-dir",
                 "--only-binary=:all:", "uv"],
                "INSTALAR uv VIA PyPI NO BOOTSTRAP ISOLADO",
                timeout=900,
            )
        if not UV_BOOTSTRAP_EXECUTABLE.is_file():
            raise FileNotFoundError(
                "pip terminou sem gerar o executável do uv: "
                f"{UV_BOOTSTRAP_EXECUTABLE}"
            )
        shutil.copy2(UV_BOOTSTRAP_EXECUTABLE, UV_EXE)
        UV_EXE.chmod(UV_EXE.stat().st_mode | 0o111)
    if not UV_EXE.is_file():
        raise FileNotFoundError(f"Executável uv ausente: {UV_EXE}")
    # Não presume que copiar um script Python preserve sua resolução.
    # Um erro aqui aparece antes de baixar Python ou mexer no tcc_env.
    UV_VERSION = executar_curto([UV_EXE, "--version"], timeout=90).stdout.strip()
    if not UV_VERSION.startswith("uv "):
        raise RuntimeError(f"Executável uv inesperado: {UV_VERSION!r}")
    print("uv (bootstrap PyPI):", UV_VERSION)
    uv_env = os.environ.copy()
    for chave in ("PYTHONPATH", "PYTHONHOME", "VIRTUAL_ENV"):
        uv_env.pop(chave, None)
    uv_env["UV_PYTHON_INSTALL_DIR"] = str(UV_PYTHON_DIR)
    uv_env["UV_CACHE_DIR"] = str(UV_CACHE_DIR)
    uv_env["UV_PYTHON_PREFERENCE"] = "only-managed"
    uv_env["UV_NO_PROGRESS"] = "1"

    # Não altera /usr/bin/python3 ou o kernel 3.13 do notebook.
    comando([UV_EXE, "python", "install", VERSIONS["python"]],
            "INSTALAR PYTHON 3.12.13 GERENCIADO", env=uv_env, timeout=900)

    # -----------------------------------------------------------------
    # 8. Retomar venv parcial ou recriar somente se incompatível
    # -----------------------------------------------------------------
    if SETUP_RETOMAR_PARCIAL:
        print("Venv parcial verificado: MANTIDO. Sem nova instalação de Torch.")
    else:
        if Path(TCC_ENV).exists():
            print("Recriando apenas /content/tcc_env (não remove dataset/checkpoints).")
            shutil.rmtree(TCC_ENV)
        comando([UV_EXE, "venv", "--python", VERSIONS["python"], "--seed", TCC_ENV],
                "CRIAR VENV PYTHON 3.12.13 SEM SYSTEM-SITE-PACKAGES",
                env=uv_env, timeout=600)
    if not Path(TCC_PYTHON).is_file():
        raise RuntimeError("uv venv terminou, mas /content/tcc_env/bin/python não existe.")
    pyversao = executar_curto([TCC_PYTHON, "-c",
                               "import sys; print('.'.join(map(str,sys.version_info[:3])))"],
                              env=ambiente_tcc()).stdout.strip()
    if pyversao != VERSIONS["python"]:
        raise RuntimeError(f"Python do tcc_env: {pyversao}; esperado {VERSIONS['python']}.")
    cfg = (Path(TCC_ENV) / "pyvenv.cfg").read_text(encoding="utf-8").lower()
    if "include-system-site-packages = false" not in cfg:
        raise RuntimeError("O tcc_env está herdando pacotes do host! Abortando.")

    # -----------------------------------------------------------------
    # 9. pip / ferramentas de build dentro do tcc_env
    # -----------------------------------------------------------------
    if not SETUP_RETOMAR_PARCIAL:
        pip_tcc("--upgrade", "pip", "wheel", f"setuptools=={VERSIONS['setuptools']}",
                "ninja", "cmake", "packaging", titulo="PIP E FERRAMENTAS DE BUILD ISOLADAS",
                timeout=1200)
    else:
        print("Ferramentas de build do venv: REUTILIZADAS.")

    # -----------------------------------------------------------------
    # 10. PyTorch CUDA 12.8 local (NÃO herdado do host Python 3.13)
    # -----------------------------------------------------------------
    if not SETUP_RETOMAR_PARCIAL:
        pip_tcc("--index-url", PYTORCH_CUDA_INDEX,
                "torch==2.11.0", "torchvision==0.26.0",
                titulo="PYTORCH 2.11 + TORCHVISION 0.26 / CUDA 12.8",
                timeout=3600)
    else:
        print("PyTorch e torchvision CUDA 12.8 do venv: REUTILIZADOS.")
    torch_probe = executar_curto(
        [TCC_PYTHON, "-c",
         "import torch,torchvision; "
         "print(torch.__version__,torchvision.__version__,torch.version.cuda); "
         "assert torch.__version__=='2.11.0+cu128'; "
         "assert torchvision.__version__=='0.26.0+cu128'; "
         "assert torch.version.cuda=='12.8'; "
         "assert torch.cuda.is_available(); "
         "assert torch.cuda.get_device_capability(0)==(8,0)"],
        env=ambiente_tcc(), timeout=240,
    )
    print("PyTorch isolado:", torch_probe.stdout.strip())

    # -----------------------------------------------------------------
    # 11. Bibliotecas fixas da V4 dentro do tcc_env
    # -----------------------------------------------------------------
    pip_tcc("-c", CONSTRAINTS_FILE,
            f"numpy=={VERSIONS['numpy_tcc']}",
            f"pandas=={VERSIONS['pandas_tcc']}",
            titulo="NUMPY E PANDAS DO TCC", timeout=1800)

    # Metadados incompatíveis entre dois pacotes que usam o mesmo namespace cv2:
    # Nerfstudio exige opencv-python-headless==4.10.0.84;
    # Ultralytics pede opencv-python>=4.6.0. Os pins ficam DISTINTOS.
    # A resolução com dois wheels é necessária para satisfazer pip check;
    # a validação funcional abaixo detecta problemas de import/cv2.
    pip_tcc("-c", CONSTRAINTS_FILE,
            f"nerfstudio=={VERSIONS['nerfstudio']}",
            f"ultralytics=={VERSIONS['ultralytics']}",
            f"opencv-python=={VERSIONS['opencv']}",
            f"opencv-python-headless=={EXPECTED_OPENCV_HEADLESS}",
            f"openpyxl=={VERSIONS['openpyxl']}",
            f"gsplat=={VERSIONS['gsplat']}",
            f"nerfacc=={VERSIONS['nerfacc']}",
            titulo="NERFSTUDIO / GSPLAT / NERFACC / YOLO",
            timeout=3600)

    # Verificar resolução, para não chegar ao treino com stack inconsistente.
    comando([TCC_PYTHON, "-m", "pip", "check"], "VALIDAR DEPENDÊNCIAS pip check",
            env=ambiente_tcc(), timeout=180)

    # -----------------------------------------------------------------
    # 12. Compilar TCNN contra PyTorch LOCAL e nvcc 12.8 separado
    # -----------------------------------------------------------------
    if REQUIRE_TCNN:
        tc_env = ambiente_tcc()
        comando(
            [TCC_PYTHON, "-m", "pip", "install", "--disable-pip-version-check",
             "--no-cache-dir", "--no-build-isolation", "-c", CONSTRAINTS_FILE,
             TCNN_SOURCE],
            "COMPILAR TINY-CUDA-NN / TCNN sm_80 / CUDA 12.8 NO PYTHON 3.12",
            env=tc_env, timeout=5400,
        )
        # TCNN pode introduzir dependências; não aceitar downgrade do torch.
        comando([TCC_PYTHON, "-m", "pip", "check"],
                "VALIDAR DEPENDÊNCIAS APÓS TCNN", env=tc_env, timeout=180)

    # -----------------------------------------------------------------
    # 13. Validar imports, CUDA, hashgrid e executáveis Nerfstudio
    # -----------------------------------------------------------------
    print("\n" + "=" * 100)
    print("VALIDAÇÃO FUNCIONAL DO AMBIENTE ISOLADO")
    print("=" * 100)
    if not validar_stack():
        raise RuntimeError("Falhou validação Python/Torch/Nerfstudio/TCNN; veja "
                           f"{SETUP_LOG_PATH} e traceback acima. NÃO execute A–G.")
    validar_nerfstudio_cli()

    # -----------------------------------------------------------------
    # 14. Manifesto reprodutível; formato plano preservado para CÉLULA 1
    # -----------------------------------------------------------------
    congelamento = executar_curto(
        [TCC_PYTHON, "-m", "pip", "freeze", "--all"],
        env=ambiente_tcc(), timeout=120,
    ).stdout
    PIP_FREEZE_PATH.write_text(congelamento, encoding="utf-8")
    try:
        detalhe_tcnn = executar_curto(
            [TCC_PYTHON, "-c",
             "import json,importlib.metadata as md; "
             "d=md.distribution('tinycudann'); "
             "print(json.dumps({'version':d.version,"
             "'direct_url':json.loads(d.read_text('direct_url.json') or 'null')}))"],
            env=ambiente_tcc(), timeout=90,
        )
        tcnn_info = json.loads(detalhe_tcnn.stdout.strip().splitlines()[-1])
    except Exception:
        tcnn_info = {"version": "metadata não disponível", "direct_url": None}

    def pkg_host(nome):
        try:
            return metadata.version(nome)
        except metadata.PackageNotFoundError:
            return "não instalado"

    host_depois = {nome: pkg_host(nome) for nome in BASE_PACKAGE_SNAPSHOT}
    if host_depois != BASE_PACKAGE_SNAPSHOT:
        raise RuntimeError("As distribuições Python do host mudaram durante o SETUP: "
                           f"{BASE_PACKAGE_SNAPSHOT} -> {host_depois}")
    info = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "environment_mode": "isolated_python_312",
        "tcc_use_system_site_packages": False,
        "python": VERSIONS["python"],
        "torch": VERSIONS["torch"],
        "torchvision": EXPECTED_TORCHVISION,
        "cuda": VERSIONS["cuda"],
        "numpy": VERSIONS["numpy_tcc"],
        "pandas": VERSIONS["pandas_tcc"],
        "nerfstudio": VERSIONS["nerfstudio"],
        "ultralytics": VERSIONS["ultralytics"],
        "opencv-python": VERSIONS["opencv"],
        "opencv-python-headless": EXPECTED_OPENCV_HEADLESS,
        "opencv_runtime": executar_curto(
            [TCC_PYTHON, "-c", "import cv2; print(cv2.__version__)"],
            env=ambiente_tcc(), timeout=60,
        ).stdout.strip(),
        "openpyxl": VERSIONS["openpyxl"],
        "gsplat": VERSIONS["gsplat"],
        "nerfacc": VERSIONS["nerfacc"],
        "setuptools": VERSIONS["setuptools"],
        "environment": str(TCC_ENV),
        "python_executable": str(TCC_PYTHON),
        "base_package_snapshot": BASE_PACKAGE_SNAPSHOT,
        "host_runtime": {
            "python": sys.version.split()[0],
            "executable": sys.executable,
            "base_packages": BASE_PACKAGE_SNAPSHOT,
        },
        "tiny-cuda-nn": {
            "available": REQUIRE_TCNN,
            "functional_cuda_test": REQUIRE_TCNN,
            "source": TCNN_SOURCE,
            "cuda_architectures": TCNN_CUDA_ARCHITECTURES,
            **tcnn_info,
        },
        "gpu_nvidia_smi": smi,
        "nvcc": nvcc_saida.strip() if nvcc else None,
        "uv": UV_VERSION,
        "uv_python_install_dir": str(UV_PYTHON_DIR),
        "constraints": str(CONSTRAINTS_FILE),
        "pip_freeze": str(PIP_FREEZE_PATH),
        "setup_log": str(SETUP_LOG_PATH),
        "colmap": executar_curto(["dpkg-query", "-W", "-f=${Version}", "colmap"],
                                  check=False).stdout.strip(),
        "ffmpeg": executar_curto(["ffmpeg", "-version"],
                                 check=False).stdout.splitlines()[0],
    }
    Path(SETUP_INFO).write_text(json.dumps(info, indent=2, ensure_ascii=False),
                                encoding="utf-8")
    print("Manifesto gravado:", SETUP_INFO)

# -----------------------------------------------------------------
# 15. Finalização (ambiente novo ou reutilizado)
# -----------------------------------------------------------------
if not Path(SETUP_INFO).exists():
    raise FileNotFoundError(f"Manifesto do SETUP ausente: {SETUP_INFO}")
if not validar_stack():
    raise RuntimeError("Validação final do tcc_env falhou; NÃO inicie o treino.")
validar_nerfstudio_cli()
# Propagar escolha de toolchain para as celulas 1/A/B/C; nao modifica
# os pacotes nem o CUDA runtime usado pelo PyTorch do host.
os.environ["CUDA_HOME"] = str(CUDA_TOOLKIT_DIR)
os.environ["CUDA_PATH"] = str(CUDA_TOOLKIT_DIR)
os.environ["CUDACXX"] = str(CUDA_NVCC)
novo_bin = str(CUDA_TOOLKIT_DIR / "bin")
path_atual = os.environ.get("PATH", "")
if path_atual.split(":")[0] != novo_bin:
    os.environ["PATH"] = novo_bin + ":" + path_atual
TCC_SUBPROCESS_ENV = ambiente_tcc()
TCC_SETUP_VALIDATED = True

print("\n" + "=" * 100)
print("CÉLULA SETUP V4 FINALIZADA COM SUCESSO")
print("=" * 100)
print("Host:", sys.version.split()[0], "(inalterado)")
print("TCC:", VERSIONS["python"], "em", TCC_PYTHON)
print("PyTorch:", VERSIONS["torch"], "/ CUDA:", VERSIONS["cuda"])
print("Ambiente totalmente isolado: SIM (sem system-site-packages)")
print("A100 / TCNN HashGrid CUDA: OK")
print("Modo:", ("RETOMADO SEM REINSTALAR TORCH" if SETUP_RETOMAR_PARCIAL
                else "CRIADO" if SETUP_NECESSARIO else "REUTILIZADO"))
print("Manifesto:", SETUP_INFO)
print("Log:", SETUP_LOG_PATH)
print("Próximo passo: CÉLULA 1 adaptada ao Python 3.12 isolado.")


In [ ]:
# ============================================================
# CÉLULA 1 — V4 | VALIDAÇÃO DO AMBIENTE EXPERIMENTAL ISOLADO
# ============================================================
# Executar após as células 0 e SETUP.
#
# HOST Colab: Python 3.13.15 (não é o Python experimental).
# tcc_env:     Python 3.12.13, torch 2.11.0+cu128, CUDA 12.8.
#
# Esta célula NÃO instala pacotes, NÃO altera o kernel do Colab,
# NÃO reinicia a VM e NÃO usa --system-site-packages.
# Os testes de bibliotecas científicas rodam em subprocessos.
# Inclui testes CUDA funcionais de torch, gsplat e tiny-cuda-nn.
# ENVIRONMENT_VALIDATED só fica True no FINAL de todos os testes.
# ============================================================

from datetime import datetime, timezone
from pathlib import Path
import hashlib
import importlib.metadata as metadata
import json
import os
import re
import shutil
import subprocess
import sys
import textwrap


# ============================================================
# 0. ESTADO E CONTRATO COM 0/SETUP
# ============================================================
ENVIRONMENT_VALIDATED = False  # Não reutilizar estado anterior.

OBRIGATORIAS_CELULA_1 = (
    "VERSIONS", "TCC_ENV", "TCC_ENV_BIN", "TCC_PYTHON",
    "NS_TRAIN", "NS_EVAL", "NS_RENDER", "SETUP_INFO",
    "BASE_PACKAGE_SNAPSHOT", "REQUIRE_TCNN", "TCNN_SOURCE",
    "TCNN_CUDA_ARCHITECTURES", "build_tcc_subprocess_env",
    "RESULTS_DIR",
)
faltando = [nome for nome in OBRIGATORIAS_CELULA_1 if nome not in globals()]
if faltando:
    raise RuntimeError(
        "Execute a CÉLULA 0 e o novo SETUP antes da CÉLULA 1. "
        "Variáveis ausentes: " + ", ".join(faltando)
    )

if "TCC_SETUP_VALIDATED" in globals() and not TCC_SETUP_VALIDATED:
    raise RuntimeError("O SETUP anterior não concluiu a validação com sucesso.")

EXPECTED_TORCHVISION = "0.26.0+cu128"
EXPECTED_OPENCV_HEADLESS = "4.10.0.84"  # Pin do Nerfstudio 1.1.5.
EXPECTED_GPU_NAME = "A100"
EXPECTED_CAPABILITY = (8, 0)
EXPECTED_MIN_VRAM_GIB = 70

EXPECTED_V4 = {
    "python": "3.12.13",
    "torch": "2.11.0+cu128",
    "cuda": "12.8",
    "numpy_tcc": "1.26.4",
    "pandas_tcc": "2.2.2",
    "nerfstudio": "1.1.5",
    "ultralytics": "8.4.132",
    "opencv": "4.11.0.86",
    "openpyxl": "3.1.5",
    "gsplat": "1.4.0",
    "nerfacc": "0.5.2",
    "setuptools": "81.0.0",
}
for chave, esperado in EXPECTED_V4.items():
    observado = VERSIONS.get(chave)
    if observado != esperado:
        raise RuntimeError(
            f"CÉLULA 0 incompatível com V4: {chave}: "
            f"esperado {esperado}, observado {observado}"
        )
if globals().get("TCC_USE_SYSTEM_SITE_PACKAGES", False):
    raise RuntimeError(
        "Esta V4 requer tcc_env isolado SEM --system-site-packages."
    )
if str(TCC_ENV) != "/content/tcc_env":
    raise RuntimeError(f"TCC_ENV inesperado: {TCC_ENV}")

# O SETUP pode ter deixado helpers, mas a CÉLULA 1 é autocontida.
# Evita que variáveis do notebook Python 3.13 alterem o Python 3.12.
# IMPORTANTE: nao sobrescrever o CUDA 12.8 selecionado pelo SETUP.
# /usr/local/cuda, nesta VM, aponta para CUDA 13.x do host.
CUDA_128 = Path("/usr/local/cuda-12.8")
CUDA_128_NVCC = CUDA_128 / "bin" / "nvcc"
if not CUDA_128_NVCC.is_file():
    raise RuntimeError(
        "CUDA Toolkit 12.8 ausente em /usr/local/cuda-12.8. "
        "Execute o SETUP com bootstrap de CUDA 12.8 antes da Celula 1."
    )
TCC_SUBPROCESS_ENV = build_tcc_subprocess_env()
for chave in ("PYTHONPATH", "PYTHONHOME", "PYTHONUSERBASE", "PIP_TARGET"):
    TCC_SUBPROCESS_ENV.pop(chave, None)
TCC_SUBPROCESS_ENV["PATH"] = (
    f"{TCC_ENV_BIN}:{CUDA_128 / 'bin'}:"
    + os.environ.get("PATH", "")
)
TCC_SUBPROCESS_ENV["VIRTUAL_ENV"] = str(TCC_ENV)
TCC_SUBPROCESS_ENV["PYTHONNOUSERSITE"] = "1"
TCC_SUBPROCESS_ENV["PYTHONUNBUFFERED"] = "1"
TCC_SUBPROCESS_ENV["TCNN_CUDA_ARCHITECTURES"] = str(TCNN_CUDA_ARCHITECTURES)
TCC_SUBPROCESS_ENV["TORCH_CUDA_ARCH_LIST"] = "8.0"
TCC_SUBPROCESS_ENV["CUDA_HOME"] = str(CUDA_128)
TCC_SUBPROCESS_ENV["CUDA_PATH"] = str(CUDA_128)
TCC_SUBPROCESS_ENV["CUDACXX"] = str(CUDA_128_NVCC)
# JIT/torch.utils.cpp_extension detectara CUDA_HOME quando for importado
# no SUBPROCESSO (Python 3.12). Nao modifica o Python do Colab.
checar_nvcc = subprocess.run(
    [str(CUDA_128_NVCC), "--version"],
    capture_output=True, text=True, check=True, timeout=30,
)
if not re.search(r"release\s+12\.8\b", checar_nvcc.stdout):
    raise RuntimeError("/usr/local/cuda-12.8/bin/nvcc nao e CUDA 12.8.")
print("C1 toolchain CUDA:", checar_nvcc.stdout.strip().splitlines()[-1])
print("C1 CUDA_HOME:", TCC_SUBPROCESS_ENV["CUDA_HOME"])


def titulo(nome):
    print("\n" + "=" * 100)
    print(nome)
    print("=" * 100)


def exigir_igual(rotulo, atual, esperado):
    if atual != esperado:
        raise RuntimeError(
            f"{rotulo} divergente.\nEsperado: {esperado!r}\nAtual:    {atual!r}"
        )


def executar(args, *, env=None, timeout=240, titulo_cmd=None,
             mostrar_saida=False, check=True):
    """Executa lista de argumentos, sem shell e sem import científico no kernel."""
    if titulo_cmd:
        titulo(titulo_cmd)
    cmd = [str(x) for x in args]
    try:
        resultado = subprocess.run(
            cmd, text=True, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, env=env, timeout=timeout,
            check=False,
        )
    except (FileNotFoundError, subprocess.TimeoutExpired, OSError) as exc:
        raise RuntimeError(f"Não foi possível executar {cmd}: {exc}") from exc
    if mostrar_saida and resultado.stdout:
        print(resultado.stdout.strip())
    if check and resultado.returncode != 0:
        raise RuntimeError(
            f"Falha em {' '.join(cmd)} (código {resultado.returncode}).\n"
            f"Saída:\n{resultado.stdout[-10000:]}"
        )
    return resultado


def executar_python(executavel, codigo, *, env=None, timeout=300,
                    titulo_cmd=None):
    return executar(
        [executavel, "-c", textwrap.dedent(codigo)],
        env=env, timeout=timeout, titulo_cmd=titulo_cmd,
    )


def json_marcado(resultado, identificador):
    prefixo = identificador + "="
    for linha in reversed(resultado.stdout.splitlines()):
        if linha.startswith(prefixo):
            try:
                return json.loads(linha[len(prefixo):])
            except json.JSONDecodeError as exc:
                raise RuntimeError(f"JSON inválido em {identificador}") from exc
    raise RuntimeError(
        f"Subprocesso não retornou {identificador}.\n"
        + resultado.stdout[-7000:]
    )


def dentro_do_tcc(arquivo):
    if not arquivo:
        return False
    return Path(arquivo).resolve().is_relative_to(Path(TCC_ENV).resolve())


def sha256(arquivo):
    h = hashlib.sha256()
    with Path(arquivo).open("rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()


def versao_host(pacote):
    try:
        return metadata.version(pacote)
    except metadata.PackageNotFoundError:
        return "não instalado"


titulo("CÉLULA 1 V4 — COLAB 3.13 / TCC PYTHON 3.12 ISOLADO")
print("Kernel do Colab:", sys.version.split()[0], sys.executable)
print("Python experimental esperado:", VERSIONS["python"])
print("PyTorch experimental esperado:", VERSIONS["torch"])
print("Ambiente experimental:", TCC_ENV)


# ============================================================
# 1/7. MANIFESTO E ESTRUTURA DO AMBIENTE EXPERIMENTAL
# ============================================================
titulo("1/7 — MANIFESTO E ESTRUTURA DO tcc_env")
for rotulo, caminho in (
    ("tcc_env", TCC_ENV), ("Python do TCC", TCC_PYTHON),
    ("ns-train", NS_TRAIN), ("ns-eval", NS_EVAL),
    ("ns-render", NS_RENDER), ("Manifesto do SETUP", SETUP_INFO),
):
    if not Path(caminho).exists():
        raise FileNotFoundError(f"{rotulo} ausente: {caminho}. Execute o SETUP.")

VENV_CFG = Path(TCC_ENV) / "pyvenv.cfg"
if not VENV_CFG.is_file():
    raise RuntimeError("pyvenv.cfg ausente: o ambiente virtual está incompleto.")
pyvenv_texto = VENV_CFG.read_text(encoding="utf-8").lower()
if not re.search(r"(?m)^\s*include-system-site-packages\s*=\s*false\s*$", pyvenv_texto):
    raise RuntimeError(
        "pyvenv.cfg indica herança de pacotes do HOST. "
        "Recrie o tcc_env SEM --system-site-packages."
    )

try:
    setup_info = json.loads(Path(SETUP_INFO).read_text(encoding="utf-8"))
except (ValueError, OSError) as exc:
    raise RuntimeError(f"Manifesto do SETUP inválido: {exc}") from exc

campos_manifesto = {
    "environment_mode": "isolated_python_312",
    "tcc_use_system_site_packages": False,
    "python": VERSIONS["python"],
    "torch": VERSIONS["torch"],
    "torchvision": EXPECTED_TORCHVISION,
    "cuda": VERSIONS["cuda"],
    "numpy": VERSIONS["numpy_tcc"],
    "pandas": VERSIONS["pandas_tcc"],
    "nerfstudio": VERSIONS["nerfstudio"],
    "ultralytics": VERSIONS["ultralytics"],
    "opencv-python": VERSIONS["opencv"],
    "opencv-python-headless": EXPECTED_OPENCV_HEADLESS,
    "openpyxl": VERSIONS["openpyxl"],
    "gsplat": VERSIONS["gsplat"],
    "nerfacc": VERSIONS["nerfacc"],
    "setuptools": VERSIONS["setuptools"],
    "environment": str(TCC_ENV),
    "python_executable": str(TCC_PYTHON),
}
for campo, esperado in campos_manifesto.items():
    exigir_igual(f"Manifesto do SETUP: {campo}", setup_info.get(campo), esperado)
exigir_igual(
    "Snapshot do host registrado no SETUP",
    setup_info.get("base_package_snapshot"), BASE_PACKAGE_SNAPSHOT,
)
tcnn_manifest = setup_info.get("tiny-cuda-nn", {})
if REQUIRE_TCNN:
    if not tcnn_manifest.get("available") or not tcnn_manifest.get("functional_cuda_test"):
        raise RuntimeError("O SETUP não registrou TCNN funcional em CUDA.")
    exigir_igual(
        "Arquitetura TCNN do SETUP",
        str(tcnn_manifest.get("cuda_architectures")), str(TCNN_CUDA_ARCHITECTURES),
    )
    exigir_igual(
        "Fonte TCNN do SETUP", tcnn_manifest.get("source"), TCNN_SOURCE,
    )

HOST_VERSION = ".".join(str(v) for v in sys.version_info[:3])
if "host_runtime" in setup_info:
    exigir_igual("Python HOST / SETUP", HOST_VERSION,
                 setup_info["host_runtime"].get("python"))
print("Modo:", setup_info["environment_mode"])
print("HOST:", HOST_VERSION)
print("tcc_env:", setup_info["python"], "(isolado, sem pacotes do HOST)")
print("Manifesto e diretórios: OK")


# ============================================================
# 2/7. INTEGRIDADE DO HOST — PYTHON 3.13 NÃO É PIN DO TCC
# ============================================================
titulo("2/7 — HOST COLAB: PACOTES E CUDA SEM ALTERAÇÃO")
BASE_PACKAGE_CURRENT = {
    nome: versao_host(nome) for nome in BASE_PACKAGE_SNAPSHOT
}
for nome, antes in BASE_PACKAGE_SNAPSHOT.items():
    depois = BASE_PACKAGE_CURRENT[nome]
    print(f"{nome}: antes={antes} | agora={depois}")
    exigir_igual(f"Pacote do HOST {nome}", depois, antes)

# O torch do HOST é somente DIAGNÓSTICO. A identidade necessária
# ao experimento será testada abaixo no Python 3.12 isolado.
HOST_PROBE_CODE = r'''
import json
import sys
import torch
ok = bool(torch.cuda.is_available())
info = {
    "python": ".".join(map(str, sys.version_info[:3])),
    "executable": sys.executable,
    "torch": torch.__version__, "cuda": torch.version.cuda,
    "cuda_available": ok,
}
if ok:
    props = torch.cuda.get_device_properties(0)
    y = torch.tensor([1., 2., 3.], device="cuda") * 2
    torch.cuda.synchronize()
    info.update({
        "gpu": props.name,
        "capability": [props.major, props.minor],
        "vram_gib": round(props.total_memory / 1024**3, 2),
        "cuda_test_result": y.cpu().tolist(),
    })
print("TCC_HOST_VALIDATION_JSON=" + json.dumps(info))
'''
host_resultado = executar_python(
    sys.executable, HOST_PROBE_CODE, timeout=180,
    titulo_cmd="PYTORCH / CUDA DO HOST (APENAS DIAGNÓSTICO)",
)
host_info = json_marcado(host_resultado, "TCC_HOST_VALIDATION_JSON")
exigir_igual("Python do HOST", host_info["python"], HOST_VERSION)
if not host_info.get("cuda_available"):
    raise RuntimeError("CUDA não está acessível nesta VM do Colab.")
if EXPECTED_GPU_NAME not in str(host_info.get("gpu", "")).upper():
    raise RuntimeError("GPU do HOST não é NVIDIA A100.")
exigir_igual("Compute capability HOST", host_info.get("capability"), [8, 0])
exigir_igual("Cálculo CUDA HOST", host_info.get("cuda_test_result"), [2., 4., 6.])
print("Host:", host_info["python"], host_info["gpu"])
print("Torch HOST:", host_info["torch"], "CUDA:", host_info["cuda"])
print("Snapshot do HOST e CUDA: OK")


# ============================================================
# 3/7. STACK REAL DO PYTHON EXPERIMENTAL 3.12 (SUBPROCESSO)
# ============================================================
# Teste GPU funcional, pequeno e determinístico; NÃO é treinamento.
# O micro-render do gsplat usa o mesmo modo antialiased da V4.
# ============================================================
TCC_PROBE_CODE = r'''
import importlib.metadata as md
import json
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import torch
import torchvision
import nerfstudio
import ultralytics
import openpyxl
import gsplat
import nerfacc
from gsplat.rendering import rasterization
from nerfacc import OccGridEstimator

required_tcnn = sys.argv[1] == "1"
root = Path(sys.argv[2]).resolve()
assert Path(sys.prefix).resolve() == root, (sys.prefix, root)
assert sys.prefix != sys.base_prefix, "Não é um venv"

# Testes básicos das bibliotecas locais.
x = np.arange(6).reshape(2, 3)
assert int(x.sum()) == 15
frame = pd.DataFrame({"valor": [1, 2, 3]})
assert int(frame["valor"].sum()) == 6
imagem = np.zeros((32, 32, 3), dtype=np.uint8)
assert cv2.cvtColor(imagem, cv2.COLOR_BGR2GRAY).shape == (32, 32)
assert cv2.resize(imagem, (16, 16)).shape == (16, 16, 3)

assert torch.cuda.is_available(), "CUDA não disponível no Python 3.12"
props = torch.cuda.get_device_properties(0)
y = torch.tensor([1., 2., 3.], device="cuda") * 2
assert y.cpu().tolist() == [2., 4., 6.]

# gsplat: JIT + rasterização CUDA usando antialiasing.
g_means = torch.tensor([[0., 0., 2.]], device="cuda")
g_quats = torch.tensor([[1., 0., 0., 0.]], device="cuda")
g_scales = torch.full((1, 3), .35, device="cuda")
g_opacities = torch.tensor([.9], device="cuda")
g_colors = torch.tensor([[1., 0., 0.]], device="cuda")
g_view = torch.eye(4, device="cuda")[None]
g_K = torch.tensor([[[16., 0., 16.], [0., 16., 16.], [0., 0., 1.]]], device="cuda")
g_rgb, g_alpha, _ = rasterization(
    g_means, g_quats, g_scales, g_opacities, g_colors,
    g_view, g_K, 32, 32, packed=False, rasterize_mode="antialiased",
)
torch.cuda.synchronize()
assert g_rgb.is_cuda and tuple(g_rgb.shape) == (1, 32, 32, 3)
assert g_alpha.is_cuda and float(g_alpha.max()) > 0

# nerfacc: importação da classe usada na aceleração do NeRF.
assert callable(OccGridEstimator)

tcnn_info = {"available": False, "functional_cuda": False}
if required_tcnn:
    import tinycudann as tcnn
    x_tcnn = torch.rand((32, 3), device="cuda", dtype=torch.float32)
    encoding = tcnn.Encoding(
        n_input_dims=3,
        encoding_config={
            "otype": "HashGrid", "n_levels": 2,
            "n_features_per_level": 2, "log2_hashmap_size": 12,
            "base_resolution": 8, "per_level_scale": 2.0,
        },
    )
    y_tcnn = encoding(x_tcnn)
    torch.cuda.synchronize()
    assert y_tcnn.is_cuda and y_tcnn.shape[0] == 32
    try:
        versao_tcnn = md.version("tinycudann")
    except md.PackageNotFoundError:
        versao_tcnn = "metadata indisponível"
    tcnn_info = {
        "available": True, "functional_cuda": True,
        "version": versao_tcnn,
        "module_path": tcnn.__file__, "output_shape": list(y_tcnn.shape),
    }

module_files = {
    "numpy": np.__file__, "pandas": pd.__file__,
    "cv2": cv2.__file__, "torch": torch.__file__,
    "torchvision": torchvision.__file__,
    "nerfstudio": nerfstudio.__file__,
    "ultralytics": ultralytics.__file__,
    "openpyxl": openpyxl.__file__,
    "gsplat": gsplat.__file__, "nerfacc": nerfacc.__file__,
}
versions = {
    "python": ".".join(map(str, sys.version_info[:3])),
    "torch": torch.__version__, "torchvision": torchvision.__version__,
    "cuda": torch.version.cuda, "numpy": np.__version__,
    "pandas": pd.__version__, "opencv-python": md.version("opencv-python"),
    "opencv-python-headless": md.version("opencv-python-headless"),
    "opencv_runtime": cv2.__version__,
    "nerfstudio": md.version("nerfstudio"),
    "ultralytics": md.version("ultralytics"),
    "openpyxl": md.version("openpyxl"),
    "gsplat": md.version("gsplat"), "nerfacc": md.version("nerfacc"),
    "setuptools": md.version("setuptools"),
}
info = {
    "prefix": sys.prefix, "base_prefix": sys.base_prefix,
    "executable": sys.executable, "versions": versions,
    "module_files": module_files,
    "cuda_available": True,
    "gpu": props.name, "capability": [props.major, props.minor],
    "vram_gib": round(props.total_memory / 1024**3, 2),
    "cuda_test_result": [2., 4., 6.],
    "gsplat_antialiased_cuda": True,
    "nerfacc_estimator_import": True,
    "tcnn": tcnn_info,
}
print("TCC_STACK_VALIDATION_JSON=" + json.dumps(info, ensure_ascii=False))
'''

# sys.argv no -c começa em índice 1 após o texto do código.
tcc_resultado = executar(
    [TCC_PYTHON, "-c", TCC_PROBE_CODE,
     "1" if REQUIRE_TCNN else "0", TCC_ENV],
    env=TCC_SUBPROCESS_ENV, timeout=900,
    titulo_cmd="3/7 — TESTES DO tcc_env: TORCH / GSPLAT / NERFACC / TCNN",
)
tcc_info = json_marcado(tcc_resultado, "TCC_STACK_VALIDATION_JSON")
versoes = tcc_info["versions"]
esperadas = {
    "python": VERSIONS["python"],
    "torch": VERSIONS["torch"],
    "torchvision": EXPECTED_TORCHVISION,
    "cuda": VERSIONS["cuda"],
    "numpy": VERSIONS["numpy_tcc"],
    "pandas": VERSIONS["pandas_tcc"],
    "opencv-python": VERSIONS["opencv"],
    "opencv-python-headless": EXPECTED_OPENCV_HEADLESS,
    "nerfstudio": VERSIONS["nerfstudio"],
    "ultralytics": VERSIONS["ultralytics"],
    "openpyxl": VERSIONS["openpyxl"],
    "gsplat": VERSIONS["gsplat"],
    "nerfacc": VERSIONS["nerfacc"],
    "setuptools": VERSIONS["setuptools"],
}
for campo, valor in esperadas.items():
    exigir_igual(f"Pacote do tcc_env {campo}", versoes.get(campo), valor)

# Os dois distributions OpenCV coexistem por causa das exigências
# declaradas de Nerfstudio/Ultralytics. Elas compartilham cv2.
# NÃO presumir que cv2.__version__ seja o número do wheel GUI.
if not versoes["opencv_runtime"].startswith(("4.10.", "4.11.")):
    raise RuntimeError("cv2 carregado é incompatível: " + versoes["opencv_runtime"])
exigir_igual("cv2 usado no SETUP", versoes["opencv_runtime"],
             setup_info.get("opencv_runtime"))
exigir_igual("sys.prefix", str(Path(tcc_info["prefix"]).resolve()),
             str(Path(TCC_ENV).resolve()))
exigir_igual("Cálculo CUDA TCC", tcc_info.get("cuda_test_result"), [2., 4., 6.])
exigir_igual("Compute capability TCC", tcc_info.get("capability"), [8, 0])
if EXPECTED_GPU_NAME not in str(tcc_info.get("gpu", "")).upper():
    raise RuntimeError("GPU do tcc_env não é NVIDIA A100.")
if tcc_info.get("vram_gib", 0) < EXPECTED_MIN_VRAM_GIB:
    raise RuntimeError("VRAM detectada é insuficiente para A100 80 GB.")
if not tcc_info.get("gsplat_antialiased_cuda"):
    raise RuntimeError("gsplat antialiased não executou na GPU.")
if not tcc_info.get("nerfacc_estimator_import"):
    raise RuntimeError("nerfacc.OccGridEstimator indisponível.")
if REQUIRE_TCNN and not tcc_info.get("tcnn", {}).get("functional_cuda"):
    raise RuntimeError("tiny-cuda-nn / HashGrid não executou na GPU.")

# ============================================================
# 4/7. ISOLAMENTO FÍSICO ENTRE O HOST E O tcc_env
# ============================================================
titulo("4/7 — ISOLAMENTO DE MÓDULOS E PYTORCH LOCAL")
for nome, caminho in tcc_info["module_files"].items():
    if not dentro_do_tcc(caminho):
        raise RuntimeError(
            f"O módulo {nome} não foi carregado do tcc_env:\n"
            f"{caminho}\nO Python 3.12 NÃO pode herdar bibliotecas do Colab 3.13."
        )
    print(f"{nome}: {caminho}")
if REQUIRE_TCNN:
    caminho_tcnn = tcc_info["tcnn"].get("module_path")
    if not dentro_do_tcc(caminho_tcnn):
        raise RuntimeError(f"TCNN não está instalado localmente: {caminho_tcnn}")
print("PyTorch é LOCAL no tcc_env:", tcc_info["module_files"]["torch"])
print("Python 3.13 do HOST não é herdado: OK")
print("gsplat CUDA antialiased: OK")
print("TCNN HashGrid CUDA:", "OK" if REQUIRE_TCNN else "não exigido")


# ============================================================
# 5/7. NERFSTUDIO / EXECUTÁVEIS / PRESETS REAIS DA V4
# ============================================================
titulo("5/7 — NERFSTUDIO 1.1.5 E PRESETS UTILIZADOS NA V4")
CLI_TESTS = (
    ("ns-train", [NS_TRAIN, "--help"]),
    ("nerfacto-huge", [NS_TRAIN, "nerfacto-huge", "--help"]),
    ("splatfacto-big", [NS_TRAIN, "splatfacto-big", "--help"]),
    ("ns-eval", [NS_EVAL, "--help"]),
    ("ns-render", [NS_RENDER, "--help"]),
)
CLI_RESULTS = {}
for nome, cmd in CLI_TESTS:
    resultado = executar(cmd, env=TCC_SUBPROCESS_ENV, timeout=300)
    if not resultado.stdout.strip():
        raise RuntimeError(f"A ajuda de {nome} está vazia.")
    CLI_RESULTS[nome] = True
    print(nome + ": OK")
    if nome == "ns-eval":
        for opcao in ("--load-config", "--output-path", "--render-output-path"):
            if opcao not in resultado.stdout:
                raise RuntimeError(f"ns-eval não contém {opcao}, exigido pela CÉLULA D.")
print("Interface ns-eval exigida pela D: OK")

# O pip check somente LÊ a resolução; não instala nada.
excutar_pip = executar(
    [TCC_PYTHON, "-m", "pip", "check"], env=TCC_SUBPROCESS_ENV,
    timeout=120, titulo_cmd="PIP CHECK — CONSISTÊNCIA DAS DEPENDÊNCIAS",
)
print(excutar_pip.stdout.strip() or "pip check: OK")


# ============================================================
# 6/7. FERRAMENTAS DO SISTEMA / GPU / COMPILADOR
# ============================================================
titulo("6/7 — GPU, NVCC, COLMAP E FFMPEG")
NVIDIA_INFO = executar(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
     "--format=csv,noheader"], timeout=60,
).stdout.strip()
if "A100" not in NVIDIA_INFO.upper():
    raise RuntimeError("nvidia-smi não identificou a A100: " + NVIDIA_INFO)

# O nvcc do host nao e o do experimento: consulte o binario explicito.
nvcc_path = str(CUDA_128_NVCC)
NVCC_VERSION = None
nvcc_saida = executar([nvcc_path, "--version"], timeout=60).stdout
if not re.search(r"release\s+12\.8\b", nvcc_saida):
    raise RuntimeError("nvcc selecionado nao e CUDA Toolkit 12.8.")
NVCC_VERSION = nvcc_saida.strip().splitlines()[-1]

COLMAP_VERSION = executar(
    ["dpkg-query", "-W", "-f=${Version}", "colmap"], timeout=60,
).stdout.strip()
executar(["colmap", "help"], timeout=120)
ffmpeg_saida = executar(["ffmpeg", "-version"], timeout=60).stdout
FFMPEG_VERSION = ffmpeg_saida.splitlines()[0] if ffmpeg_saida else ""
if not FFMPEG_VERSION:
    raise RuntimeError("FFmpeg não retornou a versão.")
print("GPU:", NVIDIA_INFO)
print("NVCC:", NVCC_VERSION)
print("COLMAP:", COLMAP_VERSION)
print("FFmpeg:", FFMPEG_VERSION)


# ============================================================
# 7/7. CONSISTÊNCIA FINAL E RELATÓRIO DE PROVENIÊNCIA
# ============================================================
titulo("7/7 — CONSISTÊNCIA FINAL E RELATÓRIO DE VALIDAÇÃO")
# Verifica novamente o snapshot: CÉLULA 1 não modificou o HOST.
for nome, esperado in BASE_PACKAGE_SNAPSHOT.items():
    exigir_igual(f"HOST após validação: {nome}", versao_host(nome), esperado)

# Os dados do manifesto precisam coincidir com as bibliotecas de fato usadas.
for chave, esperado in esperadas.items():
    if chave == "opencv_runtime":
        continue
    exigir_igual(
        f"Runtime/manifesto: {chave}",
        versoes.get(chave), setup_info.get(chave),
    )

ENVIRONMENT_REPORT_PATH = (
    Path(RESULTS_DIR) / "environment_validation_v4_isolated.json"
)
relatorio = {
    "validated_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "validated",
    "experiment": "tcc_v4",
    "setup_manifest_path": str(SETUP_INFO),
    "setup_manifest_sha256": sha256(SETUP_INFO),
    "pyvenv_cfg_sha256": sha256(VENV_CFG),
    "host": host_info,
    "host_packages": dict(BASE_PACKAGE_CURRENT),
    "tcc_env": tcc_info,
    "tcc_isolated": True,
    "system_site_packages": False,
    "nvidia_smi": NVIDIA_INFO,
    "nvcc": NVCC_VERSION,
    "colmap": COLMAP_VERSION,
    "ffmpeg": FFMPEG_VERSION,
    "nerfstudio_cli_tests": CLI_RESULTS,
    "ns_eval_required_options": [
        "--load-config", "--output-path", "--render-output-path",
    ],
}
ENVIRONMENT_REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
ENVIRONMENT_REPORT_PATH.write_text(
    json.dumps(relatorio, indent=2, ensure_ascii=False), encoding="utf-8",
)

# IMPORTANTE: Somente depois de TODAS as verificações acima.
ENVIRONMENT_VALIDATED = True

print("\n" + "=" * 100)
print("CÉLULA 1 V4 — AMBIENTE EXPERIMENTAL VALIDADO COM SUCESSO")
print("=" * 100)
print("HOST Colab: ", HOST_VERSION)
print("TCC isolado:", versoes["python"])
print("torch:       ", versoes["torch"], "/ CUDA", versoes["cuda"])
print("torchvision: ", versoes["torchvision"])
print("NumPy/Pandas:", versoes["numpy"], "/", versoes["pandas"])
print("Nerfstudio:  ", versoes["nerfstudio"])
print("gsplat/nerfacc:", versoes["gsplat"], "/", versoes["nerfacc"])
print("OpenCV GUI/headless:", versoes["opencv-python"], "/",
      versoes["opencv-python-headless"])
print("cv2 runtime: ", versoes["opencv_runtime"])
print("GPU:         ", tcc_info["gpu"], "CC", tcc_info["capability"])
print("gsplat / TCNN CUDA: OK")
print("Isolamento:  SEM system-site-packages")
print("Relatório:  ", ENVIRONMENT_REPORT_PATH)
print("ENVIRONMENT_VALIDATED =", ENVIRONMENT_VALIDATED)
print("Nenhuma dependência foi instalada ou atualizada por esta célula.")
print("Próximo passo: CÉLULA A (ou retomar checkpoints V4 já preservados).")


In [ ]:
# ============================================================
# CÉLULA A V7 — nuScenes scene-0655, CAM_FRONT: protocolo alinhado à H V3
# Objetivo (se fisicamente viável): 95 TRAIN / 10 VAL / 10 TEST.
# Split cronológico ANTES de treinamento, com guardas temporais.
# Poses e calibração oficiais do nuScenes, não COLMAP (não são vazamento).
# LiDAR: somente scans vinculados a TRAIN, coloridos somente por RGB TRAIN.
# Nenhum RGB de TEST entra na geometria, colorização ou escolha de parâmetros.
# Dataset isolado por execução; sem reutilizar checkpoints anteriores.
# As células B–G usam os hashes e os artefatos gerados nesta execução.
# ============================================================

# ============================================================
# 1. IMPORTS
# ============================================================

from datetime import datetime, timezone

from pathlib import Path
from collections import defaultdict, Counter
from functools import lru_cache
from bisect import bisect_left

from PIL import Image, ImageDraw

import hashlib
import json
import math
import shutil
import subprocess

import numpy as np


# ============================================================
# 2. VALIDAR CÉLULAS ANTERIORES
# ============================================================

VARIAVEIS_OBRIGATORIAS = (
    "TCC_ROOT",
    "OUTPUT_DATASET",
    "OUTPUT_DIR",
    "LOG_DIR",
    "RESULTS_DIR",
    "TCC_ENV",
    "TCC_PYTHON",
    "NS_TRAIN",
    "NS_EVAL",
)

faltando = [
    nome
    for nome in VARIAVEIS_OBRIGATORIAS
    if nome not in globals()
]

if faltando:
    raise RuntimeError(
        "Execute a CÉLULA 0, a CÉLULA SETUP e a CÉLULA 1 "
        "antes da CÉLULA A.\n\n"
        "Variáveis ausentes:\n"
        + "\n".join(f"- {nome}" for nome in faltando)
    )

if not globals().get("ENVIRONMENT_VALIDATED", False):
    raise RuntimeError(
        "O ambiente ainda não foi validado pela CÉLULA 1.\n"
        "Execute a CÉLULA 1 antes de preparar o dataset."
    )

if not TCC_PYTHON.exists():
    raise RuntimeError(
        "Python do tcc_env não encontrado:\n"
        f"{TCC_PYTHON}"
    )

if not NS_TRAIN.exists():
    raise RuntimeError(
        "ns-train não encontrado:\n"
        f"{NS_TRAIN}"
    )


# ============================================================
# 3. DADOS CONFIGURÁVEIS (MONTAGEM OPCIONAL E FORA DO NOTEBOOK)
# ============================================================

if not TCC_ROOT.is_dir():
    raise FileNotFoundError(
        "Diretório de dados não disponível. Defina TCC_DATA_ROOT antes da célula 0 "
        "e, se usar armazenamento externo, faça a montagem em uma célula privada."
    )


# ============================================================
# 4. CONFIGURAÇÃO DA RODADA A V7 / H V3
# ============================================================
EXPERIMENT_TAG = 'scene0655_hv3_protocol_Av7_95_10_10'
DATASET_PROFILE = 'nuscenes_front_samples_train_sweeps_holdout_10_10_v7'

# Se já houver pasta com este nome na sessão, NÃO apagá-la automaticamente.
A_ALLOW_REBUILD = False

OUTPUT_DATASET = Path('/content') / ('nuscenes_nerfstudio_' + EXPERIMENT_TAG)
OUTPUT_DIR = Path('/content') / ('outputs_nuscenes_' + EXPERIMENT_TAG)
LOG_DIR = Path('/content') / ('logs_nuscenes_' + EXPERIMENT_TAG)
RESULTS_DIR = Path('/content') / ('results_nuscenes_' + EXPERIMENT_TAG)

SCENE_NAME = 'scene-0655'
CAMERA_CHANNELS = ['CAM_FRONT']
ALL_CAMERA_CHANNELS = [
    'CAM_FRONT', 'CAM_FRONT_LEFT', 'CAM_FRONT_RIGHT',
    'CAM_BACK', 'CAM_BACK_LEFT', 'CAM_BACK_RIGHT',
]
USE_SAMPLES = True
USE_CAMERA_SWEEPS = True
USE_SWEEPS = True
MAX_FRAMES_PER_CAMERA = None

# Para tentar 95/10/10, utilizar toda a cena (aproximadamente 20 s).
# Não inflar o número de imagens duplicando frames ou outras câmeras.
TEMPORAL_WINDOW_SECONDS = None
TEMPORAL_CENTER_FRACTION = 0.5
IMAGE_DOWNSCALE_FACTOR = 2  # CAM_FRONT: 1600x900 -> 800x450
if IMAGE_DOWNSCALE_FACTOR < 1:
    raise ValueError('IMAGE_DOWNSCALE_FACTOR deve ser >=1.')

# 5. DEFINIÇÃO PRÉVIA DO PROTOCOLO / HOLDOUT
# ============================================================
A_TARGET_TRAIN = 95
A_TARGET_VAL = 10
A_TARGET_TEST = 10
A_REQUIRE_EXACT_TRAIN = True
# Se a cena NÃO suportar 95, abortar e informar o número possível.
# Não mudar a configuração silenciosamente após observar resultados.
A_HOLDOUT_STRATEGY = '20 vistas CAM_FRONT espaçadas no tempo, 95 treinos com guarda de 250ms'

# O nuScenes fornece keyframes aproximadamente a cada 0,5 s, enquanto os
# sweeps são mais densos. A guarda é em MILISSEGUNDOS, não em número de frames.
# H V3 (4 fps) reserva vizinhos em ±0,25 s; usar 250 ms aqui.
HOLDOUT_SWEEP_GUARD_MS = 250.0
# Não impor espaçamento artificial dentro do próprio TRAIN (a H não impõe).
# Há guarda temporal rigorosa entre TRAIN e imagens reservadas.
MIN_TRAIN_SWEEP_INTERVAL_MS = None
MAX_TRAIN_SWEEPS_PER_CAMERA = None
MIN_AVAILABLE_SWEEPS = 12
MIN_SELECTED_TRAIN_SWEEPS = 8

# Variáveis antigas mantidas como NULL apenas para compatibilidade do manifesto.
HOLDOUT_EVERY = None
VAL_OFFSET = None
TEST_OFFSET = None

# Restrições comparativas adicionais:
# - seleção por TEMPO e sem usar PSNR/YOLO para escolher quadros;
# - TEST/VAL podem ser keyframes ou sweeps, como a H V3;
# - normalização geométrica deriva exclusivamente de TRAIN;
# - 95/10/10 é meta, não permissão para fabricar amostras.

# 6. NORMALIZAÇÃO GEOMÉTRICA
# ============================================================

TARGET_RADIUS = 4.0


# ============================================================
# 7. LiDAR — CONFIGURAÇÃO DE DIAGNÓSTICO
# ============================================================

LIDAR_CHANNEL = "LIDAR_TOP"
CREATE_LIDAR_POINT_CLOUD = True

# Apenas scans sample/keyframe do split de treino.
USE_LIDAR_SAMPLES = True
USE_LIDAR_SWEEPS = False
LIDAR_TRAIN_SPLIT_ONLY = True

# Cena pequena; 40 é suficiente para capturar todos os scans de
# treino disponíveis sem exagerar.
MAX_LIDAR_FRAMES = 40

# Muito menor que os 500k da rodada pesada.
MAX_POINTS_TOTAL = 120_000
LIDAR_RANDOM_SEED = 42

MIN_LIDAR_DISTANCE = 2.0
MAX_LIDAR_DISTANCE = 80.0

USE_HEIGHT_FILTER_LOCAL = True
MIN_LOCAL_Z = -4.0
MAX_LOCAL_Z = 8.0

# Colorização somente com imagens de treino.
COLORIZE_LIDAR_WITH_CAMERAS = True
MAX_CAMERA_LIDAR_DELTA_MS = 60.0
LIDAR_COLOR_USE_ZBUFFER = True


# ============================================================
# 8. DIAGNÓSTICOS
# ============================================================

CREATE_GEOMETRY_DIAGNOSTICS = True
DIAGNOSTIC_OVERLAY_COUNT = 8
DIAGNOSTIC_MAX_PROJECTED_POINTS = 12_000
DIAGNOSTIC_POINT_RADIUS = 2

# Objetos dinâmicos continuam na nuvem: sem máscaras confiáveis por sweep.
# Esta é uma limitação metodológica explícita a analisar visualmente.
REMOVE_DYNAMIC_LIDAR_POINTS = False


# ============================================================
# 9. DIRETÓRIOS
# ============================================================

for directory in (
    OUTPUT_DIR,
    LOG_DIR,
    RESULTS_DIR,
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

print("\n" + "=" * 100)
print("CONFIGURAÇÃO DO DATASET — A V7, ALINHADA À H V3")
print("=" * 100)

print("Experiment tag:", EXPERIMENT_TAG)
print("Perfil:", DATASET_PROFILE)
print("Cena:", SCENE_NAME)
print("Câmeras de treino:", CAMERA_CHANNELS)
print("Samples:", USE_SAMPLES)
print("Sweeps de câmera:", USE_CAMERA_SWEEPS)
print("Janela temporal:", TEMPORAL_WINDOW_SECONDS)
print("Downscale:", IMAGE_DOWNSCALE_FACTOR, "x")
print("LiDAR somente treino:", LIDAR_TRAIN_SPLIT_ONLY)
print("LiDAR sweeps:", USE_LIDAR_SWEEPS)
print("Máximo de pontos LiDAR:", MAX_POINTS_TOTAL)
print("Diagnóstico geométrico:", CREATE_GEOMETRY_DIAGNOSTICS)
print("Meta TRAIN/VAL/TEST:", A_TARGET_TRAIN, A_TARGET_VAL, A_TARGET_TEST)
print("Exigir 95 exatos:", A_REQUIRE_EXACT_TRAIN)
print("OUTPUT_DATASET:", OUTPUT_DATASET)


# ============================================================
# 10. FUNÇÕES GERAIS
# ============================================================

def run(cmd, check=True):
    if isinstance(cmd, (list, tuple)):
        args = [str(x) for x in cmd]
        shell = False
        display = " ".join(args)
    else:
        args = str(cmd)
        shell = True
        display = args

    print("\n$", display)

    result = subprocess.run(
        args,
        shell=shell,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

    if result.stdout:
        print(result.stdout.strip())

    if check and result.returncode != 0:
        raise RuntimeError(
            "Comando falhou:\n\n"
            f"{display}\n\n"
            f"{result.stdout}"
        )

    return result


def load_json(json_dir, name, required=True):
    path = Path(json_dir) / name

    if not path.exists():
        if required:
            raise FileNotFoundError(
                f"JSON não encontrado:\n{path}"
            )
        return None

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            block = f.read(1024 * 1024)
            if not block:
                break
            digest.update(block)

    return digest.hexdigest()


def safe_rmtree_dataset(path):
    path = Path(path).resolve()

    # Proteção contra remoção acidental.
    if not str(path).startswith("/content/nuscenes_nerfstudio_"):
        raise RuntimeError(
            "Proteção de segurança: OUTPUT_DATASET inesperado:\n"
            f"{path}"
        )

    if path.exists():
        shutil.rmtree(path)


# ============================================================
# 11. TRANSFORMAÇÕES GEOMÉTRICAS
# ============================================================

def quat_trans_to_matrix(rotation, translation):
    """
    Quaternion nuScenes: [w, x, y, z].
    Retorna transformação local -> parent.
    """
    q = np.asarray(rotation, dtype=np.float64)
    norm = np.linalg.norm(q)

    if norm < 1e-12:
        raise ValueError("Quaternion inválido.")

    q = q / norm
    w, x, y, z = q

    rot = np.array(
        [
            [
                1 - 2*y*y - 2*z*z,
                2*x*y - 2*z*w,
                2*x*z + 2*y*w,
            ],
            [
                2*x*y + 2*z*w,
                1 - 2*x*x - 2*z*z,
                2*y*z - 2*x*w,
            ],
            [
                2*x*z - 2*y*w,
                2*y*z + 2*x*w,
                1 - 2*x*x - 2*y*y,
            ],
        ],
        dtype=np.float64,
    )

    mat = np.eye(4, dtype=np.float64)
    mat[:3, :3] = rot
    mat[:3, 3] = np.asarray(
        translation,
        dtype=np.float64,
    )

    return mat


def camera_to_global_opencv(
    camera_sd,
    calib_by_token,
    ego_by_token,
):
    """
    Pose individual da câmera no global do nuScenes.

    IMPORTANTE:
    usa o ego_pose_token do próprio sample_data da câmera,
    não uma pose compartilhada pelo sample.
    """
    calib = calib_by_token[
        camera_sd["calibrated_sensor_token"]
    ]
    ego = ego_by_token[
        camera_sd["ego_pose_token"]
    ]

    cam_to_ego = quat_trans_to_matrix(
        calib["rotation"],
        calib["translation"],
    )
    ego_to_global = quat_trans_to_matrix(
        ego["rotation"],
        ego["translation"],
    )

    return ego_to_global @ cam_to_ego


def lidar_to_global_matrix(
    lidar_sd,
    calib_by_token,
    ego_by_token,
):
    calib = calib_by_token[
        lidar_sd["calibrated_sensor_token"]
    ]
    ego = ego_by_token[
        lidar_sd["ego_pose_token"]
    ]

    lidar_to_ego = quat_trans_to_matrix(
        calib["rotation"],
        calib["translation"],
    )
    ego_to_global = quat_trans_to_matrix(
        ego["rotation"],
        ego["translation"],
    )

    return ego_to_global @ lidar_to_ego


# ============================================================
# 12. CAMINHOS NUSCENES
# ============================================================

def find_nuscenes_paths(tcc_root):
    required = [
        "sensor.json",
        "calibrated_sensor.json",
        "ego_pose.json",
        "sample.json",
        "sample_data.json",
        "scene.json",
    ]

    sensor_paths = sorted(
        tcc_root.rglob("sensor.json")
    )

    if not sensor_paths:
        raise FileNotFoundError(
            "sensor.json não encontrado dentro de:\n"
            f"{tcc_root}"
        )

    for sensor_path in sensor_paths:
        json_dir = sensor_path.parent

        if not all(
            (json_dir / name).exists()
            for name in required
        ):
            continue

        candidates = [
            json_dir,
            json_dir.parent,
        ]

        for candidate in candidates:
            if (
                (candidate / "samples").exists()
                or
                (candidate / "sweeps").exists()
            ):
                print("JSON_DIR:", json_dir)
                print("NUSCENES_ROOT:", candidate)
                return candidate, json_dir

    raise FileNotFoundError(
        "Não foi encontrada uma estrutura completa do nuScenes."
    )


# ============================================================
# 13. HELPERS DE IMAGEM
# ============================================================

def make_relative_image_name(filename):
    parts = Path(filename).parts
    return "_".join(parts[-2:])


@lru_cache(maxsize=64)
def load_rgb_image(path_string):
    with Image.open(path_string) as img:
        return np.asarray(img.convert("RGB"))


def save_resized_rgb(src, dst, factor):
    with Image.open(src) as img:
        img = img.convert("RGB")

        if factor == 1:
            img.save(dst, format="JPEG", quality=95)
            return img.width, img.height

        new_width = int(round(img.width / factor))
        new_height = int(round(img.height / factor))

        if new_width <= 0 or new_height <= 0:
            raise RuntimeError(
                "Dimensão inválida após downscale."
            )

        img = img.resize(
            (new_width, new_height),
            Image.Resampling.LANCZOS,
        )
        img.save(dst, format="JPEG", quality=95)

        return new_width, new_height


# ============================================================
# 14. HELPERS TEMPORAIS
# ============================================================

def is_sample_frame(sd):
    filename = sd.get("filename", "")
    return bool(
        sd.get("is_key_frame", False)
        or filename.startswith("samples/")
    )


def is_sweep_frame(sd):
    filename = sd.get("filename", "")
    return (
        filename.startswith("sweeps/")
        and not is_sample_frame(sd)
    )


def nearest_timestamp_delta_us(
    timestamp,
    sorted_timestamps,
):
    if not sorted_timestamps:
        return None

    pos = bisect_left(
        sorted_timestamps,
        timestamp,
    )

    candidates = []

    if pos < len(sorted_timestamps):
        candidates.append(sorted_timestamps[pos])

    if pos > 0:
        candidates.append(sorted_timestamps[pos - 1])

    if not candidates:
        return None

    return min(
        abs(timestamp - candidate)
        for candidate in candidates
    )


# ============================================================
# 15. LiDAR — IO
# ============================================================

def load_lidar_bin(path):
    points = np.fromfile(
        str(path),
        dtype=np.float32,
    )

    if points.size % 5 == 0:
        return points.reshape(-1, 5)

    if points.size % 4 == 0:
        return points.reshape(-1, 4)

    raise ValueError(
        "Formato LiDAR inesperado:\n"
        f"{path}\n"
        f"Valores encontrados: {points.size}"
    )


def intensity_to_rgb(intensity):
    intensity = np.asarray(
        intensity,
        dtype=np.float32,
    )

    if len(intensity) == 0:
        return np.empty((0, 3), dtype=np.uint8)

    p1, p99 = np.percentile(
        intensity,
        [1, 99],
    )

    if p99 <= p1:
        gray = np.full(
            len(intensity),
            180,
            dtype=np.uint8,
        )
    else:
        gray = np.clip(
            (intensity - p1) / (p99 - p1),
            0,
            1,
        )
        gray = (gray * 255).astype(np.uint8)

    return np.stack(
        [gray, gray, gray],
        axis=1,
    )


def save_ply_xyzrgb(path, xyz, rgb):
    path = Path(path)
    xyz = np.asarray(xyz, dtype=np.float32)
    rgb = np.asarray(rgb, dtype=np.uint8)

    if len(xyz) != len(rgb):
        raise ValueError(
            "xyz e rgb possuem tamanhos diferentes."
        )

    # Binary little endian reduz bastante o tamanho em relação
    # ao PLY ASCII usado anteriormente.
    vertex_dtype = np.dtype([
        ("x", "<f4"),
        ("y", "<f4"),
        ("z", "<f4"),
        ("red", "u1"),
        ("green", "u1"),
        ("blue", "u1"),
    ])

    vertices = np.empty(
        len(xyz),
        dtype=vertex_dtype,
    )
    vertices["x"] = xyz[:, 0]
    vertices["y"] = xyz[:, 1]
    vertices["z"] = xyz[:, 2]
    vertices["red"] = rgb[:, 0]
    vertices["green"] = rgb[:, 1]
    vertices["blue"] = rgb[:, 2]

    header = (
        "ply\n"
        "format binary_little_endian 1.0\n"
        f"element vertex {len(vertices)}\n"
        "property float x\n"
        "property float y\n"
        "property float z\n"
        "property uchar red\n"
        "property uchar green\n"
        "property uchar blue\n"
        "end_header\n"
    ).encode("ascii")

    with open(path, "wb") as f:
        f.write(header)
        vertices.tofile(f)


# ============================================================
# 16. ÍNDICE TEMPORAL DAS CÂMERAS
# ============================================================

def build_camera_time_index(camera_frames):
    index = {}

    for channel, items in camera_frames.items():
        items = sorted(
            items,
            key=lambda item: int(item["timestamp"]),
        )

        index[channel] = {
            "frames": items,
            "timestamps": [
                int(item["timestamp"])
                for item in items
            ],
        }

    return index


def nearest_camera_frame(
    timestamp,
    channel,
    camera_time_index,
):
    info = camera_time_index.get(channel)

    if info is None or not info["frames"]:
        return None, None

    timestamps = info["timestamps"]
    pos = bisect_left(timestamps, timestamp)
    candidates = []

    if pos < len(timestamps):
        candidates.append(pos)
    if pos > 0:
        candidates.append(pos - 1)

    if not candidates:
        return None, None

    best = min(
        candidates,
        key=lambda idx: abs(
            timestamps[idx] - timestamp
        ),
    )

    delta_us = abs(
        timestamps[best] - timestamp
    )

    return info["frames"][best], delta_us


# ============================================================
# 17. PROJEÇÃO LiDAR NAS CÂMERAS
# ============================================================

def visible_indices_zbuffer(
    valid_indices,
    pixel_x,
    pixel_y,
    depth,
    width,
):
    if len(valid_indices) == 0:
        return valid_indices

    flat_pixel = (
        pixel_y.astype(np.int64) * int(width)
        + pixel_x.astype(np.int64)
    )

    order = np.lexsort((depth, flat_pixel))
    flat_sorted = flat_pixel[order]
    _, first_positions = np.unique(
        flat_sorted,
        return_index=True,
    )
    selected_local = order[first_positions]

    return valid_indices[selected_local]


def project_global_points_to_camera(
    xyz_global,
    camera_sd,
    calib_by_token,
    ego_by_token,
    image_width,
    image_height,
):
    calib = calib_by_token[
        camera_sd["calibrated_sensor_token"]
    ]

    K = np.asarray(
        calib["camera_intrinsic"],
        dtype=np.float64,
    )

    cam_to_global = camera_to_global_opencv(
        camera_sd,
        calib_by_token,
        ego_by_token,
    )
    global_to_cam = np.linalg.inv(cam_to_global)

    xyz_h = np.concatenate(
        [
            xyz_global,
            np.ones((len(xyz_global), 1), dtype=np.float64),
        ],
        axis=1,
    )

    xyz_cam = (
        global_to_cam @ xyz_h.T
    ).T[:, :3]

    z = xyz_cam[:, 2]
    valid = z > 1e-3
    safe_z = np.where(valid, z, 1.0)

    u = (
        K[0, 0] * xyz_cam[:, 0] / safe_z
        + K[0, 2]
    )
    v = (
        K[1, 1] * xyz_cam[:, 1] / safe_z
        + K[1, 2]
    )

    valid &= (
        (u >= 0)
        & (u < image_width)
        & (v >= 0)
        & (v < image_height)
    )

    return u, v, z, valid


def colorize_lidar_points(
    xyz_global,
    intensity,
    lidar_timestamp,
    camera_time_index,
    calib_by_token,
    ego_by_token,
    nuscenes_root,
):
    rgb = intensity_to_rgb(intensity)
    colored = np.zeros(len(xyz_global), dtype=bool)
    best_score = np.full(
        len(xyz_global),
        np.inf,
        dtype=np.float64,
    )

    xyz_h = np.concatenate(
        [
            xyz_global,
            np.ones((len(xyz_global), 1), dtype=np.float64),
        ],
        axis=1,
    )

    for channel in CAMERA_CHANNELS:
        camera_sd, delta_us = nearest_camera_frame(
            lidar_timestamp,
            channel,
            camera_time_index,
        )

        if camera_sd is None:
            continue

        delta_ms = delta_us / 1000.0

        if delta_ms > MAX_CAMERA_LIDAR_DELTA_MS:
            continue

        calib = calib_by_token[
            camera_sd["calibrated_sensor_token"]
        ]
        K = np.asarray(
            calib["camera_intrinsic"],
            dtype=np.float64,
        )

        cam_to_global = camera_to_global_opencv(
            camera_sd,
            calib_by_token,
            ego_by_token,
        )
        global_to_cam = np.linalg.inv(cam_to_global)

        xyz_cam = (
            global_to_cam @ xyz_h.T
        ).T[:, :3]

        z = xyz_cam[:, 2]
        valid = z > 1e-3

        if not np.any(valid):
            continue

        safe_z = np.where(valid, z, 1.0)
        u = K[0, 0] * xyz_cam[:, 0] / safe_z + K[0, 2]
        v = K[1, 1] * xyz_cam[:, 1] / safe_z + K[1, 2]

        image_path = nuscenes_root / camera_sd["filename"]
        if not image_path.exists():
            continue

        image = load_rgb_image(str(image_path))
        height, width = image.shape[:2]

        valid &= (
            (u >= 0)
            & (u < width)
            & (v >= 0)
            & (v < height)
        )

        if not np.any(valid):
            continue

        valid_indices = np.where(valid)[0]
        pixel_x = np.rint(u[valid_indices]).astype(np.int64)
        pixel_y = np.rint(v[valid_indices]).astype(np.int64)
        pixel_x = np.clip(pixel_x, 0, width - 1)
        pixel_y = np.clip(pixel_y, 0, height - 1)

        if LIDAR_COLOR_USE_ZBUFFER:
            visible_indices = visible_indices_zbuffer(
                valid_indices=valid_indices,
                pixel_x=pixel_x,
                pixel_y=pixel_y,
                depth=z[valid_indices],
                width=width,
            )
        else:
            visible_indices = valid_indices

        if len(visible_indices) == 0:
            continue

        visible_x = np.rint(u[visible_indices]).astype(np.int64)
        visible_y = np.rint(v[visible_indices]).astype(np.int64)
        visible_x = np.clip(visible_x, 0, width - 1)
        visible_y = np.clip(visible_y, 0, height - 1)

        center_score = (
            ((u[visible_indices] - K[0, 2]) / max(width, 1)) ** 2
            +
            ((v[visible_indices] - K[1, 2]) / max(height, 1)) ** 2
        )
        temporal_penalty = (
            0.05
            * (
                delta_ms
                / max(MAX_CAMERA_LIDAR_DELTA_MS, 1e-6)
            ) ** 2
        )
        score = center_score + temporal_penalty

        update_local = score < best_score[visible_indices]
        if not np.any(update_local):
            continue

        update_indices = visible_indices[update_local]
        update_x = visible_x[update_local]
        update_y = visible_y[update_local]

        rgb[update_indices] = image[update_y, update_x]
        colored[update_indices] = True
        best_score[update_indices] = score[update_local]

    return rgb, colored


# ============================================================
# 18. CONSTRUIR NUVEM LiDAR
# ============================================================

def build_lidar_point_cloud(
    sample_data,
    sensor_by_token,
    calib_by_token,
    ego_by_token,
    sample_by_token,
    sample_split_by_token,
    selected_scene_token,
    nuscenes_root,
    camera_time_index,
    origin,
    scale_factor,
    time_min_us,
    time_max_us,
    holdout_times_us=None,
):
    lidar_frames = []

    for sd in sample_data:
        filename = sd.get("filename", "")
        is_sample = filename.startswith("samples/")
        is_sweep = filename.startswith("sweeps/")

        if is_sample and not USE_LIDAR_SAMPLES:
            continue
        if is_sweep and not USE_LIDAR_SWEEPS:
            continue
        if not is_sample and not is_sweep:
            continue

        timestamp = int(sd["timestamp"])
        if holdout_times_us:
            gap_us = nearest_timestamp_delta_us(timestamp, holdout_times_us)
            if gap_us is not None and gap_us < int(HOLDOUT_SWEEP_GUARD_MS*1000):
                continue

        if time_min_us is not None and timestamp < time_min_us:
            continue
        if time_max_us is not None and timestamp > time_max_us:
            continue

        sample_item = sample_by_token.get(sd["sample_token"])
        if sample_item is None:
            continue
        if sample_item["scene_token"] != selected_scene_token:
            continue

        if LIDAR_TRAIN_SPLIT_ONLY:
            split = sample_split_by_token.get(sd["sample_token"])
            if split != "train":
                continue

        calib = calib_by_token.get(sd["calibrated_sensor_token"])
        if calib is None:
            continue

        sensor_item = sensor_by_token.get(calib["sensor_token"])
        if sensor_item is None:
            continue
        if sensor_item.get("channel") != LIDAR_CHANNEL:
            continue

        lidar_path = nuscenes_root / filename
        if not lidar_path.exists():
            continue

        lidar_frames.append(sd)

    lidar_frames = sorted(
        lidar_frames,
        key=lambda item: int(item["timestamp"]),
    )

    if (
        MAX_LIDAR_FRAMES is not None
        and len(lidar_frames) > MAX_LIDAR_FRAMES
    ):
        indices = np.linspace(
            0,
            len(lidar_frames) - 1,
            MAX_LIDAR_FRAMES,
        ).astype(int)
        lidar_frames = [
            lidar_frames[i]
            for i in indices
        ]

    print("\nFrames LiDAR utilizados:", len(lidar_frames))

    if not lidar_frames:
        return None, None, {
            "total": 0,
            "rgb_colored": 0,
            "rgb_percentage": 0.0,
            "frames_used": 0,
        }

    all_xyz = []
    all_rgb = []
    all_colored = []

    for frame_index, sd in enumerate(lidar_frames):
        lidar_path = nuscenes_root / sd["filename"]
        points = load_lidar_bin(lidar_path)

        xyz_local = points[:, :3].astype(np.float64)

        if points.shape[1] >= 4:
            intensity = points[:, 3].astype(np.float32)
        else:
            intensity = np.ones(
                len(points),
                dtype=np.float32,
            )

        distance = np.linalg.norm(
            xyz_local[:, :2],
            axis=1,
        )

        mask = (
            (distance >= MIN_LIDAR_DISTANCE)
            & (distance <= MAX_LIDAR_DISTANCE)
        )

        if USE_HEIGHT_FILTER_LOCAL:
            mask &= xyz_local[:, 2] >= MIN_LOCAL_Z
            mask &= xyz_local[:, 2] <= MAX_LOCAL_Z

        xyz_local = xyz_local[mask]
        intensity = intensity[mask]

        if len(xyz_local) == 0:
            continue

        lidar_to_global = lidar_to_global_matrix(
            sd,
            calib_by_token,
            ego_by_token,
        )

        xyz_h = np.concatenate(
            [
                xyz_local,
                np.ones((len(xyz_local), 1), dtype=np.float64),
            ],
            axis=1,
        )
        xyz_global = (
            lidar_to_global @ xyz_h.T
        ).T[:, :3]

        if COLORIZE_LIDAR_WITH_CAMERAS:
            rgb, colored = colorize_lidar_points(
                xyz_global=xyz_global,
                intensity=intensity,
                lidar_timestamp=int(sd["timestamp"]),
                camera_time_index=camera_time_index,
                calib_by_token=calib_by_token,
                ego_by_token=ego_by_token,
                nuscenes_root=nuscenes_root,
            )
        else:
            rgb = intensity_to_rgb(intensity)
            colored = np.zeros(len(xyz_global), dtype=bool)

        xyz_ns = (xyz_global - origin) * scale_factor

        all_xyz.append(xyz_ns)
        all_rgb.append(rgb)
        all_colored.append(colored)

        if (frame_index + 1) % 10 == 0:
            print(
                "LiDAR processado:",
                frame_index + 1,
                "/",
                len(lidar_frames),
            )

    if not all_xyz:
        return None, None, {
            "total": 0,
            "rgb_colored": 0,
            "rgb_percentage": 0.0,
            "frames_used": 0,
        }

    xyz = np.concatenate(all_xyz, axis=0)
    rgb = np.concatenate(all_rgb, axis=0)
    colored = np.concatenate(all_colored, axis=0)

    radius = np.linalg.norm(xyz, axis=1)
    keep = radius <= TARGET_RADIUS * 2.5
    xyz = xyz[keep]
    rgb = rgb[keep]
    colored = colored[keep]

    print(
        "Pontos LiDAR antes do downsample:",
        len(xyz),
    )

    if (
        MAX_POINTS_TOTAL is not None
        and len(xyz) > MAX_POINTS_TOTAL
    ):
        rng = np.random.default_rng(
            LIDAR_RANDOM_SEED
        )
        indices = rng.choice(
            len(xyz),
            size=MAX_POINTS_TOTAL,
            replace=False,
        )
        indices.sort()
        xyz = xyz[indices]
        rgb = rgb[indices]
        colored = colored[indices]

    colored_count = int(colored.sum())
    total_count = len(xyz)
    percentage = (
        100.0 * colored_count / max(total_count, 1)
    )

    print("Pontos LiDAR finais:", total_count)
    print("Pontos com RGB real de treino:", colored_count)
    print("Cobertura RGB:", f"{percentage:.2f}%")

    return (
        xyz.astype(np.float32),
        rgb.astype(np.uint8),
        {
            "total": int(total_count),
            "rgb_colored": int(colored_count),
            "rgb_percentage": float(percentage),
            "frames_used": int(len(lidar_frames)),
        },
    )


# ============================================================
# 19. LOCALIZAR E CARREGAR NUSCENES
# ============================================================

print("\n" + "=" * 100)
print("LOCALIZAÇÃO E CARREGAMENTO DO NUSCENES")
print("=" * 100)

NUSCENES_ROOT, JSON_DIR = find_nuscenes_paths(
    TCC_ROOT
)

sensor = load_json(JSON_DIR, "sensor.json")
calibrated_sensor = load_json(
    JSON_DIR,
    "calibrated_sensor.json",
)
ego_pose = load_json(JSON_DIR, "ego_pose.json")
sample = load_json(JSON_DIR, "sample.json")
sample_data = load_json(JSON_DIR, "sample_data.json")
scene = load_json(JSON_DIR, "scene.json")

# O diagnóstico semântico é opcional: a preparação geométrica
# continua funcionando caso esses JSONs não estejam presentes.
attribute = load_json(
    JSON_DIR,
    "attribute.json",
    required=False,
)
sample_annotation = load_json(
    JSON_DIR,
    "sample_annotation.json",
    required=False,
)
instance = load_json(
    JSON_DIR,
    "instance.json",
    required=False,
)
category = load_json(
    JSON_DIR,
    "category.json",
    required=False,
)

sensor_by_token = {
    item["token"]: item
    for item in sensor
}
calib_by_token = {
    item["token"]: item
    for item in calibrated_sensor
}
ego_by_token = {
    item["token"]: item
    for item in ego_pose
}
sample_by_token = {
    item["token"]: item
    for item in sample
}


# ============================================================
# 20. SELECIONAR CENA
# ============================================================

matches = [
    item
    for item in scene
    if item.get("name") == SCENE_NAME
]

if not matches:
    raise ValueError(
        "Cena não encontrada:\n"
        f"{SCENE_NAME}"
    )

selected_scene = matches[0]
selected_scene_token = selected_scene["token"]

print("\n" + "=" * 100)
print("CENA")
print("=" * 100)
print("Nome:", selected_scene["name"])
print(
    "Descrição:",
    selected_scene.get("description", ""),
)
print("Token:", selected_scene_token)
print(
    "Samples declarados:",
    selected_scene.get("nbr_samples", "N/A"),
)


# ============================================================
# 21. SAMPLES DA CENA / INVENTÁRIO PARA O PROTOCOLO H V3
# ============================================================
scene_samples = sorted(
    [item for item in sample if item['scene_token'] == selected_scene_token],
    key=lambda x: (int(x['timestamp']), x['token']),
)
if not scene_samples:
    raise RuntimeError('Não há samples para scene-0655.')
scene_start_us = int(scene_samples[0]['timestamp'])
scene_end_us = int(scene_samples[-1]['timestamp'])
if TEMPORAL_WINDOW_SECONDS is None:
    time_min_us, time_max_us = scene_start_us, scene_end_us
else:
    mid = scene_start_us + round(
        (scene_end_us - scene_start_us) * TEMPORAL_CENTER_FRACTION)
    half = round(TEMPORAL_WINDOW_SECONDS * 500000)
    time_min_us, time_max_us = mid-half, mid+half
scene_samples_in_window = [
    item for item in scene_samples
    if time_min_us <= int(item['timestamp']) <= time_max_us
]
scene_sample_tokens = {item['token'] for item in scene_samples_in_window}
print('Scene-0655:', len(scene_samples_in_window), 'keyframes no período; '
      'duração s:', round((scene_end_us-scene_start_us)/1e6, 2))

# O split real será definido no nível de sample_data (imagem) depois do
# inventário de CAM_FRONT; pode incluir keyframe OU sweep em VAL/TEST.
# Este é o alinhamento correto à H, que não condiciona os holdouts a keyframes.
sample_split_by_token = {}
sample_split_counts = defaultdict(int)

# 22. AUDITORIA DE SINCRONIZAÇÃO DAS 6 CÂMERAS
# ============================================================

# Mesmo usando apenas CAM_FRONT no treino, verificamos a diferença
# temporal entre os keyframes das seis câmeras de cada sample.
all_camera_keyframes_by_sample = defaultdict(dict)

for sd in sample_data:
    if not is_sample_frame(sd):
        continue

    sample_item = sample_by_token.get(sd["sample_token"])
    if sample_item is None:
        continue
    if sample_item["scene_token"] != selected_scene_token:
        continue

    calib = calib_by_token.get(sd["calibrated_sensor_token"])
    if calib is None:
        continue

    sensor_item = sensor_by_token.get(calib["sensor_token"])
    if sensor_item is None:
        continue

    channel = sensor_item.get("channel")
    if channel not in ALL_CAMERA_CHANNELS:
        continue

    all_camera_keyframes_by_sample[
        sd["sample_token"]
    ][channel] = sd

sync_rows = []

for sample_item in scene_samples_in_window:
    by_channel = all_camera_keyframes_by_sample.get(
        sample_item["token"],
        {},
    )

    timestamps = [
        int(item["timestamp"])
        for item in by_channel.values()
    ]

    if len(timestamps) >= 2:
        spread_ms = (
            max(timestamps) - min(timestamps)
        ) / 1000.0
    else:
        spread_ms = None

    sync_rows.append({
        "sample_token": sample_item["token"],
        "sample_timestamp": int(sample_item["timestamp"]),
        "camera_count": len(by_channel),
        "camera_timestamp_spread_ms": spread_ms,
    })

valid_spreads = [
    row["camera_timestamp_spread_ms"]
    for row in sync_rows
    if row["camera_timestamp_spread_ms"] is not None
]

print("\n" + "=" * 100)
print("AUDITORIA TEMPORAL DAS CÂMERAS")
print("=" * 100)

if valid_spreads:
    print(
        "Δt médio entre as 6 câmeras:",
        f"{np.mean(valid_spreads):.3f} ms",
    )
    print(
        "Δt máximo entre as 6 câmeras:",
        f"{np.max(valid_spreads):.3f} ms",
    )
else:
    print("Não foi possível calcular a sincronização.")


# ============================================================
# 23. AUDITORIA DE MOVIMENTO / ANOTAÇÕES
# ============================================================

scene_dynamic_stats = {
    "available": False,
    "annotation_count": 0,
    "attribute_counts": {},
    "category_counts": {},
}

if (
    attribute is not None
    and sample_annotation is not None
    and instance is not None
    and category is not None
):
    attribute_by_token = {
        item["token"]: item
        for item in attribute
    }
    instance_by_token = {
        item["token"]: item
        for item in instance
    }
    category_by_token = {
        item["token"]: item
        for item in category
    }

    scene_sample_tokens = {
        item["token"]
        for item in scene_samples_in_window
    }

    attr_counter = Counter()
    category_counter = Counter()
    annotation_count = 0

    for ann in sample_annotation:
        if ann.get("sample_token") not in scene_sample_tokens:
            continue

        annotation_count += 1

        for attr_token in ann.get("attribute_tokens", []):
            attr_item = attribute_by_token.get(attr_token)
            if attr_item is not None:
                attr_counter[attr_item.get("name", attr_token)] += 1

        inst = instance_by_token.get(
            ann.get("instance_token")
        )
        if inst is not None:
            cat = category_by_token.get(
                inst.get("category_token")
            )
            if cat is not None:
                category_counter[
                    cat.get("name", inst.get("category_token"))
                ] += 1

    scene_dynamic_stats = {
        "available": True,
        "annotation_count": int(annotation_count),
        "attribute_counts": dict(
            sorted(attr_counter.items())
        ),
        "category_counts": dict(
            sorted(category_counter.items())
        ),
    }

    print("\n" + "=" * 100)
    print("ANOTAÇÕES / DINÂMICA DA CENA")
    print("=" * 100)
    print("Anotações na janela:", annotation_count)

    movement_keys = [
        key
        for key in attr_counter
        if (
            ".moving" in key
            or ".stopped" in key
            or ".parked" in key
            or key.startswith("pedestrian.")
            or key.startswith("cycle.")
        )
    ]

    for key in sorted(movement_keys):
        print(f"  {key:<35} {attr_counter[key]:>6}")
else:
    print("\nAnotações opcionais não encontradas; diagnóstico semântico ignorado.")


# ============================================================
# 24. INVENTÁRIO FÍSICO CAM_FRONT: SAMPLES E SWEEPS
# ============================================================

camera_candidates = {channel: {"sample": [], "sweep": []} for channel in CAMERA_CHANNELS}
physical_inventory = {"camera_sample_json_in_window": 0,
                      "camera_sweep_json_in_window": 0,
                      "camera_sweep_missing_images": 0}

for sd in sample_data:
    if sd.get("fileformat", "").lower() not in ("jpg", "jpeg"):
        continue
    timestamp = int(sd["timestamp"])
    sample_item = sample_by_token.get(sd["sample_token"])
    if sample_item is None or sample_item["scene_token"] != selected_scene_token:
        continue
    # Camera e LiDAR keyframes podem diferir ~dezenas de ms. A escolha da
    # janela é feita no timestamp do SAMPLE; não perder a imagem frontal
    # correspondente apenas porque seu próprio timestamp cruzou a borda.
    is_selected_sample = (is_sample_frame(sd) and
                          sd["sample_token"] in scene_sample_tokens)
    if is_sample_frame(sd) and not is_selected_sample:
        continue
    if not is_selected_sample and not (time_min_us <= timestamp <= time_max_us):
        continue
    calib = calib_by_token.get(sd["calibrated_sensor_token"])
    if calib is None:
        continue
    sensor_item = sensor_by_token.get(calib["sensor_token"])
    if sensor_item is None:
        continue
    channel = sensor_item.get("channel")
    if channel not in CAMERA_CHANNELS or not calib.get("camera_intrinsic"):
        continue
    if is_sample_frame(sd):
        source_type = "sample"
        physical_inventory["camera_sample_json_in_window"] += 1
    elif is_sweep_frame(sd):
        source_type = "sweep"
        physical_inventory["camera_sweep_json_in_window"] += 1
    else:
        continue
    path = NUSCENES_ROOT / sd["filename"]
    if not path.is_file():
        if source_type == "sweep":
            physical_inventory["camera_sweep_missing_images"] += 1
        continue
    camera_candidates[channel][source_type].append(sd)

print("\n" + "=" * 100)
print("INVENTÁRIO CAM_FRONT / JANELA 12 s")
print("=" * 100)
print("Samples com arquivo:", len(camera_candidates["CAM_FRONT"]["sample"]))
print("Sweeps com arquivo:", len(camera_candidates["CAM_FRONT"]["sweep"]))
print("Sweeps no JSON:", physical_inventory["camera_sweep_json_in_window"])
print("Sweeps no JSON sem JPG:", physical_inventory["camera_sweep_missing_images"])

if len(camera_candidates["CAM_FRONT"]["sweep"]) < MIN_AVAILABLE_SWEEPS:
    raise RuntimeError(
        "A V6 precisa dos JPGs de sweeps/CAM_FRONT no nuScenes mini. "
        "O dataset montado contém apenas "
        f"{len(camera_candidates['CAM_FRONT']['sweep'])} sweeps válidos nesta janela "
        f"(exigido >= {MIN_AVAILABLE_SWEEPS}). "
        "Confira se a pasta sweeps/CAM_FRONT está disponível no Drive. "
        "A V5 permanece intacta."
    )

# ============================================================
# 25. AMOSTRAGEM TEMPORAL DETERMINÍSTICA (95/10/10/40)
# ============================================================
# Alinhamento conceitual à H V3: 95 TRAIN + 10 VAL + 10 TEST e 40 GUARD.
# A H amostra a 4 FPS num vídeo de ~39 s. O nuScenes tem cadência própria,
# portanto NÃO copiar cegamente seus índices: garantir >=250ms físicos entre
# treino e holdouts. A divisão é feita antes de qualquer otimização.
A_RAW_GRID_SIZE = 155
raw_candidates = sorted(
    camera_candidates['CAM_FRONT']['sample'] + camera_candidates['CAM_FRONT']['sweep'],
    key=lambda x: (int(x['timestamp']), x['token']),
)
if len({sd['token'] for sd in raw_candidates}) != len(raw_candidates):
    raise RuntimeError('CAM_FRONT contém sample_data_token repetido.')
if len(raw_candidates) < A_RAW_GRID_SIZE:
    raise RuntimeError(
        f'CAM_FRONT tem {len(raw_candidates)} JPGs; não é possível construir '
        '95/10/10 com 40 guardas. Nunca duplicar frames.')

# 20 âncoras internas uniformemente espaçadas no tempo da cena.
# A escolha depende somente de timestamps e disponibilidade, não de PSNR/YOLO.
timestamps = np.asarray([int(sd['timestamp']) for sd in raw_candidates], dtype=np.int64)
from bisect import bisect_left as _a_bisect
n_raw = len(raw_candidates)
edge = max(5, min(8, n_raw//20))
anchor_targets = np.linspace(int(timestamps[edge]), int(timestamps[-edge-1]), 20)
anchor_positions = []
last_pos = -1
for target in anchor_targets:
    ip = _a_bisect(timestamps, target)
    choices = [j for j in range(max(last_pos+1,ip-5),min(n_raw,ip+6))]
    if not choices:
        raise RuntimeError('Não há frames para as âncoras dos holdouts.')
    index = min(choices, key=lambda j: (abs(int(timestamps[j])-target),j))
    anchor_positions.append(index)
    last_pos = index
if len(set(anchor_positions)) != 20:
    raise RuntimeError('As 20 âncoras de holdout se sobrepuseram.')
hold_positions = set(anchor_positions)
val_positions = set(anchor_positions[::2])
test_positions = set(anchor_positions[1::2])
holdout_times = sorted(int(timestamps[j]) for j in hold_positions)

# Selecionar exatamente 2 vizinhos próximos por âncora como GUARD.
# Todos os outros frames que estejam dentro da guarda de 250ms também são
# automaticamente INELEGÍVEIS para TRAIN (mesmo que não sejam listados nos 40).
guard_positions = set()
for pos in anchor_positions:
    for direction in (-1, 1):
        j = pos + direction
        while 0 <= j < n_raw and j in hold_positions | guard_positions:
            j += direction
        if not (0 <= j < n_raw):
            raise RuntimeError('Não existem 40 vizinhos de guarda disponíveis.')
        guard_positions.add(j)
if len(guard_positions) != 40:
    raise RuntimeError('A seleção de GUARD não chegou a 40 imagens únicas.')

# Pontos de treino precisam ficar FORA da vizinhança temporal de cada holdout.
guard_us = int(HOLDOUT_SWEEP_GUARD_MS * 1000)
train_pool = []
for i,sd in enumerate(raw_candidates):
    if i in hold_positions or i in guard_positions:
        continue
    delta = nearest_timestamp_delta_us(int(sd['timestamp']), holdout_times)
    if delta is not None and delta <= guard_us:
        continue
    train_pool.append(i)

if len(train_pool) < A_TARGET_TRAIN:
    raise RuntimeError(
        f'Capacidade para treino após 20 holdouts e guarda de '
        f'{HOLDOUT_SWEEP_GUARD_MS:.0f}ms: somente {len(train_pool)} frames '
        f'CAM_FRONT utilizáveis, requeridos {A_TARGET_TRAIN}. '
        'A célula não reduzirá a guarda, não incluirá TEST no treino nem '
        'criará quadros artificiais. É necessário usar mais cenas ou '
        'definir um novo protocolo com contagens explícitas.')

# Subamostra TRAIN uniformemente POR TEMPO dentro da cobertura elegível.
# Preserva os dois extremos disponíveis e congela 95 escolhas ANTES do treino.
train_pick = [int(round(x)) for x in np.linspace(0,len(train_pool)-1,A_TARGET_TRAIN)]
if len(set(train_pick)) != A_TARGET_TRAIN:
    raise RuntimeError('TRAIN duplicou uma posição da grade.')
train_positions = set(train_pool[k] for k in train_pick)

# Grade final ordenada de 155 imagens, incluindo GUARD auditável mas NÃO
# presente no transforms.json. VAL/TEST incluem keyframes e/ou sweeps.
selected_positions = sorted(train_positions | val_positions | test_positions | guard_positions)
if len(selected_positions) != A_RAW_GRID_SIZE:
    raise RuntimeError('Grade não contém 155 imagens únicas.')
position_to_split = {i:'train' for i in train_positions}
position_to_split.update({i:'val' for i in val_positions})
position_to_split.update({i:'test' for i in test_positions})
position_to_split.update({i:'guard' for i in guard_positions})
frame_split_by_token = {}
frames_raw = []
A_PLANNED_GRID = []
val_indices=[];test_indices=[]
for local_idx, original_idx in enumerate(selected_positions):
    sd=raw_candidates[original_idx]
    split=position_to_split[original_idx]
    if split == 'val': val_indices.append(local_idx)
    if split == 'test': test_indices.append(local_idx)
    if split != 'guard':
        frame_split_by_token[sd['token']]=split
        frames_raw.append(sd)
    A_PLANNED_GRID.append({
        'grid_index':local_idx, 'physical_index':original_idx,
        'timestamp_us':int(sd['timestamp']), 'sample_token':sd['sample_token'],
        'sample_data_token':sd['token'],
        'source_type':'sample' if is_sample_frame(sd) else 'sweep',
        'dataset_split':split,
    })
if (sum(x['dataset_split']=='train' for x in A_PLANNED_GRID),
    len(val_indices),len(test_indices),
    sum(x['dataset_split']=='guard' for x in A_PLANNED_GRID)) != (95,10,10,40):
    raise RuntimeError('Contagens do protocolo H V3 divergentes.')

# Somente samples sem qualquer quadro retido no mesmo grupo podem fornecer
# LiDAR de inicialização. Além disso os scans são afastados de holdouts no tempo.
held_sample_tokens={r['sample_token'] for r in A_PLANNED_GRID
                    if r['dataset_split'] in ('val','test','guard')}
train_source_samples={r['sample_token'] for r in A_PLANNED_GRID
                      if r['dataset_split']=='train'}
sample_split_by_token={}
sample_split_counts=defaultdict(int)
for item in scene_samples_in_window:
    token=item['token']
    split=('guard' if token in held_sample_tokens
           else 'train' if token in train_source_samples else 'unused')
    sample_split_by_token[token]=split
    sample_split_counts[split]+=1

A_HOLDOUT_TIMESTAMPS_US=holdout_times
A_FRAME_GAP_MS = [nearest_timestamp_delta_us(int(sd['timestamp']), holdout_times)/1000.
                  for sd in frames_raw if frame_split_by_token[sd['token']]=='train']
if min(A_FRAME_GAP_MS) <= HOLDOUT_SWEEP_GUARD_MS:
    raise RuntimeError('Vazamento temporal: imagem TRAIN muito próxima de holdout.')
pose_timestamp_deltas_ms=[]
for sd in frames_raw:
    ego=ego_by_token.get(sd['ego_pose_token'])
    if ego is None: raise RuntimeError('Pose ausente: '+sd['token'])
    if 'timestamp' in ego:
        dt=abs(int(ego['timestamp'])-int(sd['timestamp']))/1000.
        pose_timestamp_deltas_ms.append(dt)
        if dt > 2.0: raise RuntimeError(f'Ego pose defasada em {dt:.3f}ms: '+sd['token'])
sweep_timestamp_guards_ms = [nearest_timestamp_delta_us(int(sd['timestamp']),holdout_times)/1000.
    for sd in frames_raw if is_sweep_frame(sd) and frame_split_by_token[sd['token']]=='train']
if not sweep_timestamp_guards_ms: raise RuntimeError('Sem sweeps de treino.')
frames_by_camera={'CAM_FRONT':frames_raw}
selection_stats={'CAM_FRONT':{
    'raw_available':len(raw_candidates), 'raw_grid_count':155,
    'train_keyframes':sum(is_sample_frame(sd) and frame_split_by_token[sd['token']]=='train' for sd in frames_raw),
    'train_sweeps':sum(is_sweep_frame(sd) and frame_split_by_token[sd['token']]=='train' for sd in frames_raw),
    'val':10,'test':10,'guard':40,
    'holdout_anchor_positions':anchor_positions,
    'val_grid_indices':val_indices,'test_grid_indices':test_indices,
    'candidate_train_after_guard':len(train_pool),
    'minimum_train_to_holdout_ms':min(A_FRAME_GAP_MS),
}}
print('A V7, CAM_FRONT JPGs reais:',len(raw_candidates))
print('A V7 = 95 TRAIN / 10 VAL / 10 TEST / 40 GUARD')
print('TRAIN keyframes:',selection_stats['CAM_FRONT']['train_keyframes'],
      '| sweeps:',selection_stats['CAM_FRONT']['train_sweeps'])
print('Mínima distância temporal para holdout:',min(A_FRAME_GAP_MS),'ms')
print('LiDAR: samples de treino sem holdouts no grupo:',sample_split_counts['train'])


# 25.1 AUDITORIA DA DISPONIBILIDADE FÍSICA / CAPACIDADE
# ============================================================
comparison_v6 = None  # Este é um protocolo NOVO; não faz parte da B5.
print('Quantidade efetiva planejada de TRAIN:',
      sum(frame_split_by_token[x['token']] == 'train' for x in frames_raw))
print('VAL e TEST permanecem reservados durante a preparação de geometria.')

# 26. RECRIAR DATASET NERFSTUDIO
# ============================================================

if OUTPUT_DATASET.exists():
    if not A_ALLOW_REBUILD:
        raise FileExistsError(
            "A pasta da A V7 já existe: " + str(OUTPUT_DATASET) +
            ". Para reconstruir esta mesma versão, altere A_ALLOW_REBUILD=True. "
            "A B5 e seus checkpoints não serão modificados."
        )
    print("Reconstrução A V7 explicitamente autorizada:", OUTPUT_DATASET)
    safe_rmtree_dataset(OUTPUT_DATASET)

images_out = OUTPUT_DATASET / "images"
diagnostics_dir = OUTPUT_DATASET / "diagnostics"
overlay_dir = diagnostics_dir / "lidar_projection"

images_out.mkdir(parents=True, exist_ok=True)
overlay_dir.mkdir(parents=True, exist_ok=True)

# Procedência: gravar quais 155 frames entraram na grade, incluindo 40 guardas.
(OUTPUT_DATASET / 'planned_split_A_V7.json').write_text(
    json.dumps({'protocol': 'A_V7_HV3_aligned', 'frames': A_PLANNED_GRID,
                'raw_jpgs_available': len(raw_candidates),
                'train': A_TARGET_TRAIN, 'val': A_TARGET_VAL, 'test': A_TARGET_TEST,
                'guard_count': 40, 'guard_ms': HOLDOUT_SWEEP_GUARD_MS},
               ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

frames = []
normalization_train_camera_centers = []
train_filenames = []
val_filenames = []
test_filenames = []
eval_pairs = []

train_camera_sources = {
    channel: []
    for channel in CAMERA_CHANNELS
}

source_type_counts = defaultdict(int)


# ============================================================
# 27. CONVERSÃO OpenCV -> OpenGL / NERFSTUDIO
# ============================================================

# nuScenes camera frame follows OpenCV:
# +x right, +y down, +z forward.
# Nerfstudio/OpenGL camera:
# +x right, +y up, -z forward.
opencv_to_opengl = np.eye(4, dtype=np.float64)
opencv_to_opengl[1, 1] = -1.0
opencv_to_opengl[2, 2] = -1.0


# ============================================================
# 28. GERAR FRAMES + SPLITS
# ============================================================

frame_sd_by_output_path = {}

for global_index, sd in enumerate(frames_raw):
    calib = calib_by_token[
        sd["calibrated_sensor_token"]
    ]
    sensor_item = sensor_by_token[
        calib["sensor_token"]
    ]
    channel = sensor_item["channel"]

    split = frame_split_by_token.get(sd["token"])
    if split not in ("train", "val", "test"):
        raise RuntimeError(
            "Split não definido para frame:\n"
            f"{sd['token']}"
        )

    source_type = "sample" if is_sample_frame(sd) else "sweep"

    cam_to_world_cv = camera_to_global_opencv(
        sd,
        calib_by_token,
        ego_by_token,
    )
    cam_to_world = (
        cam_to_world_cv @ opencv_to_opengl
    )

    if split == "train":
        normalization_train_camera_centers.append(
            cam_to_world[:3, 3].copy()
        )

    src = NUSCENES_ROOT / sd["filename"]

    dst_name = (
        f"{global_index:05d}_"
        f"{channel}_{source_type}_"
        + make_relative_image_name(sd["filename"])
    )
    dst = images_out / dst_name

    new_width, new_height = save_resized_rgb(
        src,
        dst,
        IMAGE_DOWNSCALE_FACTOR,
    )

    relative_path = f"images/{dst_name}"
    frame_sd_by_output_path[relative_path] = sd

    K_original = np.asarray(
        calib["camera_intrinsic"],
        dtype=np.float64,
    )

    if K_original.shape != (3, 3):
        raise ValueError(
            "Matriz intrínseca inválida."
        )

    scale_x = new_width / float(sd["width"])
    scale_y = new_height / float(sd["height"])

    # Como o resize é proporcional, scale_x ~= scale_y.
    if not np.isclose(scale_x, scale_y, atol=1e-6):
        raise RuntimeError(
            "Resize não proporcional detectado."
        )

    K = K_original.copy()
    K[0, 0] *= scale_x
    K[1, 1] *= scale_y
    K[0, 2] *= scale_x
    K[1, 2] *= scale_y

    frame = {
        "file_path": relative_path,
        "transform_matrix": cam_to_world.tolist(),
        "fl_x": float(K[0, 0]),
        "fl_y": float(K[1, 1]),
        "cx": float(K[0, 2]),
        "cy": float(K[1, 2]),
        "w": int(new_width),
        "h": int(new_height),
        "camera_model": "OPENCV",
        "camera_channel": channel,
        "timestamp": int(sd["timestamp"]),
        "nuscenes_filename": sd["filename"],
        "sample_data_token": sd["token"],
        "sample_token": sd["sample_token"],
        "ego_pose_token": sd["ego_pose_token"],
        "calibrated_sensor_token": sd["calibrated_sensor_token"],
        "is_key_frame": source_type == "sample",
        "source_type": source_type,
        "dataset_split": split,
        "image_downscale_factor": IMAGE_DOWNSCALE_FACTOR,
    }

    frames.append(frame)
    source_type_counts[source_type] += 1

    if split == "train":
        train_filenames.append(relative_path)
        train_camera_sources[channel].append(sd)
    elif split == "val":
        val_filenames.append(relative_path)
    elif split == "test":
        test_filenames.append(relative_path)
        eval_pairs.append({
            "index": len(eval_pairs),
            "split": "test",
            "camera_channel": channel,
            "timestamp": int(sd["timestamp"]),
            "sample_data_token": sd["token"],
            "sample_token": sd["sample_token"],
            "nuscenes_filename": sd["filename"],
            "real_image_path": str(dst),
            "nerfstudio_file_path": relative_path,
        })


# ============================================================
# 29. VALIDAR SPLITS
# ============================================================

train_set = set(train_filenames)
val_set = set(val_filenames)
test_set = set(test_filenames)

if train_set & val_set:
    raise RuntimeError(
        "Leakage: treino e validação se sobrepõem."
    )
if train_set & test_set:
    raise RuntimeError(
        "Leakage: treino e teste se sobrepõem."
    )
if val_set & test_set:
    raise RuntimeError(
        "Leakage: validação e teste se sobrepõem."
    )

if len(frames) != (
    len(train_filenames)
    + len(val_filenames)
    + len(test_filenames)
):
    raise RuntimeError(
        "A composição dos splits não fecha com o total de frames."
    )

print("\n" + "=" * 100)
print("SPLITS")
print("=" * 100)
print("Treino:", len(train_filenames))
print("Validação:", len(val_filenames))
print("Teste:", len(test_filenames))
print("Sweeps TRAIN:", sum(f['source_type']=='sweep' and f['dataset_split']=='train' for f in frames))
print("Guarda temporal (ms):", HOLDOUT_SWEEP_GUARD_MS)
print("Interseções entre splits: 0")

# H V3 permite sweeps de CAM_FRONT também em VAL/TEST.
# Todos são imagens distintas com tokens distintos e guardas temporais.
train_tokens={f['sample_data_token'] for f in frames if f['dataset_split']=='train'}
held_tokens={f['sample_data_token'] for f in frames if f['dataset_split'] in ('val','test')}
if train_tokens & held_tokens:
    raise RuntimeError('Leakage entre tokens de treino e holdout.')
# Sample_token no nuScenes representa um grupo de aquisições próximas, NÃO
# identidade de pixels. É possível que sweeps em tempos diferentes compartilhem
# sample_token; a garantia principal é o token de sample_data e a guarda temporal.


# ============================================================
# 30. VALIDAR MATRIZES DE CÂMERA
# ============================================================

for frame in frames:
    matrix = np.asarray(
        frame["transform_matrix"],
        dtype=np.float64,
    )

    if matrix.shape != (4, 4):
        raise RuntimeError(
            "Transformação de câmera não é 4x4."
        )

    if not np.all(np.isfinite(matrix)):
        raise RuntimeError(
            "Transformação contém NaN ou infinito."
        )

    rotation = matrix[:3, :3]
    det = np.linalg.det(rotation)

    if not np.isclose(det, 1.0, atol=1e-4):
        raise RuntimeError(
            "Matriz de rotação inválida.\n"
            f"Determinante: {det}"
        )

print("Matrizes de câmera: OK")


# ============================================================
# 31. NORMALIZAÇÃO GEOMÉTRICA
# ============================================================

print("\n" + "=" * 100)
print("NORMALIZAÇÃO GEOMÉTRICA")
print("=" * 100)

if not normalization_train_camera_centers:
    raise RuntimeError(
        "Não há keyframes de treino suficientes para normalização."
    )

centers = np.asarray(
    normalization_train_camera_centers,
    dtype=np.float64,
)

# Apenas poses de treino influenciam a origem e escala.
origin = centers.mean(axis=0)

for frame in frames:
    matrix = np.asarray(
        frame["transform_matrix"],
        dtype=np.float64,
    )
    matrix[:3, 3] -= origin
    frame["transform_matrix"] = matrix.tolist()

centers_centered = centers - origin
max_camera_radius = float(
    np.max(
        np.linalg.norm(
            centers_centered,
            axis=1,
        )
    )
)

if max_camera_radius < 1e-6:
    raise RuntimeError(
        "Trajetória de câmera degenerada."
    )

scale_factor = TARGET_RADIUS / max_camera_radius

for frame in frames:
    matrix = np.asarray(
        frame["transform_matrix"],
        dtype=np.float64,
    )
    matrix[:3, 3] *= scale_factor
    frame["transform_matrix"] = matrix.tolist()

print("Origem global (treino):", origin.tolist())
print("Raio original:", max_camera_radius)
print("Raio alvo:", TARGET_RADIUS)
print("Scale factor:", scale_factor)


# ============================================================
# 32. ÍNDICE DE CÂMERAS DE TREINO PARA LiDAR
# ============================================================

camera_time_index_train = build_camera_time_index(
    train_camera_sources
)

for channel in CAMERA_CHANNELS:
    if not camera_time_index_train.get(
        channel,
        {},
    ).get("frames", []):
        raise RuntimeError(
            "Não há imagens de treino para colorização LiDAR "
            f"na câmera {channel}."
        )

print("\n" + "=" * 100)
print("COLORIZAÇÃO LiDAR — FONTES DE TREINO")
print("=" * 100)

for channel in CAMERA_CHANNELS:
    print(
        f"{channel:<20}: "
        f"{len(camera_time_index_train[channel]['frames'])} "
        "imagens de treino"
    )


# ============================================================
# 33. CRIAR NUVEM LiDAR COLORIDA
# ============================================================

ply_relative_path = None
lidar_stats = {
    "total": 0,
    "rgb_colored": 0,
    "rgb_percentage": 0.0,
    "frames_used": 0,
}

if CREATE_LIDAR_POINT_CLOUD:
    xyz_lidar, rgb_lidar, lidar_stats = build_lidar_point_cloud(
        sample_data=sample_data,
        sensor_by_token=sensor_by_token,
        calib_by_token=calib_by_token,
        ego_by_token=ego_by_token,
        sample_by_token=sample_by_token,
        sample_split_by_token=sample_split_by_token,
        selected_scene_token=selected_scene_token,
        nuscenes_root=NUSCENES_ROOT,
        camera_time_index=camera_time_index_train,
        origin=origin,
        scale_factor=scale_factor,
        time_min_us=time_min_us,
        time_max_us=time_max_us,
        holdout_times_us=A_HOLDOUT_TIMESTAMPS_US,
    )

    if xyz_lidar is not None and len(xyz_lidar) > 0:
        ply_path = OUTPUT_DATASET / "point_cloud_lidar.ply"
        save_ply_xyzrgb(
            ply_path,
            xyz_lidar,
            rgb_lidar,
        )
        ply_relative_path = "point_cloud_lidar.ply"

        print("PLY:", ply_path)
        print(
            "Tamanho PLY:",
            f"{ply_path.stat().st_size / (1024**2):.2f} MB",
        )


if CREATE_LIDAR_POINT_CLOUD and (not ply_relative_path or lidar_stats['total'] < 1000):
    raise RuntimeError('LiDAR TRAIN-only insuficiente após holdouts. Não iniciar Splatfacto. '
                       'Verifique se há samples LIDAR_TOP autorizados pela A V7.')

# ============================================================
# 34. DIAGNÓSTICO VISUAL LiDAR -> RGB
# ============================================================

# Índice de scans LiDAR sample por sample_token.
lidar_sample_by_sample_token = {}

for sd in sample_data:
    if not is_sample_frame(sd):
        continue

    sample_item = sample_by_token.get(sd["sample_token"])
    if sample_item is None:
        continue
    if sample_item["scene_token"] != selected_scene_token:
        continue

    calib = calib_by_token.get(sd["calibrated_sensor_token"])
    if calib is None:
        continue

    sensor_item = sensor_by_token.get(calib["sensor_token"])
    if sensor_item is None:
        continue

    if sensor_item.get("channel") == LIDAR_CHANNEL:
        lidar_sample_by_sample_token[
            sd["sample_token"]
        ] = sd


def depth_to_rgb(depth_values):
    if len(depth_values) == 0:
        return np.empty((0, 3), dtype=np.uint8)

    d_min = np.percentile(depth_values, 5)
    d_max = np.percentile(depth_values, 95)

    if d_max <= d_min:
        norm = np.zeros_like(depth_values)
    else:
        norm = np.clip(
            (depth_values - d_min) / (d_max - d_min),
            0.0,
            1.0,
        )

    # Azul perto -> vermelho longe, sem dependência de matplotlib.
    r = (255.0 * norm).astype(np.uint8)
    g = (
        255.0 * (1.0 - np.abs(norm - 0.5) * 2.0)
    ).clip(0, 255).astype(np.uint8)
    b = (255.0 * (1.0 - norm)).astype(np.uint8)

    return np.stack([r, g, b], axis=1)


def make_lidar_overlay(
    frame,
    camera_sd,
    lidar_sd,
    overlay_path,
):
    image_path = NUSCENES_ROOT / camera_sd["filename"]
    lidar_path = NUSCENES_ROOT / lidar_sd["filename"]

    with Image.open(image_path) as src_img:
        img = src_img.convert("RGB")
        original_width, original_height = img.size

    points = load_lidar_bin(lidar_path)
    xyz_local = points[:, :3].astype(np.float64)

    distance = np.linalg.norm(
        xyz_local[:, :2],
        axis=1,
    )
    keep = (
        (distance >= MIN_LIDAR_DISTANCE)
        & (distance <= MAX_LIDAR_DISTANCE)
    )
    if USE_HEIGHT_FILTER_LOCAL:
        keep &= xyz_local[:, 2] >= MIN_LOCAL_Z
        keep &= xyz_local[:, 2] <= MAX_LOCAL_Z

    xyz_local = xyz_local[keep]

    lidar_to_global = lidar_to_global_matrix(
        lidar_sd,
        calib_by_token,
        ego_by_token,
    )
    xyz_h = np.concatenate(
        [
            xyz_local,
            np.ones((len(xyz_local), 1), dtype=np.float64),
        ],
        axis=1,
    )
    xyz_global = (
        lidar_to_global @ xyz_h.T
    ).T[:, :3]

    u, v, depth, valid = project_global_points_to_camera(
        xyz_global=xyz_global,
        camera_sd=camera_sd,
        calib_by_token=calib_by_token,
        ego_by_token=ego_by_token,
        image_width=original_width,
        image_height=original_height,
    )

    indices = np.where(valid)[0]

    if len(indices) > DIAGNOSTIC_MAX_PROJECTED_POINTS:
        rng = np.random.default_rng(42)
        indices = rng.choice(
            indices,
            size=DIAGNOSTIC_MAX_PROJECTED_POINTS,
            replace=False,
        )

    depth_valid = depth[indices]
    colors = depth_to_rgb(depth_valid)

    draw = ImageDraw.Draw(img)

    for idx, color in zip(indices, colors):
        x = int(round(u[idx]))
        y = int(round(v[idx]))
        r = DIAGNOSTIC_POINT_RADIUS
        draw.ellipse(
            (x-r, y-r, x+r, y+r),
            fill=tuple(int(c) for c in color),
        )

    # Texto com split e delta temporal câmera-LiDAR.
    delta_ms = abs(
        int(camera_sd["timestamp"])
        - int(lidar_sd["timestamp"])
    ) / 1000.0

    caption = (
        f"{frame['dataset_split']} | "
        f"{frame['camera_channel']} | "
        f"camera-lidar dt={delta_ms:.2f} ms | "
        f"projected={len(indices)}"
    )

    draw.rectangle(
        (0, 0, min(original_width, 1100), 34),
        fill=(0, 0, 0),
    )
    draw.text(
        (10, 9),
        caption,
        fill=(255, 255, 255),
    )

    img.save(overlay_path, format="PNG")

    return {
        "file": str(overlay_path),
        "split": frame["dataset_split"],
        "camera_channel": frame["camera_channel"],
        "camera_timestamp": int(camera_sd["timestamp"]),
        "lidar_timestamp": int(lidar_sd["timestamp"]),
        "camera_lidar_delta_ms": float(delta_ms),
        "projected_points": int(len(indices)),
        "sample_token": frame["sample_token"],
    }


diagnostic_rows = []

if CREATE_GEOMETRY_DIAGNOSTICS:
    candidate_frames = [
        frame
        for frame in frames
        if frame["source_type"] == "sample" and frame["dataset_split"] == "train"
    ]

    if candidate_frames:
        if len(candidate_frames) <= DIAGNOSTIC_OVERLAY_COUNT:
            selected_indices = np.arange(
                len(candidate_frames)
            )
        else:
            selected_indices = np.linspace(
                0,
                len(candidate_frames) - 1,
                DIAGNOSTIC_OVERLAY_COUNT,
            ).round().astype(int)

        selected_indices = sorted(
            set(int(i) for i in selected_indices)
        )

        print("\n" + "=" * 100)
        print("DIAGNÓSTICO LiDAR -> CÂMERA")
        print("=" * 100)

        for diagnostic_index, frame_index in enumerate(
            selected_indices
        ):
            frame = candidate_frames[frame_index]
            camera_sd = frame_sd_by_output_path[
                frame["file_path"]
            ]
            lidar_sd = lidar_sample_by_sample_token.get(
                frame["sample_token"]
            )

            if lidar_sd is None:
                print(
                    "Sem LiDAR para sample:",
                    frame["sample_token"],
                )
                continue

            overlay_path = (
                overlay_dir
                / (
                    f"overlay_{diagnostic_index:02d}_"
                    f"{frame['dataset_split']}_"
                    f"{frame['camera_channel']}.png"
                )
            )

            row = make_lidar_overlay(
                frame=frame,
                camera_sd=camera_sd,
                lidar_sd=lidar_sd,
                overlay_path=overlay_path,
            )
            diagnostic_rows.append(row)

            print(
                overlay_path.name,
                "| dt=",
                f"{row['camera_lidar_delta_ms']:.2f} ms",
                "| projected=",
                row["projected_points"],
            )


# ============================================================
# ============================================================
# 34.5 — LINHA DO TEMPO E CONTATO VISUAL DE TREINO
# ============================================================
import csv

timeline_path = diagnostics_dir / "camera_timeline_train_val_test.csv"
with timeline_path.open('w', encoding='utf-8',newline='') as stream:
    writer=csv.DictWriter(stream,fieldnames=[
        'sample_data_token','timestamp','seconds_from_window_start',
        'source_type','split','sample_token','ego_pose_token','source_filename'])
    writer.writeheader()
    for f in sorted(frames,key=lambda x:int(x['timestamp'])):
        writer.writerow({
            'sample_data_token':f['sample_data_token'],
            'timestamp':f['timestamp'],
            'seconds_from_window_start':round((int(f['timestamp'])-time_min_us)/1e6,4),
            'source_type':f['source_type'],'split':f['dataset_split'],
            'sample_token':f['sample_token'],'ego_pose_token':f['ego_pose_token'],
            'source_filename':f['nuscenes_filename'],
        })
# Não abrir JPG de teste nos previews visuais.
preview = [f for f in frames if f['dataset_split']=='train']
preview = sorted(preview,key=lambda x:int(x['timestamp']))
if len(preview)>16:
    inds=np.linspace(0,len(preview)-1,16).round().astype(int)
    preview=[preview[int(i)] for i in sorted(set(inds))]
if preview:
    thumb_w,thumb_h=400,225
    header_h=35
    cols=4
    rows=math.ceil(len(preview)/cols)
    sheet=Image.new('RGB',(cols*thumb_w,rows*(thumb_h+header_h)),(18,30,48))
    draw=ImageDraw.Draw(sheet)
    for i,f in enumerate(preview):
        with Image.open(OUTPUT_DATASET/f['file_path']) as image:
            small=image.convert('RGB').resize((thumb_w,thumb_h),Image.Resampling.LANCZOS)
        x=(i%cols)*thumb_w; y=(i//cols)*(thumb_h+header_h)
        sheet.paste(small,(x,y+header_h))
        draw.text((x+8,y+9),
                  f"TRAIN {f['source_type'].upper()} t={(int(f['timestamp'])-time_min_us)/1e6:.2f}s",
                  fill=(255,255,255))
    sheet.save(diagnostics_dir/'contato_visual_train.jpg','JPEG',quality=90)
    print('Contato visual somente TRAIN:',diagnostics_dir/'contato_visual_train.jpg')
print('Timeline CSV (metadados de todos os splits):',timeline_path)

# ============================================================
# 35. GERAR transforms.json
# ============================================================

transforms = {
    "camera_model": "OPENCV",
    "orientation_override": "none",
    "frames": frames,
    "train_filenames": train_filenames,
    "val_filenames": val_filenames,
    "test_filenames": test_filenames,
}

if ply_relative_path:
    transforms["ply_file_path"] = ply_relative_path

transforms_path = OUTPUT_DATASET / "transforms.json"

with open(
    transforms_path,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        transforms,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 36. GERAR eval_pairs.json
# ============================================================

eval_pairs_path = OUTPUT_DATASET / "eval_pairs.json"

with open(
    eval_pairs_path,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        eval_pairs,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 37. GERAR DIAGNÓSTICO JSON
# ============================================================

diagnostics_json = diagnostics_dir / "geometry_diagnostics.json"

diagnostics_payload = {
    "protocol_v7": {
        "reference": "H V3 95/10/10; same split sizes when physically available",
        "holdout_selection": A_HOLDOUT_STRATEGY,
        "requested_counts": {"train": A_TARGET_TRAIN, "val": A_TARGET_VAL, "test": A_TARGET_TEST},
        "actual_counts": {"train": len(train_filenames), "val": len(val_filenames), "test": len(test_filenames)},
        "strict_train_count": A_REQUIRE_EXACT_TRAIN,
        "geometry_source": "official nuScenes ego poses and calibrations; not COLMAP",
        "lidar_initialization": "nuScenes LIDAR_TOP, selected TRAIN samples only",
        "holdout_rgb_in_training_geometry": False,
        "holdout_rgb_in_training_colorization": False,
        "holdout_rgb_in_model_optimization": False,
        "normalization_uses_only_train": True,
    },
    "experiment_tag": EXPERIMENT_TAG,
    "scene": {
        "name": selected_scene["name"],
        "token": selected_scene_token,
        "description": selected_scene.get("description", ""),
        "samples": len(scene_samples_in_window),
        "duration_seconds": float(
            (scene_end_us - scene_start_us) / 1_000_000.0
        ),
    },
    "camera_dataset": {
        "training_channels": CAMERA_CHANNELS,
        "samples_only": False,
        "sweeps_train_only": False,
        "sweeps_selected": sum(f['source_type']=='sweep' and f['dataset_split']=='train' for f in frames),
        "inventory": physical_inventory,
        "selection_stats": selection_stats,
        "minimum_sweep_distance_to_holdout_ms": min(sweep_timestamp_guards_ms),
        "image_downscale_factor": IMAGE_DOWNSCALE_FACTOR,
    },
    "camera_sync": {
        "sample_rows": sync_rows,
        "mean_spread_ms": (
            float(np.mean(valid_spreads))
            if valid_spreads
            else None
        ),
        "max_spread_ms": (
            float(np.max(valid_spreads))
            if valid_spreads
            else None
        ),
    },
    "ego_pose_audit": {
        "uses_sample_data_ego_pose_token": True,
        "max_sample_data_vs_ego_pose_timestamp_delta_ms": (
            float(max(pose_timestamp_deltas_ms))
            if pose_timestamp_deltas_ms
            else None
        ),
    },
    "scene_dynamic_stats": scene_dynamic_stats,
    "lidar_overlays": diagnostic_rows,
}

with open(
    diagnostics_json,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        diagnostics_payload,
        f,
        indent=2,
        ensure_ascii=False,
    )


comparison_path = diagnostics_dir / 'protocolo_A_V7_vs_H_V3.json'
comparison_path.write_text(
    json.dumps({
        'experiment': EXPERIMENT_TAG,
        'reference': 'H V3 iPhone: 95/10/10; 4 FPS; COLMAP train-only; PnP holdouts',
        'nuscenes': 'CAM_FRONT, official poses, LiDAR TRAIN-only',
        'requested': [A_TARGET_TRAIN, A_TARGET_VAL, A_TARGET_TEST],
        'observed': [len(train_filenames), len(val_filenames), len(test_filenames)],
        'holdout_guard_ms': HOLDOUT_SWEEP_GUARD_MS,
        'note': 'Split sizes and holdout/time guards are aligned, not capture frequency. '
                'Scenes, video capture, geometry modality and image aspect ratios differ.',
    }, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
)

# 38. MANIFESTO REPRODUZÍVEL
# ============================================================

manifest = {
    "protocol_v7": {
        "reference": "H V3 95/10/10; same split sizes when physically available",
        "holdout_selection": A_HOLDOUT_STRATEGY,
        "requested_counts": {"train": A_TARGET_TRAIN, "val": A_TARGET_VAL, "test": A_TARGET_TEST},
        "actual_counts": {"train": len(train_filenames), "val": len(val_filenames), "test": len(test_filenames)},
        "strict_train_count": A_REQUIRE_EXACT_TRAIN,
        "geometry_source": "official nuScenes ego poses and calibrations; not COLMAP",
        "lidar_initialization": "nuScenes LIDAR_TOP, selected TRAIN samples only",
        "holdout_rgb_in_training_geometry": False,
        "holdout_rgb_in_training_colorization": False,
        "holdout_rgb_in_model_optimization": False,
        "normalization_uses_only_train": True,
    },
    "experiment_tag": EXPERIMENT_TAG,
    "dataset_profile": DATASET_PROFILE,
    "diagnostic_run": True,
    "version": "A_V7_HV3_protocol",

    "scene": {
        "name": selected_scene["name"],
        "token": selected_scene_token,
        "description": selected_scene.get("description", ""),
    },

    "cameras": {
        "channels": CAMERA_CHANNELS,
        "use_samples": USE_SAMPLES,
        "use_camera_sweeps": USE_CAMERA_SWEEPS,
        "use_sweeps": USE_CAMERA_SWEEPS,
        "max_frames_per_camera": MAX_FRAMES_PER_CAMERA,
        "sweeps_train_only": False,
        "min_train_sweep_interval_ms": MIN_TRAIN_SWEEP_INTERVAL_MS,
        "holdout_sweep_guard_ms": HOLDOUT_SWEEP_GUARD_MS,
        "max_train_sweeps_per_camera": MAX_TRAIN_SWEEPS_PER_CAMERA,
        "selection_stats": selection_stats,
        "image_downscale_factor": IMAGE_DOWNSCALE_FACTOR,
    },

    "temporal_window": {
        "seconds": TEMPORAL_WINDOW_SECONDS,
        "center_fraction": TEMPORAL_CENTER_FRACTION,
        "min_timestamp": time_min_us,
        "max_timestamp": time_max_us,
    },

    "split": {
        "strategy": "H V3-like 95/10/10, 20 temporal anchor holdouts, >=250ms guard, 40 excluded neighbor frames",
        "holdout_every": None,
        "holdout_raw_grid_indices_val": val_indices,
        "holdout_raw_grid_indices_test": test_indices,
        "raw_grid": A_PLANNED_GRID,
        "validation_offset": None,
        "test_offset": None,
        "raw_grid_size": 155,
        "guard_frame_count": 40,
        "guard_ms": HOLDOUT_SWEEP_GUARD_MS,
        "min_train_sweep_interval_ms":MIN_TRAIN_SWEEP_INTERVAL_MS,
        "train_sweeps":sum(f['source_type']=='sweep' and f['dataset_split']=='train' for f in frames),
        "train_count": len(train_filenames),
        "validation_count": len(val_filenames),
        "test_count": len(test_filenames),
        "train_sample_count": sum(f['source_type']=='sample' and f['dataset_split']=='train' for f in frames),
        "lidar_safe_sample_groups": int(sample_split_counts['train']),
        "validation_sample_count": sum(f['source_type']=='sample' for f in frames if f['dataset_split']=='val'),
        "test_sample_count": sum(f['source_type']=='sample' for f in frames if f['dataset_split']=='test'),
    },

    "geometry": {
        "coordinate_source": (
            "per-sample_data ego_pose_token + calibrated_sensor"
        ),
        "opencv_to_opengl": [
            [1.0, 0.0, 0.0, 0.0],
            [0.0, -1.0, 0.0, 0.0],
            [0.0, 0.0, -1.0, 0.0],
            [0.0, 0.0, 0.0, 1.0],
        ],
        "target_radius": TARGET_RADIUS,
        "origin_source": "all training camera centers only; samples and sweeps included",
        "origin_global": origin.tolist(),
        "max_camera_radius_original": max_camera_radius,
        "geometry_inductive": True,
        "pose_source": "nuScenes calibrated_sensor + per sample_data ego_pose",
        "holdout_poses_from_official_metadata": True,
        "scale_factor": float(scale_factor),
    },

    "lidar": {
        "enabled": CREATE_LIDAR_POINT_CLOUD,
        "channel": LIDAR_CHANNEL,
        "use_samples": USE_LIDAR_SAMPLES,
        "use_sweeps": USE_LIDAR_SWEEPS,
        "train_split_only": LIDAR_TRAIN_SPLIT_ONLY,
        "max_frames": MAX_LIDAR_FRAMES,
        "frames_used": lidar_stats["frames_used"],
        "max_points": MAX_POINTS_TOTAL,
        "min_distance": MIN_LIDAR_DISTANCE,
        "max_distance": MAX_LIDAR_DISTANCE,
        "min_local_z": MIN_LOCAL_Z,
        "max_local_z": MAX_LOCAL_Z,
        "rgb_colorization_enabled": COLORIZE_LIDAR_WITH_CAMERAS,
        "rgb_colorization_source": "training camera frames only",
        "zbuffer_enabled": LIDAR_COLOR_USE_ZBUFFER,
        "max_camera_lidar_delta_ms": MAX_CAMERA_LIDAR_DELTA_MS,
        "dynamic_points_removed": REMOVE_DYNAMIC_LIDAR_POINTS,
        "final_point_count": lidar_stats["total"],
        "rgb_colored_points": lidar_stats["rgb_colored"],
        "rgb_colored_percentage": lidar_stats["rgb_percentage"],
    },

    "diagnostics": {
        "enabled": CREATE_GEOMETRY_DIAGNOSTICS,
        "directory": "diagnostics/",
        "json": "diagnostics/geometry_diagnostics.json",
        "lidar_projection_directory": "diagnostics/lidar_projection/",
        "overlays_created": len(diagnostic_rows),
        "scene_dynamic_stats_available": scene_dynamic_stats["available"],
    },

    "files": {
        "images": "images/",
        "transforms": "transforms.json",
        "evaluation_pairs": "eval_pairs.json",
        "point_cloud": ply_relative_path,
    },
}

manifest_path = OUTPUT_DATASET / "dataset_manifest.json"

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 39. RESUMO POR CÂMERA
# ============================================================

counter_total = defaultdict(int)
counter_train = defaultdict(int)
counter_val = defaultdict(int)
counter_test = defaultdict(int)

for frame in frames:
    channel = frame["camera_channel"]
    split = frame["dataset_split"]
    counter_total[channel] += 1

    if split == "train":
        counter_train[channel] += 1
    elif split == "val":
        counter_val[channel] += 1
    elif split == "test":
        counter_test[channel] += 1

print("\n" + "=" * 100)
print("RESUMO POR CÂMERA")
print("=" * 100)

print(
    f"{'Câmera':<20}"
    f"{'Total':>8}"
    f"{'Treino':>9}"
    f"{'Val':>7}"
    f"{'Teste':>8}"
)
print("-" * 52)

for camera in CAMERA_CHANNELS:
    print(
        f"{camera:<20}"
        f"{counter_total[camera]:>8}"
        f"{counter_train[camera]:>9}"
        f"{counter_val[camera]:>7}"
        f"{counter_test[camera]:>8}"
    )


# ============================================================
# 40. VALIDAÇÃO FINAL
# ============================================================

for required_file in (
    transforms_path,
    eval_pairs_path,
    manifest_path,
    diagnostics_json,
):
    if not required_file.exists():
        raise FileNotFoundError(
            "Arquivo não criado:\n"
            f"{required_file}"
        )

if CREATE_LIDAR_POINT_CLOUD and ply_relative_path:
    if not (OUTPUT_DATASET / ply_relative_path).exists():
        raise FileNotFoundError(
            "point_cloud_lidar.ply não foi criado."
        )

missing_images = []

for frame in frames:
    image_path = OUTPUT_DATASET / frame["file_path"]
    if not image_path.exists():
        missing_images.append(str(image_path))

if missing_images:
    raise FileNotFoundError(
        "Existem imagens ausentes:\n"
        + "\n".join(missing_images[:20])
    )

if len(eval_pairs) != len(test_filenames):
    raise RuntimeError(
        "eval_pairs não corresponde ao conjunto de teste."
    )

# Validação e teste reservados na grade de frames, como na H V3.
if not any(f['source_type']=='sweep' and f['dataset_split']=='train' for f in frames):
    raise RuntimeError('A V7 não encontrou sweeps de treino; conferir CAM_FRONT no Drive.')
if min(sweep_timestamp_guards_ms) <= HOLDOUT_SWEEP_GUARD_MS:
    raise RuntimeError('A V7 violou a guarda temporal dos holdouts.')

# Conferir a resolução física das imagens produzidas.
for frame in frames[:3]:
    path = OUTPUT_DATASET / frame["file_path"]
    with Image.open(path) as img:
        if img.width != frame["w"] or img.height != frame["h"]:
            raise RuntimeError(
                "Dimensão de imagem e intrínsecos inconsistentes."
            )

TRANSFORMS_SHA256 = sha256_file(transforms_path)
EVAL_PAIRS_SHA256 = sha256_file(eval_pairs_path)
MANIFEST_SHA256 = sha256_file(manifest_path)
DIAGNOSTICS_SHA256 = sha256_file(diagnostics_json)

# Variáveis para células posteriores.
DATASET_MANIFEST_PATH = manifest_path
EVAL_PAIRS_PATH = eval_pairs_path
TRANSFORMS_PATH = transforms_path


# ============================================================
# 41. RESUMO FINAL
# ============================================================

print("\n" + "=" * 100)
print("CÉLULA A V7 CAM_FRONT + SWEEPS FINALIZADA COM SUCESSO")
print("=" * 100)

print("Perfil:", DATASET_PROFILE)
print("Experiment tag:", EXPERIMENT_TAG)
print("Cena:", selected_scene["name"])
print(
    "Descrição:",
    selected_scene.get("description", ""),
)
print("Câmeras de treino:", CAMERA_CHANNELS)
print("Total de imagens:", len(frames))
print("Treino:", len(train_filenames))
print("Validação:", len(val_filenames))
print("Teste:", len(test_filenames))
print("Sweeps CAM_FRONT usados em TRAIN:",sum(f['source_type']=='sweep' and f['dataset_split']=='train' for f in frames))
print('Espaçamento mínimo em TRAIN: não imposto (mesma ideia da H V3)')
print("Guarda em VAL/TEST:", HOLDOUT_SWEEP_GUARD_MS,'ms')
print('Menor distância de sweep para holdout:',min(sweep_timestamp_guards_ms),'ms')
print('Contato visual TRAIN:',diagnostics_dir/'contato_visual_train.jpg')
print("Resolução:", f"{frames[0]['w']}x{frames[0]['h']}")

print("\nAuditoria geométrica:")
print(
    "  Pose por sample_data:",
    "SIM — ego_pose_token individual",
)
print(
    "  Maior delta sample_data/ego_pose:",
    (
        f"{max(pose_timestamp_deltas_ms):.6f} ms"
        if pose_timestamp_deltas_ms
        else "N/A"
    ),
)
print(
    "  Δt máximo entre câmeras da cena:",
    (
        f"{max(valid_spreads):.3f} ms"
        if valid_spreads
        else "N/A"
    ),
)
print(
    "  Overlays LiDAR -> RGB:",
    len(diagnostic_rows),
)

print("\nLiDAR:")
print("  Pontos finais:", lidar_stats["total"])
print("  Scans utilizados:", lidar_stats["frames_used"])
print(
    "  Pontos com RGB real de treino:",
    lidar_stats["rgb_colored"],
)
print(
    "  Cobertura RGB:",
    f"{lidar_stats['rgb_percentage']:.2f}%",
)

if ply_relative_path:
    ply_path = OUTPUT_DATASET / ply_relative_path
    print(
        "  Tamanho PLY:",
        f"{ply_path.stat().st_size / (1024**2):.2f} MB",
    )

print("\nArtefatos:")
print("  Dataset:", OUTPUT_DATASET)
print("  transforms.json:", transforms_path)
print("    SHA256:", TRANSFORMS_SHA256)
print("  eval_pairs.json:", eval_pairs_path)
print("    SHA256:", EVAL_PAIRS_SHA256)
print("  dataset_manifest.json:", manifest_path)
print("    SHA256:", MANIFEST_SHA256)
print("  diagnostics:", diagnostics_json)
print("    SHA256:", DIAGNOSTICS_SHA256)
print("  overlays:", overlay_dir)
print('  comparação de protocolos A V7 vs H V3:', comparison_path)

print(
    "\nAntes do treinamento, confira as projeções LiDAR "
    "em diagnostics/lidar_projection/."
)
print(
    "Se os pontos LiDAR coincidirem com rua, carros, fachadas, "
    "postes e bordas dos objetos, a cadeia de poses/calibração "
    "está visualmente coerente."
)
print(
    "Se houver deslocamento sistemático dos pontos, corrigimos "
    "a geometria antes de treinar qualquer modelo."
)


# ============================================================
# 42. IDENTIDADE DINÂMICA E SEGURANÇA PARA AS PRÓXIMAS CÉLULAS
# ============================================================
# Os artefatos são identificados por hashes da execução corrente.
# Nenhum checkpoint de outra rodada é reutilizado.
A_BACKUP_TO_DRIVE = False
A_CORE_HASHES = {
    'transforms.json': sha256_file(transforms_path),
    'eval_pairs.json': sha256_file(eval_pairs_path),
    'dataset_manifest.json': sha256_file(manifest_path),
}
if ply_relative_path:
    A_CORE_HASHES[ply_relative_path] = sha256_file(OUTPUT_DATASET / ply_relative_path)

A_IDENTITY_PATH = OUTPUT_DATASET / 'dataset_identity.json'
A_IDENTITY_PAYLOAD = {
    'identity_version': 7,
    'protocol': 'A_V7_HV3-aligned',
    'scene': SCENE_NAME,
    'camera': 'CAM_FRONT',
    'resolution': [frames[0]['w'], frames[0]['h']],
    'total_frames': len(frames),
    'train_frames': len(train_filenames),
    'validation_frames': len(val_filenames),
    'test_frames': len(test_filenames),
    'train_keyframes': sum(f['dataset_split']=='train' and f['source_type']=='sample' for f in frames),
    'train_sweeps': int(sum(f['source_type']=='sweep' and f['dataset_split']=='train' for f in frames)),
    'core_sha256': A_CORE_HASHES,
    'geometry_source': 'official nuScenes poses + TRAIN-only LiDAR and RGB',
    'drive_upload': False,
}
A_IDENTITY_PATH.write_text(
    json.dumps(A_IDENTITY_PAYLOAD, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
print('IDENTIDADE A V7:', A_IDENTITY_PATH)
print('HASHES:', json.dumps(A_CORE_HASHES, indent=2))
print('UPLOAD no Drive: NÃO. O notebook deve enviar apenas a análise final.')
print('Confira os diagnósticos antes de prosseguir para a célula B.')
print('CÉLULA A V7 CONCLUÍDA.')




In [ ]:
# =============================================================================
# CÉLULA B V7.2 — SPLATFACTO nuScenes scene-0655 / PROTOCOLO A V7 ↔ iPHONE H V3
# =============================================================================
# Executar após CÉLULAS 0 -> SETUP -> 1 -> A V7, na MESMA sessão Colab.
# Treina Splatfacto do zero, inicializado por LiDAR APENAS de TRAIN (A V7).
# 95 TRAIN / 10 VAL / 10 TEST; seleção de checkpoint exclusivamente em VAL.
# Nunca renderiza ou avalia TEST. Checkpoints e intermediários somente /content.
# NENHUM upload para Google Drive aqui. A análise final ficará a cargo de G.
# Os checkpoints e manifestos desta execução serão usados nas células seguintes.
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import deque
import csv
import hashlib
import json
import math
import os
import re
import shlex
import shutil
import subprocess
import sys
import time

import numpy as np
from PIL import Image, ImageDraw

# Amostragem externa do pico de VRAM do processo de treinamento (não é pico exato).
import threading
class _TrainingGpuSampler:
    def __init__(self,pid):
        self.pid=int(pid);self.peak=None;self.count=0
        self.stop_signal=threading.Event()
        self.thread=threading.Thread(target=self._sample,daemon=True)
    def _sample(self):
        while not self.stop_signal.is_set():
            try:
                result=subprocess.run(['nvidia-smi','--query-compute-apps=pid,used_gpu_memory',
                    '--format=csv,noheader,nounits'],capture_output=True,text=True,timeout=5)
                if result.returncode==0:
                    for line in result.stdout.splitlines():
                        fields=[x.strip() for x in line.split(',')]
                        if len(fields)>=2 and fields[0].isdigit() and int(fields[0])==self.pid:
                            try:
                                mem=float(fields[1].split()[0])
                                self.peak=mem if self.peak is None else max(self.peak,mem)
                                self.count+=1
                            except ValueError:pass
            except (OSError,subprocess.TimeoutExpired):pass
            self.stop_signal.wait(2.0)
    def start(self):self.thread.start()
    def stop(self):self.stop_signal.set();self.thread.join(timeout=7)

# ------------------------------- CONFIGURAÇÃO -------------------------------
B_RUN_TRAINING = True
B_RUN_VALIDATION = True
B_SHOW_GALLERY = True
B_TRAIN_STEPS = 15_000
B_CHECKPOINT_STEPS = (4_000, 5_000, 6_000, 8_000, 14_999)
B_SAVE_STEPS = 1_000
B_SEED = 42
B_MAX_GAUSSIANS_WARNING = 450_000  # alerta diagnóstico; NÃO limita rigidamente
B_PLY_MIN_POINTS = 1_000
B_REQUIRE_EXACT_SPLITS = (95, 10, 10)
B_EXPECTED_A_TAG = 'scene0655_hv3_protocol_Av7_95_10_10'
B_EXPECTED_A_PROFILE = 'nuscenes_front_samples_train_sweeps_holdout_10_10_v7'
B_PYTHON = Path(globals().get('TCC_PYTHON', '/content/tcc_env/bin/python'))
B_BIN = B_PYTHON.parent
B_NS_TRAIN = B_BIN / 'ns-train'
B_NS_RENDER = B_BIN / 'ns-render'
B_CUDA = Path('/usr/local/cuda-12.8')

# CRÍTICO: usar exatamente a pasta da A V7. Jamais recuar para dataset B5.
if 'OUTPUT_DATASET' not in globals():
    raise RuntimeError('Execute a CÉLULA A V7 nesta sessão antes da B.')
B_DATASET = Path(OUTPUT_DATASET).resolve()
B_A_EXPECTED_PATH = Path('/content/nuscenes_nerfstudio_' + B_EXPECTED_A_TAG).resolve()
if B_DATASET != B_A_EXPECTED_PATH:
    raise RuntimeError('Dataset não é o gerado pela A V7: ' + str(B_DATASET) +
                       '\nEsperado: ' + str(B_A_EXPECTED_PATH))
if not globals().get('ENVIRONMENT_VALIDATED', False):
    raise RuntimeError('Execute e valide a CÉLULA 1 antes da B.')
for p in (B_PYTHON, B_NS_TRAIN, B_NS_RENDER):
    if not p.is_file():
        raise FileNotFoundError('Ferramenta não encontrada: '+str(p))
if B_TRAIN_STEPS != 15_000 or B_CHECKPOINT_STEPS[-1] != B_TRAIN_STEPS - 1:
    raise RuntimeError('Agenda de checkpoints incompatível com o protocolo de 15.000 iterações.')

B_STAMP = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
B_ROOT = Path('/content/nuscenes_experiments') / ('B_V7_splatfacto_' + B_STAMP)
B_OUTPUT_ROOT = B_ROOT / 'models'
B_LOG_ROOT = B_ROOT / 'logs'
B_RESULTS_ROOT = B_ROOT / 'results'
for p in (B_OUTPUT_ROOT, B_LOG_ROOT, B_RESULTS_ROOT):
    p.mkdir(parents=True, exist_ok=False)
B_EXPERIMENT = 'nuscenes_Av7_Hv3_splatfacto'
B_LOG = B_LOG_ROOT / 'splatfacto_training.log'
B_MANIFEST = B_RESULTS_ROOT / 'b_training_manifest.json'
B_VAL_CSV = B_RESULTS_ROOT / 'b_validation_per_view.csv'
B_VAL_SUMMARY_CSV = B_RESULTS_ROOT / 'b_validation_checkpoint_summary.csv'
B_RUNNER = B_ROOT / 'splatfacto_train_guard.py'
B_RUN_STATUS = B_RESULTS_ROOT / 'checkpoint_validation_selection.json'


def b_sha(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):
            digest.update(block)
    return digest.hexdigest()


def b_json_read(p):
    return json.loads(Path(p).read_text(encoding='utf-8'))


def b_json_write(p,obj):
    p=Path(p); p.parent.mkdir(parents=True,exist_ok=True)
    p.write_text(json.dumps(obj,indent=2,ensure_ascii=False,default=str)+'\n',encoding='utf-8')


def b_csv_write(p,rows,headers):
    with Path(p).open('w',encoding='utf-8',newline='') as out:
        w=csv.DictWriter(out,fieldnames=headers); w.writeheader(); w.writerows(rows)


def b_ply_vertex_count(p):
    with Path(p).open('rb') as f:
        header=f.read(65536).split(b'end_header',1)
    if len(header)!=2:
        raise RuntimeError('PLY sem end_header: '+str(p))
    match=re.search(rb'(?m)^element vertex\s+(\d+)\s*$',header[0])
    if not match:
        raise RuntimeError('PLY sem element vertex: '+str(p))
    return int(match.group(1))

# ----------------------- PREFLIGHT DE IDENTIDADE DA A V7 -------------------
B_TF_FILE = B_DATASET/'transforms.json'
B_MANIFEST_A = B_DATASET/'dataset_manifest.json'
B_IDENTITY_FILE = B_DATASET/'dataset_identity.json'
B_EVAL_FILE = B_DATASET/'eval_pairs.json'
for p in (B_TF_FILE,B_MANIFEST_A,B_IDENTITY_FILE,B_EVAL_FILE):
    if not p.is_file(): raise FileNotFoundError('Artefato A V7 ausente: '+str(p))
B_TF=b_json_read(B_TF_FILE)
B_A_MANIFEST=b_json_read(B_MANIFEST_A)
B_A_IDENTITY=b_json_read(B_IDENTITY_FILE)
if B_A_MANIFEST.get('experiment_tag') != B_EXPECTED_A_TAG or B_A_MANIFEST.get('dataset_profile') != B_EXPECTED_A_PROFILE:
    raise RuntimeError('Manifesto A não é V7/HV3; não executar B com outro dataset.')
if B_A_IDENTITY.get('protocol') != 'A_V7_HV3-aligned':
    raise RuntimeError('Identidade do dataset não é A V7.')
if B_A_MANIFEST.get('scene',{}).get('name')!='scene-0655' or B_A_MANIFEST.get('cameras',{}).get('channels')!=['CAM_FRONT']:
    raise RuntimeError('A exige scene-0655 e CAM_FRONT.')
if B_A_MANIFEST.get('protocol_v7',{}).get('normalization_uses_only_train') is not True:
    raise RuntimeError('A V7 não declara normalização TRAIN-only.')
if B_A_MANIFEST.get('lidar',{}).get('train_split_only') is not True:
    raise RuntimeError('A V7 não declara LiDAR exclusivo TRAIN.')
if B_A_MANIFEST.get('lidar',{}).get('rgb_colorization_source')!='training camera frames only':
    raise RuntimeError('A não declara colorização LiDAR exclusiva TRAIN.')
if B_A_MANIFEST.get('geometry',{}).get('geometry_inductive') is not True:
    raise RuntimeError('Geometria indutiva não registrada na A V7.')
if B_TF.get('camera_model')!='OPENCV' or B_TF.get('orientation_override')!='none':
    raise RuntimeError('Configuração de câmera/orientação inesperada.')
B_LISTS={k:list(B_TF.get(k+'_filenames',[])) for k in ('train','val','test')}
B_SPLIT_COUNTS=tuple(len(B_LISTS[k]) for k in ('train','val','test'))
if B_SPLIT_COUNTS != B_REQUIRE_EXACT_SPLITS:
    raise RuntimeError(f'A V7 precisa fornecer {B_REQUIRE_EXACT_SPLITS}; recebido {B_SPLIT_COUNTS}')
if len(B_TF.get('frames',[])) != sum(B_SPLIT_COUNTS):
    raise RuntimeError('transforms.json deve conter somente 115 frames (excluindo as 40 guardas).')
B_FRAME_MAP={f['file_path']:f for f in B_TF['frames']}
if len(B_FRAME_MAP)!=sum(B_SPLIT_COUNTS):
    raise RuntimeError('Frames repetidos no transforms.json.')
B_UNION=set().union(*(set(v) for v in B_LISTS.values()))
if len(B_UNION)!=sum(B_SPLIT_COUNTS) or B_UNION != set(B_FRAME_MAP):
    raise RuntimeError('Vazamento entre splits ou frames sem split correspondente.')
B_IMAGE_SIZE=None
for split,names in B_LISTS.items():
    for name in names:
        frame=B_FRAME_MAP[name]
        if frame.get('dataset_split')!=split:
            raise RuntimeError('Split divergente no transforms.json: '+str(name))
        wh=(int(frame['w']),int(frame['h']))
        if B_IMAGE_SIZE is None: B_IMAGE_SIZE=wh
        if wh != B_IMAGE_SIZE or wh!=(800,450):
            raise RuntimeError('A deve gerar imagens 800x450 CAM_FRONT.')
        if frame.get('camera_channel')!='CAM_FRONT':
            raise RuntimeError('Câmera diferente de CAM_FRONT: '+name)
        if not (B_DATASET/name).is_file():
            raise FileNotFoundError('Arquivo não existe: '+str(B_DATASET/name))

B_RAW_HASHES={name:b_sha(B_DATASET/name) for name in ('transforms.json','eval_pairs.json','dataset_manifest.json')}
B_LIDAR_REL=B_TF.get('ply_file_path')
if not B_LIDAR_REL: raise RuntimeError('PLY não declarado; não aceitar inicialização aleatória.')
B_PLY=(B_DATASET/B_LIDAR_REL).resolve()
if not B_PLY.is_relative_to(B_DATASET) or not B_PLY.is_file():
    raise RuntimeError('PLY LiDAR ausente ou fora do dataset.')
B_PLY_POINTS=b_ply_vertex_count(B_PLY)
if B_PLY_POINTS<B_PLY_MIN_POINTS:
    raise RuntimeError(f'LiDAR inicial tem somente {B_PLY_POINTS} pontos.')
if B_PLY_POINTS!=int(B_A_MANIFEST.get('lidar',{}).get('final_point_count',-1)):
    raise RuntimeError('Contagem de pontos no PLY difere do manifesto A V7.')
B_RAW_HASHES[B_LIDAR_REL]=b_sha(B_PLY)
for file_name,hash_expected in B_A_IDENTITY.get('core_sha256',{}).items():
    if file_name not in B_RAW_HASHES or B_RAW_HASHES[file_name]!=hash_expected:
        raise RuntimeError(f'Dataset A V7 alterado após geração: {file_name}')
B_HOLDOUT_TIMES=[int(B_FRAME_MAP[n]['timestamp']) for s in ('val','test') for n in B_LISTS[s]]
B_TRAIN_TIMES=[int(B_FRAME_MAP[n]['timestamp']) for n in B_LISTS['train']]
B_GAP_MS=min(abs(x-y) for x in B_TRAIN_TIMES for y in B_HOLDOUT_TIMES)/1000.0
B_REQUIRED_GAP_MS=float(B_A_MANIFEST.get('cameras',{}).get('holdout_sweep_guard_ms',250))
if B_GAP_MS<=B_REQUIRED_GAP_MS:
    raise RuntimeError(f'Guarda temporal {B_GAP_MS:.3f} ms não excede {B_REQUIRED_GAP_MS} ms.')
B_PLY_COUNT_LIMIT=min(B_PLY_POINTS,120000)

print('='*90)
print('B V7 — Splatfacto nuScenes / A V7, pareado em tamanho com H V3')
print('Dataset:',B_DATASET)
print('TRAIN / VAL / TEST:',B_SPLIT_COUNTS,' | GUARD:',B_A_MANIFEST.get('split',{}).get('guard_frame_count'))
print('Menor distância TRAIN–holdout:',f'{B_GAP_MS:.1f} ms')
print('PLY LiDAR TRAIN-only:',B_PLY_POINTS,'pontos')
print('Checkpoints avaliados por VAL:',B_CHECKPOINT_STEPS)
print('Somente TEST será avaliado na célula D atualizada. Nenhum backup de checkpoints no Drive.')

# ---------------------------- PREPARAR AMBIENTE -----------------------------
B_ENV=dict(globals().get('TCC_SUBPROCESS_ENV',os.environ))
for key in ('PYTHONHOME','PYTHONPATH','PYTHONUSERBASE','PIP_TARGET'):
    B_ENV.pop(key,None)
B_ENV.update({'PATH':str(B_BIN)+':'+str(B_CUDA/'bin')+':'+os.environ.get('PATH',''),
              'VIRTUAL_ENV':str(B_BIN.parent),'PYTHONNOUSERSITE':'1',
              'PYTHONUNBUFFERED':'1','QT_QPA_PLATFORM':'offscreen'})
if (B_CUDA/'bin/nvcc').is_file():
    B_ENV.update({'CUDA_HOME':str(B_CUDA),'CUDA_PATH':str(B_CUDA),
                  'CUDACXX':str(B_CUDA/'bin/nvcc')})
if (B_CUDA/'lib64').is_dir():
    B_ENV['LD_LIBRARY_PATH']=str(B_CUDA/'lib64')+':'+B_ENV.get('LD_LIBRARY_PATH','')

# Runner rápido: prova que o Splatfacto carregou TODOS os pontos do PLY
# (e não uma nuvem aleatória). Sem manipular o loop/densificação internamente.
B_RUNNER.write_text('''
import os
from nerfstudio.engine.trainer import Trainer
original_setup = Trainer.setup
expected = int(os.environ['B_EXPECTED_LIDAR_POINTS'])
def guarded_setup(self,*args,**kwargs):
    answer=original_setup(self,*args,**kwargs)
    if self.config.method_name != 'splatfacto':
        raise RuntimeError('Modelo diferente do Splatfacto.')
    if self.config.load_checkpoint or self.config.load_dir:
        raise RuntimeError('A célula B deve começar do zero.')
    points = int(self.pipeline.model.num_points)
    print('B_INITIAL_GAUSSIANS='+str(points),flush=True)
    if points != expected:
        raise RuntimeError('Inicialização não veio do PLY LiDAR: %d != %d' % (points,expected))
    return answer
Trainer.setup=guarded_setup
from nerfstudio.scripts.train import entrypoint
entrypoint()
'''.lstrip(),encoding='utf-8')
B_ENV['B_EXPECTED_LIDAR_POINTS']=str(B_PLY_POINTS)
B_COMMAND=[str(B_PYTHON),str(B_RUNNER),'splatfacto',
           '--data',str(B_DATASET),
           '--experiment-name',B_EXPERIMENT,'--output-dir',str(B_OUTPUT_ROOT),
           '--machine.seed',str(B_SEED),
           '--max-num-iterations',str(B_TRAIN_STEPS),
           '--steps-per-save',str(B_SAVE_STEPS),
           '--save-only-latest-checkpoint','False',
           '--steps-per-eval-image','1000',
           '--steps-per-eval-all-images','5000',
           '--vis','tensorboard',
           '--pipeline.datamanager.cache-images','cpu',
           '--pipeline.datamanager.cache-images-type','uint8',
           '--pipeline.model.num-random',str(B_PLY_COUNT_LIMIT),
           '--pipeline.model.camera-optimizer.mode','off',
           '--pipeline.model.num-downscales','0',
           '--pipeline.model.sh-degree','2',
           '--pipeline.model.sh-degree-interval','500',
           '--pipeline.model.densify-grad-thresh','0.001',
           '--pipeline.model.stop-split-at','6000',
           '--pipeline.model.stop-screen-size-at','750',
           '--pipeline.model.cull-alpha-thresh','0.01',
           '--pipeline.model.use-absgrad','False',
           '--pipeline.model.use-bilateral-grid','False',
           '--pipeline.model.rasterize-mode','antialiased',
           '--pipeline.model.ssim-lambda','0.2',
           'nerfstudio-data','--downscale-factor','1','--orientation-method','none',
           '--center-method','none','--auto-scale-poses','False','--scene-scale','1.5']

B_RECORD={'status':'preflight_ok',
          'protocol':'A_V7_HV3_aligned_95_10_10',
          'created_utc':datetime.now(timezone.utc).isoformat(),
          'dataset':str(B_DATASET),'dataset_sha256':B_RAW_HASHES,
          'split_counts':dict(zip(('train','val','test'),B_SPLIT_COUNTS)),
          'guard_ms':B_REQUIRED_GAP_MS,'minimum_actual_train_holdout_gap_ms':B_GAP_MS,
          'initialization':{'type':'nuScenes LiDAR TRAIN-only','ply':str(B_PLY),'points':B_PLY_POINTS},
          'train_parameters':{'seed':B_SEED,'steps':B_TRAIN_STEPS,'stop_split_at':6000,
                              'stop_screen_size_at':750,'checkpoint_steps':list(B_CHECKPOINT_STEPS)},
          'test_accessed':False,'uploads_to_drive':False,'command':B_COMMAND,
          'artifacts':{'log':str(B_LOG),'results':str(B_RESULTS_ROOT)}}
b_json_write(B_MANIFEST,B_RECORD)
print('\nComando de treino:\n'+shlex.join(B_COMMAND))

# ------------------------------- TREINAMENTO --------------------------------
B_GS_CONFIG=None
B_GS_CHECKPOINT=None
B_SELECTED_STEP=None
B_RUN_DIRECTORY=None
B_VALIDATION_SUMMARY=None
B_SELECTED_CHECKPOINT=None
B_RETURN_CODE=None
if B_RUN_TRAINING:
    start_wall=time.time(); start_perf=time.monotonic()
    process=subprocess.Popen(B_COMMAND,env=B_ENV,stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT,text=True,bufsize=1,cwd=B_ROOT)
    B_GPU_MONITOR=_TrainingGpuSampler(process.pid)
    B_GPU_MONITOR.start()
    tail=deque(maxlen=80); initial_points=None
    try:
        with B_LOG.open('w',encoding='utf-8') as logfile:
            for line in process.stdout:
                logfile.write(line);logfile.flush();tail.append(line.rstrip())
                if 'B_INITIAL_GAUSSIANS=' in line:
                    m=re.search(r'B_INITIAL_GAUSSIANS=(\d+)',line)
                    if m: initial_points=int(m.group(1))
                if any(x in line.lower() for x in ('error','traceback','out of memory','saving config to',
                                                'training finished','saving checkpoints to')):
                    print(line.rstrip(),flush=True)
                # saída curta com progresso temporal, sem poluir o notebook
                match=re.search(r'(?m)^\s*(\d+)\s+\([\d.]+%\)',line)
                if match and int(match.group(1))%1000==0:
                    print('Treinamento B:',match.group(1),'passos',flush=True)
            B_RETURN_CODE=process.wait()
    except KeyboardInterrupt:
        process.terminate()
        try:process.wait(timeout=15)
        except subprocess.TimeoutExpired:process.kill();process.wait()
        raise
    finally:
        if process.poll() is None:
            process.terminate();process.wait(timeout=15)
    B_GPU_MONITOR.stop()
    B_TRAIN_SECONDS=time.monotonic()-start_perf
    if B_RETURN_CODE != 0:
        B_RECORD.update(status='train_failed',return_code=B_RETURN_CODE,
                        training_seconds=B_TRAIN_SECONDS)
        b_json_write(B_MANIFEST,B_RECORD)
        raise RuntimeError('Treinamento Splatfacto falhou. Últimas linhas:\n'+
                           '\n'.join(tail)+'\nLog: '+str(B_LOG))
    if initial_points != B_PLY_POINTS:
        raise RuntimeError('Sem confirmação da inicialização LiDAR no log do treino.')
    configs=[p for p in B_OUTPUT_ROOT.rglob('config.yml') if p.stat().st_mtime>=start_wall-10]
    if len(configs)!=1:
        raise RuntimeError('Era esperado um único config.yml desta rodada; encontrados '+str(configs))
    B_GS_CONFIG=configs[0]
    B_RUN_DIRECTORY=B_GS_CONFIG.parent
    B_CHECKPOINT_DIR=B_RUN_DIRECTORY/'nerfstudio_models'
    ckpt={int(re.search(r'step-(\d+)\.ckpt$',p.name).group(1)):p
          for p in B_CHECKPOINT_DIR.glob('step-*.ckpt')}
    if not set(B_CHECKPOINT_STEPS).issubset(ckpt):
        raise RuntimeError('Checkpoints necessários ausentes: '+str(sorted(set(B_CHECKPOINT_STEPS)-set(ckpt))))
    B_GS_CHECKPOINT = ckpt[B_TRAIN_STEPS - 1]
    B_RECORD.update(status='trained_waiting_validation',training_seconds=round(B_TRAIN_SECONDS,2),peak_gpu_memory_mib_sampled=B_GPU_MONITOR.peak,
                    peak_gpu_memory_sampling_interval_seconds=2,
                    peak_gpu_memory_samples=B_GPU_MONITOR.count,

                    config=str(B_GS_CONFIG),
                    checkpoint_history=[{'step':s,'file':str(p),'size_mib':round(p.stat().st_size/1048576,2)}
                                        for s,p in sorted(ckpt.items())],
                    test_accessed=False)
    b_json_write(B_MANIFEST,B_RECORD)
    print('\nTreino concluído em',round(B_TRAIN_SECONDS/60,2),'min. Checkpoints:',sorted(ckpt))

# -------------------------------- VALIDAÇÃO ---------------------------------
# A escolha do checkpoint usa SOMENTE as imagens de VAL, como na célula H V3.
if B_RUN_TRAINING and B_RUN_VALIDATION:
    # Config YAML do Nerfstudio é Python-tagged: carregar APENAS config GERADO
    # localmente pelo processo ns-train da própria execução.
    B_MAKE_CONFIG=B_ROOT/'prepare_checkpoint_config.py'
    B_MAKE_CONFIG.write_text('''
import sys,yaml
from pathlib import Path
source=Path(sys.argv[1]); output=Path(sys.argv[2]); step=int(sys.argv[3])
cfg=yaml.load(source.read_text(encoding='utf-8'),Loader=yaml.Loader)
assert cfg.method_name=='splatfacto'
assert cfg.pipeline.model.camera_optimizer.mode=='off'
assert (cfg.get_checkpoint_dir()/f'step-{step:09d}.ckpt').is_file()
cfg.load_step=step
output.write_text(yaml.dump(cfg),encoding='utf-8')
print('B_SELECTED_CONFIG_OK='+str(step))
'''.lstrip(),encoding='utf-8')
    expected={Path(p).stem:p for p in B_LISTS['val']}
    if len(expected)!=len(B_LISTS['val']):
        raise RuntimeError('Nomes de VAL colidem quando reduzidos a stem.')
    B_VAL_ROWS=[];B_VAL_SUMMARY=[]
    for step in B_CHECKPOINT_STEPS:
        base=B_RESULTS_ROOT/'validation'/str(step)
        base.mkdir(parents=True,exist_ok=False)
        selected_cfg=base/'config.yml'
        cfg_result=subprocess.run([str(B_PYTHON),str(B_MAKE_CONFIG),str(B_GS_CONFIG),str(selected_cfg),str(step)],
                                  text=True,capture_output=True,env=B_ENV,timeout=120)
        if cfg_result.returncode!=0 or 'B_SELECTED_CONFIG_OK=' not in cfg_result.stdout:
            raise RuntimeError('Falha na validação de checkpoint '+str(step)+'\n'+cfg_result.stderr[-3000:])
        render_cmd=[str(B_NS_RENDER),'dataset','--load-config',str(selected_cfg),
                    '--output-path',str(base/'renders'),'--data',str(B_DATASET),
                    '--split','val','--image-format','png',
                    '--rendered-output-names','rgb','gt-rgb']
        logpath=base/'render.log'
        print('Renderizando VAL, checkpoint',step,flush=True)
        with logpath.open('w',encoding='utf-8') as logfile:
            render_run=subprocess.run(render_cmd,stdout=logfile,stderr=subprocess.STDOUT,
                                      text=True,env=B_ENV,timeout=3600)
        if render_run.returncode:
            raise RuntimeError('ns-render VAL falhou em '+str(step)+'\n'+logpath.read_text(errors='replace')[-5000:])
        log_txt=re.sub(r'\x1b\[[0-9;]*m','',logpath.read_text(errors='replace'))
        if 'Done loading checkpoint from' not in log_txt:
            raise RuntimeError('Render não confirmou carregamento do checkpoint '+str(step))
        checkpoint_match=re.findall(r'Done loading checkpoint from[^\n]*step-(\d+)\.ckpt',log_txt)
        if checkpoint_match and int(checkpoint_match[-1])!=step:
            raise RuntimeError('Render carregou checkpoint diferente de '+str(step))
        outroot=base/'renders'/'val'
        paths={k:list((outroot/k).rglob('*.png')) for k in ('rgb','gt-rgb')}
        for kind,images in paths.items():
            if len(images)!=len(expected) or {p.stem for p in images}!=set(expected):
                raise RuntimeError('Render incompleto para '+kind+', step '+str(step))
        gt={p.stem:p for p in paths['gt-rgb']}
        rgb={p.stem:p for p in paths['rgb']}
        vals=[]
        for stem,rel in sorted(expected.items(),key=lambda x:int(B_FRAME_MAP[x[1]]['timestamp'])):
            a=np.asarray(Image.open(gt[stem]).convert('RGB'))
            p=np.asarray(Image.open(rgb[stem]).convert('RGB'))
            inp=np.asarray(Image.open(B_DATASET/rel).convert('RGB'))
            if a.shape!=p.shape or a.shape!=inp.shape or a.shape!=(450,800,3):
                raise RuntimeError('Dimensões RGB inconsistentes na vista '+stem)
            # As referências reais em B e C devem ter os mesmos pixels.
            if np.max(np.abs(a.astype(np.int16)-inp.astype(np.int16)))>1:
                raise RuntimeError('gt-rgb da renderização não corresponde à imagem original '+stem)
            mse=np.mean(((a.astype(np.float64)-p.astype(np.float64))/255.0)**2)
            value=float(-10*math.log10(max(mse,1e-15)))
            B_VAL_ROWS.append({'step':step,'image':stem,'file_path':rel,'psnr_db':round(value,7)})
            vals.append(value)
        B_VAL_SUMMARY.append({'step':step,'count':len(vals),'mean_psnr_db':float(np.mean(vals)),
                              'min_psnr_db':float(np.min(vals)),'max_psnr_db':float(np.max(vals))})
        print('  VAL:',f'{np.mean(vals):.4f} dB',flush=True)
    b_csv_write(B_VAL_CSV,B_VAL_ROWS,['step','image','file_path','psnr_db'])
    b_csv_write(B_VAL_SUMMARY_CSV,B_VAL_SUMMARY,['step','count','mean_psnr_db','min_psnr_db','max_psnr_db'])
    B_VALIDATION_SUMMARY=B_VAL_SUMMARY
    # Ordem dos steps é crescente. Em caso de empate, manter o checkpoint mais antigo.
    B_BEST=min(B_VAL_SUMMARY,key=lambda item:(-item['mean_psnr_db'],item['step']))
    B_SELECTED_STEP=int(B_BEST['step'])
    B_SELECTED_CHECKPOINT=ckpt[B_SELECTED_STEP]
    B_GS_CHECKPOINT=B_SELECTED_CHECKPOINT  # compatibilidade com as próximas células
    B_RECORD.update(status='completed_selected_by_validation',
                    selected_step=B_SELECTED_STEP,
                    selected_checkpoint=str(B_SELECTED_CHECKPOINT),
                    selected_checkpoint_sha256=b_sha(B_SELECTED_CHECKPOINT),
                    validation_mean_psnr_db=B_BEST['mean_psnr_db'],
                    validation_summary=B_VAL_SUMMARY,
                    test_accessed=False,uploads_to_drive=False)
    b_json_write(B_MANIFEST,B_RECORD)
    b_json_write(B_RUN_STATUS,{'dataset':str(B_DATASET),'dataset_hashes':B_RAW_HASHES,
                               'selected_step':B_SELECTED_STEP,'selected_checkpoint':str(B_SELECTED_CHECKPOINT),
                               'selection_criterion':'highest mean PSNR over 10 validation images',
                               'validation_steps':B_VAL_SUMMARY,
                               'test_used_for_selection':False,'test_rendered':False})
    if B_SHOW_GALLERY:
        # Visualizar apenas algumas amostras TRAIN/VAL; TEST não é aberto.
        sample_dir=B_RESULTS_ROOT/'validation'/str(B_SELECTED_STEP)/'renders'/'val'
        rgb_paths={p.stem:p for p in (sample_dir/'rgb').glob('*.png')}
        gt_paths={p.stem:p for p in (sample_dir/'gt-rgb').glob('*.png')}
        ordered=sorted(expected.items(),key=lambda x:int(B_FRAME_MAP[x[1]]['timestamp']))
        gallery=Image.new('RGB',(800,5*260),(255,255,255))
        drawer=ImageDraw.Draw(gallery)
        for i,(stem,rel) in enumerate(ordered[:5]):
            a=Image.open(gt_paths[stem]).convert('RGB').resize((400,225))
            p=Image.open(rgb_paths[stem]).convert('RGB').resize((400,225))
            y=i*260;gallery.paste(a,(0,y+28));gallery.paste(p,(400,y+28))
            drawer.text((5,y+5),f'VAL {i+1:02d} real',fill=(0,0,0))
            drawer.text((405,y+5),f'Splatfacto step {B_SELECTED_STEP}',fill=(0,0,0))
        gallery_file=B_RESULTS_ROOT/'val_gallery_5_examples.jpg'
        gallery.save(gallery_file,quality=92)
        try:
            from IPython.display import display
            display(gallery)
        except ImportError:pass
    print('\n'+'='*90)
    print('B V7 CONCLUÍDA — VAL checkpoint selecionado:',B_SELECTED_STEP)
    print('PSNR médio VAL:',f'{B_BEST["mean_psnr_db"]:.4f} dB')
    print('Config:',B_GS_CONFIG)
    print('Checkpoint selecionado:',B_SELECTED_CHECKPOINT)
    print('Manifesto B:',B_MANIFEST)
    print('TEST não avaliado. Sem upload para Google Drive.')
else:
    print('B apenas preparou/treinou; seleção VAL não concluída.')
    print('As células seguintes não devem assumir checkpoint selecionado.')

# Variáveis de integração para as células C–G.
B_DATASET_SHA256=B_RAW_HASHES
B_RESULTS_DIR=B_RESULTS_ROOT
B_MODEL_CONFIG=B_GS_CONFIG
B_BEST_CHECKPOINT=B_SELECTED_CHECKPOINT


In [ ]:
# =============================================================================
# CÉLULA C V7.2 — NERFACTO-BIG 30k | nuScenes scene-0655 | A V7 + B V7
# =============================================================================
# Executar após: CÉLULA 0 -> SETUP -> 1 -> A V7 -> B V7, na MESMA sessão.
# 95 TRAIN / 10 VAL / 10 TEST | CAM_FRONT | 800x450.
# Alinhamento com H V3: Nerfacto-big, 30k, seed 42, 8192/4096 raios,
# TCNN, poses fixas, checkpoints de 1000 em 1000 e seleção APENAS por VAL.
# A/B/C compartilham transforms.json, imagens e calibrações (poses nuScenes).
# TEST: não lê RGB, não renderiza e não utiliza PSNR para selecionar checkpoints.
# Sem backup de checkpoints/imagens no Google Drive. Resultados só em /content.
# A célula D realiza a avaliação final do conjunto TEST.
# =============================================================================

from collections import deque
from datetime import datetime, timezone
from pathlib import Path
from PIL import Image, ImageDraw
import csv
import hashlib
import json
import math
import os
import re
import shlex
import shutil
import subprocess
import time
import numpy as np

# Amostragem externa do pico de VRAM do processo de treinamento (não é pico exato).
import threading
class _TrainingGpuSampler:
    def __init__(self,pid):
        self.pid=int(pid);self.peak=None;self.count=0
        self.stop_signal=threading.Event()
        self.thread=threading.Thread(target=self._sample,daemon=True)
    def _sample(self):
        while not self.stop_signal.is_set():
            try:
                result=subprocess.run(['nvidia-smi','--query-compute-apps=pid,used_gpu_memory',
                    '--format=csv,noheader,nounits'],capture_output=True,text=True,timeout=5)
                if result.returncode==0:
                    for line in result.stdout.splitlines():
                        fields=[x.strip() for x in line.split(',')]
                        if len(fields)>=2 and fields[0].isdigit() and int(fields[0])==self.pid:
                            try:
                                mem=float(fields[1].split()[0])
                                self.peak=mem if self.peak is None else max(self.peak,mem)
                                self.count+=1
                            except ValueError:pass
            except (OSError,subprocess.TimeoutExpired):pass
            self.stop_signal.wait(2.0)
    def start(self):self.thread.start()
    def stop(self):self.stop_signal.set();self.thread.join(timeout=7)

# ----------------------------- PARÂMETROS -----------------------------------
C_RUN_TRAINING = True
C_RUN_VALIDATION = True
C_SHOW_GALLERY = True
C_VERBOSE_TRAIN = False
C_REQUIRE_B_COMPLETED = True

C_PRESET = 'nerfacto-big'
C_TOTAL_STEPS = 15_000
C_SAVE_EVERY = 1_000
C_SEED = 42
C_TRAIN_RAYS = 8_192
C_EVAL_RAYS = 4_096
C_CHECKPOINT_STEPS = (1_000, 2_000, 3_000, 4_000, 5_000,
6_000, 8_000, 10_000, 12_000, 14_999)
C_EXPECTED_SPLITS = (95, 10, 10)
C_EXPECTED_A_TAG = 'scene0655_hv3_protocol_Av7_95_10_10'
C_EXPECTED_A_PROFILE = 'nuscenes_front_samples_train_sweeps_holdout_10_10_v7'
C_EXPECTED_RESOLUTION = (800, 450)
C_TRAIN_TIMEOUT_SECONDS = 10800  # 3 horas; registra erro explícito se excedido
C_RENDER_TIMEOUT_SECONDS = 2400

C_PYTHON = Path(globals().get('TCC_PYTHON', '/content/tcc_env/bin/python'))
C_BIN = C_PYTHON.parent
C_NS_TRAIN = C_BIN / 'ns-train'
C_NS_RENDER = C_BIN / 'ns-render'
C_CUDA = Path('/usr/local/cuda-12.8')

if not globals().get('ENVIRONMENT_VALIDATED', False):
    raise RuntimeError('Execute as células 0, SETUP e 1 antes de A/B/C.')
if 'OUTPUT_DATASET' not in globals():
    raise RuntimeError('Dataset da A V7 não encontrado nas variáveis desta sessão.')
C_DATASET = Path(globals()['OUTPUT_DATASET']).resolve()
C_EXPECTED_DATASET = Path('/content/nuscenes_nerfstudio_' + C_EXPECTED_A_TAG).resolve()
if C_DATASET != C_EXPECTED_DATASET:
    raise RuntimeError('A célula C exige a A V7 exata. Recebido: '+str(C_DATASET)+
                       '\nEsperado: '+str(C_EXPECTED_DATASET))
for p in (C_PYTHON, C_NS_TRAIN, C_NS_RENDER):
    if not p.is_file():
        raise FileNotFoundError('Ferramenta do ambiente isolado ausente: '+str(p))
if C_RUN_TRAINING and not C_RUN_VALIDATION:
    raise RuntimeError('Para continuar em D com rigor, C_RUN_VALIDATION deve ser True.')
if not C_RUN_TRAINING:
    raise RuntimeError('C V7 é um treino novo. Reative C_RUN_TRAINING para evitar manifestos incompletos.')
if (C_PRESET != 'nerfacto-big' or C_TOTAL_STEPS != 15_000 or
        C_SAVE_EVERY != 1_000 or C_TRAIN_RAYS != 8_192 or
        C_EVAL_RAYS != 4_096):
    raise RuntimeError(
        'Configuração incompatível com o protocolo Nerfacto-Big de 15.000 iterações.'
    )

if C_CHECKPOINT_STEPS[-1] != C_TOTAL_STEPS - 1:
    raise RuntimeError(
        f'O checkpoint final esperado é step-{C_TOTAL_STEPS-1:09d}.ckpt.'
    )

# ------------------------------- FUNÇÕES ------------------------------------
def c_sha(p):
    dig = hashlib.sha256()
    with Path(p).open('rb') as file:
        for data in iter(lambda: file.read(1024 * 1024), b''):
            dig.update(data)
    return dig.hexdigest()


def c_read(p):
    return json.loads(Path(p).read_text(encoding='utf-8'))


def c_save(p, value):
    p = Path(p)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(value, indent=2, ensure_ascii=False, default=str)+'\n', encoding='utf-8')


def c_csv(p, rows, names):
    with Path(p).open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=names)
        writer.writeheader()
        writer.writerows(rows)


def c_checkpoint_step(p):
    result = re.fullmatch(r'step-(\d+)\.ckpt', Path(p).name)
    if result is None:
        raise RuntimeError('Checkpoint com nome inválido: '+str(p))
    return int(result.group(1))


def c_assert_within(file, root):
    file, root = Path(file).resolve(), Path(root).resolve()
    if not file.is_relative_to(root):
        raise RuntimeError('Caminho fora do dataset: '+str(file))
    return file


def c_image_rgb(p):
    with Image.open(p) as image:
        return np.asarray(image.convert('RGB'), dtype=np.uint8)


def c_psnr(a, b):
    if a.shape != b.shape or a.shape != (450, 800, 3):
        raise RuntimeError(f'Dimensões inválidas para PSNR: {a.shape} e {b.shape}')
    mse = np.mean(((a.astype(np.float64)-b.astype(np.float64))/255.0)**2)
    return float(-10.0*math.log10(max(float(mse), 1e-15)))


# --------------------------- PREFLIGHT A/B V7 -------------------------------
C_TF_PATH = C_DATASET/'transforms.json'
C_A_MANIFEST_PATH = C_DATASET/'dataset_manifest.json'
C_A_IDENTITY_PATH = C_DATASET/'dataset_identity.json'
C_EVAL_PAIRS_PATH = C_DATASET/'eval_pairs.json'
for required in (C_TF_PATH, C_A_MANIFEST_PATH, C_A_IDENTITY_PATH, C_EVAL_PAIRS_PATH):
    if not required.is_file():
        raise FileNotFoundError('A V7 incompleta; falta '+str(required))

C_TF = c_read(C_TF_PATH)
C_A_MANIFEST = c_read(C_A_MANIFEST_PATH)
C_A_IDENTITY = c_read(C_A_IDENTITY_PATH)

if (C_A_MANIFEST.get('experiment_tag') != C_EXPECTED_A_TAG or
        C_A_MANIFEST.get('dataset_profile') != C_EXPECTED_A_PROFILE or
        C_A_IDENTITY.get('protocol') != 'A_V7_HV3-aligned'):
    raise RuntimeError('Identidade do dataset A V7 não corresponde ao protocolo H V3.')
if C_A_MANIFEST.get('scene', {}).get('name') != 'scene-0655':
    raise RuntimeError('A cena não é scene-0655.')
if C_A_MANIFEST.get('cameras', {}).get('channels') != ['CAM_FRONT']:
    raise RuntimeError('A deve ser somente CAM_FRONT.')
if C_A_MANIFEST.get('geometry', {}).get('geometry_inductive') is not True:
    raise RuntimeError('A não confirmou protocolo geométrico indutivo.')
if C_A_MANIFEST.get('lidar', {}).get('train_split_only') is not True:
    raise RuntimeError('A não confirmou LiDAR somente de TRAIN.')
if C_A_MANIFEST.get('protocol_v7', {}).get('normalization_uses_only_train') is not True:
    raise RuntimeError('A não confirmou normalização baseada no treino.')
if (C_TF.get('camera_model') != 'OPENCV' or
        C_TF.get('orientation_override') != 'none'):
    raise RuntimeError('Metadados de câmera inesperados.')

C_SPLITS = {k: list(C_TF.get(k+'_filenames', [])) for k in ('train', 'val', 'test')}
C_SPLIT_COUNTS = tuple(len(C_SPLITS[k]) for k in ('train', 'val', 'test'))
if C_SPLIT_COUNTS != C_EXPECTED_SPLITS:
    raise RuntimeError(f'A V7 forneceu splits {C_SPLIT_COUNTS} em vez de {C_EXPECTED_SPLITS}.')
C_FRAME_MAP = {f['file_path']: f for f in C_TF['frames']}
C_ALL_FILES = [p for k in ('train', 'val', 'test') for p in C_SPLITS[k]]
if (len(C_TF['frames']) != 115 or len(C_FRAME_MAP) != 115 or
        len(C_ALL_FILES) != 115 or len(set(C_ALL_FILES)) != 115 or
        set(C_ALL_FILES) != set(C_FRAME_MAP)):
    raise RuntimeError('Datasets/splits incompletos, repetidos ou diferentes da A V7.')
for split, names in C_SPLITS.items():
    for relpath in names:
        frame = C_FRAME_MAP[relpath]
        if (frame.get('dataset_split') != split or
                frame.get('camera_channel') != 'CAM_FRONT' or
                (int(frame['w']),int(frame['h'])) != C_EXPECTED_RESOLUTION):
            raise RuntimeError('Imagem com split, câmera ou resolução inválida: '+relpath)
        p = c_assert_within(C_DATASET/relpath, C_DATASET)
        if not p.is_file():
            raise FileNotFoundError('Imagem da A V7 não encontrada: '+str(p))

# Apenas os metadados dos arquivos de teste são conferidos; nenhuma leitura RGB.
C_TEST_PAIRS = c_read(C_EVAL_PAIRS_PATH)
if len(C_TEST_PAIRS) != 10 or any(item.get('split') != 'test' for item in C_TEST_PAIRS):
    raise RuntimeError('eval_pairs.json deve declarar exatamente dez vistas TEST.')
C_CORE_HASHES = {name: c_sha(C_DATASET/name) for name in
                 ('transforms.json', 'eval_pairs.json', 'dataset_manifest.json')}
C_PLY_REL = C_TF.get('ply_file_path')
if not C_PLY_REL:
    raise RuntimeError('LiDAR da A V7 não declarado; C exige mesma identidade de A/B.')
C_PLY = c_assert_within(C_DATASET/C_PLY_REL, C_DATASET)
if not C_PLY.is_file():
    raise FileNotFoundError('PLY da A V7 ausente: '+str(C_PLY))
C_CORE_HASHES[C_PLY_REL] = c_sha(C_PLY)
if C_CORE_HASHES != C_A_IDENTITY.get('core_sha256', {}):
    raise RuntimeError('Hashes A V7 divergentes; dataset foi modificado após a geração.')

# Guardas validadas sobre TODOS os TRAIN (incluindo keyframes), sem processar RGB.
C_TRAIN_TIMES = [int(C_FRAME_MAP[name]['timestamp']) for name in C_SPLITS['train']]
C_HOLD_TIMES = [int(C_FRAME_MAP[name]['timestamp'])
                for split in ('val','test') for name in C_SPLITS[split]]
C_MIN_GAP_MS = min(abs(a-b) for a in C_TRAIN_TIMES for b in C_HOLD_TIMES)/1000.0
C_GUARD_MS = float(C_A_MANIFEST.get('cameras',{}).get('holdout_sweep_guard_ms',250.0))
if C_MIN_GAP_MS <= C_GUARD_MS:
    raise RuntimeError(f'Guarda temporal insuficiente: {C_MIN_GAP_MS:.3f} <= {C_GUARD_MS:.3f} ms.')

# Conferir a MESMA execução B V7, sem procurar B5 nem procurar arquivos aleatórios.
if C_REQUIRE_B_COMPLETED:
    if 'B_MANIFEST' not in globals():
        raise RuntimeError('Execute B V7 nesta mesma sessão; B_MANIFEST está ausente.')
    C_B_MANIFEST_PATH = Path(globals()['B_MANIFEST']).resolve()
    if not C_B_MANIFEST_PATH.is_file():
        raise FileNotFoundError('Manifesto da B V7 ausente: '+str(C_B_MANIFEST_PATH))
    C_B = c_read(C_B_MANIFEST_PATH)
    if C_B.get('status') != 'completed_selected_by_validation':
        raise RuntimeError('B V7 ainda não concluiu seleção por VAL: '+str(C_B.get('status')))
    if Path(C_B.get('dataset', '/missing')).resolve() != C_DATASET:
        raise RuntimeError('B V7 usa dataset diferente da A V7/C.')
    if C_B.get('dataset_sha256') != C_CORE_HASHES:
        raise RuntimeError('Dataset alterado entre B e C; hashes não coincidem.')
    if C_B.get('split_counts') != dict(zip(('train','val','test'),C_EXPECTED_SPLITS)):
        raise RuntimeError('Manifesto B registra outra divisão de dados.')
    C_B_CONFIG = Path(C_B.get('config', '/missing'))
    C_B_SELECTED = Path(C_B.get('selected_checkpoint', '/missing'))
    if not C_B_CONFIG.is_file() or not C_B_SELECTED.is_file():
        raise FileNotFoundError('Config ou checkpoint selecionado na B não existe.')
    if c_checkpoint_step(C_B_SELECTED) != int(C_B['selected_step']):
        raise RuntimeError('B selecionou checkpoint inconsistente.')
    if c_sha(C_B_SELECTED) != C_B.get('selected_checkpoint_sha256'):
        raise RuntimeError('O checkpoint da B foi modificado após seleção por VAL.')
else:
    raise RuntimeError('C_REQUIRE_B_COMPLETED deve permanecer True neste protocolo.')

# Confirmar que ferramentas e GPU estão disponíveis antes do treinamento longo.
C_ENV = dict(globals().get('TCC_SUBPROCESS_ENV', os.environ))
for var in ('PYTHONHOME','PYTHONPATH','PYTHONUSERBASE','PIP_TARGET'):
    C_ENV.pop(var, None)
C_ENV.update({
    'PATH': str(C_BIN)+':'+str(C_CUDA/'bin')+':'+os.environ.get('PATH',''),
    'VIRTUAL_ENV': str(C_BIN.parent),
    'PYTHONNOUSERSITE': '1',
    'PYTHONUNBUFFERED': '1',
    'CUDA_HOME': str(C_CUDA),
    'CUDA_PATH': str(C_CUDA),
    'CUDACXX': str(C_CUDA/'bin'/'nvcc'),
    'QT_QPA_PLATFORM': 'offscreen',
})
if (C_CUDA/'lib64').is_dir():
    C_ENV['LD_LIBRARY_PATH'] = str(C_CUDA/'lib64')+':'+C_ENV.get('LD_LIBRARY_PATH','')

C_GPU_CHECK = subprocess.run([str(C_PYTHON), '-c',
    "import torch, importlib.metadata; "
    "assert torch.cuda.is_available(), 'CUDA indisponível'; "
    "assert importlib.metadata.version('nerfstudio') == '1.1.5', 'Nerfstudio diferente de 1.1.5'; "
    "import tinycudann; "
    "print('GPU_OK='+torch.cuda.get_device_name(0))"],
    env=C_ENV, capture_output=True, text=True, timeout=180)
if C_GPU_CHECK.returncode or 'GPU_OK=' not in C_GPU_CHECK.stdout:
    raise RuntimeError('Falha de ambiente CUDA/TCNN/Nerfstudio:\n'+
                       C_GPU_CHECK.stdout[-2500:]+'\n'+C_GPU_CHECK.stderr[-2500:])

# Conferir leitura pela implementação realmente instalada, não assumir splits.
C_DATAPARSER_PROBE = r'''
import sys
from pathlib import Path
from nerfstudio.configs.method_configs import method_configs
config=method_configs['nerfacto-big'].pipeline.datamanager.dataparser
config.data=Path(sys.argv[1])
config.downscale_factor=1
config.orientation_method='none'
config.center_method='none'
config.auto_scale_poses=False
config.scene_scale=1.5
parser=config.setup()
train=parser.get_dataparser_outputs(split='train')
val=parser.get_dataparser_outputs(split='val')
assert len(train.image_filenames)==95, len(train.image_filenames)
assert len(val.image_filenames)==10, len(val.image_filenames)
assert int(train.cameras.width[0])==800 and int(train.cameras.height[0])==450
assert int(val.cameras.width[0])==800 and int(val.cameras.height[0])==450
print('C_DATAPARSER_OK=95/10;TEST_NAO_SOLICITADO')
'''
C_PROBE = subprocess.run([str(C_PYTHON),'-c',C_DATAPARSER_PROBE,str(C_DATASET)],
                         env=C_ENV,text=True,capture_output=True,timeout=240)
if C_PROBE.returncode or 'C_DATAPARSER_OK=95/10' not in C_PROBE.stdout:
    raise RuntimeError('Dataparser Nerfacto não confirmou 95 TRAIN/10 VAL:\n'+
                       C_PROBE.stdout[-2500:]+'\n'+C_PROBE.stderr[-2500:])

print('='*90)
print(
    f'C V7 — Nerfacto-Big | {C_TOTAL_STEPS:,} iterações '
    '| nuScenes scene-0655 | 95/10/10'
)
print('Dataset idêntico à B V7:',C_DATASET)
print('SHA transforms:',C_CORE_HASHES['transforms.json'])
print('Treino/VAL/TEST:',C_SPLIT_COUNTS,'| Guarda mínima:',f'{C_MIN_GAP_MS:.2f} ms')
print('B selecionado em',C_B['selected_step'],'passos | checkpoint:',C_B_SELECTED)
print(C_GPU_CHECK.stdout.strip())
print('Checkpoint C avaliado apenas por VAL, com os passos:',C_CHECKPOINT_STEPS)
print('TEST permanece reservado; nenhuma cópia de checkpoints no Google Drive.')

# -------------------- DIRETÓRIOS NOVOS E MANIFESTO -------------------------
C_STAMP = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
C_ROOT = Path('/content/nuscenes_experiments') / ('C_V7_nerfacto_'+C_STAMP)
C_OUTPUT_ROOT = C_ROOT/'models'
C_LOG_ROOT = C_ROOT/'logs'
C_RESULTS_ROOT = C_ROOT/'results'
for p in (C_OUTPUT_ROOT, C_LOG_ROOT, C_RESULTS_ROOT):
    p.mkdir(parents=True, exist_ok=False)
C_EXPERIMENT = 'nuscenes_Av7_Hv3_nerfacto_big'
C_LOG = C_LOG_ROOT/'nerfacto_training.log'
C_MANIFEST = C_RESULTS_ROOT/'c_nerfacto_training_manifest.json'
C_VAL_CSV = C_RESULTS_ROOT/'c_validation_per_view.csv'
C_VAL_SUMMARY_CSV = C_RESULTS_ROOT/'c_validation_checkpoint_summary.csv'
C_SELECTION = C_RESULTS_ROOT/'checkpoint_validation_selection.json'
C_HELPER = C_ROOT/'prepare_checkpoint_config.py'

C_RECORD = {
    'status':'prepared',
    'experiment':'C_V7_nuScenes_NerfactoBig_95_10_10',
    'protocol':'A_V7_HV3_aligned_95_10_10',
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'dataset':str(C_DATASET),
    'dataset_sha256':C_CORE_HASHES,
    'splits':dict(zip(('train','val','test'),C_SPLIT_COUNTS)),
    'source_B_manifest':str(C_B_MANIFEST_PATH),
    'source_B_checkpoint':str(C_B_SELECTED),
    'source_B_checkpoint_sha256':C_B['selected_checkpoint_sha256'],
    'model':'nerfacto', 'preset':C_PRESET,
    'configuration':{
        'seed':C_SEED, 'training_steps':C_TOTAL_STEPS,
        'save_every':C_SAVE_EVERY,
        'validation_checkpoints':list(C_CHECKPOINT_STEPS),
        'train_num_rays_per_batch':C_TRAIN_RAYS,
        'eval_num_rays_per_batch':C_EVAL_RAYS,
        'camera_optimizer':'off', 'implementation':'tcnn',
        'training_from_scratch':True, 'lidar_initialization':False,
        'nerfstudio_data':{'downscale_factor':1,'orientation_method':'none',
                          'center_method':'none','auto_scale_poses':False,'scene_scale':1.5},
    },
    'test_accessed':False,
    'uploads_to_drive':False,
}

C_COMMAND = [str(C_NS_TRAIN),C_PRESET,
    '--data',str(C_DATASET),
    '--experiment-name',C_EXPERIMENT,
    '--output-dir',str(C_OUTPUT_ROOT),
    '--machine.seed',str(C_SEED),
    '--max-num-iterations',str(C_TOTAL_STEPS),
    '--steps-per-save',str(C_SAVE_EVERY),
    '--save-only-latest-checkpoint','False',
    '--steps-per-eval-image','1000',
    '--steps-per-eval-all-images','5000',
    '--vis','tensorboard',
    '--pipeline.datamanager.train-num-rays-per-batch',str(C_TRAIN_RAYS),
    '--pipeline.datamanager.eval-num-rays-per-batch',str(C_EVAL_RAYS),
    '--pipeline.model.camera-optimizer.mode','off',
    '--pipeline.model.implementation','tcnn',
    'nerfstudio-data','--downscale-factor','1',
    '--orientation-method','none','--center-method','none',
    '--auto-scale-poses','False','--scene-scale','1.5']
C_RECORD['command']=C_COMMAND
C_RECORD['artifacts']={'log':str(C_LOG),'results':str(C_RESULTS_ROOT)}
c_save(C_MANIFEST,C_RECORD)
print('Comando Nerfacto:\n'+shlex.join(C_COMMAND))

# ------------------------------ TREINAMENTO -------------------------------
C_CONFIG = None
C_MODEL_CONFIG = None
C_SELECTED_CHECKPOINT = None
C_SELECTED_STEP = None
C_BEST_CHECKPOINT = None
C_FINAL_REPORT = None

if C_RUN_TRAINING:
    training_start_wall = time.time()
    training_start = time.monotonic()
    proc=subprocess.Popen(C_COMMAND, env=C_ENV, cwd=C_ROOT,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True,bufsize=1)
    C_GPU_MONITOR=_TrainingGpuSampler(proc.pid)
    C_GPU_MONITOR.start()
    last_lines=deque(maxlen=80)
    last_reported_step=-1
    try:
        with C_LOG.open('w',encoding='utf-8') as output:
            output.write('COMMAND: '+shlex.join(C_COMMAND)+'\n')
            for line in proc.stdout:
                output.write(line)
                output.flush()
                last_lines.append(line.rstrip())
                if C_VERBOSE_TRAIN:
                    print(line,end='',flush=True)
                elif any(term in line.lower() for term in
                          ('saving config to','saving checkpoints to','traceback',
                           'out of memory','training finished')):
                    print(line.rstrip(),flush=True)
                else:
                    m=re.search(r'^\s*(\d+)\s+\([\d.]+%\)',line)
                    if m and int(m.group(1))>=last_reported_step+1000:
                        last_reported_step=int(m.group(1))
                        print('C Nerfacto: passo',last_reported_step,flush=True)
                if time.monotonic()-training_start > C_TRAIN_TIMEOUT_SECONDS:
                    proc.terminate()
                    raise TimeoutError('Nerfacto ultrapassou '+str(C_TRAIN_TIMEOUT_SECONDS)+' s.')
            ret=proc.wait(timeout=60)
    except KeyboardInterrupt:
        if proc.poll() is None:
            proc.terminate()
            try:proc.wait(timeout=15)
            except subprocess.TimeoutExpired:proc.kill();proc.wait()
        raise
    finally:
        if proc.poll() is None:
            proc.terminate()
            try:proc.wait(timeout=15)
            except subprocess.TimeoutExpired:proc.kill();proc.wait()
    C_GPU_MONITOR.stop()
    seconds=time.monotonic()-training_start
    if ret:
        C_RECORD.update(status='training_failed',return_code=ret,training_seconds=seconds)
        c_save(C_MANIFEST,C_RECORD)
        raise RuntimeError('Treinamento Nerfacto falhou. Veja '+str(C_LOG)+'\n'+
                           '\n'.join(last_lines))
    config_candidates=[p for p in C_OUTPUT_ROOT.rglob('config.yml')
                       if p.stat().st_mtime>=training_start_wall-10]
    if len(config_candidates)!=1:
        raise RuntimeError('Esperado um único config.yml novo do treino C: '+str(config_candidates))
    C_CONFIG=config_candidates[0]
    C_MODEL_CONFIG=C_CONFIG
    C_RUN_DIRECTORY=C_CONFIG.parent
    ckpt_dir=C_RUN_DIRECTORY/'nerfstudio_models'
    C_CKPTS={c_checkpoint_step(p):p for p in ckpt_dir.glob('step-*.ckpt')}
    if not set(C_CHECKPOINT_STEPS).issubset(C_CKPTS):
        raise RuntimeError('Checkpoints obrigatórios ausentes: '+
                           str(sorted(set(C_CHECKPOINT_STEPS)-set(C_CKPTS))))
    if max(C_CKPTS) != C_TOTAL_STEPS - 1:
      raise RuntimeError(
          f'Treinamento incompleto: checkpoint final esperado '
          f'{C_TOTAL_STEPS - 1}, encontrado {max(C_CKPTS)}.'
    )
    C_RECORD.update(status='trained_waiting_validation',peak_gpu_memory_mib_sampled=C_GPU_MONITOR.peak,
        peak_gpu_memory_sampling_interval_seconds=2,
        peak_gpu_memory_samples=C_GPU_MONITOR.count,

        training_seconds=round(seconds,3),
        training_minutes=round(seconds/60.0,3),
        config=str(C_CONFIG),
        checkpoint_final=str(C_CKPTS[C_TOTAL_STEPS - 1]),
        checkpoints=[{'step':step,'path':str(path),
                      'size_mib':round(path.stat().st_size/1048576.0,3)}
                     for step,path in sorted(C_CKPTS.items())],
        test_accessed=False,uploads_to_drive=False)
    c_save(C_MANIFEST,C_RECORD)
    print('Treino C finalizado em',round(seconds/60.,2),'min; checkpoints:',sorted(C_CKPTS))

# -------------------------- VALIDAÇÃO E SELEÇÃO ----------------------------
C_HELPER.write_text(r'''
import sys,yaml
from pathlib import Path
src=Path(sys.argv[1]);dest=Path(sys.argv[2]);step=int(sys.argv[3])
# Loader só é aplicado a um config.yml gerado pelo nosso ns-train local.
cfg=yaml.load(src.read_text(encoding='utf-8'), Loader=yaml.Loader)
assert cfg.method_name=='nerfacto',cfg.method_name
assert cfg.pipeline.model.camera_optimizer.mode=='off'
assert cfg.pipeline.model.implementation=='tcnn'
assert cfg.pipeline.datamanager.dataparser.downscale_factor==1
assert (cfg.get_checkpoint_dir()/f'step-{step:09d}.ckpt').is_file()
cfg.load_step=step
dest.write_text(yaml.dump(cfg),encoding='utf-8')
print('C_SELECTED_CONFIG_OK='+str(step))
'''.lstrip(), encoding='utf-8')

if C_RUN_VALIDATION:
    C_VAL_ROWS=[]
    C_VAL_SUMMARIES=[]
    expected={Path(rel).stem:rel for rel in C_SPLITS['val']}
    if len(expected)!=10:
        raise RuntimeError('Colisão dos nomes de VAL ao usar stem; parar para auditar.')
    expected_stems=set(expected)
    val_order=sorted(expected,key=lambda stem:int(C_FRAME_MAP[expected[stem]]['timestamp']))
    gt_pixel_hashes={}
    C_VAL_RENDER_DIRS={}

    for step in C_CHECKPOINT_STEPS:
        root=C_RESULTS_ROOT/'validation'/str(step)
        root.mkdir(parents=True,exist_ok=False)
        selected_cfg=root/'config.yml'
        output=subprocess.run([str(C_PYTHON),str(C_HELPER),str(C_CONFIG),
                               str(selected_cfg),str(step)],
                              env=C_ENV,text=True,capture_output=True,timeout=180)
        if output.returncode or f'C_SELECTED_CONFIG_OK={step}' not in output.stdout:
            raise RuntimeError('Não foi possível fixar checkpoint '+str(step)+'\n'+
                               output.stdout[-2000:]+'\n'+output.stderr[-2000:])
        render_root=root/'renders'
        render_log=root/'render.log'
        command=[str(C_NS_RENDER),'dataset','--load-config',str(selected_cfg),
                 '--output-path',str(render_root),'--data',str(C_DATASET),
                 '--split','val','--image-format','png',
                 '--rendered-output-names','rgb','gt-rgb']
        print('C render VAL checkpoint',step,flush=True)
        with render_log.open('w',encoding='utf-8') as out:
            result=subprocess.run(command,env=C_ENV,stdout=out,stderr=subprocess.STDOUT,
                                  text=True,timeout=C_RENDER_TIMEOUT_SECONDS)
        if result.returncode:
            raise RuntimeError('Render de VAL falhou no passo '+str(step)+'\n'+
                               render_log.read_text(errors='replace')[-4000:])
        log_text=re.sub(r'\x1b\[[0-9;]*m','',render_log.read_text(errors='replace'))
        if 'Done loading checkpoint from' not in log_text:
            raise RuntimeError('Render não confirmou carregamento do checkpoint '+str(step))
        m=re.findall(r'Done loading checkpoint from[^\n]*step-(\d+)\.ckpt',log_text)
        if m and int(m[-1])!=step:
            raise RuntimeError('O checkpoint renderizado diverge de '+str(step))
        subroot=render_root/'val'
        rendered={kind:list((subroot/kind).rglob('*.png'))
                  for kind in ('gt-rgb','rgb')}
        for kind,paths in rendered.items():
            if len(paths)!=10 or {p.stem for p in paths}!=expected_stems:
                raise RuntimeError(f'Renders {kind} incompletos em {step}: {len(paths)}')
        gts={p.stem:p for p in rendered['gt-rgb']}
        preds={p.stem:p for p in rendered['rgb']}
        C_VAL_RENDER_DIRS[step]=subroot
        values=[]
        for stem in val_order:
            truth=c_image_rgb(gts[stem])
            pred=c_image_rgb(preds[stem])
            original=c_image_rgb(C_DATASET/expected[stem])
            if truth.shape!=original.shape or np.max(np.abs(truth.astype(np.int16)-original.astype(np.int16)))>1:
                raise RuntimeError('GT da renderização não coincide com a entrada A V7: '+stem)
            current_sha=hashlib.sha256(truth.tobytes()).hexdigest()
            if step==C_CHECKPOINT_STEPS[0]:
                gt_pixel_hashes[stem]=current_sha
            elif current_sha!=gt_pixel_hashes[stem]:
                raise RuntimeError('Ground truth VAL foi alterada entre checkpoints: '+stem)
            psnr=c_psnr(truth,pred)
            values.append(psnr)
            C_VAL_ROWS.append({'method':'nerfacto','step':step,'image':stem,
                               'file_path':expected[stem],'psnr_db':round(psnr,7)})
        C_VAL_SUMMARIES.append({'step':step,'count':len(values),
                                'mean_psnr_db':float(np.mean(values)),
                                'min_psnr_db':float(np.min(values)),
                                'max_psnr_db':float(np.max(values))})
        print('  PSNR médio VAL:',f'{np.mean(values):.4f} dB',flush=True)

    c_csv(C_VAL_CSV,C_VAL_ROWS,['method','step','image','file_path','psnr_db'])
    c_csv(C_VAL_SUMMARY_CSV,C_VAL_SUMMARIES,
          ['step','count','mean_psnr_db','min_psnr_db','max_psnr_db'])
    C_BEST=min(C_VAL_SUMMARIES,key=lambda row:(-row['mean_psnr_db'],row['step']))
    C_SELECTED_STEP=int(C_BEST['step'])
    C_SELECTED_CHECKPOINT=C_CKPTS[C_SELECTED_STEP]
    C_BEST_CHECKPOINT=C_SELECTED_CHECKPOINT
    C_SELECTION_PAYLOAD={
        'method':'nerfacto', 'preset':C_PRESET,
        'dataset':str(C_DATASET),'dataset_hashes':C_CORE_HASHES,
        'selected_step':C_SELECTED_STEP,
        'selected_checkpoint':str(C_SELECTED_CHECKPOINT),
        'selected_checkpoint_sha256':c_sha(C_SELECTED_CHECKPOINT),
        'validation_checkpoints':C_VAL_SUMMARIES,
        'selection_criterion':'highest mean PSNR on 10 VAL frames; tie earlier step',
        'test_used_for_selection':False, 'test_rendered':False}
    c_save(C_SELECTION,C_SELECTION_PAYLOAD)
    C_RECORD.update(status='completed_selected_by_validation',
        selected_step=C_SELECTED_STEP,
        selected_checkpoint=str(C_SELECTED_CHECKPOINT),
        selected_checkpoint_sha256=c_sha(C_SELECTED_CHECKPOINT),
        validation_mean_psnr_db=C_BEST['mean_psnr_db'],
        validation={'best_by_val_psnr':{'step':C_SELECTED_STEP,
                   'mean_psnr_db':C_BEST['mean_psnr_db']},
                   'checkpoints':C_VAL_SUMMARIES,
                   'per_view_csv':str(C_VAL_CSV)},
        test_accessed=False,uploads_to_drive=False)
    c_save(C_MANIFEST,C_RECORD)
    C_FINAL_REPORT=C_RECORD

    if C_SHOW_GALLERY:
        selected_root=C_VAL_RENDER_DIRS[C_SELECTED_STEP]
        gallery=Image.new('RGB',(800,5*259),'white')
        draw=ImageDraw.Draw(gallery)
        original_map={p.stem:p for p in (selected_root/'gt-rgb').glob('*.png')}
        pred_map={p.stem:p for p in (selected_root/'rgb').glob('*.png')}
        for i,stem in enumerate(val_order[:5]):
            real=Image.open(original_map[stem]).convert('RGB').resize((400,225))
            synthetic=Image.open(pred_map[stem]).convert('RGB').resize((400,225))
            y=259*i
            gallery.paste(real,(0,y+30))
            gallery.paste(synthetic,(400,y+30))
            draw.text((6,y+6),f'VAL {i+1:02d} REAL',fill='black')
            draw.text((406,y+6),f'NERFACTO {C_SELECTED_STEP} passos',fill='black')
        gallery_file=C_RESULTS_ROOT/'val_gallery_5_examples.jpg'
        gallery.save(gallery_file,quality=92)
        try:
            from IPython.display import display
            display(gallery)
        except ImportError:
            pass

    print('\n'+'='*90)
    print('C V7 CONCLUÍDA — Nerfacto-Big selecionado por VAL')
    print('Checkpoint:',C_SELECTED_STEP,'| PSNR VAL:',f'{C_BEST["mean_psnr_db"]:.4f} dB')
    print('Config:',C_CONFIG)
    print('Checkpoint selecionado:',C_SELECTED_CHECKPOINT)
    print('Manifesto:',C_MANIFEST)
    print('Métricas por VAL:',C_VAL_CSV)
    print('TEST reservado; nenhum checkpoint enviado ao Google Drive.')
else:
    print('VALIDAÇÃO desativada: não prosseguir para avaliação final sem selecionar checkpoint.')

# Alias para versões FUTURAS de D–G, que precisarão ser adaptadas ao A/B/C V7.
C_NERFACTO_CONFIG=C_CONFIG
C_NERFACTO_CHECKPOINT=C_SELECTED_CHECKPOINT
C_RESULTS_DIR=C_RESULTS_ROOT
C_DATASET_SHA256=C_CORE_HASHES


In [ ]:
# =============================================================================
# CÉLULA D V7.1 — AVALIAÇÃO FINAL PAREADA nuScenes scene-0655
# SPLATFACTO (B V7) x NERFACTO-BIG (C V7) — alinhada ao iPhone H V3
# =============================================================================
# Ordem: 0 -> SETUP -> 1 -> A V7 -> B V7 -> C V7 -> ESTA D V7.
#
# - Não treina, não escolhe ou troca checkpoints.
# - Confere que B e C já escolheram seus checkpoints em VAL, antes do TEST.
# - Confere hashes reais da A V7, identidade dos checkpoints e parâmetros.
# - Corrige a divergência Nerfstudio config.data/datamanager.data/dataparser.data
#   somente na cópia de avaliação (mantém config original e pesos intocados).
# - Avalia exatamente as mesmas 10 imagens TEST em ambos os métodos.
# - Confere igualdade pixel a pixel entre as referências dos dois modelos.
# - Calcula PSNR, SSIM (skimage), LPIPS AlexNet (torchmetrics) por imagem.
# - Gera tabelas, métricas, PNGs individuais e comparações visuais.
# - Mantém TODOS os arquivos em /content; não envia checkpoints ao Drive.
# - Não executa YOLO nesta célula: a futura F V7 reutilizará os PNGs.
#
# ATENÇÃO: D CONSOME O TESTE FINAL. Se os resultados motivarem ajustes,
# reserve um NOVO conjunto para uma avaliação confirmatória posterior.
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import math
import os
import re
import shlex
import shutil
import subprocess
import time
import zipfile

import numpy as np
from PIL import Image, ImageDraw

# ------------------------------- CONFIGURAÇÃO ------------------------------
D_SHOW_GALLERY = True
D_REQUIRE_LPIPS = True
D_RENDER_TIMEOUT_SECONDS = 3600
D_LPIPS_TIMEOUT_SECONDS = 1200
D_EXPECTED_SPLITS = {'train':95, 'val':10, 'test':10}
D_EXPECTED_A_TAG = 'scene0655_hv3_protocol_Av7_95_10_10'
D_EXPECTED_A_PROFILE = 'nuscenes_front_samples_train_sweeps_holdout_10_10_v7'
D_EXPECTED_STEPS = {
    'splatfacto': (4000, 5000, 6000, 8000, 14999),
    'nerfacto': (1000, 2000, 3000, 4000, 5000, 6000, 8000, 10000, 12000, 14999),
}
D_RESOLUTION = (800, 450)
D_PYTHON = Path(globals().get('TCC_PYTHON', '/content/tcc_env/bin/python'))
D_BIN = D_PYTHON.parent
D_NS_RENDER = D_BIN / 'ns-render'
D_CUDA = Path('/usr/local/cuda-12.8')


def d_sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as inp:
        for chunk in iter(lambda: inp.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def d_read(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def d_write(path, item):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(item, ensure_ascii=False, indent=2, default=str)+'\n', encoding='utf-8')


def d_csv(path, rows, columns):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', newline='', encoding='utf-8') as output:
        writer = csv.DictWriter(output, fieldnames=list(columns), extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)


def d_step(path):
    match = re.fullmatch(r'step-(\d+)\.ckpt', Path(path).name)
    if not match:
        raise RuntimeError('Nome de checkpoint inválido: '+str(path))
    return int(match.group(1))


def d_rgb(path):
    with Image.open(path) as image:
        return np.asarray(image.convert('RGB'), dtype=np.uint8)


def d_psnr(a, b):
    if a.shape != b.shape or a.shape != (D_RESOLUTION[1], D_RESOLUTION[0], 3):
        raise RuntimeError(f'Dimensões inesperadas: {a.shape}, {b.shape}')
    mse = float(np.mean(((a.astype(np.float64)-b.astype(np.float64))/255.)**2))
    return float(-10. * math.log10(max(mse, 1e-15)))


def d_ssim(a, b):
    from skimage.metrics import structural_similarity
    return float(structural_similarity(a, b, data_range=255, channel_axis=-1,
                 gaussian_weights=True, sigma=1.5, use_sample_covariance=False))


def d_check_under(path, parent):
    path, parent = Path(path).resolve(), Path(parent).resolve()
    if not path.is_relative_to(parent):
        raise RuntimeError('Caminho fora da pasta autorizada: '+str(path))
    return path


def d_expected_rendered(root, split, kind, source_files):
    images = list((Path(root) / split / kind).rglob('*.png'))
    expected = {Path(x).stem for x in source_files}
    mapped = {p.stem: p for p in images}
    if (len(images) != len(mapped) or len(mapped) != len(expected)
            or set(mapped) != expected):
        raise RuntimeError(f'Renders incompletos / duplicados: {root}/{split}/{kind}; '
                           f'esperado={len(expected)}, recebido={len(images)}')
    return mapped


def d_env():
    env = dict(globals().get('TCC_SUBPROCESS_ENV', os.environ))
    for name in ('PYTHONHOME', 'PYTHONPATH', 'PYTHONUSERBASE', 'PIP_TARGET'):
        env.pop(name, None)
    env.update({
        'PATH': f'{D_BIN}:{D_CUDA / "bin"}:'+os.environ.get('PATH',''),
        'VIRTUAL_ENV':str(D_BIN.parent),
        'PYTHONNOUSERSITE':'1',
        'PYTHONUNBUFFERED':'1',
        'QT_QPA_PLATFORM':'offscreen',
        'CUDA_HOME':str(D_CUDA),
        'CUDA_PATH':str(D_CUDA),
        'CUDACXX':str(D_CUDA / 'bin' / 'nvcc'),
    })
    if (D_CUDA/'lib64').is_dir():
        env['LD_LIBRARY_PATH']=str(D_CUDA/'lib64')+':'+env.get('LD_LIBRARY_PATH','')
    return env


def d_run_log(command, log_path, env, timeout):
    """Executa sem acumular logs em RAM, com timeout real por communicate()."""
    command = list(map(str, command))
    print('Executando:', shlex.join(command), flush=True)
    started = time.perf_counter()
    with Path(log_path).open('w', encoding='utf-8') as stream:
        stream.write('COMMAND: '+shlex.join(command)+'\n\n')
        stream.flush()
        try:
            result = subprocess.run(command, stdout=stream, stderr=subprocess.STDOUT,
                                    env=env, timeout=timeout, check=False)
        except subprocess.TimeoutExpired:
            raise TimeoutError('Timeout em '+str(log_path))
    seconds = time.perf_counter() - started
    if result.returncode:
        tail = Path(log_path).read_text(errors='replace')[-5000:]
        raise RuntimeError(f'Falha (exit {result.returncode}), log={log_path}:\n'+tail)
    return seconds


def d_check_render_checkpoint(log, expected):
    log = re.sub(r'\x1b\[[0-9;]*m', '', Path(log).read_text(errors='replace'))
    lines = re.findall(r'Done loading checkpoint from\s+([^\r\n]+)', log)
    if not lines:
        raise RuntimeError('ns-render não confirmou carregamento do checkpoint.')
    printed = lines[-1].strip().strip('"\'')
    match = re.search(r'step-(\d+)\.ckpt', printed)
    if match and int(match.group(1)) != d_step(expected):
        raise RuntimeError(f'Checkpoint carregado não é o escolhido: {printed}')
    # O Rich pode abreviar caminhos extensos; conferir pelo menos prefixo íntegro.
    if match:
        prefix = printed.split('/step-')[0]
        if prefix and len(prefix)>35 and not str(Path(expected).resolve()).startswith(prefix):
            raise RuntimeError('Diretório do checkpoint renderizado não confere: '+printed)


def d_validation_best(rows, allowed):
    """Audita a seleção efetuada antes do teste (desempate no passo anterior)."""
    found = {int(row['step']):float(row['mean_psnr_db']) for row in rows}
    if set(found) != set(allowed) or not all(math.isfinite(v) for v in found.values()):
        raise RuntimeError('Histórico de checkpoints VAL incompleto ou inválido.')
    return min(found, key=lambda s:(-found[s],s)), found


# ----------------------------- PREFLIGHT A/B/C -----------------------------
if globals().get('D_TEST_CONSUMED', False):
    raise RuntimeError('A célula D já consumiu TEST nesta sessão. Não reexecutar automaticamente.')
if not globals().get('ENVIRONMENT_VALIDATED', False):
    raise RuntimeError('Execute a Célula 1 antes de D.')
if 'OUTPUT_DATASET' not in globals():
    raise RuntimeError('Execute A V7 nesta sessão; OUTPUT_DATASET ausente.')
D_DATASET = Path(globals()['OUTPUT_DATASET']).resolve()
D_A_EXPECTED = Path('/content/nuscenes_nerfstudio_'+D_EXPECTED_A_TAG).resolve()
if D_DATASET != D_A_EXPECTED:
    raise RuntimeError(f'Não é o dataset A V7. Atual={D_DATASET}; esperado={D_A_EXPECTED}')
for exe in (D_PYTHON,D_NS_RENDER):
    if not exe.is_file():
        raise FileNotFoundError('Dependência ausente: '+str(exe))
for name in ('B_MANIFEST','C_MANIFEST'):
    if name not in globals():
        raise RuntimeError('Execute B V7 e C V7 nesta sessão; falta '+name)
D_B_MANIFEST_PATH = Path(globals()['B_MANIFEST']).resolve()
D_C_MANIFEST_PATH = Path(globals()['C_MANIFEST']).resolve()
if not D_B_MANIFEST_PATH.is_file() or not D_C_MANIFEST_PATH.is_file():
    raise FileNotFoundError('Manifestos B/C V7 ausentes. Execute B e C antes de D.')

D_TF_PATH = D_DATASET/'transforms.json'
D_A_MANIFEST_PATH = D_DATASET/'dataset_manifest.json'
D_A_IDENTITY_PATH = D_DATASET/'dataset_identity.json'
D_PAIRS_PATH = D_DATASET/'eval_pairs.json'
for path in (D_TF_PATH,D_A_MANIFEST_PATH,D_A_IDENTITY_PATH,D_PAIRS_PATH):
    if not path.is_file():
        raise FileNotFoundError('Artefato da A V7 não encontrado: '+str(path))
D_TF = d_read(D_TF_PATH)
D_A = d_read(D_A_MANIFEST_PATH)
D_IDENTITY = d_read(D_A_IDENTITY_PATH)
D_B = d_read(D_B_MANIFEST_PATH)
D_C = d_read(D_C_MANIFEST_PATH)

if D_A.get('experiment_tag') != D_EXPECTED_A_TAG or D_A.get('dataset_profile') != D_EXPECTED_A_PROFILE:
    raise RuntimeError('A não é a versão V7 alinhada à H V3.')
if D_IDENTITY.get('protocol') != 'A_V7_HV3-aligned':
    raise RuntimeError('dataset_identity.json não identifica A V7.')
if D_A.get('scene',{}).get('name') != 'scene-0655' or D_A.get('cameras',{}).get('channels') != ['CAM_FRONT']:
    raise RuntimeError('Somente scene-0655 / CAM_FRONT são aceitos.')
if D_A.get('protocol_v7',{}).get('normalization_uses_only_train') is not True:
    raise RuntimeError('Normalização A não é TRAIN-only.')
if D_A.get('geometry',{}).get('geometry_inductive') is not True:
    raise RuntimeError('Geometria A não é independente de VAL/TEST.')
if D_A.get('lidar',{}).get('train_split_only') is not True:
    raise RuntimeError('PLY LiDAR não é exclusivamente de TRAIN.')
if D_TF.get('camera_model')!='OPENCV' or D_TF.get('orientation_override')!='none':
    raise RuntimeError('Câmera/orientação não conferem com A V7.')

D_SPLITS = {split:list(D_TF.get(split+'_filenames',[])) for split in ('train','val','test')}
D_COUNTS = {key:len(value) for key,value in D_SPLITS.items()}
if D_COUNTS != D_EXPECTED_SPLITS:
    raise RuntimeError(f'Dataset {D_COUNTS} diferente de 95/10/10.')
D_FRAME_MAP = {frame['file_path']:frame for frame in D_TF.get('frames',[])}
D_FILES = [rel for split in ('train','val','test') for rel in D_SPLITS[split]]
if (len(D_FILES)!=115 or len(set(D_FILES))!=115 or
        len(D_FRAME_MAP)!=115 or set(D_FILES)!=set(D_FRAME_MAP)):
    raise RuntimeError('transforms.json contém imagens duplicadas, ausentes ou splits sobrepostos.')
for split,paths in D_SPLITS.items():
    for rel in paths:
        frame = D_FRAME_MAP[rel]
        if (frame.get('dataset_split')!=split or frame.get('camera_channel')!='CAM_FRONT'
            or (int(frame['w']),int(frame['h']))!=D_RESOLUTION):
            raise RuntimeError('Câmera, resolução ou split inválido: '+rel)
        if not d_check_under(D_DATASET/rel, D_DATASET).is_file():
            raise FileNotFoundError('Imagem A V7 ausente: '+rel)
# Nenhuma imagem TEST é decodificada antes de auditar a seleção dos checkpoints.
D_TEST_PAIRS = d_read(D_PAIRS_PATH)
if len(D_TEST_PAIRS)!=10 or any(p.get('split')!='test' for p in D_TEST_PAIRS):
    raise RuntimeError('eval_pairs.json precisa conter 10 imagens TEST.')
if {p.get('nerfstudio_file_path') for p in D_TEST_PAIRS} != set(D_SPLITS['test']):
    raise RuntimeError('eval_pairs.json e transforms.json possuem TEST diferentes.')
D_LIDAR_REL = D_TF.get('ply_file_path')
if not D_LIDAR_REL:
    raise RuntimeError('A V7 não registra PLY LiDAR.')
D_LIDAR = d_check_under(D_DATASET/D_LIDAR_REL,D_DATASET)
if not D_LIDAR.is_file():
    raise FileNotFoundError('PLY LiDAR da A não encontrado.')
D_HASHES = {name:d_sha(D_DATASET/name) for name in
            ('transforms.json','eval_pairs.json','dataset_manifest.json')}
D_HASHES[D_LIDAR_REL] = d_sha(D_LIDAR)
if D_HASHES != D_IDENTITY.get('core_sha256',{}):
    raise RuntimeError('A V7 foi alterada desde sua criação; hashes divergentes.')

if D_B.get('status')!='completed_selected_by_validation' or D_C.get('status')!='completed_selected_by_validation':
    raise RuntimeError('B e C devem completar a seleção por VAL antes de D.')
if (D_B.get('protocol')!='A_V7_HV3_aligned_95_10_10' or
        D_C.get('protocol')!='A_V7_HV3_aligned_95_10_10'):
    raise RuntimeError('Protocolos B/C diferentes da A V7.')
if D_B.get('test_accessed',True) or D_C.get('test_accessed',True):
    raise RuntimeError('B/C declararam uso prematuro do TEST.')
if (Path(D_B.get('dataset','/missing')).resolve()!=D_DATASET or
        Path(D_C.get('dataset','/missing')).resolve()!=D_DATASET):
    raise RuntimeError('B ou C foram treinadas em outro dataset.')
if D_B.get('dataset_sha256')!=D_HASHES or D_C.get('dataset_sha256')!=D_HASHES:
    raise RuntimeError('B ou C têm hashes diferentes do dataset A V7.')
if D_B.get('split_counts')!=D_COUNTS or D_C.get('splits')!=D_COUNTS:
    raise RuntimeError('B ou C registram outro split.')
if Path(D_C.get('source_B_manifest','/missing')).resolve()!=D_B_MANIFEST_PATH:
    raise RuntimeError('C não referencia exatamente o manifesto da B utilizada.')
if D_C.get('source_B_checkpoint_sha256')!=D_B.get('selected_checkpoint_sha256'):
    raise RuntimeError('C referencia outro checkpoint da B.')

D_MODEL = {}
for method, record, allowed in (
    ('splatfacto',D_B,D_EXPECTED_STEPS['splatfacto']),
    ('nerfacto',D_C,D_EXPECTED_STEPS['nerfacto']),
):
    selected_step = int(record['selected_step'])
    checkpoint = Path(record['selected_checkpoint']).resolve()
    config = Path(record['config']).resolve()
    if not checkpoint.is_file() or not config.is_file():
        raise FileNotFoundError(method+': config ou checkpoint não encontrado.')
    if selected_step!=d_step(checkpoint) or selected_step not in allowed:
        raise RuntimeError(method+': passo do checkpoint incompatível.')
    if d_sha(checkpoint)!=record['selected_checkpoint_sha256']:
        raise RuntimeError(method+': hash do checkpoint não confere.')
    summary=record['validation_summary'] if method=='splatfacto' else record['validation']['checkpoints']
    expected_best,scores=d_validation_best(summary,allowed)
    if selected_step!=expected_best:
        raise RuntimeError(method+': checkpoint selecionado não é o melhor registrado em VAL.')
    if not math.isclose(float(record['validation_mean_psnr_db']),scores[selected_step],
                        abs_tol=1e-6,rel_tol=0):
        raise RuntimeError(method+': PSNR de VAL do selecionado diverge do manifesto.')
    D_MODEL[method]={'method':method, 'config':config,'checkpoint':checkpoint,
                     'step':selected_step,'val_psnr_db':scores[selected_step],
                     'checkpoint_sha256':record['selected_checkpoint_sha256']}

if 'C_BEST_CHECKPOINT' in globals() and globals()['C_BEST_CHECKPOINT'] is not None:
    if Path(globals()['C_BEST_CHECKPOINT']).resolve()!=D_MODEL['nerfacto']['checkpoint']:
        raise RuntimeError('Checkpoint C em memória diverge do manifesto.')
if 'B_BEST_CHECKPOINT' in globals() and globals()['B_BEST_CHECKPOINT'] is not None:
    if Path(globals()['B_BEST_CHECKPOINT']).resolve()!=D_MODEL['splatfacto']['checkpoint']:
        raise RuntimeError('Checkpoint B em memória diverge do manifesto.')

D_TEST_NAMES=sorted(D_SPLITS['test'],key=lambda p:(int(D_FRAME_MAP[p]['timestamp']),p))
if len({Path(p).stem for p in D_TEST_NAMES})!=10:
    raise RuntimeError('Nomes de TEST colidem ao remover a extensão.')
D_ENV=d_env()

print('='*100)
print('D V7 — PRÉ-AVALIAÇÃO CONFIRMADA / PRIMEIRO ACESSO A TEST')
print('Dataset A V7:',D_DATASET)
print('Splits:',D_COUNTS,'| câmera CAM_FRONT | RGB 800x450')
print('SHA do manifesto A:',D_HASHES['dataset_manifest.json'])
for method in ('splatfacto','nerfacto'):
    rec=D_MODEL[method]
    print(method+': checkpoint',rec['step'],'| PSNR VAL',f"{rec['val_psnr_db']:.5f} dB")
print('Ambos os checkpoints foram escolhidos sem consultar TEST. Será avaliado agora.')

# ------------------------- DIRETÓRIOS DESTA EXECUÇÃO ----------------------
D_STAMP=datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
D_ROOT=Path('/content/nuscenes_experiments')/('D_V7_test_pareado_'+D_STAMP)
D_RESULTS=D_ROOT/'results'
D_LOGS=D_ROOT/'logs'
D_RENDERS=D_ROOT/'renders'
D_CONFIGS=D_ROOT/'configs'
D_FIGURES=D_RESULTS/'figures'
D_IMAGES=D_FIGURES/'test'
for folder in (D_RESULTS,D_LOGS,D_RENDERS,D_CONFIGS):
    folder.mkdir(parents=True,exist_ok=False)
for method in ('real','splatfacto','nerfacto'):
    (D_IMAGES/method).mkdir(parents=True,exist_ok=False)
(D_FIGURES/'rgb_triplets').mkdir(parents=True,exist_ok=False)

# O yaml do Nerfstudio é Python-tagged; somente ler arquivos produzidos pelo
# ns-train B e C no diretório desta mesma sessão. Executar no tcc_env isolado.
D_CONFIG_HELPER=D_ROOT/'fix_selected_checkpoint.py'
D_CONFIG_HELPER.write_text(r'''
import json, sys, yaml
from pathlib import Path

# Este script carrega APENAS config.yml das execuções B/C desta sessão,
# previamente verificadas pelos hashes da célula D.
src = Path(sys.argv[1]).resolve()
dest = Path(sys.argv[2]).resolve()
ckpt = Path(sys.argv[3]).resolve()
method = sys.argv[4]
step = int(sys.argv[5])
dataset = Path(sys.argv[6]).resolve()

cfg = yaml.load(src.read_text(encoding='utf-8'), Loader=yaml.Loader)
assert cfg.method_name == method, (cfg.method_name, method)
assert cfg.pipeline.model.camera_optimizer.mode == 'off'
dm = cfg.pipeline.datamanager
parser = dm.dataparser
assert parser.downscale_factor == 1
assert parser.orientation_method == 'none'
assert parser.center_method == 'none'
assert parser.auto_scale_poses is False
assert cfg.get_checkpoint_dir().resolve() == ckpt.parent.resolve(), (
    cfg.get_checkpoint_dir(), ckpt.parent
)
assert ckpt.is_file() and ckpt.name == f'step-{step:09d}.ckpt'

# Com --data na CLI do ns-train, o caminho efetivo pode ficar em
# TrainerConfig.data ou DataManagerConfig.data, enquanto o campo
# dataparser.data retém o padrão do preset. Esse é o erro da D V7 original.
def location(value):
    if value is None:
        return None
    return str(Path(value).expanduser().resolve())

paths_original = {
    'trainer.data': location(getattr(cfg, 'data', None)),
    'datamanager.data': location(getattr(dm, 'data', None)),
    'dataparser.data': location(getattr(parser, 'data', None)),
}
print('D_CONFIG_DATA_ORIGINAL=' + json.dumps(paths_original), flush=True)

# A célula D externa já validou que o comando de treino B/C utilizou
# --data exatamente neste dataset e que os hashes A/B/C coincidem.
# Fixar explicitamente o mesmo dataset em TODOS os níveis existentes
# somente no config selecionado para ns-render, sem mudar src ou pesos.
if hasattr(cfg, 'data'):
    cfg.data = dataset
if hasattr(dm, 'data'):
    dm.data = dataset
parser.data = dataset
cfg.load_step = step

dest.parent.mkdir(parents=True, exist_ok=True)
dest.write_text(yaml.dump(cfg), encoding='utf-8')

# Auditoria da cópia final usada para renderização.
check = yaml.load(dest.read_text(encoding='utf-8'), Loader=yaml.Loader)
assert check.method_name == method
assert int(check.load_step) == step
assert Path(check.pipeline.datamanager.dataparser.data).resolve() == dataset
if hasattr(check, 'data'):
    assert Path(check.data).resolve() == dataset
if hasattr(check.pipeline.datamanager, 'data'):
    if check.pipeline.datamanager.data is not None:
        assert Path(check.pipeline.datamanager.data).resolve() == dataset
assert check.get_checkpoint_dir().resolve() == ckpt.parent.resolve()
print('D_CONFIG_OK=' + method + ':' + str(step), flush=True)
'''.lstrip(),encoding='utf-8')

# A/B/C já comprovaram os hashes e a seleção por VAL. Confere ainda
# --data nos comandos de treino ANTES de normalizar a cópia do YAML.
for _method, _manifest in (('splatfacto', D_B), ('nerfacto', D_C)):
    _command = list(map(str, _manifest.get('command', [])))
    if _command.count('--data') != 1:
        raise RuntimeError(_method + ': manifesto não registra exatamente um --data.')
    _source_data = Path(_command[_command.index('--data') + 1]).resolve()
    if _source_data != D_DATASET:
        raise RuntimeError(_method + ': o comando ORIGINAL treinou em outro dataset: '
                           + str(_source_data) + ' != ' + str(D_DATASET))

for method,record in D_MODEL.items():
    selected_cfg=D_CONFIGS/(method+'_selected.yml')
    cmd=[str(D_PYTHON),str(D_CONFIG_HELPER),str(record['config']),str(selected_cfg),
         str(record['checkpoint']),method,str(record['step']),str(D_DATASET)]
    run=subprocess.run(cmd,env=D_ENV,text=True,capture_output=True,timeout=180)
    if run.returncode or f'D_CONFIG_OK={method}:{record["step"]}' not in run.stdout:
        raise RuntimeError('Config de avaliação do '+method+' inválido:\n'+run.stdout[-4000:]+'\n'+run.stderr[-4000:])
    record['selected_config']=selected_cfg

# -------------------------- TESTE FINAL PAREADO ---------------------------
D_RENDER_STAT={}
D_RENDER_MAP={}
for method,record in D_MODEL.items():
    renderroot=D_RENDERS/method
    log=D_LOGS/(method+'_test_render.log')
    command=[str(D_NS_RENDER),'dataset','--load-config',str(record['selected_config']),
             '--output-path',str(renderroot),'--data',str(D_DATASET),
             '--split','test','--image-format','png','--rendered-output-names','rgb','gt-rgb']
    print('\nRenderizando TEST',method,'checkpoint',record['step'],flush=True)
    seconds=d_run_log(command,log,D_ENV,D_RENDER_TIMEOUT_SECONDS)
    d_check_render_checkpoint(log,record['checkpoint'])
    D_RENDER_STAT[method]={'total_process_seconds':seconds,
                           'seconds_per_image_including_overhead':seconds/10,
                           'effective_images_per_second_including_overhead':10/seconds,
                           'not_renderer_fps':True}
    D_RENDER_MAP[method]={
        'rgb':d_expected_rendered(renderroot,'test','rgb',D_TEST_NAMES),
        'gt':d_expected_rendered(renderroot,'test','gt-rgb',D_TEST_NAMES),
    }
    print('Processo ns-render:',f'{seconds:.2f} segundos (inclui carregar e salvar).')

D_TEST_ROWS=[]
D_TRIPLET_MANIFEST=[]
D_LPIPS_PAIRS=[]
D_GALLERY_ROWS=[]
for index,rel in enumerate(D_TEST_NAMES,1):
    stem=Path(rel).stem
    a=d_rgb(D_RENDER_MAP['splatfacto']['gt'][stem])
    b=d_rgb(D_RENDER_MAP['nerfacto']['gt'][stem])
    if a.shape!=(450,800,3) or not np.array_equal(a,b):
        raise RuntimeError('GT divergente entre os métodos na imagem '+rel)
    # Mesmo critério de verificação do GT utilizado em B/C VAL.
    orig=d_rgb(D_DATASET/rel)
    if orig.shape!=a.shape or np.max(np.abs(orig.astype(np.int16)-a.astype(np.int16)))>1:
        raise RuntimeError('GT renderizado difere da imagem original A V7: '+rel)
    pred={method:d_rgb(D_RENDER_MAP[method]['rgb'][stem])
          for method in ('splatfacto','nerfacto')}
    for name,image in pred.items():
        if image.shape!=a.shape:
            raise RuntimeError('Render com tamanho inesperado '+name+'/'+rel)
    scores={method:(d_psnr(a,im),d_ssim(a,im)) for method,im in pred.items()}
    row={'test_view':index,'image':stem,'file_path':rel,
         'frame_timestamp_us':int(D_FRAME_MAP[rel]['timestamp']),
         'source_type':D_FRAME_MAP[rel].get('source_type'),
         'splatfacto_psnr_db':scores['splatfacto'][0],
         'splatfacto_ssim':scores['splatfacto'][1],
         'nerfacto_psnr_db':scores['nerfacto'][0],
         'nerfacto_ssim':scores['nerfacto'][1],
         'delta_psnr_nerf_minus_gs_db':scores['nerfacto'][0]-scores['splatfacto'][0],
         'delta_ssim_nerf_minus_gs':scores['nerfacto'][1]-scores['splatfacto'][1]}
    D_TEST_ROWS.append(row)
    paths={}
    for name,image in [('real',a),('splatfacto',pred['splatfacto']),('nerfacto',pred['nerfacto'])]:
        dest=D_IMAGES/name/(stem+'.png')
        Image.fromarray(image).save(dest,format='PNG')
        paths[name]=dest
    D_LPIPS_PAIRS.append({'test_view':index,'image':stem,
                          'real':str(paths['real']),'splatfacto':str(paths['splatfacto']),
                          'nerfacto':str(paths['nerfacto'])})
    D_TRIPLET_MANIFEST.append({'test_view':index,'image':stem,'file_path':rel,
        'real':paths['real'].relative_to(D_RESULTS).as_posix(),
        'splatfacto':paths['splatfacto'].relative_to(D_RESULTS).as_posix(),
        'nerfacto':paths['nerfacto'].relative_to(D_RESULTS).as_posix()})
    if index in (1,2,3,4,5,6,7,8,9,10):
        D_GALLERY_ROWS.append((index,stem,a,pred['splatfacto'],pred['nerfacto']))
    print(f'  TEST {index:02d}: Splatfacto {scores["splatfacto"][0]:.3f} dB | '
          f'Nerfacto {scores["nerfacto"][0]:.3f} dB')

# --------------------- LPIPS/AlexNet no ambiente isolado ------------------
D_LPIPS_INPUT=D_ROOT/'lpips_pairs_input.json'
D_LPIPS_OUTPUT=D_RESULTS/'lpips_alex_by_image.json'
D_LPIPS_HELPER=D_ROOT/'compute_lpips_alex.py'
d_write(D_LPIPS_INPUT,D_LPIPS_PAIRS)
D_LPIPS_HELPER.write_text(r'''
import json,sys,torch
import numpy as np
from PIL import Image
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity
pairs=json.load(open(sys.argv[1],encoding='utf-8'))
metric=LearnedPerceptualImagePatchSimilarity(net_type='alex',normalize=True).cpu().eval()
rows=[]
def tensor(path):
    with Image.open(path) as im:
        rgb=np.asarray(im.convert('RGB'),dtype=np.uint8)
    if rgb.shape!=(450,800,3):raise RuntimeError('Shape incorreto '+str(rgb.shape))
    return torch.from_numpy(rgb.copy()).permute(2,0,1).unsqueeze(0).float()/255.
with torch.inference_mode():
    for item in pairs:
        ref=tensor(item['real'])
        record={'test_view':item['test_view'],'image':item['image']}
        for method in ('splatfacto','nerfacto'):
            pred=tensor(item[method])
            metric.reset()
            record[method+'_lpips_alex']=float(metric(pred,ref).item())
        rows.append(record)
with open(sys.argv[2],'w',encoding='utf-8') as out:
    json.dump(rows,out,indent=2,ensure_ascii=False)
print('D_LPIPS_OK='+str(len(rows)))
'''.lstrip(),encoding='utf-8')

D_LPIPS_LOG=D_LOGS/'lpips_alex.log'
try:
    d_lpips_time=d_run_log([str(D_PYTHON),str(D_LPIPS_HELPER),
                            str(D_LPIPS_INPUT),str(D_LPIPS_OUTPUT)],
                            D_LPIPS_LOG,D_ENV,D_LPIPS_TIMEOUT_SECONDS)
    lpips_rows=d_read(D_LPIPS_OUTPUT)
    if len(lpips_rows)!=10 or {r['image'] for r in lpips_rows}!={r['image'] for r in D_TEST_ROWS}:
        raise RuntimeError('LPIPS não processou exatamente dez imagens de teste.')
    lpips_map={r['image']:r for r in lpips_rows}
    for row in D_TEST_ROWS:
        lp=lpips_map[row['image']]
        for method in ('splatfacto','nerfacto'):
            value=float(lp[method+'_lpips_alex'])
            if not math.isfinite(value):
                raise RuntimeError('LPIPS não finito na vista '+row['image'])
            row[method+'_lpips_alex']=value
        row['delta_lpips_nerf_minus_gs']=row['nerfacto_lpips_alex']-row['splatfacto_lpips_alex']
except Exception as exc:
    if D_REQUIRE_LPIPS:
        raise RuntimeError('LPIPS é obrigatório: veja o log '+str(D_LPIPS_LOG)) from exc
    print('AVISO: LPIPS indisponível, métricas incompletas:',str(exc))
    d_lpips_time=None

# ----------------------------- RESUMO E TABELAS ----------------------------
def d_mean(key):
    numbers=[float(row[key]) for row in D_TEST_ROWS if key in row]
    return float(np.mean(numbers)) if numbers else None

D_SUMMARY={}
for method in ('splatfacto','nerfacto'):
    info=D_MODEL[method]
    D_SUMMARY[method]={
        'selected_step':info['step'],
        'selection_val_psnr_db':info['val_psnr_db'],
        'test_psnr_db':d_mean(method+'_psnr_db'),
        'test_ssim':d_mean(method+'_ssim'),
        'test_lpips_alex':d_mean(method+'_lpips_alex'),
        'training_seconds':(D_B if method=='splatfacto' else D_C).get('training_seconds'),
        **D_RENDER_STAT[method],
    }
D_SUMMARY['delta_nerfacto_minus_splatfacto']={
    'psnr_db':D_SUMMARY['nerfacto']['test_psnr_db']-D_SUMMARY['splatfacto']['test_psnr_db'],
    'ssim':D_SUMMARY['nerfacto']['test_ssim']-D_SUMMARY['splatfacto']['test_ssim'],
    'lpips_alex':(D_SUMMARY['nerfacto']['test_lpips_alex']-D_SUMMARY['splatfacto']['test_lpips_alex'])
        if D_SUMMARY['nerfacto']['test_lpips_alex'] is not None else None,
}
D_METRICS_CSV=D_RESULTS/'metricas_test_pareadas.csv'
D_METHODS_CSV=D_RESULTS/'resumo_metodos.csv'
D_TRIPLETS_JSON=D_RESULTS/'test_image_manifest.json'
d_csv(D_METRICS_CSV,D_TEST_ROWS,list(D_TEST_ROWS[0]))
summary_columns=['method','selected_step','selection_val_psnr_db','test_psnr_db',
                 'test_ssim','test_lpips_alex','training_seconds',
                 'total_process_seconds','seconds_per_image_including_overhead',
                 'effective_images_per_second_including_overhead','not_renderer_fps']
d_csv(D_METHODS_CSV,[{'method':name,**D_SUMMARY[name]}
                     for name in ('splatfacto','nerfacto')],summary_columns)
d_write(D_TRIPLETS_JSON,D_TRIPLET_MANIFEST)

# Galeria em pares: uma imagem completa de cada vista (para inspecionar qualidade).
D_GALLERY_DIR=D_FIGURES/'rgb_triplets'
for idx,stem,real,gs,nf in D_GALLERY_ROWS:
    h,w=real.shape[:2]
    combined=Image.new('RGB',(w*3,h+32),'white')
    drawer=ImageDraw.Draw(combined)
    for j,(array,label) in enumerate(zip((real,gs,nf),('REAL','SPLATFACTO','NERFACTO'))):
        combined.paste(Image.fromarray(array),(j*w,32))
        drawer.text((j*w+10,8),f'TEST {idx:02d} {label}',fill='black')
    combined.save(D_GALLERY_DIR/(stem+'.jpg'),format='JPEG',quality=94)

# Galeria geral reduzida, 10 linhas para revisão rápida.
thumb_w,thumb_h,label_h=400,225,28
contact=Image.new('RGB',(3*thumb_w,len(D_GALLERY_ROWS)*(thumb_h+label_h)),(245,245,245))
drawer=ImageDraw.Draw(contact)
for i,(idx,stem,real,gs,nf) in enumerate(D_GALLERY_ROWS):
    y=i*(thumb_h+label_h)
    for j,(arr,label) in enumerate(zip((real,gs,nf),('REAL','SPLATFACTO','NERFACTO'))):
        drawer.text((j*thumb_w+6,y+5),f'{idx:02d} {label}',fill=(0,0,0))
        contact.paste(Image.fromarray(arr).resize((thumb_w,thumb_h),Image.Resampling.LANCZOS),
                      (j*thumb_w,y+label_h))
D_GALLERY_PATH=D_FIGURES/'contato_TEST_RGB.jpg'
contact.save(D_GALLERY_PATH,quality=93)
if D_SHOW_GALLERY:
    try:
        from IPython.display import display
        display(contact)
    except ImportError:
        pass

D_REPORT={
    'experiment':'D_V7_paired_final_test_scene0655_95_10_10',
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'status':'completed',
    'protocol':'A_V7_HV3_aligned_95_10_10',
    'dataset':{'path':str(D_DATASET),'scene':'scene-0655','camera':'CAM_FRONT',
               'resolution':list(D_RESOLUTION),'splits':D_COUNTS,
               'protocol_description':'95 TRAIN / 10 VAL / 10 TEST; LiDAR only TRAIN; poses oficiais nuScenes',
               'core_sha256':D_HASHES},
    'model_selection':{
        method:{'validation_step':item['step'],'val_mean_psnr_db':item['val_psnr_db'],
                'checkpoint':str(item['checkpoint']),
                'checkpoint_sha256':item['checkpoint_sha256'],
                'source_manifest':str(D_B_MANIFEST_PATH if method=='splatfacto' else D_C_MANIFEST_PATH)}
        for method,item in D_MODEL.items()},
    'metric_protocol':{
        'reference':'RGB PNG 8 bits da mesma camera, pose e arquivo para ambos os modelos',
        'psnr':'PSNR por imagem [0,1], media aritmetica das 10 imagens',
        'ssim':'skimage structural_similarity, gaussian_weights=True, sigma=1.5, use_sample_covariance=False, channel_axis=-1',
        'lpips':'torchmetrics LPIPS AlexNet, normalize=True, CPU',
        'reference_pixels_identical_between_models':True,
        'test_pixels_used_for_training_or_checkpoint_selection':False,
        'test_opened_after_val_selection':True,
        'not_an_equal_geometry_source_benchmark':True},
    'per_view':D_TEST_ROWS,
    'summary':D_SUMMARY,
    'test_image_manifest':str(D_TRIPLETS_JSON),
    'test_consumed':True,
    'timing_note':'ns-render mede processo inteiro (carregamento, render e gravacao); NAO medir como FPS puro.',
    'drive_upload':False,
}
D_REPORT_PATH=D_RESULTS/'relatorio_D_final.json'
d_write(D_REPORT_PATH,D_REPORT)

# ZIP local somente com análise, imagens e manifestos (SEM checkpoints/configs).
D_ZIP=D_RESULTS/'D_V7_avaliacao_final_nuscenes.zip'
D_ARCHIVE_FILES=[D_METRICS_CSV,D_METHODS_CSV,D_TRIPLETS_JSON,D_REPORT_PATH,D_LPIPS_OUTPUT,
                 D_GALLERY_PATH]
D_ARCHIVE_FILES += list(D_FIGURES.rglob('*.png')) + list(D_GALLERY_DIR.glob('*.jpg'))
with zipfile.ZipFile(D_ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=5) as z:
    for f in sorted(set(D_ARCHIVE_FILES)):
        if f.is_file():
            z.write(f,arcname=f.relative_to(D_RESULTS).as_posix())
with zipfile.ZipFile(D_ZIP) as z:
    if z.testzip() is not None:
        raise RuntimeError('ZIP final de D corrompido.')

D_TEST_CONSUMED=True
D_FINAL_REPORT=D_REPORT
D_RESULTS_DIR=D_RESULTS
D_FINAL_ZIP=D_ZIP
D_TEST_IMAGE_MANIFEST=D_TRIPLETS_JSON

print('\n'+'='*100)
print('D V7 — AVALIAÇÃO FINAL PAREADA CONCLUÍDA (10 TEST CONSUMIDOS)')
for method in ('splatfacto','nerfacto'):
    s=D_SUMMARY[method]
    lp='N/D' if s['test_lpips_alex'] is None else f"{s['test_lpips_alex']:.4f}"
    print(f"{method:10s} | checkpoint {s['selected_step']:5d} | PSNR {s['test_psnr_db']:.3f} dB | "
          f"SSIM {s['test_ssim']:.4f} | LPIPS {lp}")
print('CSV por vista:',D_METRICS_CSV)
print('Imagens para YOLO:',D_IMAGES)
print('Resumo:',D_REPORT_PATH)
print('ZIP LOCAL (NÃO enviado ao Drive):',D_ZIP)
print('ATENÇÃO: TEST já foi consumido. Não ajustar modelos a partir destes resultados.')


In [ ]:
# =============================================================================
# CÉLULA E V7 — TRÍADES RGB, RASTREABILIDADE E ENTRADA DA YOLO
# nuScenes scene-0655 / CAM_FRONT / 95 TRAIN + 10 VAL + 10 TEST / 800x450
# =============================================================================
# ORDEM: 0 -> SETUP -> 1 -> A V7 -> B V7 -> C V7 -> D V7 -> ESTA E V7.
#
# Protocolo alinhado à H V3 do iPhone, com diferenças próprias do nuScenes:
#   - poses e calibrações oficiais nuScenes (não COLMAP);
#   - Splatfacto inicializado pela nuvem LiDAR de TRAIN-only;
#   - Nerfacto-Big não usa a nuvem LiDAR na inicialização;
#   - checkpoints B/C já escolhidos exclusivamente por VAL, antes de TEST.
#
# Esta célula SOMENTE ORGANIZA os resultados já produzidos pela D V7:
#   - não treina, não renderiza, não executa YOLO;
#   - não seleciona checkpoints, não acessa imagens de VAL para otimizá-los;
#   - não copia checkpoints nem arquivos intermediários para o Google Drive;
#   - verifica A/D usando os hashes da execução;
#   - confere os 10 TEST em REAL x NERFACTO x SPLATFACTO;
#   - publica um manifesto 1:1:1 para a futura F V7 (YOLO);
#   - cria 10 tríades PNG, galerias e ZIP LOCAL, sem upload ao Drive.
#
# A REAL de referência é a PNG da GT usada pela D para PSNR/SSIM/LPIPS;
# a imagem original da A deve coincidir com ela, com diferença máxima 1/255
# por canal, exatamente como o critério da D V7. Isso evita referências
# ligeiramente diferentes entre as análises fotométrica e semântica.
#
# ATENÇÃO: TEST JÁ FOI CONSUMIDO NA D. E não justifica reajustar modelos.
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import math
import shutil
import zipfile

import numpy as np
from PIL import Image, ImageDraw

# ------------------------------ CONFIGURAÇÃO -------------------------------
E_PROTOCOL = 'A_V7_HV3_aligned_95_10_10'
E_EXPERIMENT = 'E_V7_triplets_yolo_scene0655_95_10_10'
E_A_TAG = 'scene0655_hv3_protocol_Av7_95_10_10'
E_A_PROFILE = 'nuscenes_front_samples_train_sweeps_holdout_10_10_v7'
E_EXPECTED_SPLITS = {'train': 95, 'val': 10, 'test': 10}
E_RESOLUTION = (800, 450)
E_MAX_GT_ORIGINAL_PIXEL_DELTA = 1  # 8-bit por canal; mesmo limite da D V7
E_PSNR_ABS_TOL_DB = 1e-4
E_SHOW_GALLERY = True
E_CREATE_LOCAL_ZIP = True
E_SEND_TO_DRIVE = False  # Apenas a análise FINAL consolidada deverá ir ao Drive.

# ------------------------------ HELPERS ------------------------------------
def e_sha256(path):
    sha = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            sha.update(block)
    return sha.hexdigest()


def e_load_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def e_write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=str)+'\n',
                    encoding='utf-8')


def e_write_csv(path, rows, columns):
    with Path(path).open('w', newline='', encoding='utf-8') as out:
        writer = csv.DictWriter(out, fieldnames=columns, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)


def e_safe_file(root, relative_path):
    """Impede caminhos absolutos ou traversals nos caminhos relativos dos manifestos."""
    root = Path(root).resolve()
    rel = Path(str(relative_path))
    if rel.is_absolute() or '..' in rel.parts:
        raise RuntimeError('Caminho relativo inseguro no manifesto: '+str(relative_path))
    path = (root / rel).resolve()
    if not path.is_relative_to(root):
        raise RuntimeError('Arquivo fora do diretório autorizado: '+str(path))
    if not path.is_file():
        raise FileNotFoundError('Arquivo ausente: '+str(path))
    return path


def e_rgb(path):
    with Image.open(path) as im:
        if im.size != E_RESOLUTION:
            raise RuntimeError('Imagem com tamanho inesperado: '+str(path)+' '+str(im.size))
        return np.asarray(im.convert('RGB'), dtype=np.uint8)


def e_psnr(ref, pred):
    if ref.shape != pred.shape:
        raise RuntimeError('PSNR: dimensões diferentes.')
    mse = float(np.mean(((ref.astype(np.float64)-pred.astype(np.float64))/255.0)**2))
    return float(-10.0 * math.log10(max(mse, 1e-15)))


def e_make_triplet(real, nerf, gs, label):
    """Tríade com dados em resolução original, sem alterar pixels das imagens."""
    width, height = E_RESOLUTION
    top = 32
    canvas = Image.new('RGB', (3*width, height+top), (255,255,255))
    draw = ImageDraw.Draw(canvas)
    for idx,(array,title) in enumerate(((real,'REAL'),(nerf,'NERFACTO-BIG'),(gs,'SPLATFACTO'))):
        canvas.paste(Image.fromarray(array), (idx*width,top))
        draw.text((idx*width+9,9),label+' — '+title,fill=(20,20,20))
    return canvas


# ---------------------- IDENTIDADE DA EXECUÇÃO D V7 ------------------------
if not globals().get('ENVIRONMENT_VALIDATED', False):
    raise RuntimeError('Execute 0 -> SETUP -> 1 antes de E.')
if globals().get('D_TEST_CONSUMED') is not True:
    raise RuntimeError('Execute a D V7 e conclua a avaliação TEST antes de executar E.')
required_globals = ('OUTPUT_DATASET','D_RESULTS_DIR','D_REPORT_PATH','D_TEST_IMAGE_MANIFEST')
missing = [name for name in required_globals if name not in globals()]
if missing:
    raise RuntimeError('E precisa da D V7 na MESMA sessão. Variáveis ausentes: '+str(missing))

E_DATASET = Path(globals()['OUTPUT_DATASET']).resolve()
E_D_RESULTS = Path(globals()['D_RESULTS_DIR']).resolve()
E_D_REPORT_PATH = Path(globals()['D_REPORT_PATH']).resolve()
E_D_IMAGE_MANIFEST_PATH = Path(globals()['D_TEST_IMAGE_MANIFEST']).resolve()
E_EXPECTED_DATASET = Path('/content/nuscenes_nerfstudio_'+E_A_TAG).resolve()

if E_DATASET != E_EXPECTED_DATASET:
    raise RuntimeError('E exige A V7: '+str(E_EXPECTED_DATASET)+'; encontrado '+str(E_DATASET))
if E_D_REPORT_PATH != E_D_RESULTS/'relatorio_D_final.json':
    raise RuntimeError('Relatório D não está na pasta D_RESULTS_DIR esperada.')
if E_D_IMAGE_MANIFEST_PATH != E_D_RESULTS/'test_image_manifest.json':
    raise RuntimeError('Manifesto de imagens da D não é o esperado.')

E_A_TF = E_DATASET/'transforms.json'
E_A_PAIRS = E_DATASET/'eval_pairs.json'
E_A_MANIFEST = E_DATASET/'dataset_manifest.json'
E_A_IDENTITY = E_DATASET/'dataset_identity.json'
for path in (E_A_TF,E_A_PAIRS,E_A_MANIFEST,E_A_IDENTITY,
             E_D_REPORT_PATH,E_D_IMAGE_MANIFEST_PATH):
    if not path.is_file():
        raise FileNotFoundError('Arquivo obrigatório da A/D ausente: '+str(path))

E_TF = e_load_json(E_A_TF)
E_A = e_load_json(E_A_MANIFEST)
E_ID = e_load_json(E_A_IDENTITY)
E_EVAL_PAIRS = e_load_json(E_A_PAIRS)
E_D = e_load_json(E_D_REPORT_PATH)
E_D_INPUTS = e_load_json(E_D_IMAGE_MANIFEST_PATH)

# Registro da D em memória, evitando confundir execuções diferentes.
if globals().get('D_FINAL_REPORT') != E_D:
    raise RuntimeError('O relatório D em disco não corresponde ao resultado D desta sessão.')
if E_A.get('experiment_tag')!=E_A_TAG or E_A.get('dataset_profile')!=E_A_PROFILE:
    raise RuntimeError('A não corresponde ao protocolo V7 esperado.')
if E_ID.get('protocol')!='A_V7_HV3-aligned':
    raise RuntimeError('Identidade da A V7 ausente ou inválida.')
if E_A.get('scene',{}).get('name')!='scene-0655':
    raise RuntimeError('Cena diferente da scene-0655.')
if E_A.get('cameras',{}).get('channels')!=['CAM_FRONT']:
    raise RuntimeError('Este experimento exige exclusivamente CAM_FRONT.')
if E_A.get('geometry',{}).get('geometry_inductive') is not True:
    raise RuntimeError('A V7 não confirma geometria independente de VAL/TEST.')
if E_A.get('lidar',{}).get('train_split_only') is not True:
    raise RuntimeError('A nuvem LiDAR não é exclusiva do TRAIN.')

E_SPLITS = {key:list(E_TF.get(key+'_filenames', [])) for key in ('train','val','test')}
E_COUNTS = {key:len(paths) for key,paths in E_SPLITS.items()}
if E_COUNTS!=E_EXPECTED_SPLITS:
    raise RuntimeError('A V7 não tem os splits 95/10/10: '+str(E_COUNTS))
E_FRAMES = E_TF.get('frames', [])
E_FRAME_MAP = {frame['file_path']:frame for frame in E_FRAMES}
E_FILES = [p for paths in E_SPLITS.values() for p in paths]
if (len(E_FRAME_MAP)!=115 or len(E_FILES)!=115 or len(set(E_FILES))!=115
        or set(E_FILES)!=set(E_FRAME_MAP)):
    raise RuntimeError('A V7: imagens ausentes, duplicadas ou fora do split.')
for rel in E_SPLITS['test']:
    f = E_FRAME_MAP[rel]
    if f.get('dataset_split')!='test' or f.get('camera_channel')!='CAM_FRONT':
        raise RuntimeError('TEST não é CAM_FRONT ou tem split incorreto: '+rel)
    if (int(f['w']),int(f['h']))!=E_RESOLUTION:
        raise RuntimeError('Resolução de TEST não é 800x450: '+rel)

E_LIDAR = E_TF.get('ply_file_path')
if not E_LIDAR:
    raise RuntimeError('A V7 não aponta para a nuvem LiDAR em transforms.json.')
E_SHA = {name:e_sha256(E_DATASET/name) for name in
         ('transforms.json','eval_pairs.json','dataset_manifest.json')}
E_LIDAR_FILE = e_safe_file(E_DATASET, E_LIDAR)
E_SHA[E_LIDAR] = e_sha256(E_LIDAR_FILE)
if E_ID.get('core_sha256')!=E_SHA:
    raise RuntimeError('Hashes A V7 alterados após construção do dataset.')

if E_D.get('experiment')!='D_V7_paired_final_test_scene0655_95_10_10':
    raise RuntimeError('E recebeu relatório da D de outro experimento.')
if E_D.get('protocol')!=E_PROTOCOL or E_D.get('status')!='completed':
    raise RuntimeError('A D V7 não confirma execução concluída do protocolo 95/10/10.')
if E_D.get('test_consumed') is not True:
    raise RuntimeError('A D não registra TEST consumido.')
E_D_DS = E_D.get('dataset',{})
if Path(E_D_DS.get('path','/invalid')).resolve()!=E_DATASET:
    raise RuntimeError('A D foi executada em outro dataset.')
if E_D_DS.get('splits')!=E_COUNTS or E_D_DS.get('core_sha256')!=E_SHA:
    raise RuntimeError('Dataset ou hashes da D divergentes da A atual.')
if E_D_DS.get('resolution')!=list(E_RESOLUTION):
    raise RuntimeError('D tem outra resolução.')
E_D_METRIC = E_D.get('metric_protocol',{})
for key in ('reference_pixels_identical_between_models',
            'test_opened_after_val_selection'):
    if E_D_METRIC.get(key) is not True:
        raise RuntimeError('D não confirmou: '+key)
if E_D_METRIC.get('test_pixels_used_for_training_or_checkpoint_selection') is not False:
    raise RuntimeError('D indica uso indevido do TEST na seleção ou treinamento.')

for method in ('splatfacto','nerfacto'):
    selection=E_D.get('model_selection',{}).get(method,{})
    summary=E_D.get('summary',{}).get(method,{})
    if (selection.get('validation_step')!=summary.get('selected_step')
            or not selection.get('checkpoint_sha256')):
        raise RuntimeError('Seleção de '+method+' não corresponde ao resumo D.')
    if not math.isclose(float(selection['val_mean_psnr_db']),
                        float(summary['selection_val_psnr_db']),abs_tol=1e-6,rel_tol=0):
        raise RuntimeError('PSNR VAL divergente para '+method)

if len(E_D.get('per_view',[]))!=10 or len(E_D_INPUTS)!=10 or len(E_EVAL_PAIRS)!=10:
    raise RuntimeError('A, D e manifesto D precisam conter exatamente 10 TEST.')

# Ordenação canônica segundo os pares originais de TEST da A V7.
E_TEST_PAIRS = sorted(E_EVAL_PAIRS, key=lambda p:int(p['index']))
if [int(p['index']) for p in E_TEST_PAIRS]!=list(range(10)):
    raise RuntimeError('eval_pairs.json não contém índices TEST de 0 a 9.')
E_TEST_PATHS = [p.get('nerfstudio_file_path') for p in E_TEST_PAIRS]
if len(set(E_TEST_PATHS))!=10 or set(E_TEST_PATHS)!=set(E_SPLITS['test']):
    raise RuntimeError('eval_pairs.json e transforms.json não possuem o mesmo TEST.')
if any(p.get('split')!='test' or p.get('camera_channel')!='CAM_FRONT'
       for p in E_TEST_PAIRS):
    raise RuntimeError('eval_pairs.json contém imagem não-TEST ou câmera inesperada.')


def e_index_unique(items, key, label):
    result={}
    for item in items:
        item_key=item.get(key)
        if not item_key or item_key in result:
            raise RuntimeError('Chave ausente ou duplicada em '+label+': '+str(item_key))
        result[item_key]=item
    return result

E_D_BY_PATH=e_index_unique(E_D['per_view'],'file_path','D/per_view')
E_D_IMG_BY_PATH=e_index_unique(E_D_INPUTS,'file_path','D/test_image_manifest')
if set(E_D_BY_PATH)!=set(E_TEST_PATHS) or set(E_D_IMG_BY_PATH)!=set(E_TEST_PATHS):
    raise RuntimeError('A e D não referenciam as mesmas dez imagens TEST.')

print('='*100)
print('E V7 — PRÉ-VALIDAÇÃO DAS TRÍADES PARA YOLO')
print('Dataset:',E_DATASET)
print('Protocolo:',E_PROTOCOL)
print('Splits:',E_COUNTS,'| CAM_FRONT | 800x450')
print('Relatório D:',E_D_REPORT_PATH)
print('Hashes A V7/D correspondentes: OK')
print('Os checkpoints já estavam selecionados por VAL antes do TEST: OK')

# ------------------------ ARQUIVOS NORMALIZADOS ----------------------------
E_STAMP=datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
E_RESULTS=E_D_RESULTS/('E_triplets_yolo_'+E_STAMP)
E_REAL_DIR=E_RESULTS/'real'
E_NERF_DIR=E_RESULTS/'nerfacto'
E_3DGS_DIR=E_RESULTS/'splatfacto'
E_TRIADES_DIR=E_RESULTS/'galerias'/ 'individual'
E_GALLERY_PATH=E_RESULTS/'galerias'/'galeria_triplets_TEST.jpg'
E_PAIRS_MANIFEST_PATH=E_RESULTS/'render_pairs_manifest.json'
E_PAIRS_CSV=E_RESULTS/'render_pairs.csv'
E_README_PATH=E_RESULTS/'LEIA_ME.txt'
E_ZIP_PATH=E_RESULTS/'E_V7_triplets_yolo_scene0655.zip'
for folder in (E_RESULTS,E_REAL_DIR,E_NERF_DIR,E_3DGS_DIR,E_TRIADES_DIR):
    folder.mkdir(parents=True,exist_ok=False)

E_ROWS=[]
E_THUMBNAILS=[]
E_GALLERY_CONTACT=[]
E_D_REPORT_SHA=e_sha256(E_D_REPORT_PATH)

# --------------------------- DEZ TRÍADES ----------------------------------
for image_index, pair in enumerate(E_TEST_PAIRS):
    rel=pair['nerfstudio_file_path']
    drow=E_D_BY_PATH[rel]
    dimages=E_D_IMG_BY_PATH[rel]
    stem=Path(rel).stem
    if drow.get('image')!=stem or dimages.get('image')!=stem:
        raise RuntimeError('D vincula a imagem incorreta ao caminho: '+rel)
    if drow.get('test_view')!=dimages.get('test_view'):
        raise RuntimeError('Índices TEST divergentes dentro da D: '+rel)
    if int(pair['timestamp'])!=int(drow['frame_timestamp_us']):
        raise RuntimeError('Timestamps A/D divergentes: '+rel)
    expected_original=e_safe_file(E_DATASET,rel)
    registered_original=Path(pair.get('real_image_path','/missing')).resolve()
    if registered_original!=expected_original:
        raise RuntimeError('A/eval_pairs aponta para outra REAL: '+str(registered_original))

    # D já produz PNGs RGB (e não apenas referências com ajustes de RGB).
    source={name:e_safe_file(E_D_RESULTS,dimages[name]) for name in
            ('real','nerfacto','splatfacto')}
    for name,p in source.items():
        if p.suffix.lower()!='.png':
            raise RuntimeError('Imagem da D não está em PNG: '+str(p))
    real,nerf,gs=(e_rgb(source[domain]) for domain in ('real','nerfacto','splatfacto'))
    original=e_rgb(expected_original)

    # Exatamente o mesmo limite que a D impôs à GT em relação à A.
    delta=np.abs(real.astype(np.int16)-original.astype(np.int16))
    maximum=int(delta.max()); mae=float(delta.mean())
    if maximum>E_MAX_GT_ORIGINAL_PIXEL_DELTA:
        raise RuntimeError(f'TEST {image_index+1}: GT da D difere do original A; '
                           f'máximo={maximum} níveis, limite={E_MAX_GT_ORIGINAL_PIXEL_DELTA}.')

    # D publicou os PSNR por imagem. Recalcular sem alterar os modelos garante
    # correspondência 1:1:1 dos PNGs selecionados.
    for method, image in (('nerfacto',nerf),('splatfacto',gs)):
        reported=float(drow[method+'_psnr_db'])
        calculated=e_psnr(real,image)
        if not math.isclose(calculated,reported,abs_tol=E_PSNR_ABS_TOL_DB,rel_tol=0):
            raise RuntimeError(f'PSNR {method} diverge do relatório D, imagem {rel}: '
                               f'{calculated:.6f} != {reported:.6f}.')

    basename=f'eval_img_{image_index:04d}.png'
    target={'real':E_REAL_DIR/basename,'nerfacto':E_NERF_DIR/basename,
            'splatfacto':E_3DGS_DIR/basename}
    for domain in ('real','nerfacto','splatfacto'):
        shutil.copy2(source[domain],target[domain])
        if e_sha256(source[domain])!=e_sha256(target[domain]):
            raise RuntimeError('Cópia PNG divergente: '+str(target[domain]))

    row={
        'image_index':image_index,
        'd_test_view':int(drow['test_view']),
        'split':'test','camera_channel':'CAM_FRONT',
        'timestamp':int(pair['timestamp']),
        'source_type':E_FRAME_MAP[rel].get('source_type'),
        'sample_data_token':pair.get('sample_data_token'),
        'sample_token':pair.get('sample_token'),
        'nuscenes_filename':pair.get('nuscenes_filename'),
        'nerfstudio_file_path':rel,
        'real_original_path':str(expected_original),
        'real_rgb_path':str(target['real']),
        'real_sha256':e_sha256(target['real']),
        'nerfacto_rgb_path':str(target['nerfacto']),
        'nerfacto_sha256':e_sha256(target['nerfacto']),
        'splatfacto_rgb_path':str(target['splatfacto']),
        'splatfacto_sha256':e_sha256(target['splatfacto']),
        'real_rgb_relative':'real/'+basename,
        'nerfacto_rgb_relative':'nerfacto/'+basename,
        'splatfacto_rgb_relative':'splatfacto/'+basename,
        'd_real_png':str(source['real']),
        'd_nerfacto_png':str(source['nerfacto']),
        'd_splatfacto_png':str(source['splatfacto']),
        'gt_vs_original_mae':round(mae,8),
        'gt_vs_original_max_abs':maximum,
        'width':E_RESOLUTION[0], 'height':E_RESOLUTION[1],
        'd_splatfacto_psnr_db':float(drow['splatfacto_psnr_db']),
        'd_splatfacto_ssim':float(drow['splatfacto_ssim']),
        'd_splatfacto_lpips_alex':float(drow['splatfacto_lpips_alex']),
        'd_nerfacto_psnr_db':float(drow['nerfacto_psnr_db']),
        'd_nerfacto_ssim':float(drow['nerfacto_ssim']),
        'd_nerfacto_lpips_alex':float(drow['nerfacto_lpips_alex']),
    }
    E_ROWS.append(row)

    trip=e_make_triplet(real,nerf,gs,f'TEST {image_index+1:02d}')
    trip.save(E_TRIADES_DIR/(f'eval_img_{image_index:04d}.jpg'),format='JPEG',quality=94)
    thumb=trip.resize((1200,241),Image.Resampling.LANCZOS)
    E_GALLERY_CONTACT.append(thumb)

    print(f'[{image_index+1:02d}/10] {stem} | GT original: max={maximum} '
          f'| PSNR Splatfacto {row["d_splatfacto_psnr_db"]:.2f} dB '
          f'| Nerfacto {row["d_nerfacto_psnr_db"]:.2f} dB')

expected_names={f'eval_img_{i:04d}.png' for i in range(10)}
for domain,folder in (('real',E_REAL_DIR),('nerfacto',E_NERF_DIR),('splatfacto',E_3DGS_DIR)):
    names={p.name for p in folder.glob('*.png')}
    if names!=expected_names:
        raise RuntimeError('Imagens '+domain+' incompletas: '+str(names^expected_names))

# ------------------------------ GALERIA -----------------------------------
height=sum(item.height for item in E_GALLERY_CONTACT)
E_CONTACT=Image.new('RGB',(1200,height),(255,255,255))
y=0
for item in E_GALLERY_CONTACT:
    E_CONTACT.paste(item,(0,y))
    y+=item.height
E_CONTACT.save(E_GALLERY_PATH,format='JPEG',quality=93)
if E_SHOW_GALLERY:
    try:
        from IPython.display import display
        display(E_CONTACT)
    except ImportError:
        print('Galeria visual salva em:',E_GALLERY_PATH)

# ------------------------ MANIFESTO PARA A F V7 ----------------------------
E_MANIFEST={
    'experiment':E_EXPERIMENT,
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'status':'completed',
    'protocol':E_PROTOCOL,
    'purpose':'10 tríades RGB 1:1:1 de TEST para YOLO, sem novo treino/render/seleção',
    'source_D':{
        'report':str(E_D_REPORT_PATH),
        'report_sha256':E_D_REPORT_SHA,
        'test_image_manifest':str(E_D_IMAGE_MANIFEST_PATH),
        'test_image_manifest_sha256':e_sha256(E_D_IMAGE_MANIFEST_PATH),
        'results_root':str(E_D_RESULTS),
        'test_consumed':True,
        'selection_before_test_verified':True,
        'same_gt_between_methods_verified':True,
    },
    'dataset':{
        'path':str(E_DATASET),'profile':E_A_PROFILE,
        'scene':'scene-0655','camera':'CAM_FRONT',
        'resolution':list(E_RESOLUTION),
        'total_frames':115,'train_frames':95,'validation_frames':10,'test_frames':10,
        'train_keyframes':int(E_ID.get('train_keyframes',0)),
        'train_sweeps':int(E_ID.get('train_sweeps',0)),
        'core_sha256':E_SHA,
        'transforms_sha256':E_SHA['transforms.json'],
        'eval_pairs_sha256':E_SHA['eval_pairs.json'],
        'point_cloud_lidar_sha256':E_SHA[E_LIDAR],
        'point_cloud_lidar_path':E_LIDAR,
        'dataset_manifest_sha256_actual_run':E_SHA['dataset_manifest.json'],
        'geometry_source':'nuScenes official poses + TRAIN-only LiDAR (Splatfacto)',
    },
    'models':{
        method:{
            'selected_step':E_D['model_selection'][method]['validation_step'],
            'checkpoint':E_D['model_selection'][method]['checkpoint'],
            'checkpoint_sha256':E_D['model_selection'][method]['checkpoint_sha256'],
            'selection_val_psnr_db':E_D['summary'][method]['selection_val_psnr_db'],
        } for method in ('splatfacto','nerfacto')
    },
    'directories':{
        'real':str(E_REAL_DIR),'nerfacto':str(E_NERF_DIR),'splatfacto':str(E_3DGS_DIR),
        'triplet_images':str(E_TRIADES_DIR),
    },
    'normalization':{
        'output_format':'PNG RGB',
        'output_resolution':list(E_RESOLUTION),
        'same_output_names_across_domains':True,
        'reference':'GT PNG das mesmas vistas usado pela D; conferido com o original A',
        'max_pixel_delta_GT_vs_A_allowed':E_MAX_GT_ORIGINAL_PIXEL_DELTA,
        'no_training_in_E':True,'no_rendering_in_E':True,
        'no_model_inference_in_E':True,'no_model_selection_in_E':True,
        'no_checkpoint_upload_to_drive':True,
    },
    'pairs':E_ROWS,
}
e_write_json(E_PAIRS_MANIFEST_PATH,E_MANIFEST)
E_CSV_FIELDS=['image_index','d_test_view','nerfstudio_file_path','timestamp',
              'source_type','sample_data_token','sample_token',
              'real_rgb_path','nerfacto_rgb_path','splatfacto_rgb_path',
              'gt_vs_original_mae','gt_vs_original_max_abs',
              'd_splatfacto_psnr_db','d_splatfacto_ssim','d_splatfacto_lpips_alex',
              'd_nerfacto_psnr_db','d_nerfacto_ssim','d_nerfacto_lpips_alex']
e_write_csv(E_PAIRS_CSV,E_ROWS,E_CSV_FIELDS)

E_README_PATH.write_text(
    'CÉLULA E V7 — nuScenes scene-0655 / CAM_FRONT / 95 TRAIN, 10 VAL, 10 TEST\n'
    'Resultado local para a Célula F V7 (YOLO), sem alterar modelos.\n\n'
    'real/              = GT RGB 8-bit usada pela D nas métricas.\n'
    'nerfacto/          = reconstruções do Nerfacto-Big escolhidas por VAL.\n'
    'splatfacto/        = reconstruções do Splatfacto escolhidas por VAL.\n'
    'galerias/          = comparações visuais.\n'
    'render_pairs.csv   = correspondência e qualidade por imagem.\n'
    'render_pairs_manifest.json = paths locais e paths relativos ao ZIP.\n\n'
    'As três pastas compartilham 10 nomes eval_img_0000.png a eval_img_0009.png.\n'
    'As imagens de TEST já tinham sido consumidas na D. O teste não deve guiar\n'
    'reajuste de pesos/hiperparâmetros.\n'
    'Os caminhos absolutos do manifesto são válidos na sessão original do Colab.\n'
    'Para arquivos extraídos, use *_rgb_relative relativos à raiz deste ZIP.\n',
    encoding='utf-8',
)

# ZIP E local: conveniente para auditoria. Não é o ZIP final a exportar ao Drive.
if E_CREATE_LOCAL_ZIP:
    with zipfile.ZipFile(E_ZIP_PATH,'w',zipfile.ZIP_DEFLATED,compresslevel=5) as archive:
        for path in sorted(E_RESULTS.rglob('*')):
            if path.is_file() and path!=E_ZIP_PATH:
                if path.is_symlink():
                    raise RuntimeError('Symlink não permitido dentro do ZIP de análise: '+str(path))
                archive.write(path,path.relative_to(E_RESULTS).as_posix())
    with zipfile.ZipFile(E_ZIP_PATH) as archive:
        corrupt=archive.testzip()
        if corrupt is not None:
            raise RuntimeError('ZIP E corrompido no arquivo '+str(corrupt))

# ------------------------ INTERFACE PARA A F V7 ----------------------------
YOLO_REAL_DIR=E_REAL_DIR
YOLO_NERF_DIR=E_NERF_DIR
YOLO_3DGS_DIR=E_3DGS_DIR
YOLO_PAIRS_MANIFEST_PATH=E_PAIRS_MANIFEST_PATH
E_FINAL_REPORT=E_MANIFEST

print('\n'+'='*100)
print('E V7 — TRÍADES RGB CONCLUÍDAS (NENHUM TREINAMENTO / RENDER / YOLO)')
print('Cena: scene-0655 | CAM_FRONT | 95 TRAIN / 10 VAL / 10 TEST')
print('10 imagens REAL:',YOLO_REAL_DIR)
print('10 imagens NERFACTO:',YOLO_NERF_DIR)
print('10 imagens SPLATFACTO:',YOLO_3DGS_DIR)
print('Manifesto para F:',YOLO_PAIRS_MANIFEST_PATH)
print('CSV:',E_PAIRS_CSV)
print('Galeria:',E_GALLERY_PATH)
if E_CREATE_LOCAL_ZIP: print('ZIP LOCAL:',E_ZIP_PATH)
print('Google Drive: nenhum arquivo enviado por E (somente análise final consolidada).')
print('Próximo passo: F V7 — YOLOv8x nas mesmas 10 vistas de cada domínio.')


In [ ]:
# =============================================================================
# CÉLULA F V7.1 — YOLOv8x, concordância semântica REAL / NERFACTO / SPLATFACTO
# nuScenes scene-0655, CAM_FRONT, 95 TRAIN / 10 VAL / 10 TEST, 800x450
# =============================================================================
# ORDEM: 0 -> SETUP -> 1 -> A V7 -> B V7 -> C V7 -> D V7 -> E V7 -> F V7.
#
# Procedimento pareado com iPhone H V3:
#  - YOLOv8x, conf=0.25, NMS IoU=0.70, imgsz=1280, COCO [0,1,2,3,5,7];
#  - compara YOLO(REAL) com YOLO(RECONSTRUÍDA) pelo MESMO detector;
#  - matching 1:1 de cardinalidade máxima POR CLASSE em IoU .3/.5/.7;
#  - reporta concordância recall-like/precision-like/F1-like, NÃO mAP;
#  - não altera os checkpoints B/C nem usa TEST para retreinar modelos;
#  - testa hashes, 10 tríades, correspondência D/E, protocolo e versão YOLO;
#  - executa PyTorch/Ultralytics apenas no Python isolado de /content/tcc_env;
#  - produz CSVs/JSON, PNGs anotados, galeria e ZIP LOCAL em /content;
#  - NENHUM checkpoint ou artefato intermediário enviado ao Google Drive.
# A célula G consolida os resultados finais e permite exportação opcional.
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import csv
import hashlib
import json
import math
import os
import re
import shlex
import subprocess
import sys
import threading
import time
import zipfile
import numpy as np
from PIL import Image, ImageDraw, ImageFont

# -------------------------- CONFIGURAÇÃO EXPERIMENTAL ----------------------
F_EXPECTED_PROTOCOL = 'A_V7_HV3_aligned_95_10_10'
F_EXPECTED_E_EXPERIMENT = 'E_V7_triplets_yolo_scene0655_95_10_10'
F_EXPECTED_D_EXPERIMENT = 'D_V7_paired_final_test_scene0655_95_10_10'
F_A_TAG = 'scene0655_hv3_protocol_Av7_95_10_10'
F_EXPECTED_PROFILE = 'nuscenes_front_samples_train_sweeps_holdout_10_10_v7'
F_SPLITS = {'train':95, 'val':10, 'test':10}
F_WIDTH, F_HEIGHT = 800, 450
F_YOLO_MODEL = str(globals().get('YOLO_MODEL_OVERRIDE', 'yolov8x.pt'))
F_EXPECTED_WEIGHTS_SHA256 = globals().get('YOLO_EXPECTED_SHA256', None)
F_CONF = 0.25
F_NMS_IOU = 0.70
F_IMGSZ = 1280
F_MAX_DET = 300
F_CLASSES = [0,1,2,3,5,7]  # COCO: person, bicycle, car, motorcycle, bus, truck
F_MATCH_IOU_LEVELS = (0.30, 0.50, 0.70)
F_PRIMARY_MATCH_IOU = 0.50
F_SHOW_GALLERY = True
F_USE_GPU = True
F_SEED = 42
F_WORKER_TIMEOUT_SECONDS = 3600
F_CREATE_LOCAL_ZIP = True
F_SEND_INTERMEDIATES_TO_DRIVE = False
# V7.1: NÃO usar Path.resolve() no executável do venv. O bin/python costuma
# ser um symlink para o interpretador-base; resolvê-lo faz perder pyvenv.cfg
# e os pacotes instalados exclusivamente no /content/tcc_env.
F_TCC_ENV = Path(globals().get('TCC_ENV', '/content/tcc_env')).expanduser()
F_BIN = F_TCC_ENV / 'bin'
F_PYTHON = F_BIN / 'python'  # manter este caminho textual, mesmo se for symlink
F_CUDA = Path('/usr/local/cuda-12.8')

# ------------------------------- HELPERS -----------------------------------
def f_hash(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as fh:
        for chunk in iter(lambda:fh.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()


def f_read(p):
    return json.loads(Path(p).read_text(encoding='utf-8'))


def f_write(p,obj):
    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)
    p.write_text(json.dumps(obj,indent=2,ensure_ascii=False,default=str)+'\n',encoding='utf-8')


def f_csv(p,rows,fields):
    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)
    with p.open('w',newline='',encoding='utf-8') as fh:
        writer=csv.DictWriter(fh,fieldnames=list(fields),extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)


def f_under(path,root):
    root=Path(root).resolve();path=Path(path).resolve()
    if not path.is_relative_to(root):
        raise RuntimeError(f'Caminho fora da pasta autorizada: {path} (raiz {root})')
    if not path.is_file():
        raise FileNotFoundError(path)
    return path


def f_file_sha(path,expected,label):
    if not expected or not re.fullmatch(r'[0-9a-fA-F]{64}',str(expected)):
        raise RuntimeError('SHA256 ausente ou inválido: '+label)
    actual=f_hash(path)
    if actual!=str(expected).lower():
        raise RuntimeError(f'SHA256 divergente em {label}: {actual} != {expected}')


def f_rgb(path):
    with Image.open(path) as im:
        if im.size!=(F_WIDTH,F_HEIGHT):
            raise RuntimeError(f'Resolução incorreta em {path}: {im.size}')
        return im.convert('RGB').copy()


def f_agreement(nreal,npred,nmatched):
    if nmatched>min(nreal,npred) or min(nreal,npred)<0:
        raise RuntimeError('Contagem inválida no matching YOLO.')
    recall=nmatched/nreal if nreal else None
    precision=nmatched/npred if npred else None
    f1=2*nmatched/(nreal+npred) if (nreal+npred) else None
    return recall,precision,f1


def f_iou(a,b):
    x1=max(float(a['x1']),float(b['x1']))
    y1=max(float(a['y1']),float(b['y1']))
    x2=min(float(a['x2']),float(b['x2']))
    y2=min(float(a['y2']),float(b['y2']))
    iw=max(0.0,x2-x1);ih=max(0.0,y2-y1)
    inter=iw*ih
    aa=max(0.,float(a['x2'])-float(a['x1']))*max(0.,float(a['y2'])-float(a['y1']))
    ab=max(0.,float(b['x2'])-float(b['x1']))*max(0.,float(b['y2'])-float(b['y1']))
    union=aa+ab-inter
    return float(inter/union) if union>0 else 0.0


def f_matches(real,pred,threshold):
    """Máxima cardinalidade bipartida 1:1; ordena arestas por IoU (sem otimizar soma global)."""
    if not real or not pred:
        return []
    a=np.zeros((len(real),len(pred)),dtype=np.float64)
    adjacency={}
    for i,r in enumerate(real):
        for j,p in enumerate(pred):
            if int(r['class_id'])==int(p['class_id']):
                a[i,j]=f_iou(r,p)
        adjacency[i]=sorted((j for j in range(len(pred)) if a[i,j]>=threshold),
                            key=lambda j:(-a[i,j],j))
    order=sorted(range(len(real)),key=lambda i:(-max((a[i,j] for j in adjacency[i]),default=0.0),i))
    assigned={}
    def augment(i,seen):
        for j in adjacency[i]:
            if j in seen:continue
            seen.add(j)
            if j not in assigned or augment(assigned[j],seen):
                assigned[j]=i
                return True
        return False
    for i in order:augment(i,set())
    return [{'real_index':i,'render_index':j,'iou':float(a[i,j])}
            for j,i in sorted(assigned.items())]


def f_aggregate(rows,domain,threshold):
    subset=[r for r in rows if r['render_domain']==domain and r['match_iou']==threshold]
    nr=sum(int(r['real_count']) for r in subset)
    npred=sum(int(r['render_count']) for r in subset)
    nm=sum(int(r['matched_count']) for r in subset)
    recall,precision,f1=f_agreement(nr,npred,nm)
    iousum=sum(float(r['matched_iou_sum']) for r in subset)
    return {'render_domain':domain,'match_iou':threshold,
            'real_count':nr,'render_count':npred,'matched_count':nm,
            'recall_like':recall,'precision_like':precision,'f1_like':f1,
            'mean_matched_iou':iousum/nm if nm else None,
            'detection_retention_ratio':npred/nr if nr else None}

# ----------------------- VALIDAR PROVENIÊNCIA A -> D -> E -------------------
if not globals().get('ENVIRONMENT_VALIDATED',False):
    raise RuntimeError('Execute 0 -> SETUP -> 1 antes das células A–F.')
if not (F_TCC_ENV / 'pyvenv.cfg').is_file():
    raise FileNotFoundError('Ambiente virtual não encontrado: '+str(F_TCC_ENV / 'pyvenv.cfg')+
                            '. Não reinstale pacotes no Python-base; reexecute SETUP se necessário.')
if not F_PYTHON.is_file():
    raise FileNotFoundError('Python isolado do tcc_env não encontrado: '+str(F_PYTHON))
if F_EXPECTED_WEIGHTS_SHA256 is not None and not re.fullmatch(
        r'[0-9a-fA-F]{64}',str(F_EXPECTED_WEIGHTS_SHA256)):
    raise ValueError('YOLO_EXPECTED_SHA256 deve conter 64 dígitos hexadecimais.')
if not F_USE_GPU:
    raise RuntimeError('A comparação com H V3 exige YOLO executado com GPU.')
if F_SEND_INTERMEDIATES_TO_DRIVE:
    raise RuntimeError('F não deve transferir intermediários. Somente a G exportará análise final.')

if 'YOLO_PAIRS_MANIFEST_PATH' not in globals():
    raise RuntimeError('Execute a célula E V7 nesta mesma sessão (YOLO_PAIRS_MANIFEST_PATH ausente).')
F_E_MANIFEST_PATH=Path(globals()['YOLO_PAIRS_MANIFEST_PATH']).resolve()
F_E_ROOT=F_E_MANIFEST_PATH.parent
F_E=f_read(F_E_MANIFEST_PATH)
if F_E_MANIFEST_PATH.name!='render_pairs_manifest.json':
    raise RuntimeError('A F precisa do manifesto de tríades da E V7.')
if (F_E.get('experiment')!=F_EXPECTED_E_EXPERIMENT or F_E.get('status')!='completed'
        or F_E.get('protocol')!=F_EXPECTED_PROTOCOL):
    raise RuntimeError('Manifesto da E não corresponde ao protocolo E V7.')
if 'E_FINAL_REPORT' in globals() and globals()['E_FINAL_REPORT']!=F_E:
    raise RuntimeError('Manifesto E em disco é diferente do guardado na memória desta sessão.')

F_DATASET=Path(F_E.get('dataset',{}).get('path','/invalid')).resolve()
F_EXPECTED_DATASET=Path('/content/nuscenes_nerfstudio_'+F_A_TAG).resolve()
if F_DATASET!=F_EXPECTED_DATASET:
    raise RuntimeError('Dataset não é o produzido pela A V7: '+str(F_DATASET))
if 'OUTPUT_DATASET' not in globals() or Path(globals()['OUTPUT_DATASET']).resolve()!=F_DATASET:
    raise RuntimeError('Dataset global da A não é o mesmo do manifesto E.')
F_DS=F_E['dataset'];F_D_SRC=F_E.get('source_D',{})
if (F_DS.get('profile')!=F_EXPECTED_PROFILE or F_DS.get('scene')!='scene-0655'
        or F_DS.get('camera')!='CAM_FRONT' or F_DS.get('resolution')!=[F_WIDTH,F_HEIGHT]):
    raise RuntimeError('Cena, perfil, câmera ou resolução diferentes da A V7.')
if any(F_DS.get(k)!=v for k,v in [('total_frames',115),('train_frames',95),
                                    ('validation_frames',10),('test_frames',10)]):
    raise RuntimeError('A E não registra 95/10/10 corretamente.')
if F_D_SRC.get('test_consumed') is not True or F_D_SRC.get('selection_before_test_verified') is not True:
    raise RuntimeError('A E não confirma seleção VAL antes do TEST da D.')
if F_D_SRC.get('same_gt_between_methods_verified') is not True:
    raise RuntimeError('A E não confirma RGB real/GT comum aos dois modelos.')

F_CORE={}
for name in ('transforms.json','eval_pairs.json','dataset_manifest.json'):
    path=F_DATASET/name
    if not path.is_file():raise FileNotFoundError(path)
    F_CORE[name]=f_hash(path)
F_PLY=F_DS.get('point_cloud_lidar_path')
if not F_PLY or Path(F_PLY).is_absolute() or '..' in Path(F_PLY).parts:
    raise RuntimeError('PLY LiDAR com caminho inseguro/ausente no manifesto E.')
F_CORE[F_PLY]=f_hash(f_under(F_DATASET/F_PLY,F_DATASET))
if F_CORE!=F_DS.get('core_sha256'):
    raise RuntimeError('Dados A V7 foram modificados desde a E: core_sha256 divergente.')
if F_DS.get('point_cloud_lidar_sha256')!=F_CORE[F_PLY]:
    raise RuntimeError('SHA da nuvem LiDAR divergente.')
F_IDENTITY_PATH=F_DATASET/'dataset_identity.json'
if not F_IDENTITY_PATH.is_file():raise FileNotFoundError(F_IDENTITY_PATH)
F_ID=f_read(F_IDENTITY_PATH)
if F_ID.get('protocol')!='A_V7_HV3-aligned' or F_ID.get('core_sha256')!=F_CORE:
    raise RuntimeError('Identidade da A V7 não corresponde ao manifesto E.')

F_D_REPORT_PATH=Path(F_D_SRC.get('report','/invalid')).resolve()
F_D=f_read(F_D_REPORT_PATH)
f_file_sha(F_D_REPORT_PATH,F_D_SRC.get('report_sha256'),'relatorio_D_final.json')
if (F_D.get('experiment')!=F_EXPECTED_D_EXPERIMENT or F_D.get('status')!='completed'
        or F_D.get('protocol')!=F_EXPECTED_PROTOCOL or F_D.get('test_consumed') is not True):
    raise RuntimeError('A D não está concluída com o protocolo correspondente.')
if F_D.get('dataset',{}).get('core_sha256')!=F_CORE:
    raise RuntimeError('A D foi calculada com outro dataset.')
if (F_D.get('metric_protocol',{}).get('reference_pixels_identical_between_models') is not True
        or F_D.get('metric_protocol',{}).get('test_opened_after_val_selection') is not True
        or F_D.get('metric_protocol',{}).get('test_pixels_used_for_training_or_checkpoint_selection') is not False):
    raise RuntimeError('A D não certificou GT e seleção por VAL.')
if len(F_D.get('per_view',[]))!=10:
    raise RuntimeError('A D não possui dez imagens TEST.')
for method in ('nerfacto','splatfacto'):
    e_sel=F_E['models'][method]
    d_sel=F_D['model_selection'][method]
    if (e_sel.get('selected_step')!=d_sel.get('validation_step')
            or e_sel.get('checkpoint_sha256')!=d_sel.get('checkpoint_sha256')):
        raise RuntimeError('Checkpoint selecionado por VAL diverge na cadeia D/E: '+method)
    ckpt=Path(e_sel.get('checkpoint','/invalid'))
    if not ckpt.is_file():raise FileNotFoundError(ckpt)
    f_file_sha(ckpt,e_sel.get('checkpoint_sha256'),'checkpoint selecionado '+method)

if len(F_E.get('pairs',[]))!=10:
    raise RuntimeError('E precisa conter dez tríades TEST (30 PNGs).')
F_PAIRS=sorted(F_E['pairs'],key=lambda r:int(r['image_index']))
if [int(p['image_index']) for p in F_PAIRS]!=list(range(10)):
    raise RuntimeError('image_index da E precisa ser 0..9 sem repetições.')
F_A_PAIRS=sorted(f_read(F_DATASET/'eval_pairs.json'),key=lambda r:int(r['index']))
if (len(F_A_PAIRS)!=10 or [x['nerfstudio_file_path'] for x in F_A_PAIRS]!=
        [x['nerfstudio_file_path'] for x in F_PAIRS]):
    raise RuntimeError('Ordem ou identidade dos TEST divergente entre A e E.')

F_PAIR_ROWS=[]
for expected_index,pair in enumerate(F_PAIRS):
    if pair.get('split')!='test' or pair.get('camera_channel')!='CAM_FRONT':
        raise RuntimeError('F recebeu vista fora do TEST/CAM_FRONT.')
    if int(pair.get('width',0))!=F_WIDTH or int(pair.get('height',0))!=F_HEIGHT:
        raise RuntimeError('Dimensão declarada da tríade incorreta.')
    if int(pair['timestamp'])!=int(F_A_PAIRS[expected_index]['timestamp']):
        raise RuntimeError('Timestamp da tríade não corresponde à A.')
    expected_basename=f'eval_img_{expected_index:04d}.png'
    for domain in ('real','nerfacto','splatfacto'):
        path=f_under(pair[f'{domain}_rgb_path'],F_E_ROOT)
        if path.name!=expected_basename or path.suffix.lower()!='.png':
            raise RuntimeError('Nome/formato incorreto da imagem '+domain+': '+str(path))
        f_file_sha(path,pair[f'{domain}_sha256'],f'{domain} TEST {expected_index}')
        with Image.open(path) as im:
            if im.size!=(F_WIDTH,F_HEIGHT):
                raise RuntimeError('Imagem com tamanho errado: '+str(path))
    F_PAIR_ROWS.append({'image_index':expected_index,
                        'file_path':pair['nerfstudio_file_path'],
                        'timestamp':int(pair['timestamp']),
                        'source_type':pair.get('source_type'),
                        'camera_channel':'CAM_FRONT',
                        'real_image':pair['real_rgb_path'],
                        'nerfacto_image':pair['nerfacto_rgb_path'],
                        'splatfacto_image':pair['splatfacto_rgb_path']})

F_STAMP=datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
F_RESULTS=F_E_ROOT/('F_yolo_'+F_STAMP)
F_WORKERS=F_RESULTS/'_workers'
F_ANNOTATED=F_RESULTS/'annotated'
F_TRIPLETS=F_RESULTS/'yolo_triplets'
for folder in (F_RESULTS,F_WORKERS,F_ANNOTATED,F_TRIPLETS):
    folder.mkdir(parents=True,exist_ok=False)
F_LOG=F_RESULTS/'yolo_worker.log'
F_DET_CSV=F_RESULTS/'yolo_detections_all.csv'
F_INF_CSV=F_RESULTS/'yolo_inference_per_image.csv'
F_WORKER_JSON=F_RESULTS/'yolo_worker_metadata.json'
F_BY_IMAGE_CSV=F_RESULTS/'yolo_summary_by_image.csv'
F_BY_CLASS_CSV=F_RESULTS/'yolo_summary_by_class.csv'
F_OVERALL_CSV=F_RESULTS/'yolo_summary_overall.csv'
F_SENS_CSV=F_RESULTS/'yolo_matching_sensitivity.csv'
F_MATCH_CSV=F_RESULTS/'yolo_match_details.csv'
F_COUNTS_CSV=F_RESULTS/'yolo_summary_detection_counts.csv'
F_BY_CAMERA_CSV=F_RESULTS/'yolo_summary_by_camera.csv'
F_COMMON_VALID_CSV=F_RESULTS/'yolo_common_valid_examples.csv'
F_TIMING_CSV=F_RESULTS/'yolo_timing.csv'
F_COMPARISON_CSV=F_RESULTS/'yolo_comparison_by_image_and_class.csv'
F_GALLERY_PATH=F_RESULTS/'galeria_YOLO_REAL_NERFACTO_SPLATFACTO.jpg'
F_METADATA_JSON=F_RESULTS/'yolo_run_metadata.json'
F_ZIP_PATH=F_RESULTS/'F_V7_yolo_semantic_preservation_nuscenes.zip'
F_INPUT_CSV=F_RESULTS/'yolo_image_pairs_used.csv'
f_csv(F_INPUT_CSV,F_PAIR_ROWS,list(F_PAIR_ROWS[0]))

print('='*98)
print('F V7.1 — YOLOv8x / TEST 10 vistas CAM_FRONT / 30 inferências')
print('Proveniência A -> D -> E íntegra; imagem original, Nerfacto e Splatfacto conferidos.')
print('Dataset:',F_DATASET)
print('Modelo:',F_YOLO_MODEL,'conf:',F_CONF,'NMS:',F_NMS_IOU,'imgsz:',F_IMGSZ)
print('Matching principal:',F_PRIMARY_MATCH_IOU,'sensibilidade:',F_MATCH_IOU_LEVELS)
print('Saída local:',F_RESULTS)

# ------------------------------ WORKER YOLO --------------------------------
F_WORKER=F_WORKERS/'yolo_worker.py'
F_WORKER_SOURCE=r'''
from pathlib import Path
import argparse,csv,hashlib,json,random,sys,time
import numpy as np
import torch
import ultralytics
from ultralytics import YOLO

DETECTION_COLUMNS=['domain','image_index','camera_channel','timestamp','image_path',
 'det_index','class_id','class_name','confidence','x1','y1','x2','y2']
INFERENCE_COLUMNS=['domain','image_index','camera_channel','timestamp','image_path',
 'detection_count','preprocess_ms','inference_ms','postprocess_ms','total_reported_ms']

def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as stream:
        for chunk in iter(lambda:stream.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()

def csv_out(path,header,records):
    with open(path,'w',encoding='utf-8',newline='') as stream:
        writer=csv.DictWriter(stream,fieldnames=header)
        writer.writeheader();writer.writerows(records)

def main():
    ap=argparse.ArgumentParser()
    for option in ('manifest','detections','inference','metadata','model'):
        ap.add_argument('--'+option,required=True)
    ap.add_argument('--conf',type=float,required=True)
    ap.add_argument('--nms',type=float,required=True)
    ap.add_argument('--imgsz',type=int,required=True)
    ap.add_argument('--max-det',type=int,required=True)
    ap.add_argument('--classes',required=True)
    args=ap.parse_args()
    random.seed(42);np.random.seed(42);torch.manual_seed(42)
    if not torch.cuda.is_available():raise RuntimeError('YOLO: CUDA indisponível.')
    torch.cuda.manual_seed_all(42)
    pairs=json.loads(Path(args.manifest).read_text(encoding='utf-8'))['pairs']
    if len(pairs)!=10:raise RuntimeError('Worker espera 10 TEST.')
    classes=[int(c) for c in args.classes.split(',')]
    start=time.perf_counter();model=YOLO(args.model)
    model_load_seconds=time.perf_counter()-start
    weights=None
    for value in (getattr(model,'ckpt_path',None),args.model):
        if value and Path(str(value)).is_file():
            weights=Path(str(value)).resolve();break
    if weights is None:raise RuntimeError('Arquivo de pesos YOLO não encontrado para SHA256.')
    weights_sha=sha(weights)
    first=pairs[0]['real_rgb_path']
    model.predict(str(first),imgsz=args.imgsz,conf=args.conf,iou=args.nms,
                  classes=classes,max_det=args.max_det,device=0,verbose=False,save=False)
    torch.cuda.synchronize()
    det=[];inf=[];start=time.perf_counter();counter=0
    for pair in pairs:
        index=int(pair['image_index'])
        for domain in ('real','nerfacto','splatfacto'):
            path=str(pair[domain+'_rgb_path'])
            r=model.predict(str(path),imgsz=args.imgsz,conf=args.conf,iou=args.nms,
                            classes=classes,max_det=args.max_det,device=0,
                            verbose=False,save=False)
            if len(r)!=1:raise RuntimeError('YOLO retornou número inesperado de resultados.')
            result=r[0]
            boxes=result.boxes
            ndet=len(boxes) if boxes is not None else 0
            if ndet:
                xyxy=boxes.xyxy.cpu().numpy()
                confidences=boxes.conf.cpu().numpy()
                category=boxes.cls.cpu().numpy().astype(int)
                for j,(box,score,cls) in enumerate(zip(xyxy,confidences,category)):
                    if int(cls) not in classes:raise RuntimeError('YOLO retornou classe não solicitada.')
                    det.append({'domain':domain,'image_index':index,'camera_channel':'CAM_FRONT',
                        'timestamp':int(pair['timestamp']),'image_path':path,'det_index':j,
                        'class_id':int(cls),'class_name':str(result.names[int(cls)]),
                        'confidence':float(score),'x1':float(box[0]),'y1':float(box[1]),
                        'x2':float(box[2]),'y2':float(box[3])})
            speed=result.speed if isinstance(result.speed,dict) else {}
            preprocess=float(speed.get('preprocess',0.) or 0.)
            inference=float(speed.get('inference',0.) or 0.)
            post=float(speed.get('postprocess',0.) or 0.)
            inf.append({'domain':domain,'image_index':index,'camera_channel':'CAM_FRONT',
                'timestamp':int(pair['timestamp']),'image_path':path,'detection_count':ndet,
                'preprocess_ms':preprocess,'inference_ms':inference,'postprocess_ms':post,
                'total_reported_ms':preprocess+inference+post})
            counter+=1
            print(f'YOLO [{counter:02d}/30] TEST {index+1:02d} {domain:<10} detecções={ndet}',flush=True)
    torch.cuda.synchronize();wall_seconds=time.perf_counter()-start
    csv_out(args.detections,DETECTION_COLUMNS,det)
    csv_out(args.inference,INFERENCE_COLUMNS,inf)
    metadata={'model_requested':args.model,'weights_path':str(weights),
      'weights_sha256':weights_sha,'model_load_seconds':model_load_seconds,
      'torch':torch.__version__,'ultralytics':ultralytics.__version__,
      'python':sys.version.split()[0],'cuda_runtime':torch.version.cuda,
      'gpu':torch.cuda.get_device_name(0),'conf':args.conf,'nms_iou':args.nms,
      'imgsz':args.imgsz,'max_det':args.max_det,'classes':classes,
      'triplets':len(pairs),'images_processed':counter,'detections_total':len(det),
      'inference_wall_seconds':wall_seconds,'warmup_excluded':True}
    Path(args.metadata).write_text(json.dumps(metadata,indent=2)+'\n',encoding='utf-8')
    print('F_WORKER_COMPLETED=30',flush=True)

if __name__=='__main__':main()
'''
F_WORKER.write_text(F_WORKER_SOURCE,encoding='utf-8')
compile(F_WORKER_SOURCE,str(F_WORKER),'exec')

F_ENV=dict(globals().get('TCC_SUBPROCESS_ENV',os.environ))
for var in ('PYTHONHOME','PYTHONPATH','PYTHONUSERBASE','PIP_TARGET'):
    F_ENV.pop(var,None)
F_ENV.update({'PATH':str(F_BIN)+':'+str(F_CUDA/'bin')+':'+os.environ.get('PATH',''),
              'VIRTUAL_ENV':str(F_TCC_ENV),'PYTHONNOUSERSITE':'1',
              'PYTHONUNBUFFERED':'1','QT_QPA_PLATFORM':'offscreen'})
if (F_CUDA/'lib64').is_dir():
    F_ENV['LD_LIBRARY_PATH']=str(F_CUDA/'lib64')+':'+F_ENV.get('LD_LIBRARY_PATH','')

# Pré-teste do próprio interpretador que será usado pelo subprocesso YOLO.
# Não instalar nada aqui nem fazer fallback para Python global: SETUP/1
# precisa ter estabelecido um tcc_env reprodutível.
F_ENV_CHECK_SOURCE = r'''
import json
import sys
from pathlib import Path
expected = Path(sys.argv[1]).resolve()
actual = Path(sys.prefix).resolve()
if actual != expected:
    raise RuntimeError(
        'Python fora do tcc_env. sys.prefix=' + str(actual) +
        ', esperado=' + str(expected) + '. Execute pelo bin/python do venv, sem resolve().'
    )
import numpy
import torch
import ultralytics
from PIL import Image
if not torch.cuda.is_available():
    raise RuntimeError('PyTorch no tcc_env não detecta GPU CUDA.')
print('F_TCC_ENV_OK=' + json.dumps({
    'sys_executable': sys.executable,
    'sys_prefix': sys.prefix,
    'numpy': numpy.__version__,
    'torch': torch.__version__,
    'ultralytics': ultralytics.__version__,
    'gpu': torch.cuda.get_device_name(0),
}), flush=True)
'''
print('F V7.1 — validando interpretador virtual:', F_PYTHON, flush=True)
F_ENV_PROBE = subprocess.run(
    [str(F_PYTHON), '-c', F_ENV_CHECK_SOURCE, str(F_TCC_ENV)],
    env=F_ENV, capture_output=True, text=True, timeout=240, check=False,
)
F_PROBE_LINE = next((line[len('F_TCC_ENV_OK='):]
                     for line in F_ENV_PROBE.stdout.splitlines()
                     if line.startswith('F_TCC_ENV_OK=')), None)
if F_ENV_PROBE.returncode != 0 or F_PROBE_LINE is None:
    raise RuntimeError(
        'Falha no ambiente virtual da F (antes da inferência YOLO). '
        'Não instale numpy no Python-base. Se faltarem bibliotecas dentro de '
        '/content/tcc_env, reexecute SETUP e Célula 1.\n'
        + F_ENV_PROBE.stdout[-3000:] + '\n' + F_ENV_PROBE.stderr[-3000:]
    )
F_ENV_DETAILS = json.loads(F_PROBE_LINE)
print('Ambiente isolado validado:', F_ENV_DETAILS, flush=True)

F_CMD=[str(F_PYTHON),str(F_WORKER), '--manifest',str(F_E_MANIFEST_PATH),
       '--detections',str(F_DET_CSV),'--inference',str(F_INF_CSV),
       '--metadata',str(F_WORKER_JSON),'--model',F_YOLO_MODEL,
       '--conf',str(F_CONF),'--nms',str(F_NMS_IOU),'--imgsz',str(F_IMGSZ),
       '--max-det',str(F_MAX_DET),'--classes',','.join(map(str,F_CLASSES))]
started=time.perf_counter()
print('Executando YOLO pelo tcc_env:',shlex.join(F_CMD),flush=True)
F_WORKER_TIMED_OUT=[False]
try:
    with F_LOG.open('w',encoding='utf-8') as logfile:
        with subprocess.Popen(F_CMD,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                              text=True,bufsize=1,env=F_ENV) as process:
            assert process.stdout is not None
            def f_timeout_kill():
                if process.poll() is None:
                    F_WORKER_TIMED_OUT[0]=True
                    process.kill()
            timer=threading.Timer(F_WORKER_TIMEOUT_SECONDS, f_timeout_kill)
            timer.daemon=True;timer.start()
            try:
                for line in process.stdout:
                    logfile.write(line);logfile.flush()
                    if 'YOLO [' in line or 'F_WORKER_COMPLETED=' in line:
                        print(line.rstrip(),flush=True)
                code=process.wait()
            finally:
                timer.cancel()
except KeyboardInterrupt:
    raise
F_WALL_SECONDS=time.perf_counter()-started
if F_WORKER_TIMED_OUT[0]:
    raise TimeoutError('YOLO excedeu o limite de execução. Veja '+str(F_LOG))
if code!=0:
    raise RuntimeError('Worker F falhou, veja log '+str(F_LOG)+'\n'+
                       F_LOG.read_text(errors='replace')[-5000:])
for path in (F_DET_CSV,F_INF_CSV,F_WORKER_JSON):
    if not path.is_file():raise FileNotFoundError(path)
F_WORKER_INFO=f_read(F_WORKER_JSON)
if F_WORKER_INFO.get('images_processed')!=30 or F_WORKER_INFO.get('triplets')!=10:
    raise RuntimeError('Worker não processou exatamente 30 imagens em 10 tríades.')
if not F_WORKER_INFO.get('weights_sha256'):
    raise RuntimeError('YOLO não registrou SHA256 dos pesos.')
if (F_EXPECTED_WEIGHTS_SHA256 is not None and
    F_WORKER_INFO['weights_sha256']!=str(F_EXPECTED_WEIGHTS_SHA256).lower()):
    raise RuntimeError('Pesos YOLO divergentes do SHA256 exigido.')

# -------------------------- DETECÇÕES E MATCHING ---------------------------
def f_load_csv(path):
    with Path(path).open('r',newline='',encoding='utf-8') as file:
        return list(csv.DictReader(file))

F_DETECTIONS=f_load_csv(F_DET_CSV)
F_INFERENCE=f_load_csv(F_INF_CSV)
if len(F_INFERENCE)!=30:
    raise RuntimeError('CSV de inferência não tem as 30 imagens previstas.')
F_INDEX={}
for r in F_INFERENCE:
    key=(int(r['image_index']),r['domain'])
    if key in F_INDEX or key[0] not in range(10) or key[1] not in ('real','nerfacto','splatfacto'):
        raise RuntimeError('Registro de inferência ausente, duplicado ou fora de escopo.')
    F_INDEX[key]=r
if len(F_INDEX)!=30:raise RuntimeError('Tríades de inferência incompletas.')
F_DET_BY=defaultdict(list)
for r in F_DETECTIONS:
    index=int(r['image_index']);domain=r['domain']
    if (index,domain) not in F_INDEX or int(r['class_id']) not in F_CLASSES:
        raise RuntimeError('Detecção associada a uma imagem/classe inválida.')
    F_DET_BY[index,domain].append(r)
for key,r in F_INDEX.items():
    if int(r['detection_count'])!=len(F_DET_BY[key]):
        raise RuntimeError('Contagem de detecções divergente: '+str(key))

F_COMP_ROWS=[];F_MATCH_ROWS=[];F_BY_IMAGE=[];F_BY_CLASS={}
for pair in F_PAIRS:
    idx=int(pair['image_index'])
    rr=F_DET_BY[idx,'real']
    for domain in ('nerfacto','splatfacto'):
        pp=F_DET_BY[idx,domain]
        for threshold in F_MATCH_IOU_LEVELS:
            match=f_matches(rr,pp,threshold)
            nr,npred,nm=len(rr),len(pp),len(match)
            rec,prec,f1=f_agreement(nr,npred,nm)
            F_BY_IMAGE.append({'image_index':idx,'camera_channel':'CAM_FRONT',
                               'render_domain':domain,'match_iou':threshold,
                               'real_count':nr,'render_count':npred,'matched_count':nm,
                               'recall_like':rec,'precision_like':prec,'f1_like':f1})
            if threshold==F_PRIMARY_MATCH_IOU:
                classes=sorted({int(x['class_id']) for x in rr+pp})
                for cls in classes:
                    rc=[x for x in rr if int(x['class_id'])==cls]
                    pc=[x for x in pp if int(x['class_id'])==cls]
                    mc=[m for m in match if int(rr[m['real_index']]['class_id'])==cls]
                    nrr,npp,nmm=len(rc),len(pc),len(mc)
                    rrec,rprec,rf1=f_agreement(nrr,npp,nmm)
                    label=(rc or pc)[0]['class_name']
                    F_COMP_ROWS.append({'image_index':idx,'camera_channel':'CAM_FRONT',
                        'render_domain':domain,'match_iou':threshold,'class_id':cls,'class_name':label,
                        'real_count':nrr,'render_count':npp,'matched_count':nmm,
                        'recall_like':rrec,'precision_like':rprec,'f1_like':rf1,
                        'matched_iou_sum':sum(m['iou'] for m in mc)})
                for m in match:
                    a=rr[m['real_index']];b=pp[m['render_index']]
                    F_MATCH_ROWS.append({'image_index':idx,'render_domain':domain,
                        'class_id':int(a['class_id']),'class_name':a['class_name'],
                        'real_det_index':int(a['det_index']),
                        'render_det_index':int(b['det_index']),
                        'real_confidence':float(a['confidence']),
                        'render_confidence':float(b['confidence']),
                        'iou':m['iou'],
                        'real_box':[float(a[x]) for x in ('x1','y1','x2','y2')],
                        'render_box':[float(b[x]) for x in ('x1','y1','x2','y2')]})

F_OVERALL=[f_aggregate(F_COMP_ROWS,domain,F_PRIMARY_MATCH_IOU)
           for domain in ('nerfacto','splatfacto')]
F_SENSITIVITY=[]
for threshold in F_MATCH_IOU_LEVELS:
    for domain in ('nerfacto','splatfacto'):
        selected=[r for r in F_BY_IMAGE if r['render_domain']==domain and r['match_iou']==threshold]
        nr=sum(r['real_count'] for r in selected)
        npred=sum(r['render_count'] for r in selected)
        nm=sum(r['matched_count'] for r in selected)
        rec,prec,f1=f_agreement(nr,npred,nm)
        F_SENSITIVITY.append({'render_domain':domain,'match_iou':threshold,
            'real_count':nr,'render_count':npred,'matched_count':nm,
            'recall_like':rec,'precision_like':prec,'f1_like':f1})

# Resumo por classe (micro-agregado nas 10 vistas) e contagem bruta por domínio.
F_CLASS_ROWS=[]
for domain in ('nerfacto','splatfacto'):
    classes=sorted({int(row['class_id']) for row in F_COMP_ROWS
                    if row['render_domain']==domain})
    for cls in classes:
        subset=[x for x in F_COMP_ROWS if x['render_domain']==domain and x['class_id']==cls]
        nr=sum(x['real_count'] for x in subset)
        npred=sum(x['render_count'] for x in subset)
        nm=sum(x['matched_count'] for x in subset)
        rec,prec,f1=f_agreement(nr,npred,nm)
        F_CLASS_ROWS.append({'render_domain':domain,'class_id':cls,
            'class_name':subset[0]['class_name'],'real_count':nr,'render_count':npred,
            'matched_count':nm,'recall_like':rec,'precision_like':prec,'f1_like':f1,
            'mean_matched_iou':sum(x['matched_iou_sum'] for x in subset)/nm if nm else None})
F_COUNTS=[]
for domain in ('real','nerfacto','splatfacto'):
    for cls in sorted({int(r['class_id']) for r in F_DETECTIONS if r['domain']==domain}):
        subset=[r for r in F_DETECTIONS if r['domain']==domain and int(r['class_id'])==cls]
        F_COUNTS.append({'domain':domain,'class_id':cls,'class_name':subset[0]['class_name'],
                         'detections':len(subset),
                         'mean_confidence':sum(float(r['confidence']) for r in subset)/len(subset)})
F_TIMING=[]
for domain in ('real','nerfacto','splatfacto'):
    subset=[r for r in F_INFERENCE if r['domain']==domain]
    times=[float(r['inference_ms']) for r in subset]
    F_TIMING.append({'domain':domain,'images':len(subset),
      'detections':sum(int(r['detection_count']) for r in subset),
      'mean_inference_ms':sum(times)/len(times),
      'mean_preprocess_ms':sum(float(r['preprocess_ms']) for r in subset)/len(subset),
      'mean_postprocess_ms':sum(float(r['postprocess_ms']) for r in subset)/len(subset),
      'note':'Tempo inferência Ultralytics; não é FPS da reconstrução'})


# CAM_FRONT é a única câmera; publicar tabela dedicada para facilitar consolidação.
F_BY_CAMERA=[{'camera_channel':'CAM_FRONT',**row} for row in F_OVERALL]

# Objetos reais que permanecem detectados por AMBOS os métodos (não só a mesma classe).
F_MATCH_MAP={(int(m['image_index']),int(m['real_det_index']),m['render_domain']):m
             for m in F_MATCH_ROWS}
F_COMMON_VALID=[]
for m in F_MATCH_ROWS:
    if m['render_domain']!='nerfacto':continue
    idx=int(m['image_index']);rid=int(m['real_det_index'])
    other=F_MATCH_MAP.get((idx,rid,'splatfacto'))
    if other is None:continue
    F_COMMON_VALID.append({
        'image_index':idx,'real_det_index':rid,
        'class_id':m['class_id'],'class_name':m['class_name'],
        'nerfacto_render_det_index':m['render_det_index'],
        'splatfacto_render_det_index':other['render_det_index'],
        'nerfacto_iou':m['iou'],'splatfacto_iou':other['iou']})
F_COMMON_VALID.sort(key=lambda r:(r['image_index'],r['real_det_index']))


f_csv(F_COMPARISON_CSV,F_COMP_ROWS,
      ['image_index','camera_channel','render_domain','match_iou','class_id','class_name','real_count',
       'render_count','matched_count','recall_like','precision_like','f1_like','matched_iou_sum'])
f_csv(F_MATCH_CSV,F_MATCH_ROWS,
      ['image_index','render_domain','class_id','class_name','real_det_index',
       'render_det_index','real_confidence','render_confidence','iou','real_box','render_box'])
f_csv(F_BY_IMAGE_CSV,F_BY_IMAGE,
      ['image_index','camera_channel','render_domain','match_iou','real_count',
       'render_count','matched_count','recall_like','precision_like','f1_like'])
f_csv(F_BY_CLASS_CSV,F_CLASS_ROWS,
      ['render_domain','class_id','class_name','real_count','render_count','matched_count',
       'recall_like','precision_like','f1_like','mean_matched_iou'])
f_csv(F_OVERALL_CSV,F_OVERALL,
      ['render_domain','match_iou','real_count','render_count','matched_count',
       'recall_like','precision_like','f1_like','mean_matched_iou','detection_retention_ratio'])
f_csv(F_SENS_CSV,F_SENSITIVITY,
      ['render_domain','match_iou','real_count','render_count','matched_count',
       'recall_like','precision_like','f1_like'])
f_csv(F_COUNTS_CSV,F_COUNTS,
      ['domain','class_id','class_name','detections','mean_confidence'])
f_csv(F_BY_CAMERA_CSV,F_BY_CAMERA,
      ['camera_channel','render_domain','match_iou','real_count','render_count',
       'matched_count','recall_like','precision_like','f1_like',
       'mean_matched_iou','detection_retention_ratio'])
f_csv(F_COMMON_VALID_CSV,F_COMMON_VALID,
      ['image_index','real_det_index','class_id','class_name',
       'nerfacto_render_det_index','splatfacto_render_det_index',
       'nerfacto_iou','splatfacto_iou'])
f_csv(F_TIMING_CSV,F_TIMING,
      ['domain','images','detections','mean_inference_ms','mean_preprocess_ms',
       'mean_postprocess_ms','note'])

# --------------------- ANOTAÇÕES E GALERIA DE 10 TRÍADES --------------------
def f_draw_boxes(path,detections):
    im=f_rgb(path)
    draw=ImageDraw.Draw(im)
    font=ImageFont.load_default()
    for r in detections:
        x1,y1,x2,y2=[float(r[k]) for k in ('x1','y1','x2','y2')]
        x1=max(0,min(F_WIDTH-1,x1));x2=max(0,min(F_WIDTH-1,x2))
        y1=max(0,min(F_HEIGHT-1,y1));y2=max(0,min(F_HEIGHT-1,y2))
        if x1>=x2 or y1>=y2:continue
        draw.rectangle((x1,y1,x2,y2),outline=(255,255,255),width=2)
        caption=f"{r['class_name']} {float(r['confidence']):.2f}"
        box=draw.textbbox((0,0),caption,font=font)
        w,h=box[2]-box[0],box[3]-box[1]
        tx=max(0,min(F_WIDTH-w-6,int(x1)))
        ty=max(0,int(y1)-h-6)
        draw.rectangle((tx,ty,tx+w+4,ty+h+4),fill=(0,0,0))
        draw.text((tx+2,ty+2),caption,fill=(255,255,255),font=font)
    return im

F_CONTACT=[]
for pair in F_PAIRS:
    idx=int(pair['image_index'])
    canvas=Image.new('RGB',(3*400,225+34),(248,249,250))
    draw=ImageDraw.Draw(canvas)
    for col,(domain,label) in enumerate((('real','REAL'),('nerfacto','NERFACTO-BIG'),
                                          ('splatfacto','SPLATFACTO'))):
        img=f_draw_boxes(pair[domain+'_rgb_path'],F_DET_BY[idx,domain])
        out=F_ANNOTATED/f'eval_img_{idx:04d}_{domain}.png'
        img.save(out,format='PNG')
        draw.text((col*400+8,9),f'TEST {idx+1:02d} {label} / det={len(F_DET_BY[idx,domain])}',
                  fill=(15,25,35))
        canvas.paste(img.resize((400,225),Image.Resampling.LANCZOS),(col*400,34))
    canvas.save(F_TRIPLETS/f'eval_img_{idx:04d}_YOLO.jpg',format='JPEG',quality=94)
    F_CONTACT.append(canvas)
F_GALLERY=Image.new('RGB',(1200,len(F_CONTACT)*259),(255,255,255))
for i,item in enumerate(F_CONTACT):F_GALLERY.paste(item,(0,i*259))
F_GALLERY.save(F_GALLERY_PATH,format='JPEG',quality=93)
if F_SHOW_GALLERY:
    try:
        from IPython.display import display
        display(F_GALLERY)
    except ImportError:
        print('Galeria criada:',F_GALLERY_PATH)

# ------------------------- RELATÓRIO, ZIP LOCAL, INTERFACE G ----------------
F_REPORT={
    'experiment':'F_V7_yolo_semantic_preservation_scene0655_95_10_10',
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'status':'completed',
    'protocol':F_EXPECTED_PROTOCOL,
    'methodology':{
       'purpose':'Concordância de detecções nas 10 vistas TEST REAL x Nerfacto x Splatfacto',
       'reference':'YOLOv8x na REAL, pseudo-referência; NAO ground truth humano',
       'not_mean_average_precision':True,
       'not_human_annotated':True,
       'same_weights_and_thresholds_all_domains':True,
       'matching':'Bipartido 1:1 máxima cardinalidade por classe; limiar IoU',
       'primary_match_iou':F_PRIMARY_MATCH_IOU,
       'sensitivity_iou':list(F_MATCH_IOU_LEVELS),
       'aggregation':'micro (soma detecções nas 10 imagens antes de calcular proporções)',
       'test_used_only_for_final_evaluation':True,
    },
    'source_E':{'manifest':str(F_E_MANIFEST_PATH),'manifest_sha256':f_hash(F_E_MANIFEST_PATH),
                'source_D_report':str(F_D_REPORT_PATH),
                'source_D_report_sha256':f_hash(F_D_REPORT_PATH)},
    'dataset':{'path':str(F_DATASET),'scene':'scene-0655','camera':'CAM_FRONT',
               'split':'test','triplets':10,'images_processed':30,
               'resolution':[F_WIDTH,F_HEIGHT],
               'core_sha256':F_CORE,
               'dataset_manifest_sha256_actual_run':F_CORE['dataset_manifest.json']},
    'models':F_E['models'],
    'yolo':{'model':F_YOLO_MODEL,'weights_sha256':F_WORKER_INFO['weights_sha256'],
             'ultralytics_version':F_WORKER_INFO['ultralytics'],
              'isolated_python_environment':F_ENV_DETAILS,
             'torch_version':F_WORKER_INFO['torch'], 'gpu':F_WORKER_INFO['gpu'],
             'conf':F_CONF,'nms_iou':F_NMS_IOU,'imgsz':F_IMGSZ,
             'classes':F_CLASSES,'max_det':F_MAX_DET,'seed':F_SEED},
    'inference':{'images':30,'worker_wall_seconds':F_WORKER_INFO['inference_wall_seconds'],
                  'outer_process_seconds_including_load':F_WALL_SECONDS,
                  'timing_note':'YOLO inference não mede FPS do 3DGS/NeRF',
                  'detections_total':len(F_DETECTIONS)},
    'overall_iou_0_50':F_OVERALL,
    'sensitivity':F_SENSITIVITY,
    'semantic_agreement_by_class':F_CLASS_ROWS,
    'same_real_detections_matched_by_both':len(F_COMMON_VALID),
    'images':F_BY_IMAGE,
    'artifacts':{'root':str(F_RESULTS),'overall_csv':str(F_OVERALL_CSV),
                  'by_image_csv':str(F_BY_IMAGE_CSV),
                  'detections_csv':str(F_DET_CSV),
                  'matches_csv':str(F_MATCH_CSV),
                  'gallery':str(F_GALLERY_PATH)},
    'drive_upload':False,
}
f_write(F_METADATA_JSON,F_REPORT)
F_README=F_RESULTS/'LEIA_ME.txt'
F_README.write_text(
    'F V7 — YOLO nas dez tríades do nuScenes (95 TRAIN / 10 VAL / 10 TEST).\n'
    'Pesos: '+F_YOLO_MODEL+'; SHA256: '+F_WORKER_INFO['weights_sha256']+'\n'
    '0.25 conf / 0.70 NMS / imgsz 1280 / classes COCO 0,1,2,3,5,7.\n'
    'IoU matching: 0.3, 0.5 (principal), 0.7; matching por classe 1:1.\n'
    'PSNR/SSIM/LPIPS são calculados na D; F calcula concordância YOLO.\n'
    'A referência YOLO na REAL NÃO é anotação humana nem mAP.\n'
    'TEST não deve ser usado para modificar ou selecionar checkpoints posteriormente.\n'
    'Este ZIP F é intermediário LOCAL. Somente G V7 exportará análise final ao Drive.\n',
    encoding='utf-8')

if F_CREATE_LOCAL_ZIP:
    with zipfile.ZipFile(F_ZIP_PATH,'w',zipfile.ZIP_DEFLATED,compresslevel=5) as archive:
        for path in sorted(F_RESULTS.rglob('*')):
            if path.is_file() and path!=F_ZIP_PATH:
                if path.is_symlink():raise RuntimeError('Não incluir symlinks no ZIP da F.')
                archive.write(path,path.relative_to(F_RESULTS).as_posix())
    with zipfile.ZipFile(F_ZIP_PATH) as archive:
        if archive.testzip() is not None:
            raise RuntimeError('ZIP F corrompido.')

# Alias de integração para a célula G.
YOLO_RESULTS_DIR=F_RESULTS
YOLO_METADATA_JSON=F_METADATA_JSON
YOLO_GALLERY_PATH=F_GALLERY_PATH
F_FINAL_REPORT=F_REPORT

print('\n'+'='*98)
print('F V7 — YOLO CONCLUÍDO: 10 TRÍADES / 30 INFERÊNCIAS; TEST NÃO USADO PARA TREINO')
for item in F_OVERALL:
    print(f"{item['render_domain']:10s} | REAL {item['real_count']} "
          f"| RENDER {item['render_count']} | MATCH {item['matched_count']} "
          f"| F1-like@0.50: {item['f1_like'] if item['f1_like'] is not None else 'N/D'}")
print('Metadados:',F_METADATA_JSON)
print('Métricas YOLO:',F_OVERALL_CSV)
print('Galeria:',F_GALLERY_PATH)
if F_CREATE_LOCAL_ZIP:print('ZIP LOCAL:',F_ZIP_PATH)
print('Google Drive: nenhum upload (reservado à consolidação na G V7).')


In [ ]:
# =============================================================================
# CÉLULA G V10.1 — CONSOLIDAÇÃO FINAL + TODOS OS FRAMES scene-0655 / CAM_FRONT / 95-10-10
# =============================================================================
# Executar após: 0 -> SETUP -> 1 -> A V7 -> B V7 -> C V7 -> D V7
#                -> E V7 -> F V7 -> ESTA G V7.
# Não treina, não executa YOLO e não seleciona checkpoints; renderiza 10 VAL APÓS seleção e reutiliza 10 TEST; não renderiza TRAIN.
# Entrega um pacote de análise completo e exporta SOMENTE O ZIP FINAL ao Drive.
# Nenhum checkpoint (.ckpt), PLY ou outro artefato intermediário é exportado.
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
from html import escape as html_escape
from PIL import Image, ImageDraw, ImageFont
import csv
import hashlib
import json
import os
import math
import re
import shutil
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------- CONFIGURAÇÃO --------------------------------
G_PROTOCOL = 'A_V7_HV3_aligned_95_10_10'
G_EXPECTED_F = 'F_V7_yolo_semantic_preservation_scene0655_95_10_10'
G_EXPECTED_E = 'E_V7_triplets_yolo_scene0655_95_10_10'
G_EXPECTED_D = 'D_V7_paired_final_test_scene0655_95_10_10'
G_EXPECTED_PROFILE = 'nuscenes_front_samples_train_sweeps_holdout_10_10_v7'
G_SPLITS = {'train':95, 'val':10, 'test':10}
G_RESOLUTION = (800,450)
G_EXPECTED_CLASSES = [0,1,2,3,5,7]
G_EXPECTED_IOU_LEVELS = [0.30,0.50,0.70]
G_SHOW_PREVIEW = True
G_MAX_SEMANTIC_EXAMPLES = 4
G_CREATE_EXCEL = True
G_SEND_FINAL_ZIP_TO_DRIVE = False  # só o pacote CANÔNICO será enviado ao Drive
G_DRIVE_TARGET = Path(os.environ.get('TCC_EXPORT_DIR', '/content/tcc_results'))
G_STAMP = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')

# ----------------------------- HELPERS --------------------------------------
def g_load(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def g_write(path,obj):
    path=Path(path)
    path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(obj,ensure_ascii=False,indent=2,allow_nan=False,default=str)+'\n',encoding='utf-8')

def g_hash(path):
    sha=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for part in iter(lambda:stream.read(1024*1024),b''):
            sha.update(part)
    return sha.hexdigest()

def g_require(path,label):
    if path is None or str(path).strip() in ('','None'):
        raise RuntimeError('Caminho obrigatório ausente: '+label)
    p=Path(path).resolve()
    if not p.is_file():
        raise FileNotFoundError(f'{label}: {p}')
    return p

def g_eq(a,b,label):
    if a!=b:
        raise RuntimeError(f'{label} diverge: esperado {b!r}; obtido {a!r}')

def g_near(a,b,label,tol=1e-4):
    if a is None or not math.isclose(float(a),float(b),rel_tol=0.,abs_tol=tol):
        raise RuntimeError(f'{label} diverge: {a} != {b}')

def g_num(v):
    try:
        z=float(v)
        return z if math.isfinite(z) else None
    except (ValueError,TypeError):
        return None

def g_csv(path,rows,fields=None):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    if fields is None:
        fields=list(rows[0]) if rows else []
    with path.open('w',newline='',encoding='utf-8') as stream:
        writer=csv.DictWriter(stream,fieldnames=fields,extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)

def g_copy(src,dest):
    src=g_require(src,'arquivo de origem');dest=Path(dest)
    dest.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(src,dest)
    if g_hash(src)!=g_hash(dest):
        raise RuntimeError('Cópia com hash diferente: '+str(dest))
    return dest

def g_img(path):
    with Image.open(path) as im:
        im=im.convert('RGB')
        if im.size!=G_RESOLUTION:
            raise RuntimeError(f'Imagem RGB inválida: {path}, {im.size}')
        return im.copy()

def g_label_image(im,title):
    panel=Image.new('RGB',(im.width,im.height+34),'white')
    panel.paste(im,(0,34))
    ImageDraw.Draw(panel).text((8,9),title,fill='black')
    return panel

def g_triplet(imlist,titles,notes=()):
    panels=[g_label_image(im,title) for im,title in zip(imlist,titles)]
    w,h=panels[0].size
    footer_h=14+20*len(notes) if notes else 0
    result=Image.new('RGB',(w*3,h+footer_h),'white')
    for k,p in enumerate(panels):
        result.paste(p,(k*w,0))
    drawer=ImageDraw.Draw(result)
    for i,note in enumerate(notes):
        drawer.text((8,h+5+i*20),str(note),fill='black')
    return result

def g_draw_yolo(image,detections,class_id=None):
    image=image.copy();drawer=ImageDraw.Draw(image)
    font=ImageFont.load_default()
    for det in detections:
        if class_id is not None and int(det['class_id'])!=int(class_id):
            continue
        xy=[g_num(det[k]) for k in ('x1','y1','x2','y2')]
        if any(x is None for x in xy):continue
        x1,y1,x2,y2=xy
        x1=max(0.,min(G_RESOLUTION[0]-1.,x1))
        x2=max(0.,min(G_RESOLUTION[0]-1.,x2))
        y1=max(0.,min(G_RESOLUTION[1]-1.,y1))
        y2=max(0.,min(G_RESOLUTION[1]-1.,y2))
        if x1>=x2 or y1>=y2:continue
        drawer.rectangle((x1,y1,x2,y2),outline='white',width=2)
        label=f"{det['class_name']} {float(det['confidence']):.2f}"
        area=drawer.textbbox((0,0),label,font=font)
        tw,th=area[2]-area[0],area[3]-area[1]
        tx=max(0,min(int(x1),G_RESOLUTION[0]-tw-6))
        ty=max(0,int(y1)-th-6)
        drawer.rectangle((tx,ty,tx+tw+4,ty+th+4),fill='black')
        drawer.text((tx+2,ty+2),label,font=font,fill='white')
    return image

def g_bar(chart_path,metric,title,ylabel,ylim=None):
    values=[g_num(x[metric]) for x in G_ROWS]
    if any(v is None for v in values):
        print('Gráfico não criado (métrica indisponível):',metric)
        return False
    fig,ax=plt.subplots(figsize=(7,4.5))
    bars=ax.bar([r['method'] for r in G_ROWS],values)
    ax.set_title(title);ax.set_ylabel(ylabel);ax.grid(axis='y',alpha=.2)
    if ylim is not None:ax.set_ylim(*ylim)
    for bar,num in zip(bars,values):
        ax.text(bar.get_x()+bar.get_width()/2,bar.get_height(),f'{num:.3f}',
                ha='center',va='bottom',fontsize=9)
    fig.tight_layout();fig.savefig(chart_path,dpi=160);plt.close(fig)
    return True

# ---------------------- FONTE CANÔNICA DESTA SESSÃO -------------------------
for variable in ('OUTPUT_DATASET','YOLO_METADATA_JSON','YOLO_PAIRS_MANIFEST_PATH',
                 'D_REPORT_PATH','B_MANIFEST','C_MANIFEST'):
    if variable not in globals():
        raise RuntimeError('Execute A–F V7 na mesma sessão. Falta: '+variable)

G_DATASET=Path(globals()['OUTPUT_DATASET']).resolve()
G_F_META_PATH=g_require(globals()['YOLO_METADATA_JSON'],'F/yolo_run_metadata.json')
G_E_META_PATH=g_require(globals()['YOLO_PAIRS_MANIFEST_PATH'],'E/render_pairs_manifest.json')
G_D_REPORT_PATH=g_require(globals()['D_REPORT_PATH'],'D/relatorio_D_final.json')
G_B_META_PATH=g_require(globals()['B_MANIFEST'],'B/b_training_manifest.json')
G_C_META_PATH=g_require(globals()['C_MANIFEST'],'C/c_nerfacto_training_manifest.json')
G_F=g_load(G_F_META_PATH);G_E=g_load(G_E_META_PATH);G_D=g_load(G_D_REPORT_PATH)
G_B=g_load(G_B_META_PATH);G_C=g_load(G_C_META_PATH)

print('='*100)
print('CÉLULA G V7 — CONFERÊNCIA DO PIPELINE A -> B/C -> D -> E -> F')
print('='*100)
for label,obj,name in (
    ('F',G_F,G_EXPECTED_F),('E',G_E,G_EXPECTED_E),('D',G_D,G_EXPECTED_D)):
    g_eq(obj.get('experiment'),name,label+'.experiment')
    g_eq(obj.get('status'),'completed',label+'.status')
    g_eq(obj.get('protocol'),G_PROTOCOL,label+'.protocol')
for label,record in (('B',G_B),('C',G_C)):
    g_eq(record.get('status'),'completed_selected_by_validation',label+'.status')
    g_eq(record.get('protocol'),G_PROTOCOL,label+'.protocol')
    g_eq(Path(record['dataset']).resolve(),G_DATASET,label+'.dataset')
for label,value in (('D',G_D['dataset']['path']),('E',G_E['dataset']['path']),
                    ('F',G_F['dataset']['path'])):
    g_eq(Path(value).resolve(),G_DATASET,label+'.dataset')

G_F_DS=G_F['dataset'];G_F_YOLO=G_F['yolo'];G_E_DS=G_E['dataset'];G_D_DS=G_D['dataset']
g_eq(G_F_DS.get('triplets'),10,'F.tríades')
g_eq(G_F_DS.get('images_processed'),30,'F.imagens')
g_eq(G_F_DS.get('resolution'),[800,450],'F.resolução')
g_eq(G_E_DS.get('total_frames'),115,'E.frames totais')
g_eq([G_E_DS.get(k+'_frames') for k in ('train','validation','test')],[95,10,10],'E.splits')
g_eq(G_D_DS.get('splits'),G_SPLITS,'D.splits')
g_eq(G_E_DS.get('profile'),G_EXPECTED_PROFILE,'E.dataset_profile')
g_eq(G_F_DS.get('scene'),'scene-0655','F.scene')
g_eq(G_F_YOLO.get('classes'),G_EXPECTED_CLASSES,'F.classes')
g_eq(G_F_YOLO.get('imgsz'),1280,'F.imgsz')
g_eq(G_F_YOLO.get('max_det'),300,'F.max_det')
g_near(G_F_YOLO.get('conf'),.25,'F.conf')
g_near(G_F_YOLO.get('nms_iou'),.70,'F.NMS')
g_near(G_F['methodology'].get('primary_match_iou'),.50,'F.match IoU')
g_eq(G_F['methodology'].get('sensitivity_iou'),G_EXPECTED_IOU_LEVELS,'F.sensibilidade IoU')
if not G_F_YOLO.get('weights_sha256') or not re.fullmatch(r'[0-9a-f]{64}',G_F_YOLO['weights_sha256'].lower()):
    raise RuntimeError('SHA dos pesos YOLO ausente.')
if not G_D.get('test_consumed') or not G_D['metric_protocol'].get('test_opened_after_val_selection'):
    raise RuntimeError('A D não comprova seleção por VAL antes de TEST.')
if not G_D['metric_protocol'].get('reference_pixels_identical_between_models'):
    raise RuntimeError('A D não confirmou GT pareada.')
g_eq(G_E['source_D']['report_sha256'],g_hash(G_D_REPORT_PATH),'E -> D SHA')
g_eq(G_F['source_E']['manifest_sha256'],g_hash(G_E_META_PATH),'F -> E SHA')
g_eq(G_F['source_E']['source_D_report_sha256'],g_hash(G_D_REPORT_PATH),'F -> D SHA')
g_eq(Path(G_F['source_E']['manifest']).resolve(),G_E_META_PATH,'F -> E caminho')
g_eq(Path(G_E['source_D']['report']).resolve(),G_D_REPORT_PATH,'E -> D caminho')
g_eq(Path(G_C['source_B_manifest']).resolve(),G_B_META_PATH,'C -> B caminho')
g_eq(G_C['source_B_checkpoint_sha256'],G_B['selected_checkpoint_sha256'],'C -> B checkpoint SHA')

G_A_SOURCES={
    'transforms.json':G_DATASET/'transforms.json',
    'eval_pairs.json':G_DATASET/'eval_pairs.json',
    'dataset_manifest.json':G_DATASET/'dataset_manifest.json',
    'dataset_identity.json':G_DATASET/'dataset_identity.json',
}
G_A_HASHES={name:g_hash(g_require(path,'A/'+name)) for name,path in G_A_SOURCES.items()}
G_LIDAR_REL=G_E_DS.get('point_cloud_lidar_path')
if not G_LIDAR_REL:
    raise RuntimeError('Célula E não registrou point_cloud_lidar_path.')
G_PLY=(G_DATASET/G_LIDAR_REL).resolve()
if not G_PLY.is_relative_to(G_DATASET) or not G_PLY.is_file():
    raise RuntimeError('PLY da A inexistente ou fora do dataset.')
G_A_HASHES[G_LIDAR_REL]=g_hash(G_PLY)
for label,values in (('B',G_B['dataset_sha256']),('C',G_C['dataset_sha256']),
                     ('D',G_D_DS['core_sha256']),('E',G_E_DS['core_sha256']),
                     ('F',G_F_DS['core_sha256'])):
    for name,expected in G_A_HASHES.items():
        if name not in values:
            # Identidade em algumas saídas contém somente os três hashes de núcleo.
            if name=='dataset_identity.json':continue
            raise RuntimeError(label+' não registrou SHA obrigatório: '+name)
        g_eq(values[name],expected,label+'/'+name)
g_eq(G_E_DS.get('dataset_manifest_sha256_actual_run'),
     G_A_HASHES['dataset_manifest.json'],'E/A manifesto')
g_eq(G_F_DS.get('dataset_manifest_sha256_actual_run'),
     G_A_HASHES['dataset_manifest.json'],'F/A manifesto')

G_CHECKPOINTS={}
for name,record in (('splatfacto',G_B),('nerfacto',G_C)):
    selection=G_D['model_selection'][name]
    g_eq(int(selection['validation_step']),int(record['selected_step']),name+' seleção')
    g_near(selection['val_mean_psnr_db'],record['validation_mean_psnr_db'],name+' VAL PSNR',1e-6)
    path=g_require(record['selected_checkpoint'],name+' checkpoint')
    g_eq(Path(selection['checkpoint']).resolve(),path,name+' checkpoint do D')
    fingerprint=g_hash(path)
    g_eq(fingerprint,record['selected_checkpoint_sha256'],name+' SHA B/C')
    g_eq(fingerprint,selection['checkpoint_sha256'],name+' SHA D')
    G_CHECKPOINTS[name]={'step':int(record['selected_step']),
        'sha256':fingerprint,'size_mib':round(path.stat().st_size/1048576,3)}

G_D_VIEWS=G_D['per_view']
G_E_PAIRS=sorted(G_E['pairs'],key=lambda p:int(p['image_index']))
g_eq(len(G_D_VIEWS),10,'D/per_view')
g_eq(len(G_E_PAIRS),10,'E/pairs')
g_eq({int(p['image_index']) for p in G_E_PAIRS},set(range(10)),'Índices E')
g_eq({int(p['test_view']) for p in G_D_VIEWS},set(range(1,11)),'Índices D')
G_D_BY_PATH={p['file_path']:p for p in G_D_VIEWS}
g_eq({p['nerfstudio_file_path'] for p in G_E_PAIRS},set(G_D_BY_PATH),'Pares D/E')

print('Protocolos, hashes A, proveniência B/C/D/E/F e checkpoints: OK')

# ---------------------- TABELAS YOLO E INTEGRIDADE --------------------------
G_F_ROOT=G_F_META_PATH.parent
G_F_FILES={
    'pares':'yolo_image_pairs_used.csv',
    'deteccoes':'yolo_detections_all.csv',
    'inferencias':'yolo_inference_per_image.csv',
    'matching':'yolo_match_details.csv',
    'comparacao_classes':'yolo_comparison_by_image_and_class.csv',
    'por_imagem':'yolo_summary_by_image.csv',
    'por_classe':'yolo_summary_by_class.csv',
    'geral':'yolo_summary_overall.csv',
    'sensibilidade':'yolo_matching_sensitivity.csv',
    'exemplos_comuns':'yolo_common_valid_examples.csv',
    'contagens':'yolo_summary_detection_counts.csv',
    'por_camera':'yolo_summary_by_camera.csv',
    'tempos_yolo':'yolo_timing.csv',
}
for key,filename in G_F_FILES.items():
    g_require(G_F_ROOT/filename,'F/'+filename)
G_DET=pd.read_csv(G_F_ROOT/G_F_FILES['deteccoes'])
G_INF=pd.read_csv(G_F_ROOT/G_F_FILES['inferencias'])
G_SEM=pd.read_csv(G_F_ROOT/G_F_FILES['geral'])
G_PER_IMAGE=pd.read_csv(G_F_ROOT/G_F_FILES['por_imagem'])
G_BY_CLASS=pd.read_csv(G_F_ROOT/G_F_FILES['por_classe'])
G_SENS=pd.read_csv(G_F_ROOT/G_F_FILES['sensibilidade'])
G_COMMON=pd.read_csv(G_F_ROOT/G_F_FILES['exemplos_comuns'])
G_MATCH=pd.read_csv(G_F_ROOT/G_F_FILES['matching'])
g_eq(len(G_INF),30,'Inferências YOLO')
for domain in ('real','nerfacto','splatfacto'):
    g_eq(int((G_INF['domain']==domain).sum()),10,'YOLO/'+domain)
for domain in ('nerfacto','splatfacto'):
    g_eq(int((G_SEM['render_domain']==domain).sum()),1,'Resumo YOLO/'+domain)
G_DET_INDEX={}
for record in G_DET.to_dict('records'):
    G_DET_INDEX.setdefault((int(record['image_index']),record['domain']),[]).append(record)

# ----------------------- DIRETÓRIO FINAL LOCAL -----------------------------
G_FINAL_DIR=Path('/content/nuscenes_experiments')/('G_V7_final_'+G_STAMP)
if G_FINAL_DIR.exists():
    raise FileExistsError('Saída já existe; não sobrescrever: '+str(G_FINAL_DIR))
G_FIGURES=G_FINAL_DIR/'figures'
G_CHARTS=G_FIGURES/'charts'
G_RGB_FIG=G_FIGURES/'rgb_triplets'
G_YOLO_FIG=G_FIGURES/'yolo_triplets'
G_SEM_FIG=G_FIGURES/'semantic_examples'
G_IMAGES=G_FINAL_DIR/'images'
G_TABLES=G_FINAL_DIR/'tables'
G_SOURCES=G_FINAL_DIR/'provenance'
for directory in (G_CHARTS,G_RGB_FIG,G_YOLO_FIG,G_SEM_FIG,G_TABLES,G_SOURCES,
                  G_IMAGES/'real',G_IMAGES/'nerfacto',G_IMAGES/'splatfacto'):
    directory.mkdir(parents=True,exist_ok=True)

# Metadados em proveniência; NUNCA incluir o PLY nem checkpoints no ZIP.
G_PROVENANCE_FILES={**G_A_SOURCES,
    'B_training_manifest.json':G_B_META_PATH,
    'C_training_manifest.json':G_C_META_PATH,
    'D_final_report.json':G_D_REPORT_PATH,
    'E_render_pairs_manifest.json':G_E_META_PATH,
    'F_yolo_run_metadata.json':G_F_META_PATH,
    'D_test_image_manifest.json':g_require(G_D['test_image_manifest'],'D/test_image_manifest'),
}
for name,src in G_PROVENANCE_FILES.items():
    g_copy(src,G_SOURCES/name)
for filename in G_F_FILES.values():
    g_copy(G_F_ROOT/filename,G_TABLES/filename)
for extra in ('metricas_test_pareadas.csv','resumo_metodos.csv'):
    g_copy(G_D_REPORT_PATH.parent/extra,G_TABLES/extra)
for filename in ('render_pairs.csv',):
    g_copy(G_E_META_PATH.parent/filename,G_TABLES/filename)

# ----------------------- IMAGENS PAREADAS ----------------------------------
G_IMAGES_INDEX=[]
G_VISUAL_PAIRS=[]
for pair in G_E_PAIRS:
    idx=int(pair['image_index'])
    rel=pair['nerfstudio_file_path']
    drow=G_D_BY_PATH[rel]
    g_eq(int(pair['d_test_view']),int(drow['test_view']),'TEST índice')
    g_eq(Path(pair['real_rgb_path']).name,f'eval_img_{idx:04d}.png','Nome TEST')
    original={}
    for domain in ('real','nerfacto','splatfacto'):
        src=g_require(pair[domain+'_rgb_path'],domain+' TEST '+str(idx))
        g_eq(g_hash(src),pair[domain+'_sha256'],domain+' SHA TEST '+str(idx))
        target=G_IMAGES/domain/(f'eval_img_{idx:04d}.png')
        g_copy(src,target)
        original[domain]=g_img(target)
    for method in ('splatfacto','nerfacto'):
        g_near(pair['d_'+method+'_psnr_db'],drow[method+'_psnr_db'],method+' PSNR TEST',1e-4)
        g_near(pair['d_'+method+'_ssim'],drow[method+'_ssim'],method+' SSIM TEST',1e-5)
        g_near(pair['d_'+method+'_lpips_alex'],drow[method+'_lpips_alex'],method+' LPIPS TEST',1e-5)
    notes=(f"TEST {idx+1:02d} | Splatfacto PSNR {float(drow['splatfacto_psnr_db']):.2f} dB | "
           f"Nerfacto PSNR {float(drow['nerfacto_psnr_db']):.2f} dB",)
    ordered=('real','nerfacto','splatfacto')
    rgb_figure=g_triplet([original[k] for k in ordered],
                        ('REAL / referência','NERFACTO-BIG','SPLATFACTO'),notes)
    rgb_target=G_RGB_FIG/f'RGB_TEST_{idx:02d}.jpg'
    rgb_figure.save(rgb_target,quality=94)
    det_counts={}
    annotated=[]
    for domain in ordered:
        detections=G_DET_INDEX.get((idx,domain),[])
        det_counts[domain]=len(detections)
        annotated.append(g_draw_yolo(original[domain],detections))
    yolo_figure=g_triplet(annotated,
        tuple(f'{domain.upper()} / det={det_counts[domain]}' for domain in ordered),
        (f'TEST {idx+1:02d} | YOLO na REAL é pseudorreferência, não GT humano',))
    yolo_target=G_YOLO_FIG/f'YOLO_TEST_{idx:02d}.jpg'
    yolo_figure.save(yolo_target,quality=94)
    G_IMAGES_INDEX.append({
        'image_index':idx,'source_type':pair.get('source_type'),
        'file_path':rel,'timestamp_us':pair.get('timestamp'),
        'rgb_triplet':rgb_target.relative_to(G_FINAL_DIR).as_posix(),
        'yolo_triplet':yolo_target.relative_to(G_FINAL_DIR).as_posix(),
        'real_detections':det_counts['real'],
        'nerfacto_detections':det_counts['nerfacto'],
        'splatfacto_detections':det_counts['splatfacto'],
        'splatfacto_psnr_db':drow['splatfacto_psnr_db'],
        'nerfacto_psnr_db':drow['nerfacto_psnr_db'],
        'splatfacto_ssim':drow['splatfacto_ssim'],
        'nerfacto_ssim':drow['nerfacto_ssim'],
        'splatfacto_lpips_alex':drow['splatfacto_lpips_alex'],
        'nerfacto_lpips_alex':drow['nerfacto_lpips_alex'],
    })
    if idx in (0,4,9):
        G_VISUAL_PAIRS.append(rgb_target)
    print(f"Tríade {idx+1:02d}/10: REAL={det_counts['real']} "
          f"Nerfacto={det_counts['nerfacto']} Splatfacto={det_counts['splatfacto']}")
g_csv(G_TABLES/'test_images_consolidated.csv',G_IMAGES_INDEX)

# Exemplos objetivos: caixas da MESMA detecção REAL pareadas nos dois métodos.
G_SEMANTIC_SELECTION=[]
if not G_COMMON.empty:
    common=G_COMMON.copy()
    for col in ('nerfacto_iou','splatfacto_iou'):
        common[col]=pd.to_numeric(common[col],errors='coerce')
    common['sum_iou']=common['nerfacto_iou'].fillna(0)+common['splatfacto_iou'].fillna(0)
    used=set()
    for _,record in common.sort_values(['sum_iou','image_index'],ascending=[False,True]).iterrows():
        idx=int(record['image_index'])
        if idx in used:continue
        used.add(idx)
        G_SEMANTIC_SELECTION.append({'image_index':idx,'class_id':int(record['class_id']),
           'class_name':str(record['class_name']),'selection':'same_real_object_matched_in_both',
           'mean_matched_iou_nerfacto':g_num(record['nerfacto_iou']),
           'mean_matched_iou_splatfacto':g_num(record['splatfacto_iou'])})
        if len(G_SEMANTIC_SELECTION)>=G_MAX_SEMANTIC_EXAMPLES:break
if not G_SEMANTIC_SELECTION and not G_DET.empty:
    real_det=G_DET[G_DET['domain']=='real'].copy()
    used=set()
    for _,record in real_det.sort_values('confidence',ascending=False).iterrows():
        idx=int(record['image_index'])
        if idx in used:continue
        used.add(idx)
        G_SEMANTIC_SELECTION.append({'image_index':idx,'class_id':int(record['class_id']),
            'class_name':str(record['class_name']),'selection':'real_detection_only',
            'mean_matched_iou_nerfacto':None,'mean_matched_iou_splatfacto':None})
        if len(G_SEMANTIC_SELECTION)>=G_MAX_SEMANTIC_EXAMPLES:break
for seq,record in enumerate(G_SEMANTIC_SELECTION,1):
    idx=record['image_index'];cid=record['class_id']
    imgs=[g_draw_yolo(g_img(G_IMAGES/domain/f'eval_img_{idx:04d}.png'),
            G_DET_INDEX.get((idx,domain),[]),class_id=cid)
          for domain in ('real','nerfacto','splatfacto')]
    name=f"semantic_{seq:02d}_TEST_{idx:02d}_class_{cid}.jpg"
    target=G_SEM_FIG/name
    g_triplet(imgs,(f'REAL / {record["class_name"]}',
             f'NERFACTO / {record["class_name"]}',
             f'SPLATFACTO / {record["class_name"]}'),
             (f'TEST {idx+1:02d} | {record["selection"]}',)).save(target,quality=94)
    record['figure']=target.relative_to(G_FINAL_DIR).as_posix()
g_csv(G_TABLES/'semantic_examples.csv',G_SEMANTIC_SELECTION,
      ['image_index','class_id','class_name','selection',
       'mean_matched_iou_nerfacto','mean_matched_iou_splatfacto','figure'])

# ------------------------ RESUMO QUANTITATIVO -------------------------------
G_SEM_ROWS={r['render_domain']:r for r in G_SEM.to_dict('records')}
G_ROWS=[]
for method,label,record in (
    ('splatfacto','Splatfacto (3DGS)',G_B),
    ('nerfacto','Nerfacto-Big (NeRF)',G_C)):
    d=G_D['summary'][method]
    semantic=G_SEM_ROWS[method]
    G_ROWS.append({
        'method':label,'method_key':method,
        'selected_step':int(d['selected_step']),
        'val_psnr_db':g_num(d['selection_val_psnr_db']),
        'test_psnr_db':g_num(d['test_psnr_db']),
        'test_ssim':g_num(d['test_ssim']),
        'test_lpips_alex':g_num(d['test_lpips_alex']),
        'training_minutes':g_num(record.get('training_seconds'))/60.0
             if g_num(record.get('training_seconds')) is not None else None,
        'checkpoint_size_mib':G_CHECKPOINTS[method]['size_mib'],
        'render_total_seconds_including_overhead':g_num(d.get('total_process_seconds')),
        'render_seconds_per_image_including_overhead':g_num(d.get('seconds_per_image_including_overhead')),
        'real_yolo_count':int(semantic['real_count']),
        'render_yolo_count':int(semantic['render_count']),
        'matched_yolo_count':int(semantic['matched_count']),
        'recall_like':g_num(semantic.get('recall_like')),
        'precision_like':g_num(semantic.get('precision_like')),
        'f1_like':g_num(semantic.get('f1_like')),
        'mean_matched_iou':g_num(semantic.get('mean_matched_iou')),
        'detection_retention_ratio':g_num(semantic.get('detection_retention_ratio')),
    })
g_csv(G_TABLES/'final_experiment_summary.csv',G_ROWS)
g_write(G_TABLES/'final_experiment_summary.json',G_ROWS)

# Verificação independente das médias de PSNR/SSIM/LPIPS na D.
for method in ('splatfacto','nerfacto'):
    for col in ('psnr_db','ssim','lpips_alex'):
        values=[g_num(row[method+'_'+col]) for row in G_D_VIEWS]
        if any(v is None for v in values):
            raise RuntimeError('Métrica TEST incompleta: '+method+'/'+col)
        g_near(sum(values)/10,G_D['summary'][method]['test_'+col],
               method+' média '+col,5e-6)

G_PLOTS=[]
for key,title,ylabel,limit in (
    ('test_psnr_db','PSNR médio nas 10 vistas TEST','PSNR (dB)',None),
    ('test_ssim','SSIM médio nas 10 vistas TEST','SSIM',(0,1)),
    ('test_lpips_alex','LPIPS Alex nas 10 vistas TEST (menor melhor)','LPIPS',None),
    ('training_minutes','Tempo de treinamento','Minutos',None),
    ('render_total_seconds_including_overhead','Processo completo ns-render TEST','Segundos',None),
    ('f1_like','Concordância YOLO com pseudorreferência REAL','F1-like',(0,1)),
    ('recall_like','Concordância YOLO: recall-like','Recall-like',(0,1)),
):
    target=G_CHARTS/(key+'.png')
    if g_bar(target,key,title,ylabel,limit):
        G_PLOTS.append(target)

if not G_SENS.empty and set(('render_domain','match_iou','f1_like')).issubset(G_SENS.columns):
    fig,ax=plt.subplots(figsize=(7,4.5))
    for domain in ('splatfacto','nerfacto'):
        rows=G_SENS[G_SENS['render_domain']==domain].sort_values('match_iou')
        xs=pd.to_numeric(rows['match_iou'],errors='coerce')
        ys=pd.to_numeric(rows['f1_like'],errors='coerce')
        ax.plot(xs,ys,marker='o',label=domain)
    ax.set(xlabel='IoU de matching',ylabel='F1-like',title='Sensibilidade de concordância YOLO')
    ax.set_ylim(0,1);ax.grid(alpha=.2);ax.legend()
    fig.tight_layout();target=G_CHARTS/'matching_iou_sensitivity.png'
    fig.savefig(target,dpi=160);plt.close(fig);G_PLOTS.append(target)

# ---------------------- EXCEL (COMPLEMENTAR, OPCIONAL) ----------------------
G_XLSX=G_FINAL_DIR/'tabelas_resultados_nuScenes_V7.xlsx'
G_XLSX_OK=False
if G_CREATE_EXCEL:
    try:
        tables={
            'resumo':pd.DataFrame(G_ROWS),
            'qualidade_por_vista':pd.DataFrame(G_D_VIEWS),
            'yolo_geral':G_SEM,
            'yolo_por_imagem':G_PER_IMAGE,
            'yolo_por_classe':G_BY_CLASS,
            'yolo_sensibilidade':G_SENS,
            'yolo_inferencia':G_INF,
            'imagens':pd.DataFrame(G_IMAGES_INDEX),
            'exemplos_semanticos':pd.DataFrame(G_SEMANTIC_SELECTION),
        }
        with pd.ExcelWriter(G_XLSX,engine='openpyxl') as writer:
            for name,data in tables.items():
                data.to_excel(writer,sheet_name=name[:31],index=False)
                ws=writer.sheets[name[:31]]
                ws.freeze_panes='A2'
                ws.auto_filter.ref=ws.dimensions
                for cells in ws.columns:
                    try:
                        maxlen=max(len(str(c.value)) for c in cells[:min(len(cells),70)] if c.value is not None)
                    except ValueError:
                        maxlen=12
                    ws.column_dimensions[cells[0].column_letter].width=min(42,max(12,maxlen+2))
        G_XLSX_OK=True
    except Exception as exc:
        print('AVISO: Excel não gerado; todas as tabelas CSV/JSON permanecem válidas.',repr(exc))
        G_XLSX.unlink(missing_ok=True)

# ----------------------------- RELATÓRIO HTML -------------------------------
def g_html_value(v,decimals=4):
    value=g_num(v)
    return f'{value:.{decimals}f}' if value is not None else 'N/D'

def g_html_table(headers,data):
    head=''.join(f'<th>{html_escape(str(k))}</th>' for k in headers)
    body=''.join('<tr>'+''.join(f'<td>{html_escape(str(value))}</td>' for value in row)+'</tr>' for row in data)
    return '<table><thead><tr>'+head+'</tr></thead><tbody>'+body+'</tbody></table>'

G_HTML=G_FINAL_DIR/'ANALISE_FINAL.html'
summary_headers=['Método','PSNR TEST','SSIM TEST','LPIPS TEST','PSNR VAL',
                 'Treino (min)','F1-like YOLO','Recall-like YOLO']
summary_data=[[r['method'],g_html_value(r['test_psnr_db'],3),g_html_value(r['test_ssim']),
               g_html_value(r['test_lpips_alex']),g_html_value(r['val_psnr_db'],3),
               g_html_value(r['training_minutes'],2),g_html_value(r['f1_like']),
               g_html_value(r['recall_like'])] for r in G_ROWS]
per_headers=['TEST','Origem','PSNR Splat','PSNR Nerfacto','SSIM Splat','SSIM Nerfacto']
per_data=[[p['image_index']+1,p['source_type'],g_html_value(p['splatfacto_psnr_db'],2),
           g_html_value(p['nerfacto_psnr_db'],2),g_html_value(p['splatfacto_ssim']),
           g_html_value(p['nerfacto_ssim'])] for p in G_IMAGES_INDEX]
figure_tags='\n'.join(f'<figure><img loading="lazy" src="{html_escape(p.relative_to(G_FINAL_DIR).as_posix())}"><figcaption>TEST {i+1:02d}: imagem real, Nerfacto e Splatfacto</figcaption></figure>'
                       for i,p in enumerate(sorted(G_RGB_FIG.glob('*.jpg'))))
yolo_tags='\n'.join(f'<figure><img loading="lazy" src="{html_escape(p.relative_to(G_FINAL_DIR).as_posix())}"><figcaption>YOLO — TEST {i+1:02d}</figcaption></figure>'
                     for i,p in enumerate(sorted(G_YOLO_FIG.glob('*.jpg'))))
chart_tags='\n'.join(f'<figure><img src="{html_escape(p.relative_to(G_FINAL_DIR).as_posix())}"></figure>' for p in G_PLOTS)
G_HTML.write_text(f'''<!DOCTYPE html><html lang="pt-BR"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>TCC — nuScenes scene-0655 — Análise V7</title>
<style>body{{font:15px/1.6 system-ui,Arial,sans-serif;color:#203044;background:#f5f7fa;margin:0}}main{{max-width:1160px;margin:36px auto;padding:34px;background:white;border-radius:14px}}h1,h2{{color:#15263d}}table{{border-collapse:collapse;width:100%;font-size:13px;margin:15px 0 26px}}th,td{{border-bottom:1px solid #e1e5eb;padding:9px;text-align:left}}th{{background:#eaf0f7}}figure{{margin:17px 0}}img{{display:block;max-width:100%;height:auto;border:1px solid #edf0f3;border-radius:8px}}figcaption{{font-size:12px;color:#5b6978;padding:6px 0}}.note{{background:#f2f5fa;border-left:3px solid #687e9e;padding:12px 17px}}</style></head><body><main>
<h1>TCC — Comparação Nerfacto-Big × Splatfacto</h1><p>nuScenes scene-0655 · CAM_FRONT · resolução 800×450 · protocolo 95 TRAIN / 10 VAL / 10 TEST.</p>
<p class="note"><strong>Metodologia:</strong> checkpoints escolhidos exclusivamente por PSNR de validação; TEST usado uma vez na avaliação final. Splatfacto inicia com LiDAR apenas de treino; Nerfacto não usa inicialização LiDAR. Poses oficiais nuScenes. YOLO na imagem real funciona como <em>pseudorreferência</em>, não rótulo humano nem mAP. Comparações de PSNR entre diferentes cenas/datasets não isolam diferenças de método.</p>
<h2>Resultados quantitativos</h2>{g_html_table(summary_headers,summary_data)}
<h2>Qualidade das dez vistas de TEST</h2>{g_html_table(per_headers,per_data)}
<h2>Gráficos</h2>{chart_tags}
<h2>Reconstruções: REAL × Nerfacto × Splatfacto</h2>{figure_tags}
<h2>Detecções YOLO nas mesmas vistas</h2>{yolo_tags}
<p>Os CSVs, manifestos, versões e checksums encontram-se nas pastas <code>tables/</code> e <code>provenance/</code>. Os PNGs originais estão em <code>images/</code>.</p>
</main></body></html>''',encoding='utf-8')

G_README=G_FINAL_DIR/'LEIA_ME.txt'
G_README.write_text(
    'RESULTADO FINAL — nuScenes scene-0655 / CAM_FRONT / A–G V7\n'
    'Protocolo 95 TRAIN / 10 VAL / 10 TEST, imagens 800x450.\n'
    'Abra ANALISE_FINAL.html após extrair o ZIP.\n\n'
    'tables/        CSV e Excel de resultados por método, vista, YOLO e tempo.\n'
    'images/        10 REAL + 10 Nerfacto + 10 Splatfacto em PNG RGB.\n'
    'figures/       Gráficos, comparações e exemplos semânticos.\n'
    'provenance/    Manifestos A–F e seus metadados, sem pesos ou PLY.\n'
    'final_experiment_manifest.json  Resumo, seleção, hashes, arquivos do pacote.\n\n'
    'PSNR/SSIM/LPIPS: mesma GT pareada; checkpoints escolhidos em VAL.\n'
    'YOLO: pseudorreferência automática na imagem REAL; não mAP humano.\n'
    'Tempo do ns-render contém carga do modelo e gravação, não FPS puro.\n'
    'Este pacote é uma análise final, NÃO um pacote para restauração de treino.\n'
    'As poses nuScenes e a inicialização LiDAR diferem da geometria COLMAP iPhone.\n',
    encoding='utf-8')

# ------------------------- MANIFESTO REPRODUTÍVEL ---------------------------
# Lista hashes de todos os componentes anteriores ao próprio manifesto,
# evitando a circularidade SHA(manifesto) dentro de si mesmo.
G_FILE_RECORDS=[]
for path in sorted(G_FINAL_DIR.rglob('*')):
    if not path.is_file():continue
    if path.is_symlink():raise RuntimeError('Symlink não permitido no ZIP final: '+str(path))
    G_FILE_RECORDS.append({'path':path.relative_to(G_FINAL_DIR).as_posix(),
                           'sha256':g_hash(path),'bytes':path.stat().st_size})
G_MANIFEST=G_FINAL_DIR/'final_experiment_manifest.json'
G_FINAL={
    'experiment':'G_V7_final_nuScenes_scene0655_95_10_10',
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'status':'completed', 'protocol':G_PROTOCOL,
    'scene':'scene-0655','camera':'CAM_FRONT','resolution':list(G_RESOLUTION),
    'splits':G_SPLITS,'test_image_count':10,
    'dataset_hashes':G_A_HASHES,
    'train_geometry':'official nuScenes poses; Splatfacto TRAIN-only LiDAR, Nerfacto no LiDAR initialization',
    'test_selected_checkpoints':G_CHECKPOINTS,
    'quality':G_D['summary'],
    'yolo':{'model':G_F_YOLO['model'],'weights_sha256':G_F_YOLO['weights_sha256'],
            'confidence':G_F_YOLO['conf'],'nms_iou':G_F_YOLO['nms_iou'],
            'image_size':G_F_YOLO['imgsz'],'classes':G_F_YOLO['classes'],
            'matching_iou':.50,'sensitivity_iou':G_EXPECTED_IOU_LEVELS,
            'reference':'same YOLO detections on REAL frames, not human GT',
            'overall':G_F['overall_iou_0_50']},
    'summary':G_ROWS,
    'provenance_sha256':{
         'B_manifest':g_hash(G_B_META_PATH),'C_manifest':g_hash(G_C_META_PATH),
         'D_report':g_hash(G_D_REPORT_PATH),'E_manifest':g_hash(G_E_META_PATH),
         'F_metadata':g_hash(G_F_META_PATH)},
    'limitations':['one scene and 10 held-out TEST views',
        'LiDAR initialization vs SfM initialization are not controlled across datasets',
        'YOLO pseudoreference is not human-annotated ground truth',
        'render wall-clock includes initialization and PNG I/O',
        'TEST must not guide later model hyperparameter tuning'],
    'files':G_FILE_RECORDS,
    'drive_export':'final ZIP only; no checkpoints, no raw LiDAR PLY',
}
g_write(G_MANIFEST,G_FINAL)

# ---------------------------- ZIP FINAL -------------------------------------
G_ZIP=G_FINAL_DIR.parent/('analise_final_nuscenes_scene0655_A_G_V7_'+G_STAMP+'.zip')
if G_ZIP.exists():raise FileExistsError('ZIP final já existe: '+str(G_ZIP))
with zipfile.ZipFile(G_ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6,
                     allowZip64=True) as z:
    for file in sorted(G_FINAL_DIR.rglob('*')):
        if not file.is_file():continue
        if file.suffix.lower() in ('.ckpt','.ply','.pt'):
            raise RuntimeError('Arquivo proibido no ZIP final: '+str(file))
        if file.is_symlink():
            raise RuntimeError('Symlink proibido no ZIP final: '+str(file))
        z.write(file,arcname=file.relative_to(G_FINAL_DIR).as_posix())
with zipfile.ZipFile(G_ZIP) as z:
    corrupt=z.testzip()
    if corrupt is not None:raise RuntimeError('ZIP corrompido em '+corrupt)
    names=set(z.namelist())
    if 'ANALISE_FINAL.html' not in names or 'final_experiment_manifest.json' not in names:
        raise RuntimeError('ZIP incompleto: faltando HTML ou manifesto.')
    for domain in ('real','nerfacto','splatfacto'):
        files=[x for x in names if x.startswith(f'images/{domain}/') and x.endswith('.png')]
        if len(files)!=10:raise RuntimeError(domain+' possui '+str(len(files))+' imagens, esperado 10.')
    if any(name.lower().endswith(('.ckpt','.ply','.pt')) for name in names):
        raise RuntimeError('O ZIP inclui artefato de treino proibido.')

G_DRIVE_ZIP=None
if G_SEND_FINAL_ZIP_TO_DRIVE:
    G_DRIVE_TARGET.mkdir(parents=True,exist_ok=True)
    G_DRIVE_ZIP=G_DRIVE_TARGET/G_ZIP.name
    if G_DRIVE_ZIP.exists():
        raise FileExistsError('Arquivo final já existe no Drive: '+str(G_DRIVE_ZIP))
    shutil.copy2(G_ZIP,G_DRIVE_ZIP)
    if g_hash(G_ZIP)!=g_hash(G_DRIVE_ZIP):
        raise RuntimeError('ZIP final enviado com hash diferente no Google Drive.')

# --------------------------- SAÍDAS PARA NOTEBOOK ---------------------------
FINAL_RESULTS_DIR=G_FINAL_DIR
FINAL_SUMMARY_CSV=G_TABLES/'final_experiment_summary.csv'
FINAL_SUMMARY_XLSX=G_XLSX if G_XLSX_OK else None
FINAL_MANIFEST_PATH=G_MANIFEST
FINAL_ZIP_PATH=G_ZIP
FINAL_DRIVE_ZIP_PATH=G_DRIVE_ZIP
G_FINAL_REPORT=G_FINAL

print('\n'+'='*100)
print('CÉLULA G V7 — ANÁLISE FINAL CONCLUÍDA / 10 VISTAS TEST')
print('='*100)
for row in G_ROWS:
    print(f"{row['method']:22s} | checkpoint {row['selected_step']:5d} | "
          f"PSNR {row['test_psnr_db']:.3f} dB | SSIM {row['test_ssim']:.4f} | "
          f"LPIPS {row['test_lpips_alex']:.4f} | F1-like {g_html_value(row['f1_like'])}")
print('HTML:',G_HTML)
print('Imagens de cada domínio: 10 (30 no total)')
print('Gráficos:',len(G_PLOTS),'| Exemplos semânticos:',len(G_SEMANTIC_SELECTION))
print('Tabelas:',G_TABLES)
print('Excel:',G_XLSX if G_XLSX_OK else 'indisponível; CSV válido')
print('ZIP LOCAL:',G_ZIP)
print('ZIP SHA256:',g_hash(G_ZIP))
if G_DRIVE_ZIP:print('ÚNICO ARQUIVO EXPORTADO AO DRIVE:',G_DRIVE_ZIP)
else:print('Envio ao Drive desabilitado.')
print('TEST não reutilizado para escolha de modelo; nenhum checkpoint exportado.')

if G_SHOW_PREVIEW and G_VISUAL_PAIRS:
    try:
        from IPython.display import display
        for pic in G_VISUAL_PAIRS[:2]:
            with Image.open(pic) as im:
                display(im.copy().resize((1200,242),Image.Resampling.LANCZOS))
    except ImportError:
        pass


# ----------------- ESQUEMA ÚNICO DE RESULTADOS (V1) -----------------
# Biblioteca embutida: esta célula funciona independentemente de H e I.
from pathlib import Path
import importlib.util
_UNIFIED_MODULE_SOURCE = r'''
"""TCC — esquema canônico de resultados v1.
Biblioteca autocontida para uso pelo notebook no Colab. NÃO treina, não infere e
não usa resultados de TEST para alterar modelos. Só organiza e audita artefatos.
"""
from pathlib import Path
from datetime import datetime, timezone
import csv, hashlib, html, json, math, os, shutil, zipfile
from PIL import Image, ImageDraw

SCHEMA = 'TCC_UNIFIED_RESULTS_V1_95_10_10'
DOMAINS = ('real', 'nerfacto', 'splatfacto')
REQUIRED = (
    'ANALISE_FINAL.html', 'LEIA_ME.txt', 'final_experiment_manifest.json',
    'tables/final_experiment_summary.csv', 'tables/quality_by_image.csv',
    'tables/validation_per_view.csv', 'tables/yolo_overall.csv',
    'tables/yolo_by_image.csv', 'tables/yolo_detections.csv',
    'tables/yolo_matches.csv', 'tables/pose_registration_quality.csv',
    'tables/image_index.csv', 'tables/compute_cost.csv',
    'provenance/source_manifest.json', 'provenance/source_run_metadata.json',
)
SUMMARY_COLUMNS = ['dataset_id','method','selected_step','val_psnr_db','test_psnr_db',
 'test_ssim','test_lpips_alex','training_minutes','checkpoint_size_mib',
 'render_total_seconds_including_overhead','real_yolo_count','render_yolo_count',
 'matched_yolo_count','recall_like','precision_like','f1_like',
 'mean_matched_iou','detection_retention_ratio']
QUALITY_COLUMNS = ['dataset_id','image_index','source_image','method','psnr_db','ssim','lpips_alex']
VALIDATION_COLUMNS = ['dataset_id','method','step','image','psnr_db']
YOLO_COLUMNS = ['dataset_id','method','match_iou','real_count','render_count','matched_count',
 'recall_like','precision_like','f1_like','mean_matched_iou','detection_retention_ratio']
YOLO_IMAGE_COLUMNS = ['dataset_id','image_index','method','match_iou','real_count','render_count','matched_count',
 'recall_like','precision_like','f1_like']
DETECTION_COLUMNS = ['dataset_id','image_index','domain','det_index','class_id','confidence','x1','y1','x2','y2']
MATCH_COLUMNS = ['dataset_id','image_index','method','match_iou','real_det_index','render_det_index','class_id','iou']
POSE_COLUMNS = ['dataset_id','image_index','split','pose_source','pose_method','registered','inliers','reprojection_error_px','notes']
INDEX_COLUMNS = ['dataset_id','image_index','source_image','timestamp_or_index', 'real_sha256',
 'nerfacto_sha256','splatfacto_sha256','real_image','nerfacto_image','splatfacto_image',
 'rgb_triplet','yolo_triplet']
COST_COLUMNS = ['dataset_id','method','training_minutes','checkpoint_size_mib','render_total_seconds_including_overhead']

def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for part in iter(lambda:stream.read(1024*1024),b''):h.update(part)
    return h.hexdigest()

def read(path):return json.loads(Path(path).read_text(encoding='utf-8'))

def write(path,obj):
    p=Path(path);p.parent.mkdir(parents=True,exist_ok=True)
    p.write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str)+'\n',encoding='utf-8')

def read_csv(p):
    if not Path(p).is_file():return []
    with open(p,encoding='utf-8-sig',newline='') as stream:return list(csv.DictReader(stream))

def save_csv(p,fields,rows):
    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)
    with p.open('w',encoding='utf-8',newline='') as f:
        wr=csv.DictWriter(f,fieldnames=fields,extrasaction='ignore');wr.writeheader();wr.writerows(rows)

def num(x):
    try:
        v=float(x)
        return v if math.isfinite(v) else None
    except (TypeError,ValueError):return None

def eqfloat(a,b,tol=1e-4):
    if a is None or b is None:return a is None and b is None
    return abs(float(a)-float(b))<=tol

def img(src,target):
    src=Path(src);target=Path(target)
    if not src.is_file():raise FileNotFoundError('Imagem TEST ausente: '+str(src))
    with Image.open(src) as s:
        im=s.convert('RGB')
        if im.size != (800,450):
            raise RuntimeError(f'Imagem {src} com resolução {im.size}, esperado 800x450')
        target.parent.mkdir(parents=True,exist_ok=True)
        im.save(target,format='PNG')

def triplet(files,out,i,title):
    canvas=Image.new('RGB',(3*800,450+38),'white');draw=ImageDraw.Draw(canvas)
    for j,(domain,p) in enumerate(zip(DOMAINS,files)):
        with Image.open(p) as src:canvas.paste(src.convert('RGB'),(j*800,38))
        draw.text((j*800+14,11),f'{title} {i+1:02d} / {domain.upper()}',fill='#17283a')
    out.parent.mkdir(parents=True,exist_ok=True);canvas.save(out,format='JPEG',quality=93)

def normalize(source_kind,source_root,dataset_id,output_root,source_extra=None):
    """Normaliza final G (nuScenes) ou final H/I (capturas próprias).

    Retorna (diretorio_unificado, arquivo_zip); copia somente análise, nunca
    checkpoints, PLY, frames de TRAIN ou vídeo. Tabelas não aplicáveis são vazias,
    com cabeçalhos canônicos e declaração explícita de indisponibilidade.
    """
    if source_kind not in ('nuscenes','video') or dataset_id not in ('nuscenes_scene0655','estacionamento','uber'):
        raise ValueError('Origem ou dataset_id inesperado.')
    src=Path(source_root).resolve();out=Path(output_root)
    if not src.is_dir():raise FileNotFoundError(src)
    if out.exists():raise FileExistsError('Não sobrescrever análise anterior: '+str(out))
    out.mkdir(parents=True)
    for name in ['tables','figures/charts','figures/rgb_triplets','figures/yolo_triplets',
                 'images/real','images/nerfacto','images/splatfacto','provenance']:
        (out/name).mkdir(parents=True,exist_ok=True)
    extra=dict(source_extra or {})
    values={key:[] for key in ('summary','quality','validation','yolo','yolo_image',
                              'detections','matches','pose','index','cost')}
    raw_images={}
    rgb_fig={};yolo_fig={}
    manifest_source=src/('final_experiment_manifest.json' if source_kind=='nuscenes' else 'manifest.json')
    if not manifest_source.is_file():raise FileNotFoundError(manifest_source)
    native=read(manifest_source)
    shutil.copy2(manifest_source,out/'provenance/source_manifest.json')
    metadata={'original_source_kind':source_kind,'original_source_directory':str(src),
              'original_manifest_sha256':sha(manifest_source),
              'dataset_id':dataset_id,'notes':extra}
    if source_kind=='nuscenes':
        sm=read_csv(src/'tables/final_experiment_summary.csv')
        d=read(src/'provenance/D_final_report.json')
        for row in sm:
            method=row['method_key']
            values['summary'].append({'dataset_id':dataset_id, 'method':method,**{k:row.get(k) for k in SUMMARY_COLUMNS if k not in ('dataset_id','method')}})
        for imrow in d['per_view']:
            i=int(imrow['test_view'])-1
            for method in ('nerfacto','splatfacto'):
                values['quality'].append({'dataset_id':dataset_id,'image_index':i,'source_image':imrow['image'],
                    'method':method,'psnr_db':imrow[method+'_psnr_db'],'ssim':imrow[method+'_ssim'],
                    'lpips_alex':imrow[method+'_lpips_alex']})
        # Evidência VAL: caminhos provenientes desta mesma sessão B/C; sem suposições.
        csvs=extra.get('validation_csvs',{})
        for method in ('splatfacto','nerfacto'):
            valpath=Path(csvs.get(method,'/invalid_validation_csv'))
            rows=read_csv(valpath)
            if not rows:
                raise FileNotFoundError('CSV VAL da B/C não encontrado: '+str(valpath))
            for row in rows:
                values['validation'].append({'dataset_id':dataset_id,'method':method,
                    'step':row.get('step'),'image':row.get('image'),'psnr_db':row.get('psnr_db')})
        for row in read_csv(src/'tables/yolo_summary_overall.csv'):
            vals={'dataset_id':dataset_id,'method':row.get('render_domain'),'match_iou':.50}
            vals.update({k:row.get(k) for k in YOLO_COLUMNS if k not in vals});values['yolo'].append(vals)
        for row in read_csv(src/'tables/yolo_summary_by_image.csv'):
            if not eqfloat(num(row.get('match_iou')),0.50):
                continue
            vals={'dataset_id':dataset_id,'method':row.get('render_domain'),'image_index':row.get('image_index'),'match_iou':.50}
            vals.update({k:row.get(k) for k in YOLO_IMAGE_COLUMNS if k not in vals});values['yolo_image'].append(vals)
        for row in read_csv(src/'tables/yolo_detections_all.csv'):
            values['detections'].append({'dataset_id':dataset_id,**{k:row.get(k) for k in DETECTION_COLUMNS if k!='dataset_id'}})
        for row in read_csv(src/'tables/yolo_match_details.csv'):
            values['matches'].append({'dataset_id':dataset_id,'image_index':row.get('image_index'),
                 'method':row.get('render_domain'),'match_iou':.5,'real_det_index':row.get('real_det_index'),
                 'render_det_index':row.get('render_det_index'),'class_id':row.get('class_id'),'iou':row.get('iou')})
        tf=read(src/'provenance/transforms.json')
        fm={Path(p).stem:p for p in tf['test_filenames']}
        for i in range(10):
            for domain in DOMAINS:raw_images[i,domain]=src/f'images/{domain}/eval_img_{i:04d}.png'
            rgb_fig[i]=src/f'figures/rgb_triplets/RGB_TEST_{i:02d}.jpg'
            yolo_fig[i]=src/f'figures/yolo_triplets/YOLO_TEST_{i:02d}.jpg'
            values['pose'].append({'dataset_id':dataset_id,'image_index':i,'split':'test',
               'pose_source':'official_nuscenes_ego_pose_calibrated_sensor',
               'pose_method':'provided_extrinsics','registered':'not_applicable',
               'notes':'Sem COLMAP; sem reprojeção PnP estimada'})
        metadata['geometry']='nuScenes official sensor poses; TRAIN-only LiDAR for Splatfacto'
    else:
        sm=read_csv(src/'summary.csv')
        for row in sm:
            name=row['method']
            models=native['models'][name]
            values['summary'].append({
               'dataset_id':dataset_id,'method':name,'selected_step':row.get('selected_step'),
               'val_psnr_db':row.get('val_psnr_db'),'test_psnr_db':row.get('test_psnr_db'),
               'test_ssim':row.get('test_ssim'),'test_lpips_alex':row.get('test_lpips_alex'),
               'training_minutes':row.get('training_minutes'),
               'checkpoint_size_mib':models.get('checkpoint_size_mib'),
               'render_total_seconds_including_overhead':models.get('render_process_seconds')})
        for row in read_csv(src/'tables/test_per_view.csv'):
            values['quality'].append({'dataset_id':dataset_id,'image_index':-1,
                'source_image':row.get('image'),'method':row.get('method'),
                'psnr_db':row.get('psnr_db'),'ssim':row.get('ssim'),'lpips_alex':row.get('lpips_alex')})
        for row in read_csv(src/'tables/validation_per_view.csv'):
            values['validation'].append({'dataset_id':dataset_id,'method':row.get('method'),
                'step':row.get('step'),'image':row.get('image'),'psnr_db':row.get('psnr_db')})
        for row in read_csv(src/'tables/yolo_overall.csv'):
            if eqfloat(num(row.get('match_iou')),0.5):
                values['yolo'].append({'dataset_id':dataset_id,'method':row.get('method'),
                    **{k:row.get(k) for k in YOLO_COLUMNS if k not in ('dataset_id','method')}})
        for row in read_csv(src/'tables/yolo_by_image.csv'):
            if eqfloat(num(row.get('match_iou')),0.5):
                values['yolo_image'].append({'dataset_id':dataset_id,'method':row.get('method'),
                    'image_index':None,'source_image':row.get('image'),
                    **{k:row.get(k) for k in YOLO_IMAGE_COLUMNS if k not in ('dataset_id','method','image_index')}})
        for row in read_csv(src/'tables/yolo_detections.csv'):
            values['detections'].append({'dataset_id':dataset_id,'image_index':None,
                'source_image':row.get('image'),'domain':row.get('domain'), 'det_index':row.get('id'),
                **{k:row.get(k) for k in ('class_id','confidence','x1','y1','x2','y2')}})
        for row in read_csv(src/'tables/yolo_matches.csv'):
            if eqfloat(num(row.get('match_iou')),0.5):
                values['matches'].append({'dataset_id':dataset_id,'image_index':None,
                    'source_image':row.get('image'),'method':row.get('method'),'match_iou':row.get('match_iou'),
                    'real_det_index':row.get('real_id'),'render_det_index':row.get('render_id'),
                    'class_id':row.get('class_id'),'iou':row.get('iou')})
        posed=read_csv(src/'tables/pose_registration_quality.csv')
        metadata['pose_registration_original_records'] = posed  # Auditoria completa, sem perder colunas nativas.
        for row in posed:
            values['pose'].append({'dataset_id':dataset_id,**{k:row.get(k) for k in POSE_COLUMNS if k!='dataset_id'}})
        allstems=sorted(p.stem for p in (src/'figures/test/real').glob('*.png'))
        if len(allstems)!=10 or len(set(allstems))!=10:
            raise RuntimeError('H/I precisa conter 10 TEST em figures/test/real')
        index_for_stem={stem:i for i,stem in enumerate(allstems)}
        for i,stem in enumerate(allstems):
            for domain in DOMAINS:raw_images[i,domain]=src/'figures/test'/domain/(stem+'.png')
            rgb_fig[i]=src/'figures/rgb_triplets'/(stem+'.jpg')
            yolo_fig[i]=src/'figures/yolo_triplets'/(stem+'.jpg')
        for kind in ('quality','yolo_image','detections','matches'):
            for row in values[kind]:
                stem=row.get('source_image')
                if stem not in index_for_stem:raise RuntimeError(f'Imagem {stem} de {kind} fora do TEST')
                row['image_index']=index_for_stem[stem]
        # Reutilização das métricas reais; NÃO calcular métricas não medidas.
        sem={r['method']:r for r in values['yolo']}
        for row in values['summary']:
            y=sem.get(row['method'],{})
            for key in ('real_count','render_count','matched_count','recall_like','precision_like','f1_like',
                        'mean_matched_iou','detection_retention_ratio'):
                row[{'real_count':'real_yolo_count','render_count':'render_yolo_count',
                     'matched_count':'matched_yolo_count'}.get(key,key)]=y.get(key)
        metadata['geometry']='COLMAP TRAIN only; VAL/TEST PnP against frozen TRAIN model'
        metadata['source_video_hash']=native.get('video_sha256')
    if len(values['summary'])!=2 or set(r['method'] for r in values['summary'])!={'splatfacto','nerfacto'}:
        raise RuntimeError('Resumo deve conter exatamente os dois métodos.')
    if len(values['quality'])!=20 or len(values['yolo'])!=2:
        raise RuntimeError('Qualidade deve ter 20 linhas e resumo semântico duas linhas.')
    if len(values['validation'])!=150:
        raise RuntimeError(f'VAL incompleta: {len(values["validation"])} linhas, esperado 150 (5×10 + 10×10).')
    # Verificação independente das médias originais por vista em TEST.
    for summ in values['summary']:
        m=summ['method'];observed=[q for q in values['quality'] if q['method']==m]
        if len(observed)!=10 or len(set(int(r['image_index']) for r in observed))!=10:
            raise RuntimeError('10 imagens TEST não únicas para '+m)
        for sm_col,col in [('test_psnr_db','psnr_db'),('test_ssim','ssim'),('test_lpips_alex','lpips_alex')]:
            raw=[num(r[col]) for r in observed]
            if any(v is None for v in raw):raise RuntimeError('Métrica incompleta '+m+'/'+col)
            mean=sum(raw)/10
            if not eqfloat(mean,num(summ[sm_col]),tol=1e-4):
                raise RuntimeError(f'Média {m}/{col} não fecha: {mean}, manifesto {summ[sm_col]}')
    for i in range(10):
        paths=[]
        for domain in DOMAINS:
            path=out/'images'/domain/f'eval_img_{i:04d}.png'
            img(raw_images[i,domain],path);paths.append(path)
        for domain,path in zip(DOMAINS,paths):
            if not path.is_file():raise RuntimeError('Imagem ausente '+str(path))
        index={'dataset_id':dataset_id,'image_index':i,
               'source_image':next(q['source_image'] for q in values['quality'] if int(q['image_index'])==i),
               'timestamp_or_index':i}
        for domain,path in zip(DOMAINS,paths):
            index[domain+'_image']=path.relative_to(out).as_posix()
            index[domain+'_sha256']=sha(path)
        for kind,oldmap in [('rgb',rgb_fig),('yolo',yolo_fig)]:
            target=out/f'figures/{kind}_triplets/eval_img_{i:04d}.jpg'
            old=oldmap[i]
            if old.is_file():
                shutil.copy2(old,target)
            elif kind=='rgb':
                triplet(paths,target,i,'TEST')
            else:
                raise FileNotFoundError('Figura YOLO deve existir; não refazer inferência: '+str(old))
            index[kind+'_triplet']=target.relative_to(out).as_posix()
        values['index'].append(index)
    # Integridade: vistas originais versus métricas e inferências.
    if len(values['yolo_image'])!=20 or len(values['detections'])<0:
        raise RuntimeError('Tabela YOLO por vista incompleta (esperado 20).')
    for summ in values['summary']:
        values['cost'].append({key:summ.get(key) for key in COST_COLUMNS})
    files={
        'final_experiment_summary.csv':(SUMMARY_COLUMNS,'summary'),
        'quality_by_image.csv':(QUALITY_COLUMNS,'quality'),
        'validation_per_view.csv':(VALIDATION_COLUMNS,'validation'),
        'yolo_overall.csv':(YOLO_COLUMNS,'yolo'),
        'yolo_by_image.csv':(YOLO_IMAGE_COLUMNS,'yolo_image'),
        'yolo_detections.csv':(DETECTION_COLUMNS,'detections'),
        'yolo_matches.csv':(MATCH_COLUMNS,'matches'),
        'pose_registration_quality.csv':(POSE_COLUMNS,'pose'),
        'image_index.csv':(INDEX_COLUMNS,'index'),
        'compute_cost.csv':(COST_COLUMNS,'cost'),
    }
    for filename,(header,key) in files.items():save_csv(out/'tables'/filename,header,values[key])
    # Gráficos compartilhados, com os mesmos nomes e eixos; dados ausentes indicados como N/D.
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    plots=[('test_psnr_db','PSNR TEST (dB)','test_psnr.png'),
           ('test_ssim','SSIM TEST','test_ssim.png'),
           ('test_lpips_alex','LPIPS Alex TEST (menor e melhor)','test_lpips.png'),
           ('training_minutes','Treinamento (minutos)','training_minutes.png'),
           ('f1_like','YOLO F1-like IoU 0.50','yolo_f1_like.png'),
           ('recall_like','YOLO Recall-like IoU 0.50','yolo_recall_like.png')]
    for key,title,filename in plots:
        fig,ax=plt.subplots(figsize=(7,4.2))
        scores=[num(x.get(key)) for x in values['summary']]
        ax.bar(['Splatfacto','Nerfacto'],[x if x is not None else 0. for x in scores])
        ax.set_title(dataset_id+' — '+title);ax.grid(axis='y',alpha=.2)
        for i,value in enumerate(scores):ax.text(i,(value or 0),f'{value:.3f}' if value is not None else 'N/D',ha='center',va='bottom')
        fig.tight_layout();fig.savefig(out/'figures/charts'/filename,dpi=150);plt.close(fig)
    # HTML único, com links relativos para as imagens presentes no ZIP.
    rows=''.join('<tr>'+''.join(f'<td>{html.escape(str(r.get(k,"")))}</td>' for k in SUMMARY_COLUMNS)+'</tr>' for r in values['summary'])
    hdr=''.join('<th>'+html.escape(k)+'</th>' for k in SUMMARY_COLUMNS)
    figtags=''.join(f'<h3>TEST {i+1:02d}</h3><img src="figures/rgb_triplets/eval_img_{i:04d}.jpg" alt="Tríade RGB TEST {i+1}"><img src="figures/yolo_triplets/eval_img_{i:04d}.jpg" alt="Tríade YOLO TEST {i+1}">' for i in range(10))
    (out/'ANALISE_FINAL.html').write_text('<!doctype html><html lang="pt-BR"><head><meta charset="utf-8"><title>TCC - '+html.escape(dataset_id)+'</title><style>body{max-width:1200px;margin:32px auto;font:15px/1.6 system-ui}table{border-collapse:collapse;display:block;overflow:auto}td,th{border:1px solid #bbb;padding:5px}img{max-width:100%;margin:8px 0}aside{background:#eee;padding:12px}</style></head><body><h1>Comparação Nerfacto × Splatfacto — '+html.escape(dataset_id)+'</h1><aside>95 TRAIN / 10 VAL / 10 TEST. YOLO na imagem real é pseudorreferência, não ground truth humano. Comparação entre datasets tem confundidores geométricos e de conteúdo. Tempos ns-render incluem sobrecarga.</aside><h2>Resumo</h2><table><tr>'+hdr+'</tr>'+rows+'</table><h2>Gráficos</h2>'+''.join(f'<img src="figures/charts/{p[2]}">' for p in plots)+'<h2>Comparações RGB e YOLO</h2>'+figtags+'</body></html>',encoding='utf-8')
    (out/'LEIA_ME.txt').write_text('TCC — ESQUEMA DE RESULTADOS '+SCHEMA+'\n'
        'Dataset: '+dataset_id+'\nTrês domínios: real, nerfacto, splatfacto\n'
        'images/: 30 PNG TEST com nomes idênticos nas três subpastas\n'
        'figures/: 10 tríades RGB, 10 YOLO, seis gráficos com nomes fixos\n'
        'tables/: mesmas tabelas e colunas nos três datasets\n'
        'provenance/: manifesto da execução original, metadados, hashes\n'
        'Não contém checkpoints, vídeos, PLY nem imagens de treino.\n'
        'Colunas não aplicáveis permanecem vazias, não são medidas fabricadas.\n'
        'TEST é usado exclusivamente para avaliação confirmatória; não retunar.\n',encoding='utf-8')
    metadata.update(schema=SCHEMA,split_counts={'train':95,'val':10,'test':10},
                    yolo_real_is_pseudoreference=True,
                    no_new_training=True,no_new_inference=True,
                    aggregation='mean of 10 images; YOLO micro counts',
                    resolution=[800,450],sources={'native_manifest':str(manifest_source)})
    write(out/'provenance/source_run_metadata.json',metadata)
    file_records=[]
    for p in sorted(out.rglob('*')):
        if p.is_file():
            if p.is_symlink() or p.suffix.lower() in ('.ckpt','.ply','.pt','.mov','.mp4'):
                raise RuntimeError('Arquivo proibido no pacote: '+str(p))
            file_records.append({'path':p.relative_to(out).as_posix(),'sha256':sha(p),'bytes':p.stat().st_size})
    manifest={'experiment':'unified_TCC_result','schema':SCHEMA,'dataset_id':dataset_id,
             'created_utc':datetime.now(timezone.utc).isoformat(),
             'status':'completed','splits':{'train':95,'val':10,'test':10},
             'test_views':10,'rgb_images':30,
             'metrics_not_available':{
                  'nuscenes_scene0655':['COLMAP reprojection errors','matched sensor trajectories with other datasets'],
                  'estacionamento':['nuScenes LiDAR initialisation','official nuScenes poses'],
                  'uber':['nuScenes LiDAR initialisation','official nuScenes poses']
              }[dataset_id],
             'models':values['summary'],
             'limitations':['single sequence per scenario','correlated held-out views',
                 'not a complete closed-loop autonomous driving simulator',
                 'cross-dataset scores are not controlled experiments',
                 'YOLO on REAL is pseudoreference, not human ground truth'],
             'file_integrity_excluding_self':file_records}
    write(out/'final_experiment_manifest.json',manifest)
    for name in REQUIRED:
        if not (out/name).is_file():raise RuntimeError('Falta arquivo canônico: '+name)
    for domain in DOMAINS:
        imgs=list((out/'images'/domain).glob('eval_img_*.png'))
        if len(imgs)!=10:raise RuntimeError(f'{domain} possui {len(imgs)} imagens, não 10')
    archive=out.parent/(f'TCC_RESULTADOS_{dataset_id}_{out.name}.zip')
    if archive.exists():raise FileExistsError(archive)
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=5,allowZip64=True) as z:
        for p in sorted(out.rglob('*')):
            if p.is_file():z.write(p,p.relative_to(out).as_posix())
    with zipfile.ZipFile(archive) as z:
        corrupt=z.testzip()
        if corrupt:raise RuntimeError('ZIP corrompido em '+corrupt)
    return out,archive

def upload_single_zip(path,dataset_id,drive_root=None):
    """Exporta ZIP somente para diretório explicitamente indicado; verifica SHA."""
    path=Path(path)
    if not path.is_file():raise FileNotFoundError(path)
    export_dir = drive_root or os.environ.get('TCC_EXPORT_DIR', '').strip()
    if not export_dir:
        raise ValueError('Defina TCC_EXPORT_DIR explicitamente para habilitar exportação.')
    dest = Path(export_dir).expanduser()
    if not dest.is_absolute():
        raise ValueError('TCC_EXPORT_DIR deve ser um caminho absoluto.')
    dest.mkdir(parents=True, exist_ok=True)
    final=dest/path.name;partial=dest/(path.name+'.partial')
    if final.exists() or partial.exists():raise FileExistsError(final)
    shutil.copy2(path,partial)
    if sha(partial)!=sha(path):
        partial.unlink(missing_ok=True)
        raise RuntimeError('Cópia ao Drive falhou no SHA256')
    partial.replace(final)
    return final

'''
_UNIFIED_MODULE_FILE = Path('/content/tcc_unified_results_v1.py')
_UNIFIED_MODULE_FILE.write_text(_UNIFIED_MODULE_SOURCE, encoding='utf-8')
_spec = importlib.util.spec_from_file_location('tcc_unified_results_v1', _UNIFIED_MODULE_FILE)
_unified = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_unified)

G_UNIFIED_ROOT = Path('/content/tcc_resultados_padronizados') / ('nuscenes_scene0655_' + G_STAMP)
G_UNIFIED_DIR, G_UNIFIED_ZIP = _unified.normalize(
    'nuscenes', G_FINAL_DIR, 'nuscenes_scene0655', G_UNIFIED_ROOT,
    source_extra={'camera':'CAM_FRONT','geometry':'official_nuscenes_poses',
       'dataset_tag':G_EXPECTED_PROFILE,
       'validation_csvs':{'splatfacto':str(B_VAL_CSV),'nerfacto':str(C_VAL_CSV)}}
)

# ------- EVIDÊNCIAS V4: SEM RENDER TRAIN; VAL + TEST / CUSTO / VIZINHOS -------
from pathlib import Path
import importlib.util
_TCC_EVIDENCE_SOURCE = r'''
"""Extensão V4: imagens reais de TRAIN/VAL/TEST e renderizações somente VAL/TEST.

Mantém 95 RGB reais TRAIN, 10 VAL e 10 TEST; gera 10 VAL por modelo usando
checkpoints já escolhidos na validação. As 10 TEST por modelo são copiadas
dos resultados originais, sem nova avaliação nem retreinamento.
Não renderiza TRAIN nem modifica transforms.json: mantém 95 TRAIN para
carregar sem conflito os embeddings de aparência do Nerfacto-Big.
"""
from pathlib import Path
from datetime import datetime, timezone
from PIL import Image, ImageDraw, ImageOps
import csv, hashlib, html, json, math, os, re, shutil, subprocess, time, zipfile

SPLITS = {'train':95,'val':10,'test':10}
DOMAINS = ('real','nerfacto','splatfacto')
FRAMES_FIELDS = ['dataset_id','split','split_index','chronological_index','source_image','source_type',
 'source_timestamp_us','nominal_time_seconds','real_png','nerfacto_png','splatfacto_png',
 'real_sha256','nerfacto_sha256','splatfacto_sha256','preceding_source_frame',
 'preceding_source_timestamp_us','preceding_gap_ms','following_source_frame',
 'following_source_timestamp_us','following_gap_ms','temporal_neighbor_scope',
 'nearest_train_before','nearest_train_before_gap_ms','nearest_train_after',
 'nearest_train_after_gap_ms','temporal_relation_to_train','render_provenance']
COST_FIELDS = ['dataset_id','method','training_minutes','training_wall_seconds',
 'train_gpu_peak_mib_sampled','train_gpu_peak_sample_interval_seconds',
 'checkpoint_size_mib','val_candidate_render_wall_seconds','test_render_wall_seconds',
 'train_final_render_wall_seconds','val_final_render_wall_seconds',
 'additional_final_renders_wall_seconds','additional_final_render_gpu_peak_mib_sampled',
 'render_total_seconds_including_overhead','test_render_seconds_per_view_including_overhead',
 'gpu_name','gpu_memory_measurement_note','render_timing_note','train_final_render_views','val_final_render_views','test_render_views']


def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(1024*1024),b''):h.update(block)
    return h.hexdigest()


def rd(path):return json.loads(Path(path).read_text(encoding='utf-8'))


def wr(path,object_):
    p=Path(path);p.parent.mkdir(parents=True,exist_ok=True)
    p.write_text(json.dumps(object_,indent=2,ensure_ascii=False,default=str,allow_nan=False)+'\n',encoding='utf-8')


def table(path,fields,rows):
    p=Path(path);p.parent.mkdir(parents=True,exist_ok=True)
    with p.open('w',newline='',encoding='utf-8') as f:
        writer=csv.DictWriter(f,fieldnames=fields,extrasaction='ignore')
        writer.writeheader();writer.writerows(rows)


def readtable(path):
    with Path(path).open('r',newline='',encoding='utf-8-sig') as f:return list(csv.DictReader(f))


def numeric(value):
    try:
        if value is None or value=='':return None
        n=float(value)
        return n if math.isfinite(n) else None
    except (ValueError,TypeError):return None


def pixel_png(source,destination):
    source=Path(source);destination=Path(destination)
    if not source.is_file():raise FileNotFoundError(source)
    destination.parent.mkdir(parents=True,exist_ok=True)
    with Image.open(source) as im:
        if im.size!=(800,450):
            raise RuntimeError(f'{source}: resolução {im.size}, esperado 800x450')
        if im.mode=='RGB' and (im.format or '').upper()=='PNG':
            # Identidade pixel-a-pixel; evita transcoding sem necessidade.
            shutil.copy2(source,destination)
        else:
            im.convert('RGB').save(destination,format='PNG')
    return sha(destination)


def png_index(root,split,kind,stems):
    root=Path(root)/split/kind
    paths=list(root.rglob('*.png'))
    mapping={p.stem:p for p in paths}
    if len(paths)!=len(mapping) or set(mapping)!=set(stems):
        raise RuntimeError(f'{root}: faltam ou sobram PNGs; previsto={len(stems)}, obtido={len(paths)}')
    return mapping


def gpu_used_for_process(pid):
    try:
        p=subprocess.run(['nvidia-smi','--query-compute-apps=pid,used_gpu_memory',
                          '--format=csv,noheader,nounits'],capture_output=True,text=True,timeout=7)
        if p.returncode:return None
        readings=[]
        for line in p.stdout.splitlines():
            parts=[x.strip() for x in line.split(',')]
            if len(parts)<2:continue
            if parts[0].isdigit() and int(parts[0])==pid:
                value=numeric(parts[1].split()[0])
                if value is not None:readings.append(value)
        return max(readings) if readings else None
    except (FileNotFoundError,subprocess.TimeoutExpired):return None


def execute_render(command,log_path,timeout=7200):
    log_path=Path(log_path);log_path.parent.mkdir(parents=True,exist_ok=True)
    started=time.monotonic();peak=None;sample_count=0
    env=dict(os.environ)
    for key in ('PYTHONPATH','PYTHONHOME','PYTHONUSERBASE','PIP_TARGET'):
        env.pop(key,None)
    env['PYTHONNOUSERSITE']='1';env['PYTHONUNBUFFERED']='1'
    # Compatibilidade: Nerfstudio 1.1.5 usa torch.load sem weights_only explícito.
    # No PyTorch >= 2.6 o valor padrão mudou para True; alguns checkpoints
    # próprios da execução contêm escalares NumPy não permitidos nesse modo.
    # Aplicar apenas ao subprocesso ns-render com checkpoints confiáveis,
    # previamente validados pelos hashes do pipeline. NÃO altera os pesos.
    if env.get('TORCH_FORCE_WEIGHTS_ONLY_LOAD','').strip().lower() in ('1','true','yes','y'):
        raise RuntimeError(
            'TORCH_FORCE_WEIGHTS_ONLY_LOAD está ativado no ambiente: '
            'remova-o antes de renderizar checkpoints próprios do Nerfstudio 1.1.5.'
        )
    env['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD']='1'
    with log_path.open('w',encoding='utf-8') as log:
        log.write('CMD: '+' '.join(map(str,command))+'\n\n');log.flush()
        process=subprocess.Popen(list(map(str,command)),stdout=log,stderr=subprocess.STDOUT,env=env)
        try:
            while process.poll() is None:
                if time.monotonic()-started>timeout:
                    process.kill();process.wait()
                    raise TimeoutError(f'ns-render demorou mais que {timeout}s: {log_path}')
                sample=gpu_used_for_process(process.pid)
                if sample is not None:
                    peak=sample if peak is None else max(peak,sample)
                    sample_count+=1
                time.sleep(2)
            if process.returncode:
                raise RuntimeError(f'Falha do ns-render: {log_path}\n'+log_path.read_text(errors='replace')[-7000:])
        finally:
            if process.poll() is None:process.kill();process.wait()
    text=log_path.read_text(errors='replace')
    if not re.search('Done loading checkpoint from',text):
        raise RuntimeError('ns-render não confirmou carregamento de checkpoint: '+str(log_path))
    return {'wall_seconds':round(time.monotonic()-started,4),
            'gpu_peak_mib_sampled':peak,'gpu_memory_observations':sample_count,
            'gpu_memory_sampling_period_seconds':2,'log':str(log_path)}


def video_source_neighbor(run,stem,offset):
    m=re.fullmatch(r'frame_(\d+)',stem)
    if not m:return None
    n=int(m.group(1))+offset
    if n<1:return None
    source=Path(run)/'raw_frames'/f'frame_{n:06d}.png'
    return source if source.is_file() else None


def picture_neighbors(before,current,after,output,title):
    output=Path(output);output.parent.mkdir(parents=True,exist_ok=True)
    image=Image.new('RGB',(1200,260),(248,248,248))
    drawer=ImageDraw.Draw(image)
    for col,(src,label) in enumerate(((before,'ANTERIOR'),(current,'ATUAL'),(after,'POSTERIOR'))):
        drawer.text((col*400+6,5),label,fill=(17,28,43))
        if src is not None and Path(src).is_file():
            with Image.open(src) as im:
                im=ImageOps.contain(im.convert('RGB'),(400,225),Image.Resampling.LANCZOS)
                image.paste(im,(col*400+(400-im.width)//2,30+(225-im.height)//2))
        else:
            drawer.text((col*400+75,135),'INDISPONÍVEL',fill=(105,105,105))
    image.save(output,quality=88)


def extend(dataset_id,source_kind,run_root,dataset,unified_dir,unified_zip,
           selected_configs,selected_steps,ns_render,training_stats=None,
           extra_source_frames=None):
    """Atualiza diretório e ZIP canônico já normalizados; nunca treina/seleciona.

    source_kind = 'nuscenes' ou 'video', run_root contém outputs originais
    dataset tem transforms.json e as 115 imagens retificadas/normalizadas.
    """
    root=Path(run_root);dataset=Path(dataset);out=Path(unified_dir);zipfile_path=Path(unified_zip)
    if not all(x.exists() for x in (root,dataset,out,zipfile_path)):
        raise FileNotFoundError('Dados locais A–G/H/I ou ZIP canônico ausentes; é necessário manter a sessão.')
    manifest_path=out/'final_experiment_manifest.json'
    current=rd(manifest_path)
    if current.get('dataset_id')!=dataset_id or current.get('splits')!=SPLITS:
        raise RuntimeError('Arquivo canônico original não é do cenário/split esperado.')
    tf=rd(dataset/'transforms.json')
    frame_by_path={f['file_path']:f for f in tf['frames']}
    bysplit={}
    for sp,count in SPLITS.items():
        paths=tf[sp+'_filenames']
        if len(paths)!=count or len(set(paths))!=count:
            raise RuntimeError(f'Split {sp} não contém {count} frames únicos.')
        if not set(paths)<=set(frame_by_path):raise RuntimeError('Frames ausentes de transforms.json.')
        bysplit[sp]=sorted(paths,key=lambda p:(int(frame_by_path[p].get('timestamp',0)),p))
    if len({x for paths in bysplit.values() for x in paths})!=115:
        raise RuntimeError('Treino/validação/teste não disjuntos.')
    if source_kind not in ('nuscenes','video'):raise ValueError(source_kind)
    extra_source_frames={int(k):Path(v) for k,v in (extra_source_frames or {}).items()}
    for model in ('splatfacto','nerfacto'):
        config=Path(selected_configs[model]);step=int(selected_steps[model])
        if not config.is_file():raise FileNotFoundError('Config selecionado ausente: '+str(config))
        if step<0:raise ValueError(step)
    if not Path(ns_render).is_file():raise FileNotFoundError(ns_render)
    # Este manifesto é de 115 capturas, não das 40 imagens excluídas.
    canon_test=readtable(out/'tables/image_index.csv')
    if len(canon_test)!=10:raise RuntimeError('Manifesto canônico de TEST não tem dez linhas.')
    test_source_by_stem={str(r['source_image']):int(r['image_index']) for r in canon_test}
    if set(test_source_by_stem)!={Path(p).stem for p in bysplit['test']}:
        raise RuntimeError('Identidade TEST do pacote não corresponde a transforms.json.')

    # Só renderiza as 10 vistas VAL após seleção do checkpoint. As dez TEST
    # são copiadas dos resultados já avaliados, preservando o protocolo final.
    # O transforms original conserva 95 TRAIN, requisito para o embedding
    # de aparência [95,128] aprendido pelo Nerfacto-Big. Não editar TRAIN.
    source_transforms_sha=sha(dataset/'transforms.json')
    results={};base_render=out.parent/('_render_val_test_'+dataset_id+'_'+out.name)
    base_render.mkdir(parents=True,exist_ok=False)
    for method in ('splatfacto','nerfacto'):
        results[method]={}
        split='val'
        output=base_render/method/split
        output.mkdir(parents=True,exist_ok=False)
        paths=bysplit['val']
        command=[str(ns_render),'dataset','--load-config',str(selected_configs[method]),
          '--output-path',str(output),'--data',str(dataset),'--split','val',
          '--image-format','png','--rendered-output-names','rgb','gt-rgb']
        print(f'[EVIDÊNCIA V4] {dataset_id}/{method}/val: 10 vistas. '
              'TRAIN não será renderizado; TEST será reutilizado.',flush=True)
        results[method]['val']=execute_render(command,base_render/f'{method}_val.log')
        results[method]['val']['original_data_split']='val'
        results[method]['val']['render_split']='val'
        results[method]['val']['temporary_dataset']=False
        stems=[Path(p).stem for p in paths]
        rgb=png_index(output,'val','rgb',stems)
        gt=png_index(output,'val','gt-rgb',stems)
        # Referências de validação devem continuar idênticas às imagens originais.
        import numpy as np
        for rel in paths:
            stem=Path(rel).stem
            with Image.open(dataset/rel) as a, Image.open(gt[stem]) as b:
                ar=np.asarray(a.convert('RGB'),dtype=np.int16)
                br=np.asarray(b.convert('RGB'),dtype=np.int16)
                if ar.shape!=(450,800,3) or br.shape!=ar.shape or int(np.max(np.abs(ar-br)))>1:
                    raise RuntimeError('Referência VAL diferente da original: '+method+'/'+rel)
        results[method]['val']['rgb_map']={k:str(v) for k,v in rgb.items()}
    if sha(dataset/'transforms.json')!=source_transforms_sha:
        raise RuntimeError('O transforms.json mudou durante renderização de VAL.')
    # Evidências reais e geradas, sem alterar os arquivos TEST da etapa D/H/I.
    all_frames=[];chronological=[]
    for sp in ('train','val','test'):
        for i,rel in enumerate(bysplit[sp]):
            frame=frame_by_path[rel]
            raw_ts=frame.get('timestamp')
            if source_kind=='video':
                # Frame original possui nome frame_NNNNNN.png; tempo do vídeo 4 FPS.
                number=re.search(r'(\d+)$',Path(rel).stem)
                raw_ts=int(number.group(1)) if number else None
                nominal=(raw_ts-1)/4.0 if raw_ts is not None else None
                ordering=nominal if nominal is not None else i
                unit='video_sample_at_4fps'
            else:
                raw_ts=int(raw_ts)
                nominal=raw_ts/1e6
                ordering=raw_ts/1e6
                unit='nuscenes_timestamp_us'
            chronological.append((ordering,rel,sp,i,raw_ts,nominal,unit))
    chronological.sort(key=lambda x:(x[0],x[1]))
    pos={p[1]:idx for idx,p in enumerate(chronological)}
    train_times=sorted([(t,rel) for t,rel,sp,_,_,_,_ in chronological if sp=='train'])
    # nuScenes preserva um grid de 155 amostras incluindo 40 guardas; vídeo preserva os frames brutos.
    raw_grid=[]
    if source_kind=='nuscenes':
        raw_grid=rd(dataset/'dataset_manifest.json').get('split',{}).get('raw_grid',[])
        raw_grid=sorted(raw_grid,key=lambda x:int(x['timestamp_us']))
    time_to_selected_rel={int(f.get('timestamp',-1)):rel for rel,f in frame_by_path.items()} if source_kind=='nuscenes' else {}
    for position,(order,rel,split,i,raw_ts,nominal,unit) in enumerate(chronological):
        stem=Path(rel).stem
        sources={'real':dataset/rel}
        for method in ('nerfacto','splatfacto'):
            if split=='test':
                original_index=test_source_by_stem[stem]
                sources[method]=out/f'images/{method}/eval_img_{original_index:04d}.png'
            elif split=='train':
                sources[method]=None  # Somente imagens reais para TRAIN.
            else:
                sources[method]=Path(results[method][split]['rgb_map'][stem])
        target_paths={};hashes={}
        for domain,src in sources.items():
            if src is None:
                target_paths[domain]=None;hashes[domain]=None
                continue
            dest=out/'images'/split/domain/f'frame_{i:04d}.png'
            hashes[domain]=pixel_png(src,dest)
            target_paths[domain]=dest.relative_to(out).as_posix()
        if split=='test':
            # As dez vistas usadas nas métricas TEST têm de permanecer pixel-idênticas.
            for method in ('nerfacto','splatfacto'):
                if sha(sources[method])!=hashes[method]:
                    with Image.open(sources[method]) as a,Image.open(out/target_paths[method]) as b:
                        import numpy as np
                        if not np.array_equal(np.asarray(a.convert('RGB')),np.asarray(b.convert('RGB'))):
                            raise RuntimeError('TEST PNG alterou pixels ao normalizar: '+stem)
        prev=chronological[position-1] if position else None
        foll=chronological[position+1] if position+1<len(chronological) else None
        scope='selected_115_frames_only'
        previous_source=dataset/prev[1] if prev else None
        following_source=dataset/foll[1] if foll else None
        before_ts=prev[4] if prev else None;after_ts=foll[4] if foll else None
        # Contexto de captura completo quando disponível.
        if source_kind=='video':
            raw_prev=video_source_neighbor(root,stem,-1)
            raw_next=video_source_neighbor(root,stem,+1)
            if raw_prev is not None:
                previous_source=raw_prev
                before_ts=int(re.search(r'(\d+)$',raw_prev.stem).group(1))
            if raw_next is not None:
                following_source=raw_next
                after_ts=int(re.search(r'(\d+)$',raw_next.stem).group(1))
            if raw_prev is not None and raw_next is not None:scope='all_extracted_video_frames'
        elif raw_grid:
            idx=next((n for n,x in enumerate(raw_grid) if int(x['timestamp_us'])==raw_ts),None)
            if idx is not None:
                if idx>0:
                    b=raw_grid[idx-1];before_ts=int(b['timestamp_us'])
                    selected_rel=time_to_selected_rel.get(before_ts)
                    previous_source=dataset/selected_rel if selected_rel else extra_source_frames.get(before_ts)
                    # Captura de guarda sem PNG do próprio A: usar arquivo nuScenes original, se disponível.
                if idx+1<len(raw_grid):
                    after_ts=int(raw_grid[idx+1]['timestamp_us'])
                    selected_rel=time_to_selected_rel.get(after_ts)
                    following_source=dataset/selected_rel if selected_rel else extra_source_frames.get(after_ts)
                scope='nuscenes_155_grid_timestamps_with_raw_neighbors' if extra_source_frames else 'nuscenes_155_grid_timestamps; neighbor_PNG_only_if_selected'
        # segundos no vídeo, microssegundos no nuScenes
        gap_before=((raw_ts-before_ts)*250.0 if source_kind=='video' else (raw_ts-before_ts)/1000.) if before_ts is not None else None
        gap_after=((after_ts-raw_ts)*250.0 if source_kind=='video' else (after_ts-raw_ts)/1000.) if after_ts is not None else None
        lower=[x for x in train_times if x[0]<order]
        upper=[x for x in train_times if x[0]>order]
        tprev=lower[-1] if lower else None;tnext=upper[0] if upper else None
        if split=='train':relation='training_observation'
        elif tprev and tnext:relation='temporal_interpolation_between_train_observations'
        elif tprev or tnext:relation='temporal_extrapolation_outside_train_span'
        else:relation='no_surrounding_train_observation'
        row={
          'dataset_id':dataset_id,'split':split,'split_index':i,'chronological_index':position,
          'source_image':rel,'source_type':frame.get('source_type','video_frame' if source_kind=='video' else None),
          'source_timestamp_us':raw_ts if source_kind=='nuscenes' else None,
          'nominal_time_seconds':nominal if source_kind=='video' else round(nominal-chronological[0][0],6),
          'real_png':target_paths['real'],'nerfacto_png':target_paths['nerfacto'],
          'splatfacto_png':target_paths['splatfacto'],
          'real_sha256':hashes['real'],'nerfacto_sha256':hashes['nerfacto'],'splatfacto_sha256':hashes['splatfacto'],
          'preceding_source_frame':str(before_ts) if before_ts is not None else None,
          'preceding_source_timestamp_us':before_ts if source_kind=='nuscenes' else None,
          'preceding_gap_ms':gap_before,'following_source_frame':str(after_ts) if after_ts is not None else None,
          'following_source_timestamp_us':after_ts if source_kind=='nuscenes' else None,
          'following_gap_ms':gap_after,'temporal_neighbor_scope':scope,
          'nearest_train_before':tprev[1] if tprev else None,
          'nearest_train_before_gap_ms':round((order-tprev[0])*1000,5) if tprev else None,
          'nearest_train_after':tnext[1] if tnext else None,
          'nearest_train_after_gap_ms':round((tnext[0]-order)*1000,5) if tnext else None,
          'temporal_relation_to_train':relation,
          'render_provenance':('original TEST render from final evaluation' if split=='test'
             else ('selected checkpoint VAL post-selection' if split=='val'
             else 'no TRAIN rendering requested'))
        }
        all_frames.append(row)
        figure=out/'figures/temporal_context'/split/f'frame_{i:04d}.jpg'
        if split in ('val','test'):
            picture_neighbors(previous_source,dataset/rel,following_source,figure,stem)
    assert len(all_frames)==115
    table(out/'tables/all_frames_manifest.csv',FRAMES_FIELDS,sorted(all_frames,key=lambda row:(row['split'],int(row['split_index']))))
    table(out/'tables/frame_temporal_context.csv',FRAMES_FIELDS,sorted(all_frames,key=lambda row:int(row['chronological_index'])))

    # Custos computacionais idênticos em todos os pacotes. Informação ausente NÃO vira zero.
    original_cost=readtable(out/'tables/compute_cost.csv')
    cost_by_method={r['method']:r for r in original_cost}
    summary=readtable(out/'tables/final_experiment_summary.csv')
    original_sum={r['method']:r for r in summary}
    native_manifest=rd(root/'final/manifest.json') if source_kind=='video' else None
    native_model=(native_manifest or {}).get('models',{})
    computed=[]
    for method in ('splatfacto','nerfacto'):
        old=cost_by_method.get(method,{})
        sm=original_sum[method]
        info=native_model.get(method,{}) if source_kind=='video' else {}
        training_min=numeric(old.get('training_minutes'))
        test_seconds=numeric(old.get('render_total_seconds_including_overhead'))
        trainstat=info.get('training_peak_gpu_memory_mib_observed') if source_kind=='video' else (training_stats or {}).get(method,{}).get('peak_gpu_memory_mib_sampled')
        trainvals=info.get('training_peak_gpu_memory_sampling_interval_seconds') if source_kind=='video' else (training_stats or {}).get(method,{}).get('peak_gpu_memory_sampling_interval_seconds')
        rec={
            'dataset_id':dataset_id,'method':method,'training_minutes':training_min,
            'training_wall_seconds':training_min*60 if training_min is not None else None,
            'train_gpu_peak_mib_sampled':trainstat,
            'train_gpu_peak_sample_interval_seconds':trainvals,
            'checkpoint_size_mib':numeric(sm.get('checkpoint_size_mib')),
            'val_candidate_render_wall_seconds':info.get('validation_candidate_render_wall_seconds'),
            'test_render_wall_seconds':test_seconds,
            'train_final_render_wall_seconds':None,
            'val_final_render_wall_seconds':results[method]['val']['wall_seconds'],
            'additional_final_renders_wall_seconds':round(results[method]['val']['wall_seconds'],5),
            'additional_final_render_gpu_peak_mib_sampled':max((v for v in
               (results[method]['val']['gpu_peak_mib_sampled'],)
                 if v is not None),default=None),
            'render_total_seconds_including_overhead':test_seconds,
            'test_render_seconds_per_view_including_overhead':test_seconds/10 if test_seconds is not None else None,
            'gpu_name':(native_manifest or {}).get('gpu') if source_kind=='video' else None,
            'gpu_memory_measurement_note':'nvidia-smi sampled process memory, 2s on extra renders; training GPU N/D if not recorded; never an exact peak',
            'render_timing_note':'TRAIN not rendered; VAL 10 views after checkpoint selection; TEST preserved from original evaluation. Times include startup and PNG writing, not pure FPS',
            'train_final_render_views':0,
            'val_final_render_views':10,
            'test_render_views':10,
        }
        computed.append(rec)
    table(out/'tables/compute_cost.csv',COST_FIELDS,computed)
    wr(out/'provenance/full_frame_render_experiment.json',{
      'dataset_id':dataset_id,'extension_version':'V4_val_test_only',
      'checkpoint_selection':{m:{'selected_step':int(selected_steps[m]),
          'config':str(selected_configs[m])} for m in selected_steps},
      'test_re_rendered':False,'test_heldout_unchanged':True,
      'train_rendered':False,'train_real_count':95,'validation_final_render_count_per_method':10,
      'validation_post_selection_only':True,'original_train_count_preserved':95,
      'original_transforms_sha256_before_and_after':
          [source_transforms_sha,sha(dataset/'transforms.json')],
      'render_profiling':results,'dataset_transforms_sha256':sha(dataset/'transforms.json'),
      'notes':['Nenhuma imagem TRAIN foi renderizada; as 95 observações reais permanecem arquivadas.',
               'VAL foi renderizada usando os checkpoints já selecionados sem nova seleção.',
               'TEST é reutilizado pixel a pixel, sem rerenderização nem reajuste.',
               'Para vídeo, tempo é posição nominal na amostragem 4 FPS, não timestamp de hardware.',
               'Para nuScenes, vizinhos físicos guardados podem não ter PNG exportado.',
               'Tempos de subprocesso não são FPS em tempo real.']})
    if source_kind=='video':
        (out/'provenance/video_sampling_and_splits.json').write_text(
            (root/'planned_split.json').read_text(encoding='utf-8'),encoding='utf-8')
    else:
        wr(out/'provenance/nuscenes_temporal_sampling.json',{
           'raw_grid':raw_grid,'grid_source':'A V7 dataset_manifest.split.raw_grid',
           'guard_count':sum(x.get('dataset_split')=='guard' for x in raw_grid)})
    # Novos gráficos focados em custo, ausentes são declarados como N/D.
    try:
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt
        for col,filename,title,unit in (
          ('training_minutes','training_cost_minutes.png','Treinamento (wall time)','min'),
          ('test_render_wall_seconds','test_process_time_seconds.png','TEST: processo completo ns-render','s'),
          ('additional_final_renders_wall_seconds','audit_renders_seconds.png','VAL (10): geração de evidências','s'),
          ('train_gpu_peak_mib_sampled','gpu_train_peak_sampled.png','VRAM observada no treinamento','MiB'),
          ('checkpoint_size_mib','checkpoint_size_mib.png','Tamanho do checkpoint selecionado','MiB')):
            values=[numeric(r.get(col)) for r in computed]
            fig,ax=plt.subplots(figsize=(7,4))
            bars=ax.bar(['Splatfacto','Nerfacto'],[v or 0.0 for v in values])
            ax.set_ylabel(unit);ax.set_title(title);ax.grid(axis='y',alpha=.2)
            for bar,value in zip(bars,values):
                ax.text(bar.get_x()+bar.get_width()/2,bar.get_height(),
                        f'{value:.2f}' if value is not None else 'N/D',ha='center',va='bottom')
            fig.tight_layout();fig.savefig(out/'figures/charts'/filename,dpi=145);plt.close(fig)
    except ImportError:
        print('AVISO: matplotlib ausente; métricas e PNGs ainda são válidos.')
    # Galeria contém 115 imagens reais; apenas VAL/TEST incluem as reconstruções.
    gallery_parts=["<!doctype html><html lang='pt-BR'><meta charset='utf-8'><title>TCC - evidências RGB</title><style>body{font:15px/1.5 system-ui;max-width:1400px;margin:30px auto;padding:12px}.trio{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:12px}img{max-width:100%}article{border-top:1px solid #ccc;padding:15px 0}summary{font-size:21px;cursor:pointer}</style><h1>Evidências RGB: todas as capturas reais</h1><p>115 imagens reais: 95 TRAIN, 10 VAL, 10 TEST. TRAIN contém somente originais. VAL (10) e TEST (10) têm renderizações por método. TEST foi reutilizado da avaliação final, sem rerenderização ou reajuste.</p>"]
    for split in ('train','val','test'):
        subset=sorted([r for r in all_frames if r['split']==split],key=lambda r:int(r['split_index']))
        gallery_parts.append('<details '+('open' if split=='test' else '')+'><summary>'+split.upper()+' — '+str(len(subset))+' capturas</summary>')
        for row in subset:
            i=int(row['split_index']);stem=html.escape(str(row['source_image']))
            gallery_parts.append('<article><h3>'+split.upper()+' '+str(i+1)+' — '+stem+'</h3><p>t = '+html.escape(str(row['nominal_time_seconds']))+' segundos</p><div class="trio">')
            for label,key in [('REAL','real_png'),('NERFACTO','nerfacto_png'),('SPLATFACTO','splatfacto_png')]:
                image_rel=row[key]
                if image_rel:
                    gallery_parts.append('<div><b>'+label+'</b><img loading="lazy" src="'+html.escape(image_rel,quote=True)+'"></div>')
                else:
                    gallery_parts.append('<div><b>'+label+'</b><p>Não renderizada (TRAIN preservado apenas como imagem real)</p></div>')
            gallery_parts.append('</div>')
            neighbor=f'figures/temporal_context/{split}/frame_{i:04d}.jpg'
            if (out/neighbor).exists():
                gallery_parts.append('<p>Captura anterior / atual / posterior:</p><img loading="lazy" src="'+neighbor+'">')
            gallery_parts.append('</article>')
        gallery_parts.append('</details>')
    gallery_parts.append('</html>')
    (out/'GALERIA_EVIDENCIAS_RGB.html').write_text(''.join(gallery_parts),encoding='utf-8')
    old_html=(out/'ANALISE_FINAL.html').read_text(encoding='utf-8')
    old_html=old_html.replace('<body>','<body><p><a href="GALERIA_EVIDENCIAS_RGB.html">Ver imagens reais e reconstruções VAL / TEST</a></p>',1)
    (out/'ANALISE_FINAL.html').write_text(old_html,encoding='utf-8')
    readme=out/'LEIA_ME.txt'
    readme.write_text(readme.read_text(encoding='utf-8')+'\nV4: 115 imagens reais (95 TRAIN / 10 VAL / 10 TEST). Nenhuma TRAIN renderizada. As 10 VAL e 10 TEST possuem reconstruções por método; TEST é cópia pixel-idêntica da avaliação original. Custos documentados em tables/compute_cost.csv; medições ausentes não são zero. Contexto temporal em tables/frame_temporal_context.csv; imagens originais excluídas pela guarda aparecem em metadados, se conhecidos. Galeria: GALERIA_EVIDENCIAS_RGB.html\n',encoding='utf-8')
    m=rd(manifest_path)
    m['evidence_version']='V4_95_10_10_val_test_only'
    m['evidence_counts']={'train_real':95,'val_real':10,'test_real':10,
        'train_reconstructed_per_method':0,
        'val_reconstructed_per_method':10,'test_reconstructed_per_method':10}
    m.pop('selected_train_indices',None)
    m.pop('selected_train_relpaths',None)
    m['test_rerendered']=False
    m['test_metric_protocol_unchanged']=True
    m['full_gallery']='GALERIA_EVIDENCIAS_RGB.html'
    m['temporal_evidence_csv']='tables/frame_temporal_context.csv'
    m['computational_cost_csv']='tables/compute_cost.csv'
    records=[]
    for p in sorted(out.rglob('*')):
        if p.is_file() and p!=manifest_path:
            if p.is_symlink() or p.suffix.lower() in ('.mov','.mp4','.ply','.ckpt','.pt'):
                raise RuntimeError('Arquivo proibido no ZIP final: '+str(p))
            records.append({'path':p.relative_to(out).as_posix(),'sha256':sha(p),'bytes':p.stat().st_size})
    m['file_integrity_excluding_self']=records
    wr(manifest_path,m)
    temporary=zipfile_path.with_suffix('.temporary.zip')
    with zipfile.ZipFile(temporary,'w',zipfile.ZIP_DEFLATED,compresslevel=4,allowZip64=True) as z:
        for p in sorted(out.rglob('*')):
            if p.is_file():z.write(p,p.relative_to(out).as_posix())
    with zipfile.ZipFile(temporary) as z:
        if z.testzip() is not None:raise RuntimeError('ZIP enriquecido corrompido')
        if len([x for x in z.namelist() if x.startswith('images/') and x.endswith('.png')])!=185:
            # 155 imagens organizadas (115 reais + 20 VAL + 20 TEST) + 30 TEST canônicos.
            raise RuntimeError('ZIP V4 incompleto: esperado 185 PNGs no total.')
    temporary.replace(zipfile_path)
    if sha(dataset/'transforms.json')!=source_transforms_sha:
        raise RuntimeError('Dataset original foi alterado pela extensão ilustrativa.')
    print('EVIDÊNCIAS VAL/TEST V4:',out,'| ZIP:',zipfile_path)
    print('TRAIN 95 reais / 0 renderizações; VAL 10/10/10; TEST 10/10/10; 155 imagens organizadas (+30 TEST canônicas).')
    return {'out':out,'zip':zipfile_path,'cost':computed,'frames':len(all_frames),'renders':results}

'''
_TCC_EVIDENCE_FILE = Path('/content/tcc_evidencias_val_test_v4.py')
_TCC_EVIDENCE_FILE.write_text(_TCC_EVIDENCE_SOURCE, encoding='utf-8')
_spec_evidence = importlib.util.spec_from_file_location('tcc_evidencias_val_test_v4',_TCC_EVIDENCE_FILE)
_evidencias = importlib.util.module_from_spec(_spec_evidence)
_spec_evidence.loader.exec_module(_evidencias)

_G_RAW_CAPTURE_IMAGES = {}
if 'raw_candidates' in globals() and 'NUSCENES_ROOT' in globals():
    for sd in globals()['raw_candidates']:
        candidate=Path(globals()['NUSCENES_ROOT'])/sd['filename']
        if candidate.is_file():_G_RAW_CAPTURE_IMAGES[int(sd['timestamp'])]=candidate
_G_CONFIG_DIR = G_D_REPORT_PATH.parent.parent/'configs'
_G_SELECTED_CONFIGS = {
 'splatfacto': _G_CONFIG_DIR/'splatfacto_selected.yml',
 'nerfacto': _G_CONFIG_DIR/'nerfacto_selected.yml',
}
G_EVIDENCE = _evidencias.extend(
 dataset_id='nuscenes_scene0655',source_kind='nuscenes',
 run_root=G_D_REPORT_PATH.parent.parent,dataset=G_DATASET,
 unified_dir=G_UNIFIED_DIR,unified_zip=G_UNIFIED_ZIP,
 selected_configs=_G_SELECTED_CONFIGS,
 selected_steps={k:int(v['validation_step']) for k,v in G_D['model_selection'].items()},
 ns_render=Path(globals().get('TCC_PYTHON','/content/tcc_env/bin/python')).parent/'ns-render',
 extra_source_frames=_G_RAW_CAPTURE_IMAGES,
 training_stats={'splatfacto':G_B,'nerfacto':G_C}
)
G_UNIFIED_DRIVE = (_unified.upload_single_zip(G_UNIFIED_ZIP,'nuscenes_scene0655')
                   if os.environ.get('TCC_EXPORT_DIR', '').strip() else None)
FINAL_RESULTS_DIR = G_UNIFIED_DIR
FINAL_SUMMARY_CSV = G_UNIFIED_DIR/'tables/final_experiment_summary.csv'
FINAL_MANIFEST_PATH = G_UNIFIED_DIR/'final_experiment_manifest.json'
FINAL_ZIP_PATH = G_UNIFIED_ZIP
print('G V10 — ESQUEMA COMUM:', _unified.SCHEMA)
print('Pasta final padronizada:', G_UNIFIED_DIR)
print('Exportação opcional do ZIP:', G_UNIFIED_DRIVE)


In [ ]:
# =============================================================================
# CÉLULA H V6.5 — iPhone | 15.000 iteracoes para Splatfacto e Nerfacto-Big | 95 TRAIN / 10 VAL / 10 TEST
# NeRF (nerfacto-big), Splatfacto, PSNR / SSIM / LPIPS e YOLO
# Executar após as células 0 -> SETUP -> 1 (Nerfstudio 1.1.5)
# COLMAP: treino-only + localização PnP e refinamento de pose em mapa FIXO.
# Mantém intermediários em /content; exportação do ZIP é opcional.
# =============================================================================
from pathlib import Path
from datetime import datetime, timezone
import os, subprocess, json, hashlib, shutil

# -----------------------------------------------------------------------------
# CONFIGURAÇÕES EDITÁVEIS
# -----------------------------------------------------------------------------
H_VIDEO = Path(os.environ['TCC_H_VIDEO']).expanduser() if os.environ.get('TCC_H_VIDEO') else None
H_DRIVE_RESULTS = Path(os.environ.get('TCC_EXPORT_DIR', '/content/tcc_results'))
H_SAMPLE_FPS = 2.32
H_SPLIT_MODE = 'ampliado_95_10_10'  # Experimento H V3 fixo; não altere no meio da rodada
H_YOLO_MODEL = 'yolov8x.pt'
H_PYTHON = Path(globals().get('TCC_PYTHON', '/content/tcc_env/bin/python'))
H_SEND_FINAL_TO_DRIVE = False  # Exportação desativada por padrão; TCC_EXPORT_DIR habilita ZIP padronizado.
H_RESUME_FROM = None  # Reconstrução e treinamento iniciados do zero.
H_REFINE_QUERY_POSES = True  # Apenas pose VAL/TEST no mapa TRAIN congelado
H_QUERY_MATCH_NEIGHBORS = 32  # Mais suporte visual para localização PnP; fixo antes do teste

# -----------------------------------------------------------------------------
# VALIDAR CAMINHO DO VÍDEO FORNECIDO PELO USUÁRIO
# -----------------------------------------------------------------------------
if not H_PYTHON.is_file():
    raise FileNotFoundError('Execute as células 0, SETUP e 1 primeiro: ' + str(H_PYTHON))
if not globals().get('ENVIRONMENT_VALIDATED', False):
    raise RuntimeError('Célula 1 não validou o ambiente. Execute 0, SETUP e 1.')
if H_SPLIT_MODE != 'ampliado_95_10_10':
    raise ValueError('A H V3 usa exclusivamente 95/10/10; mantenha H_SPLIT_MODE=ampliado_95_10_10.')
if not isinstance(H_REFINE_QUERY_POSES, bool):
    raise TypeError('H_REFINE_QUERY_POSES deve ser booleano.')
if not (8 <= H_QUERY_MATCH_NEIGHBORS <= 60):
    raise ValueError('H_QUERY_MATCH_NEIGHBORS fora da faixa permitida (8 a 60).')
if H_RESUME_FROM is not None:
    raise RuntimeError('O novo protocolo exige reconstrução nova. Deixe H_RESUME_FROM=None.')

# Arquivo de entrada fornecido explicitamente por quem executa.
if H_VIDEO is None:
    raise ValueError('Defina TCC_H_VIDEO em configuração privada antes de executar esta célula.')
if not H_VIDEO.is_file():
    raise FileNotFoundError('Vídeo de entrada não encontrado. Verifique TCC_H_VIDEO.')

H_STAMP = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
H_RUN = Path('/content/iphone_experiments') / (H_SPLIT_MODE + '_' + H_STAMP)
H_RUN.mkdir(parents=True, exist_ok=False)
H_CONFIG = {
    'video': str(H_VIDEO.resolve()), 'run': str(H_RUN), 'python': str(H_PYTHON),
    'fps': H_SAMPLE_FPS, 'split_mode': H_SPLIT_MODE, 'yolo_model': H_YOLO_MODEL,
    'protocol_version': 'H_V6_5_train_only_95_10_10_equal_15000_steps_pose_refined',
    'pose_refinement': H_REFINE_QUERY_POSES, 'query_train_neighbors': H_QUERY_MATCH_NEIGHBORS
}
(H_RUN / 'run_config.json').write_text(json.dumps(H_CONFIG, indent=2), encoding='utf-8')

H_WORKER_SOURCE = r'''
"""Pipeline iPhone: Nerfstudio 1.1.5, execução pelo Python isolado do Colab."""
from pathlib import Path
from datetime import datetime, timezone
import argparse, csv, hashlib, html, json, math, os, re, shutil, subprocess, time, zipfile
import numpy as np
from PIL import Image, ImageDraw


def sha(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()


def save(p,obj):
    Path(p).parent.mkdir(parents=True,exist_ok=True)
    Path(p).write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str)+'\n')


def table(p,rows):
    if not rows: return
    with Path(p).open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)


RUN_SAMPLES = {}

def _gpu_process_vram_mib(process_pid):
    try:
        p = subprocess.run(['nvidia-smi','--query-compute-apps=pid,used_gpu_memory',
            '--format=csv,noheader,nounits'],capture_output=True,text=True,timeout=6)
        if p.returncode:return None
        readings=[]
        for line in p.stdout.splitlines():
            cols=[v.strip() for v in line.split(',')]
            if len(cols)<2:continue
            if cols[0].isdigit() and int(cols[0])==process_pid:
                try:readings.append(float(cols[1].split()[0]))
                except ValueError:pass
        return max(readings) if readings else None
    except (OSError,subprocess.TimeoutExpired):return None

def run(cmd,log,timeout=14400):
    cmd=list(map(str,cmd));print('Executando:', ' '.join(cmd),flush=True)
    start=time.monotonic();peak=None;observations=0
    with Path(log).open('w') as f:
        p=subprocess.Popen(cmd,stdout=f,stderr=subprocess.STDOUT)
        try:
            while p.poll() is None:
                if time.monotonic()-start>timeout: raise TimeoutError('Tempo excedido: '+str(log))
                vram=_gpu_process_vram_mib(p.pid)
                if vram is not None:
                    peak=vram if peak is None else max(peak,vram)
                    observations+=1
                time.sleep(2)
                if int(time.monotonic()-start)//30 != int(time.monotonic()-start-2)//30:
                    print(f'  {time.monotonic()-start:.0f}s; log local: {log}',flush=True)
            if p.returncode: raise RuntimeError(Path(log).read_text(errors='replace')[-5000:])
        finally:
            if p.poll() is None:
                p.terminate()
                try:p.wait(timeout=10)
                except subprocess.TimeoutExpired:p.kill();p.wait()
    seconds=time.monotonic()-start
    RUN_SAMPLES[str(Path(log))]={'wall_seconds':seconds,'gpu_peak_mib_sampled':peak,
        'gpu_memory_sample_count':observations,'gpu_memory_sampling_interval_seconds':2}
    return seconds

def pixel_hash(p):
    with Image.open(p) as im:
        im=im.convert('RGB');return hashlib.sha256(str(im.size).encode()+im.tobytes()).hexdigest()


def split_indices(n, mode):
    """Divisões temporais planejadas ANTES de qualquer reconstrução.

    comparavel_48_3_3: 48/3/3 (n>=80), guardas de um frame junto a holdouts.
    ampliado_95_10_10: grade histórica, exige exatamente os 95/10/10 se n=155.
    """
    if mode == 'comparavel_48_3_3':
        if n < 80:
            raise RuntimeError('Modo 48/3/3 precisa de pelo menos 80 frames.')
        anchors = sorted({int(round(x)) for x in np.linspace(8, n - 9, 6)})
        if len(anchors) != 6:
            raise RuntimeError('Não foi possível criar seis imagens reservadas.')
        val, test = anchors[::2], anchors[1::2]
        hold = set(val + test)
        guard = {j for i in hold for j in (i-1, i+1) if 0 <= j < n} - hold
        pool = [i for i in range(n) if i not in hold | guard]
        if len(pool) < 48:
            raise RuntimeError('Não restaram 48 imagens para treino.')
        train = [pool[int(round(x))] for x in np.linspace(0, len(pool)-1, 48)]
        if len(set(train)) != 48:
            raise RuntimeError('Divisão 48/3/3 produziu treino duplicado.')
    elif mode == 'ampliado_95_10_10':
        val = list(range(5, n-2, 15))
        test = list(range(10, n-2, 15))
        hold = set(val + test)
        guard = {j for i in hold for j in (i-1, i+1) if 0 <= j < n} - hold
        train = [i for i in range(n) if i not in hold | guard]
        if (len(train), len(val), len(test)) != (95, 10, 10):
            raise RuntimeError(
                'O modo ampliado foi definido para 155 frames (95/10/10). '
                f'Este vídeo extraiu {n} frames: {(len(train), len(val), len(test))}. '
                'Ajuste H_SAMPLE_FPS ou defina um novo protocolo antes da execução.'
            )
    else:
        raise ValueError('H_SPLIT_MODE não reconhecido: ' + mode)
    assert len(set(train) & (set(val) | set(test))) == 0
    return {'train': sorted(train), 'val': val, 'test': test, 'guard': sorted(guard)}



def psnr(a,b):
    mse=np.mean(((a.astype(np.float64)-b.astype(np.float64))/255.)**2)
    return float(-10*np.log10(max(mse,1e-15)))


def image_map(root,split,kind,expected):
    paths=list((Path(root)/split/kind).rglob('*.png'))
    found={p.stem:p for p in paths}
    want={Path(p).stem for p in expected}
    if len(paths)!=len(found) or set(found)!=want:
        raise RuntimeError(f'Renders incompletos: {root}/{split}/{kind}; esperado {len(want)}, obtido {len(found)}')
    return found


def render(config,step,split,dataset,out,nsrender):
    import yaml
    # Config confiável, gerado nesta execução pelo Nerfstudio.
    cfg=yaml.load(config.read_text(),Loader=yaml.Loader)
    cfg.load_step=int(step)
    assert cfg.pipeline.model.camera_optimizer.mode=='off'
    expected=cfg.get_checkpoint_dir()/f'step-{step:09d}.ckpt'
    assert expected.is_file(), expected
    out.mkdir(parents=True,exist_ok=False)
    selected=out/'config.yml';selected.write_text(yaml.dump(cfg))
    seconds=run([nsrender,'dataset','--load-config',selected,'--output-path',out/'renders',
                 '--data',dataset,'--split',split,'--image-format','png',
                 '--rendered-output-names','rgb','gt-rgb'],out/'render.log',3600)
    log=re.sub(r'\x1b\[[0-9;]*m','',(out/'render.log').read_text(errors='replace'))
    if not re.search(r'Done loading checkpoint from',log):
        raise RuntimeError('Render não confirmou carregamento do checkpoint.')
    # load_step e diretório são fixados no config; verificar número quando log não truncado.
    printed=re.findall(r'Done loading checkpoint from[^\n]*step-(\d+)\.ckpt',log)
    if printed and int(printed[-1])!=step:raise RuntimeError('Checkpoint de render divergente.')
    return out/'renders',seconds,expected


def init_train_points(dataset,tf,train_names,source_by_output,raw_dir,stats):
    from nerfstudio.data.utils.colmap_parsing_utils import read_images_binary,read_points3D_binary
    model=dataset/'sparse'
    if not (model/'images.bin').is_file():raise RuntimeError('Modelo COLMAP retificado não encontrado.')
    ims=read_images_binary(model/'images.bin');pts=read_points3D_binary(model/'points3D.bin')
    basename_train={Path(p).name for p in train_names}
    colors={}
    for iid,im in ims.items():
        if Path(im.name).name not in basename_train:continue
        # Coordenadas COLMAP referem-se às imagens copiadas sem redimensionamento.
        path=dataset/'images'/im.name
        if not path.is_file():raise FileNotFoundError(path)
        colors[iid]=np.asarray(Image.open(path).convert('RGB'))
    transform=np.asarray(tf.get('applied_transform',np.eye(4)[:3]),dtype=float)
    if transform.shape!=(3,4):raise RuntimeError('applied_transform inválida')
    rows=[]
    for p in pts.values():
        if not np.isfinite(p.xyz).all() or p.error>2.0:continue
        obs=[]
        for iid,idx in zip(p.image_ids,p.point2D_idxs):
            if iid not in colors:continue
            xy=np.rint(ims[iid].xys[idx]).astype(int);arr=colors[iid]
            if 0<=xy[0]<arr.shape[1] and 0<=xy[1]<arr.shape[0]:obs.append(arr[xy[1],xy[0]])
        if len(obs)<3:continue
        xyz=transform[:,:3]@p.xyz+transform[:,3]
        rgb=np.median(obs,axis=0).astype(np.uint8)
        rows.append((*xyz,*rgb))
    if len(rows)<500:raise RuntimeError(f'Somente {len(rows)} pontos confiáveis com >=3 observações TRAIN. Não iniciar treino.')
    if len(rows)>120000:
        idx=np.random.default_rng(42).choice(len(rows),120000,replace=False);rows=[rows[i] for i in sorted(idx)]
    ply=dataset/'train_supported_points.ply'
    with ply.open('w') as f:
        f.write('ply\nformat ascii 1.0\nelement vertex '+str(len(rows))+'\nproperty float x\nproperty float y\nproperty float z\nproperty uchar red\nproperty uchar green\nproperty uchar blue\nend_header\n')
        for r in rows:f.write('%.8f %.8f %.8f %d %d %d\n'%r)
    tf['ply_file_path']=ply.name
    stats.update(initial_points=len(rows),initial_points_sha256=sha(ply),point_color_source='TRAIN only',
                 point_geometry_source='TRAIN-only SfM, >=3 TRAIN observations, reprojection error <=2px')


def iou(a,b):
    xy=np.maximum(a[:2],b[:2]);zw=np.minimum(a[2:],b[2:]);inter=np.maximum(zw-xy,0).prod()
    area=lambda c:np.maximum(c[2:]-c[:2],0).prod()
    return float(inter/max(area(a)+area(b)-inter,1e-12))


def matches(real,pred,threshold):
    # Mesmo princípio de F: matching bipartido de cardinalidade máxima por classe.
    matrix=np.zeros((len(real),len(pred)))
    adjacency={}
    for i,a in enumerate(real):
        for j,b in enumerate(pred):
            if a['class_id']==b['class_id']:matrix[i,j]=iou(np.array(a['xyxy']),np.array(b['xyxy']))
        adjacency[i]=sorted([j for j in range(len(pred)) if matrix[i,j]>=threshold],key=lambda j:matrix[i,j],reverse=True)
    order=sorted(range(len(real)),key=lambda i:max((matrix[i,j] for j in adjacency[i]),default=0),reverse=True)
    assigned={}
    def augment(i,seen):
        for j in adjacency[i]:
            if j in seen:continue
            seen.add(j)
            if j not in assigned or augment(assigned[j],seen):assigned[j]=i;return True
        return False
    for i in order:augment(i,set())
    return [(i,j,float(matrix[i,j])) for j,i in sorted(assigned.items())]


def verify_input_gt(gt,input_path):
    a=np.asarray(Image.open(gt).convert('RGB')).astype(np.int16)
    b=np.asarray(Image.open(input_path).convert('RGB')).astype(np.int16)
    if a.shape!=b.shape or np.max(np.abs(a-b))>1:
        raise RuntimeError('GT renderizada diverge da referência retificada: '+str(input_path))



def refine_query_poses(root, original_model, train_model, query_splits, logs, stats):
    """Audita e opcionalmente refina somente as 6-DOF poses das consultas.

    Os pontos 3D / intrinsecos / poses TRAIN permanecem byte-identicos. A LM
    utiliza apenas 2D-3D das proprias consultas para estimar suas cameras,
    nunca a cor de referencia para otimizar o modelo. Nao usa PSNR de TEST.
    """
    import cv2
    from nerfstudio.data.utils.colmap_parsing_utils import (
        read_images_binary, read_points3D_binary, read_cameras_binary,
        qvec2rotmat, rotmat2qvec, write_images_binary
    )
    registered = read_images_binary(original_model / 'images.bin')
    training = read_images_binary(train_model / 'images.bin')
    points = read_points3D_binary(train_model / 'points3D.bin')
    cameras = read_cameras_binary(train_model / 'cameras.bin')
    image_by_name = {im.name: im for im in registered.values()}
    if any(name not in image_by_name for name in query_splits):
        raise RuntimeError('Faltam imagens reservadas no mapa para auditar poses.')
    # Evitar ajuste com matriz intrinseca incorreta.
    if len(cameras) != 1 or next(iter(cameras.values())).model != 'SIMPLE_RADIAL':
        raise RuntimeError('A H V3 espera o modelo SIMPLE_RADIAL de uma unica camera COLMAP.')
    camera = next(iter(cameras.values()))
    f, cx, cy, radial = map(float, camera.params)
    K = np.array([[f,0,cx],[0,f,cy],[0,0,1]], dtype=np.float64)
    D = np.array([radial,0.,0.,0.,0.], dtype=np.float64)
    centers=[]
    for im in training.values():
        R=qvec2rotmat(im.qvec)
        centers.append(-R.T@np.asarray(im.tvec))
    extent=float(np.linalg.norm(np.ptp(np.stack(centers),axis=0)))
    if extent<=1e-8:raise RuntimeError('Mapa de treino degenerado ao auditar poses.')
    rows=[]
    updated=dict(registered)
    min_correspondences=12
    threshold_px=6.0
    for name in sorted(query_splits):
        im=image_by_name[name]
        ids=np.asarray(im.point3D_ids)
        idx=np.array([j for j,pid in enumerate(ids) if int(pid) in points],dtype=int)
        if len(idx)<min_correspondences:
            raise RuntimeError(f'Pose {name} com somente {len(idx)} associacoes 2D-3D no mapa TRAIN.')
        X=np.asarray([points[int(ids[j])].xyz for j in idx],dtype=np.float64).reshape(-1,3)
        xy=np.asarray(im.xys[idx],dtype=np.float64).reshape(-1,2)
        R=qvec2rotmat(im.qvec)
        rvec,_=cv2.Rodrigues(R)
        tvec=np.asarray(im.tvec,dtype=np.float64).reshape(3,1)
        def errors(rr,tt):
            pts,_=cv2.projectPoints(X,rr,tt,K,D)
            return np.linalg.norm(pts.reshape(-1,2)-xy,axis=1)
        before=errors(rvec,tvec)
        inliers=before<=threshold_px
        n_inlier=int(inliers.sum())
        if n_inlier<min_correspondences:
            raise RuntimeError(
                f'Pose {name}: somente {n_inlier} correspondencias com erro <= {threshold_px} px. '
                'A localizacao nao tem suporte suficiente; nao avaliar TEST silenciosamente.'
            )
        center0=-R.T@tvec.reshape(3)
        after=before.copy()
        applied=False
        shift=0.0
        angle=0.0
        if stats['pose_refinement_requested'] and n_inlier>=min_correspondences:
            # Inliers predefinidos pela reprojecao da pose robusta obtida no COLMAP.
            # O filtro nao utiliza PSNR, SSIM ou YOLO.
            try:
                rr,tt=cv2.solvePnPRefineLM(X[inliers],xy[inliers],K,D,
                                            rvec.copy(),tvec.copy(),
                                            criteria=(cv2.TERM_CRITERIA_EPS+cv2.TERM_CRITERIA_COUNT,50,1e-8))
                proposed=errors(rr,tt)
                R1,_=cv2.Rodrigues(rr)
                center1=-R1.T@tt.reshape(3)
                shift=float(np.linalg.norm(center1-center0)/extent)
                relative_rot=R1@R.T
                angle=float(np.degrees(np.arccos(np.clip((np.trace(relative_rot)-1)/2,-1,1))))
                # Aceitacao predeterminada; protege contra saltos e degradacao.
                improved=(float(np.median(proposed[inliers]))<=float(np.median(before[inliers]))+1e-7)
                safe=bool(np.all(np.isfinite(proposed)) and shift<=0.10 and angle<=8.0)
                if improved and safe:
                    q=np.asarray(rotmat2qvec(R1),dtype=np.float64)
                    q/=np.linalg.norm(q)
                    updated[im.id]=im._replace(qvec=q,tvec=tt.reshape(3))
                    applied=True
                    after=proposed
            except cv2.error as exc:
                print(f'AVISO: refinamento LM nao aplicado a {name}: {exc}',flush=True)
        rows.append({
            'image':name, 'split':query_splits[name], 'matched_2d3d':len(idx), 'inliers_within_6px':n_inlier,
            'inlier_fraction':round(n_inlier/len(idx),6),
            'median_reprojection_before_px':float(np.median(before[inliers])),
            'median_reprojection_after_px':float(np.median(after[inliers])),
            'p90_reprojection_after_px':float(np.percentile(after[inliers],90)),
            'pose_only_lm_applied':applied,
            'pose_translation_change_over_train_extent':shift,
            'pose_rotation_change_degrees':angle,
        })
    # Copiar modelo: apenas images.bin pode mudar, camera.bin e points3D.bin
    # permanecem originais. Se nao houve melhora a localizacao original e mantida.
    refined_path=root/'sfm'/'registered_with_queries_pose_only'
    shutil.copytree(original_model,refined_path)
    write_images_binary(updated,refined_path/'images.bin')
    original_points=read_points3D_binary(refined_path/'points3D.bin')
    if set(points)!=set(original_points):
        raise RuntimeError('IDs dos pontos de treino mudaram durante a localizacao.')
    for pid,p in points.items():
        if np.max(np.abs(np.asarray(p.xyz)-np.asarray(original_points[pid].xyz)))>1e-12:
            raise RuntimeError('Coordenadas da nuvem TRAIN mudaram durante a localizacao.')
    if sha(refined_path/'cameras.bin')!=sha(train_model/'cameras.bin'):
        raise RuntimeError('Intrinsecos de treino foram modificados por engano.')
    for iid,orig in training.items():
        new=updated[iid]
        if np.max(np.abs(new.qvec-orig.qvec))>1e-12 or np.max(np.abs(new.tvec-orig.tvec))>1e-12:
            raise RuntimeError('Refinamento de consulta alterou uma pose TRAIN.')
    stats['query_pose_diagnostics']=rows
    stats['pose_refinement_applied_count']=sum(bool(r['pose_only_lm_applied']) for r in rows)
    stats['query_pose_min_inliers']=min(r['inliers_within_6px'] for r in rows)
    stats['query_pose_reference']='fixed TRAIN points / SIMPLE_RADIAL / per-query 2D-3D'
    print('Diagnostico de poses reservado:', [(r['image'],r['inliers_within_6px'],r['pose_only_lm_applied']) for r in rows],flush=True)
    return refined_path,rows


def colmap_inductive(root, originals, split, logs, stats):
    """Reconstrói somente TRAIN; VAL/TEST recebem pose via registro PnP.

    Sequência: extrator TRAIN -> matcher TRAIN -> mapper TRAIN ->
    extrator consultas -> pares CONSULTA/TRAIN -> image_registrator.
    Não roda bundle_adjuster nem triangulação após inserir consultas.
    """
    from nerfstudio.data.utils.colmap_parsing_utils import (
        read_images_binary, read_points3D_binary, read_cameras_binary
    )
    source = root / 'sfm_source'
    images_dir = source / 'images'
    images_dir.mkdir(parents=True)
    sfm = root / 'sfm'
    sfm.mkdir()
    database = sfm / 'database.db'
    active = sorted(split['train'] + split['val'] + split['test'])
    for i in active:
        shutil.copy2(originals[i], images_dir / originals[i].name)

    # O banco COLMAP contém inicialmente apenas as imagens de treinamento.
    train_files = [originals[i].name for i in split['train']]
    query_files = [originals[i].name for i in sorted(split['val'] + split['test'])]
    train_list = sfm / 'train.txt'
    query_list = sfm / 'query.txt'
    train_list.write_text('\n'.join(train_files) + '\n')
    query_list.write_text('\n'.join(query_files) + '\n')
    run(['colmap', 'feature_extractor',
         '--database_path', database, '--image_path', images_dir,
         '--image_list_path', train_list,
         '--ImageReader.single_camera', '1',
         '--ImageReader.camera_model', 'SIMPLE_RADIAL',
         '--SiftExtraction.use_gpu', '0'], logs / 'colmap_train_features.log', 7200)
    stats['train_features_only_before_mapper'] = True
    run(['colmap', 'exhaustive_matcher', '--database_path', database,
         '--SiftMatching.use_gpu', '0'], logs / 'colmap_train_matches.log', 14400)
    out = sfm / 'train_sparse'
    out.mkdir()
    run(['colmap', 'mapper', '--database_path', database,
         '--image_path', images_dir, '--output_path', out],
        logs / 'colmap_train_mapper.log', 14400)
    candidates = []
    for p in sorted(out.glob('*/images.bin')):
        ims = read_images_binary(p)
        candidates.append((len(ims), p.parent))
    if not candidates:
        raise RuntimeError('COLMAP TRAIN não produziu reconstrução. Verifique logs e vídeo.')
    count, chosen = max(candidates, key=lambda x: x[0])
    stats['colmap_train_candidates'] = [
        {'model': p.name, 'train_registered': c} for c, p in candidates
    ]
    stats['colmap_train_selected'] = chosen.name
    if count != len(train_files):
        raise RuntimeError(
            f'COLMAP registrou {count}/{len(train_files)} imagens TRAIN. '
            'Para o protocolo comparável, não reduzirei silenciosamente o conjunto. '
            'Verifique o log colmap_train_mapper.log; aumente a sobreposição '
            'ou utilize o modo ampliado como experimento separado.'
        )
    train_images_before = read_images_binary(chosen / 'images.bin')
    train_points_before = read_points3D_binary(chosen / 'points3D.bin')
    cam_ids = sorted(read_cameras_binary(chosen / 'cameras.bin'))
    if len(cam_ids) != 1:
        raise RuntimeError('Esperava uma câmera intrínseca única para o mesmo vídeo.')
    camera_id = cam_ids[0]
    stats['train_point_count_colmap'] = len(train_points_before)
    stats['colmap_camera_id'] = int(camera_id)

    # Agora acrescentar imagens reservadas ao banco, sem recalcular o mapa.
    run(['colmap', 'feature_extractor',
         '--database_path', database, '--image_path', images_dir,
         '--image_list_path', query_list,
         '--ImageReader.existing_camera_id', str(camera_id),
         '--SiftExtraction.use_gpu', '0'], logs / 'colmap_query_features.log', 7200)
    pairs = []
    for qi in sorted(split['val'] + split['test']):
        # Pares SOMENTE consulta-treino. Não há pares consulta-consulta.
        closest = sorted(split['train'], key=lambda i: (abs(i-qi), i))[:stats['colmap_query_match_nearest_train_count']]
        for ti in closest:
            pairs.append(f'{originals[qi].name} {originals[ti].name}')
    pair_path = sfm / 'query_train_pairs.txt'
    pair_path.write_text('\n'.join(pairs) + '\n')
    run(['colmap', 'matches_importer', '--database_path', database,
         '--match_list_path', pair_path, '--match_type', 'pairs',
         '--SiftMatching.use_gpu', '0'], logs / 'colmap_query_matches.log', 14400)
    registered_path = sfm / 'registered_with_queries'
    registered_path.mkdir()
    run(['colmap', 'image_registrator', '--database_path', database,
         '--input_path', chosen, '--output_path', registered_path],
        logs / 'colmap_query_registration.log', 7200)

    train_images_after = read_images_binary(registered_path / 'images.bin')
    points_after = read_points3D_binary(registered_path / 'points3D.bin')
    expected_train = {im.name for im in train_images_before.values()}
    actual = {im.name for im in train_images_after.values()}
    missing = sorted(set(query_files) - actual)
    stats['train_registered'] = len(expected_train)
    stats['query_registered'] = len(set(query_files) & actual)
    stats['query_registration_missing'] = missing
    if not expected_train <= actual:
        raise RuntimeError('O registro de consultas perdeu imagens de treino.')
    if missing:
        raise RuntimeError(
            'Falha ao localizar todas as imagens reservadas no mapa de treino. '
            f'Faltantes ({len(missing)}): {missing}. '
            'NÃO vou substituir as imagens VAL/TEST nem fazer ajuste conjunto. '
            'Consulte os logs de consulta, aumente a cobertura do vídeo ou '
            'teste H_SPLIT_MODE="ampliado_95_10_10" como experimento separado.'
        )
    for iid, im in train_images_before.items():
        newer = train_images_after[iid]
        if (np.max(np.abs(np.asarray(im.qvec) - np.asarray(newer.qvec))) > 1e-10 or
                np.max(np.abs(np.asarray(im.tvec) - np.asarray(newer.tvec))) > 1e-10):
            raise RuntimeError('image_registrator alterou pose TRAIN; protocolo abortado.')
    if set(train_points_before) != set(points_after):
        raise RuntimeError('O conjunto de pontos SfM mudou após o registro das consultas.')
    for pid, p in train_points_before.items():
        if np.max(np.abs(np.asarray(p.xyz) - np.asarray(points_after[pid].xyz))) > 1e-10:
            raise RuntimeError('Geometria SfM mudou após o registro de consultas.')
    stats['training_geometry_frozen_after_query_registration'] = True
    stats['sfm_protocol'] = 'inductive: TRAIN-only SfM; query PnP + optional pose-only LM in fixed map; no joint BA'
    stats['sfm_uses_holdout_images_for_training_geometry'] = False
    stats['holdout_rgb_used_in_radiance_optimization'] = False

    # Refinamento estritamente de pose nas consultas: nenhum BA conjunto.
    query_splits={originals[i].name:'val' for i in split['val']}
    query_splits.update({originals[i].name:'test' for i in split['test']})
    final_model, pose_rows = refine_query_poses(
        root, registered_path, chosen, query_splits, logs, stats
    )
    table(root / 'final' / 'tables' / 'pose_registration_quality.csv', pose_rows)
    # A retificação usa as poses e intrínsecos estimados, SEM recalibrar geometria.
    undistorted = root / 'undistorted'
    run(['colmap', 'image_undistorter', '--image_path', images_dir,
         '--input_path', final_model, '--output_path', undistorted,
         '--output_type', 'COLMAP', '--max_image_size', '800'],
        logs / 'colmap_undistort.log', 3600)
    from nerfstudio.process_data.colmap_utils import colmap_to_json
    colmap_to_json(undistorted / 'sparse', undistorted)
    return undistorted


def normalize_fixed_from_train(dataset, tf, train_names, stats):
    """Define origem/escala usando SOMENTE as câmeras de TRAIN.

    Como orientação/centragem/autoscale do dataparser serão desligados,
    o mesmo sistema de coordenadas valerá em treino, validação e teste.
    """
    train_set = set(train_names)
    poses = np.array([np.asarray(f['transform_matrix'], dtype=np.float64)
                      for f in tf['frames'] if f['file_path'] in train_set])
    if len(poses) != len(train_names):
        raise RuntimeError('Poses TRAIN incompletas durante normalização.')
    centers = poses[:, :3, 3]
    centroid = centers.mean(axis=0)
    extent = float(np.max(np.abs(centers - centroid)))
    if extent <= 1e-5:
        raise RuntimeError('Geometria TRAIN degenerada sem translação útil.')
    scaling = 0.85 / extent
    for f in tf['frames']:
        mat = np.array(f['transform_matrix'], dtype=np.float64)
        mat[:3, 3] = (mat[:3, 3] - centroid) * scaling
        f['transform_matrix'] = mat.tolist()
    # Os pontos SfM serão transformados exatamente como as câmeras.
    a = np.eye(4)
    a[:3, :4] = np.asarray(tf.get('applied_transform', np.eye(4)[:3]), dtype=np.float64)
    n = np.eye(4)
    n[:3, :3] = np.eye(3) * scaling
    n[:3, 3] = -centroid * scaling
    tf['applied_transform'] = (n @ a)[:3].tolist()
    tf['orientation_override'] = 'none'
    stats['normalization'] = {
        'source': 'train_camera_centers_only',
        'center_saved_xyz': centroid.tolist(),
        'scale': scaling,
        'poses_max_abs_train_after': 0.85,
    }


def main():
    ap=argparse.ArgumentParser();ap.add_argument('--config',required=True);args=ap.parse_args()
    cfg=json.loads(Path(args.config).read_text());root=Path(cfg['run']);video=Path(cfg['video'])
    final=root/'final';final.mkdir();tables=final/'tables';tables.mkdir();figures=final/'figures';figures.mkdir()
    logs=root/'logs';logs.mkdir();raw=root/'raw_frames';raw.mkdir();dataset=root/'dataset'
    bin_dir=Path(cfg['python']).parent
    train=bin_dir/'ns-train';process=bin_dir/'ns-process-data';nsrender=bin_dir/'ns-render'
    import torch, importlib.metadata
    assert importlib.metadata.version('nerfstudio')=='1.1.5','Use o SETUP fornecido (Nerfstudio 1.1.5).'
    assert torch.cuda.is_available(),'GPU indisponível'
    if shutil.disk_usage(root).free<30*1024**3:raise RuntimeError('Necessários 30 GiB livres para intermediários locais.')
    for exe in (train,process,nsrender):assert exe.is_file(),exe
    for exe in ('ffmpeg','ffprobe','colmap'):assert shutil.which(exe), 'Execute SETUP: falta '+exe
    probe=json.loads(subprocess.check_output(['ffprobe','-v','error','-show_streams','-show_format','-of','json',str(video)],text=True))
    vs=next(s for s in probe['streams'] if s['codec_type']=='video')
    if vs.get('color_transfer') in ('smpte2084','arib-std-b67'):
        raise RuntimeError('Vídeo HDR detectado: converter para SDR com tone mapping antes desta célula.')
    duration=float(probe['format']['duration'])
    if not 15<=duration<=90:raise RuntimeError('Perfil desta célula espera vídeo de 15–90s; revise a amostragem.')
    stats={'video_name':video.name,'video_sha256':sha(video),'duration_seconds':duration,
           'video_stream':{k:vs.get(k) for k in ('width','height','avg_frame_rate','pix_fmt','color_transfer')},
           'seed':42,'gpu':torch.cuda.get_device_name(0),'fps_sampled':cfg['fps'],
           'pose_refinement_requested':bool(cfg['pose_refinement']),
           'pose_optimization':'off','sfm_protocol':'train-only COLMAP; holdouts PnP localization',
           'test_pixels_used_in_radiance_optimization':False,'sfm_uses_holdout_images':False,
           'point_cloud_is_lidar':False,'storage_policy':'final_zip_only',
           'versions':{x:importlib.metadata.version(x) for x in ('nerfstudio','torch','gsplat','ultralytics')}}
    # A sessão nova usa mapas diferentes; proibir reaproveitamento de SfM conjunto.
    if cfg.get('resume_preprocessing'):
        raise RuntimeError('Não reutilizar SfM do protocolo anterior: H_RESUME_FROM=None.')
    run(['ffmpeg', '-hide_banner', '-loglevel', 'error', '-i', video,
         '-map', '0:v:0', '-an',
         '-vf', f"fps={cfg['fps']},scale=w=800:h=800:force_original_aspect_ratio=decrease:force_divisible_by=2",
         raw / 'frame_%06d.png'], logs / 'extract.log', 600)
    originals = sorted(raw.glob('*.png'))
    n = len(originals)
    if n < 60:
        raise RuntimeError(f'Apenas {n} frames extraídos; revise o FPS/amostragem.')
    split = split_indices(n, cfg['split_mode'])
    save(root / 'planned_split.json', split)
    stats['split_mode'] = cfg['split_mode']
    stats['protocol_version'] = cfg['protocol_version']
    stats['planned_split'] = {k: len(v) for k, v in split.items()}
    stats['colmap_query_match_nearest_train_count'] = int(cfg['query_train_neighbors'])
    print('Divisão planejada:', stats['planned_split'], flush=True)
    dataset = colmap_inductive(root, originals, split, logs, stats)
    tf = json.loads((dataset / 'transforms.json').read_text())
    frames = tf['frames']
    expected_names = {originals[i].name: i for i in split['train'] + split['val'] + split['test']}
    mapped = {}
    for frame in frames:
        name = Path(frame['file_path']).name
        if name not in expected_names:
            raise RuntimeError('Nome inesperado no transforms: ' + name)
        if not (dataset / frame['file_path']).is_file():
            raise RuntimeError('Imagem retificada ausente: ' + name)
        mapped[frame['file_path']] = expected_names[name]
    if len(mapped) != len(expected_names):
        raise RuntimeError('Faltam imagens após a retificação.')
    for sp in ('train', 'val', 'test'):
        names = sorted((p for p in mapped if mapped[p] in split[sp]), key=lambda p: mapped[p])
        if len(names) != len(split[sp]):
            raise RuntimeError(f'Split {sp} incompleto: {len(names)}/{len(split[sp])}')
        tf[sp + '_filenames'] = names
    # Sanidade de referência e checagem do gap temporal planejado.
    stats['splits'] = {sp: len(tf[sp + '_filenames']) for sp in ('train', 'val', 'test')}
    stats['minimum_train_holdout_gap_seconds'] = min(
        abs(mapped[a] - mapped[b]) / cfg['fps']
        for a in tf['train_filenames']
        for b in tf['val_filenames'] + tf['test_filenames']
    )
    stats['evaluation_rgb'] = 'COLMAP-undistorted SDR RGB PNG; one GT shared by both models'
    normalize_fixed_from_train(dataset, tf, tf['train_filenames'], stats)
    init_train_points(dataset, tf, tf['train_filenames'], mapped, raw, stats)
    save(dataset / 'transforms.json', tf)
    stats['transforms_sha256'] = sha(dataset / 'transforms.json')
    save(final / 'transforms.json', tf)
    image_rows=[]
    import cv2
    for s in ('train','val','test'):
        d=figures/'inputs'/s;d.mkdir(parents=True)
        for p in tf[s+'_filenames']:
            dest=d/Path(p).name;shutil.copy2(dataset/p,dest)
            arr=np.asarray(Image.open(dest).convert('RGB'))
            image_rows.append({'split':s,'frame_index':mapped[p],'nominal_time_seconds':mapped[p]/cfg['fps'],
                               'relative_path':dest.relative_to(final).as_posix(),'sha256':sha(dest),
                               'laplacian_variance':float(cv2.Laplacian(cv2.cvtColor(arr,cv2.COLOR_RGB2GRAY),cv2.CV_64F).var())})
    table(tables/'input_frames.csv',image_rows);print('Dataset pronto:',stats['splits'],flush=True)
    # Poses fixas para o primeiro baseline iPhone. Não importar configs/dataset scene-0655.
    # Orçamento experimental igualado: 15.000 iteracoes por metodo.
    # Preserva 5 checkpoints candidatos para 3DGS e 10 para Nerfacto-Big (150 vistas VAL).
    methods={
        'splatfacto':{
            'preset':'splatfacto',
            'iterations':15000,
            'eval_steps':[4000,5000,6000,8000,14999],
        },
        'nerfacto':{
            'preset':'nerfacto-big',
            'iterations':15000,
            'eval_steps':[1000,2000,3000,4000,5000,6000,8000,10000,12000,14999],
        },
    }
    for _method, _profile in methods.items():
        if _profile['iterations']!=15000 or _profile['eval_steps'][-1]!=14999:
            raise RuntimeError(f'Agenda de 15.000 iteracoes invalida para {_method}')
    valrows=[];models={}
    for method,profile in methods.items():
        output=root/'models'/method
        command=[train,profile['preset'],'--data',dataset,'--output-dir',output,'--experiment-name','iphone_'+cfg['split_mode']+'_'+method,
                 '--machine.seed','42','--max-num-iterations',profile['iterations'],'--steps-per-save','1000',
                 '--save-only-latest-checkpoint','False','--steps-per-eval-image','1000',
                 '--steps-per-eval-all-images','5000','--vis','tensorboard',
                 '--pipeline.model.camera-optimizer.mode','off']
        if method=='splatfacto':
            command+=['--pipeline.datamanager.cache-images','cpu','--pipeline.datamanager.cache-images-type','uint8',
                      '--pipeline.model.num-downscales','0','--pipeline.model.sh-degree','2',
                      '--pipeline.model.sh-degree-interval','500','--pipeline.model.densify-grad-thresh','0.001',
                      '--pipeline.model.stop-split-at','6000','--pipeline.model.stop-screen-size-at','750',
                      '--pipeline.model.cull-alpha-thresh','0.01','--pipeline.model.use-absgrad','False',
                      '--pipeline.model.use-bilateral-grid','False','--pipeline.model.rasterize-mode','antialiased',
                      '--pipeline.model.ssim-lambda','0.2']
        else:
            command+=['--pipeline.datamanager.train-num-rays-per-batch','8192',
                      '--pipeline.datamanager.eval-num-rays-per-batch','4096','--pipeline.model.implementation','tcnn']
        command+=['nerfstudio-data', '--downscale-factor', '1',
                     '--orientation-method', 'none', '--center-method', 'none',
                     '--auto-scale-poses', 'False', '--scene-scale', '1.5']
        seconds=run(command,logs/(method+'_train.log'),7200)
        configs=list(output.rglob('config.yml'))
        if len(configs)!=1:raise RuntimeError('Esperado um config novo por método.')
        config=configs[0];checkpoints={int(p.stem.split('-')[-1]):p for p in config.parent.glob('nerfstudio_models/step-*.ckpt')}
        if not set(profile['eval_steps'])<=set(checkpoints):raise RuntimeError('Treino incompleto; faltam checkpoints.')
        models[method]={'profile':profile,'training_seconds':seconds,'config':str(config),'command':list(map(str,command))}
        models[method]['training_peak_gpu_memory_mib_observed']=RUN_SAMPLES.get(str(logs/(method+'_train.log')),{}).get('gpu_peak_mib_sampled')
        models[method]['training_peak_gpu_memory_sampling_interval_seconds']=2
        for step in profile['eval_steps']:
            r,_,_=render(config,step,'val',dataset,root/'validation'/method/str(step),nsrender)
            gt=image_map(r,'val','gt-rgb',tf['val_filenames']);pred=image_map(r,'val','rgb',tf['val_filenames'])
            for stem in sorted(gt):
                a=np.asarray(Image.open(gt[stem]).convert('RGB'));b=np.asarray(Image.open(pred[stem]).convert('RGB'))
                assert a.shape==b.shape
                verify_input_gt(gt[stem],dataset/next(x for x in tf['val_filenames'] if Path(x).stem==stem))
                valrows.append({'method':method,'step':step,'image':stem,'psnr_db':psnr(a,b)})
            print(method,step,'VAL',np.mean([x['psnr_db'] for x in valrows if x['method']==method and x['step']==step]),flush=True)
        models[method]['validation_candidate_render_wall_seconds']=sum(
            RUN_SAMPLES.get(str(root/'validation'/method/str(s)/'render.log'),{}).get('wall_seconds',0)
            for s in profile['eval_steps'])
        means={s:float(np.mean([r['psnr_db'] for r in valrows if r['method']==method and r['step']==s])) for s in profile['eval_steps']}
        best=max(means,key=means.get);models[method].update(selected_step=best,val_psnr_db=means[best],validation_means=means)
    table(tables/'validation_per_view.csv',valrows)
    save(final/'selection_before_test.json',models)
    # Ambos os checkpoints já foram selecionados. Só agora avaliar TEST.
    from skimage.metrics import structural_similarity
    from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity
    lpips=LearnedPerceptualImagePatchSimilarity(net_type='alex',normalize=True).cpu().eval()
    metrics=[];renders={};real={};summary=[]
    for method in methods:
        selected=models[method];step=selected['selected_step']
        r,seconds,ckpt=render(Path(selected['config']),step,'test',dataset,root/'test'/method,nsrender)
        gt=image_map(r,'test','gt-rgb',tf['test_filenames']);pred=image_map(r,'test','rgb',tf['test_filenames'])
        renders[method]={};out=figures/'test'/method;out.mkdir(parents=True)
        for stem in sorted(gt):
            a=np.asarray(Image.open(gt[stem]).convert('RGB'));b=np.asarray(Image.open(pred[stem]).convert('RGB'))
            assert a.shape==b.shape
            input_path=dataset/next(x for x in tf['test_filenames'] if Path(x).stem==stem)
            verify_input_gt(gt[stem],input_path)
            if stem in real and pixel_hash(real[stem])!=pixel_hash(gt[stem]):raise RuntimeError('GT diferente entre métodos.')
            real_dir=figures/'test'/'real';real_dir.mkdir(parents=True,exist_ok=True)
            real[stem]=real_dir/(stem+'.png');shutil.copy2(gt[stem],real[stem])
            dest=out/(stem+'.png');shutil.copy2(pred[stem],dest);renders[method][stem]=dest
            to_tensor=lambda x:torch.from_numpy(x.copy()).permute(2,0,1).unsqueeze(0).float()/255.
            with torch.no_grad():distance=float(lpips(to_tensor(a),to_tensor(b)));lpips.reset()
            metrics.append({'method':method,'image':stem,'psnr_db':psnr(a,b),
                            'ssim':float(structural_similarity(a,b,channel_axis=-1,data_range=255,gaussian_weights=True,sigma=1.5,use_sample_covariance=False)),
                            'lpips_alex':distance})
        selected.update(checkpoint_sha256=sha(ckpt),checkpoint_size_mib=ckpt.stat().st_size/1024**2,render_process_seconds=seconds)
        selected['test_render_gpu_peak_mib_observed']=RUN_SAMPLES.get(str(root/'test'/method/'render.log'),{}).get('gpu_peak_mib_sampled')
        subset=[x for x in metrics if x['method']==method]
        summary.append({'method':method,'selected_step':step,'val_psnr_db':selected['val_psnr_db'],
                        'test_psnr_db':float(np.mean([x['psnr_db'] for x in subset])),
                        'test_ssim':float(np.mean([x['ssim'] for x in subset])),
                        'test_lpips_alex':float(np.mean([x['lpips_alex'] for x in subset])),
                        'training_minutes':selected['training_seconds']/60})
    del lpips
    table(tables/'test_per_view.csv',metrics);table(final/'summary.csv',summary)
    # YOLO idêntico ao protocolo anterior: classes COCO de trânsito, 1280, conf .25, NMS .70.
    from ultralytics import YOLO
    yolo=YOLO(cfg['yolo_model']);weights=Path(yolo.ckpt_path)
    stats['yolo']={'weights_sha256':sha(weights),'model':cfg['yolo_model'],'conf':0.25,'nms_iou':0.70,'imgsz':1280,
                   'classes':[0,1,2,3,5,7],'matching':'maximum cardinality per class','match_iou':[0.3,0.5,0.7],'reference':'YOLO real; not human GT'}
    det={};detrows=[];matchrows=[];byimage=[]
    triplets=figures/'rgb_triplets';triplets.mkdir();ytriplets=figures/'yolo_triplets';ytriplets.mkdir()
    for stem in sorted(real):
        rgb_panels=[];yolo_panels=[]
        for domain,path in [('real',real[stem]),('nerfacto',renders['nerfacto'][stem]),('splatfacto',renders['splatfacto'][stem])]:
            res=yolo.predict(str(path),imgsz=1280,conf=.25,iou=.70,classes=[0,1,2,3,5,7],max_det=300,device=0,verbose=False)[0]
            rows=[]
            for j,(box,score,cls) in enumerate(zip(res.boxes.xyxy.cpu().numpy(),res.boxes.conf.cpu().numpy(),res.boxes.cls.cpu().numpy())):
                d={'class_id':int(cls),'confidence':float(score),'xyxy':box.tolist()};rows.append(d)
                detrows.append({'image':stem,'domain':domain,'id':j,'class_id':int(cls),'confidence':float(score),
                                'x1':float(box[0]),'y1':float(box[1]),'x2':float(box[2]),'y2':float(box[3])})
            det[(stem,domain)]=rows
            rgb_panels.append(Image.open(path).convert('RGB'));yolo_panels.append(Image.fromarray(res.plot()[...,::-1]))
        for panels,target in ((rgb_panels,triplets),(yolo_panels,ytriplets)):
            w,h=panels[0].size;canvas=Image.new('RGB',(3*w,h+30),'white');draw=ImageDraw.Draw(canvas)
            for j,(im,title) in enumerate(zip(panels,['REAL','NERFACTO','SPLATFACTO'])):
                canvas.paste(im,(j*w,30));draw.text((j*w+8,7),title,fill='black')
            canvas.save(target/(stem+'.jpg'),quality=94)
        for method in methods:
            rr=det[(stem,'real')];pp=det[(stem,method)]
            for threshold in (.3,.5,.7):
                matched=matches(rr,pp,threshold)
                byimage.append({'image':stem,'method':method,'match_iou':threshold,'real_count':len(rr),'render_count':len(pp),'matched_count':len(matched)})
                for i,j,overlap in matched:matchrows.append({'image':stem,'method':method,'match_iou':threshold,'real_id':i,'render_id':j,'iou':overlap,'class_id':rr[i]['class_id']})
    semantic=[]
    for method in methods:
        for threshold in (.3,.5,.7):
            subset=[r for r in byimage if r['method']==method and r['match_iou']==threshold]
            nr=sum(x['real_count'] for x in subset);npred=sum(x['render_count'] for x in subset);nm=sum(x['matched_count'] for x in subset)
            semantic.append({'method':method,'match_iou':threshold,'real_count':nr,'render_count':npred,'matched_count':nm,
                             'recall_like':nm/nr if nr else None,'precision_like':nm/npred if npred else None,
                             'f1_like':2*nm/(nr+npred) if nr+npred else None})
    table(tables/'yolo_detections.csv',detrows);table(tables/'yolo_matches.csv',matchrows)
    table(tables/'yolo_by_image.csv',byimage);table(tables/'yolo_overall.csv',semantic)
    # Preservar validação selecionada como evidência visual.
    for method in methods:
        r=root/'validation'/method/str(models[method]['selected_step'])/'renders'
        out=figures/'validation'/method;out.mkdir(parents=True)
        for p in (r/'val'/'rgb').rglob('*.png'):shutil.copy2(p,out/p.name)
    stats['models']=models;stats['quality']=summary;stats['semantic']=semantic
    save(final/'manifest.json',stats)
    (final/'LEIA_ME.txt').write_text(
        'CÉLULA H V6.5 - 15000 iteracoes por metodo - protocolo ' + cfg['split_mode'] + '\n'
        'COLMAP: SfM apenas TRAIN; VAL/TEST por PnP e LM opcional apenas das poses, mapa congelado.\n'
        'Auditoria por vista: tables/pose_registration_quality.csv.\n'
        'CHECKPOINT: selecionado por PSNR médio de VAL antes da avaliação TEST.\n'
        'MÉTRICAS: PSNR/SSIM/LPIPS calculadas em TEST; detecções YOLO = pseudorreferência.\n'
        'OBS: estes resultados NÃO são diretamente comparáveis com nuScenes apenas pela igualdade do número de imagens.\n'
        'Não há checkpoints no ZIP final. Ver manifest.json para rastreabilidade.\n',
        encoding='utf-8'
    )
    def html_table(rows):
        if not rows:return '<p>Sem registros.</p>'
        cols=list(rows[0]);return '<table><tr>'+''.join('<th>'+html.escape(x)+'</th>' for x in cols)+'</tr>'+''.join('<tr>'+''.join('<td>'+html.escape(f'{r[c]:.5f}' if isinstance(r[c],float) else str(r[c]))+'</td>' for c in cols)+'</tr>' for r in rows)+'</table>'
    report="""<!doctype html><html lang="pt-BR"><meta charset="utf-8"><title>iPhone: NeRF x 3DGS</title>
<style>body{font:16px/1.5 sans-serif;max-width:1200px;margin:30px auto;padding:15px}table{border-collapse:collapse;font-size:13px}td,th{border:1px solid #ccc;padding:6px}img{max-width:100%}</style>
<h1>iPhone H V6.5 — NeRF e 3DGS: 15000 iteracoes por metodo, 95/10/10</h1><p>Experimento separado de scene-0655. Poses estimadas por COLMAP.
O COLMAP reconstrói a geometria usando apenas imagens TRAIN. VAL e TEST são localizados posteriormente
por registro PnP e, quando seguro, otimização 6-DOF somente da câmera no mapa fixo, sem bundle adjustment conjunto.
A nuvem inicial usa pontos com >=3 observações TRAIN e cores
amostradas somente em TRAIN. Não é LiDAR. Guardas temporais reduzem proximidade imediata, mas não eliminam correlação.</p>
<p>Seleção de checkpoint por PSNR médio em VAL antes da avaliação TEST. PSNR/SSIM/LPIPS em RGB 8-bit,
na resolução vertical preservada. F1-like YOLO mede concordância com detecções em imagens reais, não mAP.</p><h2>Qualidade</h2>"""+html_table(summary)+'<h2>YOLO</h2>'+html_table([x for x in semantic if x['match_iou']==.5])
    report+='<h2>Auditoria geometrica VAL / TEST</h2>'+html_table(stats['query_pose_diagnostics'])
    for folder,label in [(triplets,'Comparações RGB'),(ytriplets,'Comparações YOLO')]:
        report+='<h2>'+label+'</h2>'
        for p in sorted(folder.glob('*.jpg')):report+='<p>'+html.escape(p.stem)+'</p><img loading="lazy" src="'+p.relative_to(final).as_posix()+'">'
    report+="""<h2>Limitações</h2><p>Um vídeo, uma execução, mesma trajetória. Mesmo com SfM TRAIN-only e PnP nos holdouts, as
poses de VAL/TEST dependem de correspondências visuais com o mapa de treino, e suas poses são refinadas
somente por reprojeção 2D-3D em geometria congelada. Reflexos e objetos móveis
podem dificultar reconstrução estática. Não comparar numericamente com nuScenes como se fosse o mesmo benchmark.
Os tempos de treino diferem entre presets; não é um experimento de orçamento computacional igual.</p>
<p>Referência de processamento: <a href="https://docs.nerf.studio/quickstart/custom_dataset.html">Nerfstudio: custom data</a>.</p></html>"""
    (final/'ANALISE_FINAL.html').write_text(report,encoding='utf-8')
    allowed={'.json','.csv','.html','.png','.jpg','.jpeg','.txt'}
    archive=root/'analise_iphone.zip'
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(final.rglob('*')):
            if p.is_symlink():raise RuntimeError('Link não permitido no pacote final.')
            if p.is_file():
                if p.suffix.lower() not in allowed:raise RuntimeError('Intermediário indevido na análise final: '+str(p))
                z.write(p,p.relative_to(final))
    with zipfile.ZipFile(archive) as z:assert z.testzip() is None
    save(root/'completed.json',{'archive':str(archive),'sha256':sha(archive),'summary':summary})
    print('CONCLUÍDO: somente análise pronta para exportar:',archive,flush=True)


if __name__=='__main__':main()

'''
H_WORKER = H_RUN / 'iphone_worker.py'
H_WORKER.write_text(H_WORKER_SOURCE, encoding='utf-8')
compile(H_WORKER_SOURCE, str(H_WORKER), 'exec')

H_ENV = dict(globals().get('TCC_SUBPROCESS_ENV', os.environ))
for key in ('PYTHONHOME', 'PYTHONPATH', 'PYTHONUSERBASE', 'PIP_TARGET'):
    H_ENV.pop(key, None)
H_ENV.update({
    'PATH': str(H_PYTHON.parent) + ':/usr/local/cuda-12.8/bin:' + os.environ.get('PATH', ''),
    'PYTHONNOUSERSITE': '1', 'PYTHONUNBUFFERED': '1', 'QT_QPA_PLATFORM': 'offscreen',
    'CUDA_HOME': '/usr/local/cuda-12.8', 'CUDA_PATH': '/usr/local/cuda-12.8',
    'CUDACXX': '/usr/local/cuda-12.8/bin/nvcc', 'VIRTUAL_ENV': str(H_PYTHON.parent.parent)
})
if Path('/usr/local/cuda-12.8/lib64').is_dir():
    H_ENV['LD_LIBRARY_PATH'] = '/usr/local/cuda-12.8/lib64:' + H_ENV.get('LD_LIBRARY_PATH', '')

print('Vídeo de entrada configurado para o cenário H.')
print('Protocolo H V6.5 (15000 passos por metodo):', H_SPLIT_MODE, '| refino pose-only:', H_REFINE_QUERY_POSES, '| vizinhos:', H_QUERY_MATCH_NEIGHBORS)
print('Execução local:', H_RUN)
print('Geometria COLMAP: somente TRAIN. VAL/TEST: PnP e refinamento pose-only (sem BA).')
H_PROC = subprocess.Popen(
    [str(H_PYTHON), str(H_WORKER), '--config', str(H_RUN / 'run_config.json')],
    cwd=H_RUN, env=H_ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
try:
    with (H_RUN / 'worker.log').open('w') as log:
        for line in H_PROC.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
    H_CODE = H_PROC.wait()
finally:
    if H_PROC.poll() is None:
        H_PROC.terminate()
        H_PROC.wait(timeout=15)
if H_CODE:
    raise RuntimeError('Pipeline interrompido. Consulte o log local: ' + str(H_RUN / 'worker.log'))
H_DONE = json.loads((H_RUN / 'completed.json').read_text())
H_ZIP = Path(H_DONE['archive'])

def h_sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

if h_sha(H_ZIP) != H_DONE['sha256']:
    raise RuntimeError('O ZIP produzido não passou na verificação SHA256.')
if H_SEND_FINAL_TO_DRIVE and os.environ.get('TCC_EXPORT_DIR', '').strip():
    H_DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
    H_DEST = H_DRIVE_RESULTS / ('analise_iphone_' + H_SPLIT_MODE + '_' + H_STAMP + '.zip')
    H_TEMP = H_DEST.with_suffix('.zip.partial')
    if H_DEST.exists() or H_TEMP.exists():
        raise FileExistsError(H_DEST)
    try:
        shutil.copy2(H_ZIP, H_TEMP)
        if h_sha(H_TEMP) != H_DONE['sha256']:
            raise RuntimeError('Falha SHA256 ao copiar o ZIP para o Drive.')
        H_TEMP.replace(H_DEST)
    except Exception:
        H_TEMP.unlink(missing_ok=True)
        raise
    print('ZIP FINAL enviado ao Drive:', H_DEST)
print('\nMÉTRICAS DOS MODELOS:')
for row in H_DONE['summary']:
    print(row)
print('Abra ANALISE_FINAL.html dentro do ZIP para inspecionar as imagens.')


# ----------------- ESQUEMA ÚNICO DE RESULTADOS (V1) -----------------
# Biblioteca embutida: esta célula funciona independentemente de H e I.
from pathlib import Path
import importlib.util
_UNIFIED_MODULE_SOURCE = r'''
"""TCC — esquema canônico de resultados v1.
Biblioteca autocontida para uso pelo notebook no Colab. NÃO treina, não infere e
não usa resultados de TEST para alterar modelos. Só organiza e audita artefatos.
"""
from pathlib import Path
from datetime import datetime, timezone
import csv, hashlib, html, json, math, os, shutil, zipfile
from PIL import Image, ImageDraw

SCHEMA = 'TCC_UNIFIED_RESULTS_V1_95_10_10'
DOMAINS = ('real', 'nerfacto', 'splatfacto')
REQUIRED = (
    'ANALISE_FINAL.html', 'LEIA_ME.txt', 'final_experiment_manifest.json',
    'tables/final_experiment_summary.csv', 'tables/quality_by_image.csv',
    'tables/validation_per_view.csv', 'tables/yolo_overall.csv',
    'tables/yolo_by_image.csv', 'tables/yolo_detections.csv',
    'tables/yolo_matches.csv', 'tables/pose_registration_quality.csv',
    'tables/image_index.csv', 'tables/compute_cost.csv',
    'provenance/source_manifest.json', 'provenance/source_run_metadata.json',
)
SUMMARY_COLUMNS = ['dataset_id','method','selected_step','val_psnr_db','test_psnr_db',
 'test_ssim','test_lpips_alex','training_minutes','checkpoint_size_mib',
 'render_total_seconds_including_overhead','real_yolo_count','render_yolo_count',
 'matched_yolo_count','recall_like','precision_like','f1_like',
 'mean_matched_iou','detection_retention_ratio']
QUALITY_COLUMNS = ['dataset_id','image_index','source_image','method','psnr_db','ssim','lpips_alex']
VALIDATION_COLUMNS = ['dataset_id','method','step','image','psnr_db']
YOLO_COLUMNS = ['dataset_id','method','match_iou','real_count','render_count','matched_count',
 'recall_like','precision_like','f1_like','mean_matched_iou','detection_retention_ratio']
YOLO_IMAGE_COLUMNS = ['dataset_id','image_index','method','match_iou','real_count','render_count','matched_count',
 'recall_like','precision_like','f1_like']
DETECTION_COLUMNS = ['dataset_id','image_index','domain','det_index','class_id','confidence','x1','y1','x2','y2']
MATCH_COLUMNS = ['dataset_id','image_index','method','match_iou','real_det_index','render_det_index','class_id','iou']
POSE_COLUMNS = ['dataset_id','image_index','split','pose_source','pose_method','registered','inliers','reprojection_error_px','notes']
INDEX_COLUMNS = ['dataset_id','image_index','source_image','timestamp_or_index', 'real_sha256',
 'nerfacto_sha256','splatfacto_sha256','real_image','nerfacto_image','splatfacto_image',
 'rgb_triplet','yolo_triplet']
COST_COLUMNS = ['dataset_id','method','training_minutes','checkpoint_size_mib','render_total_seconds_including_overhead']

def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for part in iter(lambda:stream.read(1024*1024),b''):h.update(part)
    return h.hexdigest()

def read(path):return json.loads(Path(path).read_text(encoding='utf-8'))

def write(path,obj):
    p=Path(path);p.parent.mkdir(parents=True,exist_ok=True)
    p.write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str)+'\n',encoding='utf-8')

def read_csv(p):
    if not Path(p).is_file():return []
    with open(p,encoding='utf-8-sig',newline='') as stream:return list(csv.DictReader(stream))

def save_csv(p,fields,rows):
    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)
    with p.open('w',encoding='utf-8',newline='') as f:
        wr=csv.DictWriter(f,fieldnames=fields,extrasaction='ignore');wr.writeheader();wr.writerows(rows)

def num(x):
    try:
        v=float(x)
        return v if math.isfinite(v) else None
    except (TypeError,ValueError):return None

def eqfloat(a,b,tol=1e-4):
    if a is None or b is None:return a is None and b is None
    return abs(float(a)-float(b))<=tol

def img(src,target):
    # Preserva os pixels da imagem retificada pelo COLMAP, sem redimensionar.
    # O limite 800 no ffmpeg/COLMAP nao garante resolucao exata 800x450.
    src=Path(src);target=Path(target)
    if not src.is_file():raise FileNotFoundError('Imagem TEST ausente: '+str(src))
    with Image.open(src) as original:
        im=original.convert('RGB')
        if im.width < 2 or im.height < 2:
            raise RuntimeError('Imagem TEST com dimensoes invalidas: '+str(src))
        target.parent.mkdir(parents=True,exist_ok=True)
        im.save(target,format='PNG')

def triplet(files,out,i,title):
    # Três dominios precisam ter as MESMAS dimensoes para uma vista;
    # nao fabricar alinhamento redimensionando um dominio silenciosamente.
    with Image.open(files[0]) as first:
        w,h=first.size
    canvas=Image.new('RGB',(3*w,h+38),'white');draw=ImageDraw.Draw(canvas)
    for j,(domain,p) in enumerate(zip(DOMAINS,files)):
        with Image.open(p) as src:
            if src.size!=(w,h):
                raise RuntimeError(f'Triade TEST {i+1}: {domain} com {src.size}; esperado {(w,h)}')
            canvas.paste(src.convert('RGB'),(j*w,38))
        draw.text((j*w+14,11),f'{title} {i+1:02d} / {domain.upper()}',fill='#17283a')
    out.parent.mkdir(parents=True,exist_ok=True);canvas.save(out,format='JPEG',quality=93)

def normalize(source_kind,source_root,dataset_id,output_root,source_extra=None):
    """Normaliza final G (nuScenes) ou final H/I (capturas próprias).

    Retorna (diretorio_unificado, arquivo_zip); copia somente análise, nunca
    checkpoints, PLY, frames de TRAIN ou vídeo. Tabelas não aplicáveis são vazias,
    com cabeçalhos canônicos e declaração explícita de indisponibilidade.
    """
    if source_kind not in ('nuscenes','video') or dataset_id not in ('nuscenes_scene0655','estacionamento','uber'):
        raise ValueError('Origem ou dataset_id inesperado.')
    src=Path(source_root).resolve();out=Path(output_root)
    if not src.is_dir():raise FileNotFoundError(src)
    if out.exists():raise FileExistsError('Não sobrescrever análise anterior: '+str(out))
    out.mkdir(parents=True)
    for name in ['tables','figures/charts','figures/rgb_triplets','figures/yolo_triplets',
                 'images/real','images/nerfacto','images/splatfacto','provenance']:
        (out/name).mkdir(parents=True,exist_ok=True)
    extra=dict(source_extra or {})
    values={key:[] for key in ('summary','quality','validation','yolo','yolo_image',
                              'detections','matches','pose','index','cost')}
    raw_images={}
    rgb_fig={};yolo_fig={}
    manifest_source=src/('final_experiment_manifest.json' if source_kind=='nuscenes' else 'manifest.json')
    if not manifest_source.is_file():raise FileNotFoundError(manifest_source)
    native=read(manifest_source)
    shutil.copy2(manifest_source,out/'provenance/source_manifest.json')
    metadata={'original_source_kind':source_kind,'original_source_directory':str(src),
              'original_manifest_sha256':sha(manifest_source),
              'dataset_id':dataset_id,'notes':extra}
    if source_kind=='nuscenes':
        sm=read_csv(src/'tables/final_experiment_summary.csv')
        d=read(src/'provenance/D_final_report.json')
        for row in sm:
            method=row['method_key']
            values['summary'].append({'dataset_id':dataset_id, 'method':method,**{k:row.get(k) for k in SUMMARY_COLUMNS if k not in ('dataset_id','method')}})
        for imrow in d['per_view']:
            i=int(imrow['test_view'])-1
            for method in ('nerfacto','splatfacto'):
                values['quality'].append({'dataset_id':dataset_id,'image_index':i,'source_image':imrow['image'],
                    'method':method,'psnr_db':imrow[method+'_psnr_db'],'ssim':imrow[method+'_ssim'],
                    'lpips_alex':imrow[method+'_lpips_alex']})
        # Evidência VAL: caminhos provenientes desta mesma sessão B/C; sem suposições.
        csvs=extra.get('validation_csvs',{})
        for method in ('splatfacto','nerfacto'):
            valpath=Path(csvs.get(method,'/invalid_validation_csv'))
            rows=read_csv(valpath)
            if not rows:
                raise FileNotFoundError('CSV VAL da B/C não encontrado: '+str(valpath))
            for row in rows:
                values['validation'].append({'dataset_id':dataset_id,'method':method,
                    'step':row.get('step'),'image':row.get('image'),'psnr_db':row.get('psnr_db')})
        for row in read_csv(src/'tables/yolo_summary_overall.csv'):
            vals={'dataset_id':dataset_id,'method':row.get('render_domain'),'match_iou':.50}
            vals.update({k:row.get(k) for k in YOLO_COLUMNS if k not in vals});values['yolo'].append(vals)
        for row in read_csv(src/'tables/yolo_summary_by_image.csv'):
            if not eqfloat(num(row.get('match_iou')),0.50):
                continue
            vals={'dataset_id':dataset_id,'method':row.get('render_domain'),'image_index':row.get('image_index'),'match_iou':.50}
            vals.update({k:row.get(k) for k in YOLO_IMAGE_COLUMNS if k not in vals});values['yolo_image'].append(vals)
        for row in read_csv(src/'tables/yolo_detections_all.csv'):
            values['detections'].append({'dataset_id':dataset_id,**{k:row.get(k) for k in DETECTION_COLUMNS if k!='dataset_id'}})
        for row in read_csv(src/'tables/yolo_match_details.csv'):
            values['matches'].append({'dataset_id':dataset_id,'image_index':row.get('image_index'),
                 'method':row.get('render_domain'),'match_iou':.5,'real_det_index':row.get('real_det_index'),
                 'render_det_index':row.get('render_det_index'),'class_id':row.get('class_id'),'iou':row.get('iou')})
        tf=read(src/'provenance/transforms.json')
        fm={Path(p).stem:p for p in tf['test_filenames']}
        for i in range(10):
            for domain in DOMAINS:raw_images[i,domain]=src/f'images/{domain}/eval_img_{i:04d}.png'
            rgb_fig[i]=src/f'figures/rgb_triplets/RGB_TEST_{i:02d}.jpg'
            yolo_fig[i]=src/f'figures/yolo_triplets/YOLO_TEST_{i:02d}.jpg'
            values['pose'].append({'dataset_id':dataset_id,'image_index':i,'split':'test',
               'pose_source':'official_nuscenes_ego_pose_calibrated_sensor',
               'pose_method':'provided_extrinsics','registered':'not_applicable',
               'notes':'Sem COLMAP; sem reprojeção PnP estimada'})
        metadata['geometry']='nuScenes official sensor poses; TRAIN-only LiDAR for Splatfacto'
    else:
        sm=read_csv(src/'summary.csv')
        for row in sm:
            name=row['method']
            models=native['models'][name]
            values['summary'].append({
               'dataset_id':dataset_id,'method':name,'selected_step':row.get('selected_step'),
               'val_psnr_db':row.get('val_psnr_db'),'test_psnr_db':row.get('test_psnr_db'),
               'test_ssim':row.get('test_ssim'),'test_lpips_alex':row.get('test_lpips_alex'),
               'training_minutes':row.get('training_minutes'),
               'checkpoint_size_mib':models.get('checkpoint_size_mib'),
               'render_total_seconds_including_overhead':models.get('render_process_seconds')})
        for row in read_csv(src/'tables/test_per_view.csv'):
            values['quality'].append({'dataset_id':dataset_id,'image_index':-1,
                'source_image':row.get('image'),'method':row.get('method'),
                'psnr_db':row.get('psnr_db'),'ssim':row.get('ssim'),'lpips_alex':row.get('lpips_alex')})
        for row in read_csv(src/'tables/validation_per_view.csv'):
            values['validation'].append({'dataset_id':dataset_id,'method':row.get('method'),
                'step':row.get('step'),'image':row.get('image'),'psnr_db':row.get('psnr_db')})
        for row in read_csv(src/'tables/yolo_overall.csv'):
            if eqfloat(num(row.get('match_iou')),0.5):
                values['yolo'].append({'dataset_id':dataset_id,'method':row.get('method'),
                    **{k:row.get(k) for k in YOLO_COLUMNS if k not in ('dataset_id','method')}})
        for row in read_csv(src/'tables/yolo_by_image.csv'):
            if eqfloat(num(row.get('match_iou')),0.5):
                values['yolo_image'].append({'dataset_id':dataset_id,'method':row.get('method'),
                    'image_index':None,'source_image':row.get('image'),
                    **{k:row.get(k) for k in YOLO_IMAGE_COLUMNS if k not in ('dataset_id','method','image_index')}})
        for row in read_csv(src/'tables/yolo_detections.csv'):
            values['detections'].append({'dataset_id':dataset_id,'image_index':None,
                'source_image':row.get('image'),'domain':row.get('domain'), 'det_index':row.get('id'),
                **{k:row.get(k) for k in ('class_id','confidence','x1','y1','x2','y2')}})
        for row in read_csv(src/'tables/yolo_matches.csv'):
            if eqfloat(num(row.get('match_iou')),0.5):
                values['matches'].append({'dataset_id':dataset_id,'image_index':None,
                    'source_image':row.get('image'),'method':row.get('method'),'match_iou':row.get('match_iou'),
                    'real_det_index':row.get('real_id'),'render_det_index':row.get('render_id'),
                    'class_id':row.get('class_id'),'iou':row.get('iou')})
        posed=read_csv(src/'tables/pose_registration_quality.csv')
        metadata['pose_registration_original_records'] = posed  # Auditoria completa, sem perder colunas nativas.
        for row in posed:
            values['pose'].append({'dataset_id':dataset_id,**{k:row.get(k) for k in POSE_COLUMNS if k!='dataset_id'}})
        allstems=sorted(p.stem for p in (src/'figures/test/real').glob('*.png'))
        if len(allstems)!=10 or len(set(allstems))!=10:
            raise RuntimeError('H/I precisa conter 10 TEST em figures/test/real')
        index_for_stem={stem:i for i,stem in enumerate(allstems)}
        for i,stem in enumerate(allstems):
            for domain in DOMAINS:raw_images[i,domain]=src/'figures/test'/domain/(stem+'.png')
            rgb_fig[i]=src/'figures/rgb_triplets'/(stem+'.jpg')
            yolo_fig[i]=src/'figures/yolo_triplets'/(stem+'.jpg')
        for kind in ('quality','yolo_image','detections','matches'):
            for row in values[kind]:
                stem=row.get('source_image')
                if stem not in index_for_stem:raise RuntimeError(f'Imagem {stem} de {kind} fora do TEST')
                row['image_index']=index_for_stem[stem]
        # Reutilização das métricas reais; NÃO calcular métricas não medidas.
        sem={r['method']:r for r in values['yolo']}
        for row in values['summary']:
            y=sem.get(row['method'],{})
            for key in ('real_count','render_count','matched_count','recall_like','precision_like','f1_like',
                        'mean_matched_iou','detection_retention_ratio'):
                row[{'real_count':'real_yolo_count','render_count':'render_yolo_count',
                     'matched_count':'matched_yolo_count'}.get(key,key)]=y.get(key)
        metadata['geometry']='COLMAP TRAIN only; VAL/TEST PnP against frozen TRAIN model'
        metadata['source_video_hash']=native.get('video_sha256')
    if len(values['summary'])!=2 or set(r['method'] for r in values['summary'])!={'splatfacto','nerfacto'}:
        raise RuntimeError('Resumo deve conter exatamente os dois métodos.')
    if len(values['quality'])!=20 or len(values['yolo'])!=2:
        raise RuntimeError('Qualidade deve ter 20 linhas e resumo semântico duas linhas.')
    if len(values['validation'])!=150:
        raise RuntimeError(f'VAL incompleta: {len(values["validation"])} linhas, esperado 150 (5×10 + 10×10).')
    # Verificação independente das médias originais por vista em TEST.
    for summ in values['summary']:
        m=summ['method'];observed=[q for q in values['quality'] if q['method']==m]
        if len(observed)!=10 or len(set(int(r['image_index']) for r in observed))!=10:
            raise RuntimeError('10 imagens TEST não únicas para '+m)
        for sm_col,col in [('test_psnr_db','psnr_db'),('test_ssim','ssim'),('test_lpips_alex','lpips_alex')]:
            raw=[num(r[col]) for r in observed]
            if any(v is None for v in raw):raise RuntimeError('Métrica incompleta '+m+'/'+col)
            mean=sum(raw)/10
            if not eqfloat(mean,num(summ[sm_col]),tol=1e-4):
                raise RuntimeError(f'Média {m}/{col} não fecha: {mean}, manifesto {summ[sm_col]}')
    original_size=None
    for i in range(10):
        sizes={}
        for domain in DOMAINS:
            with Image.open(raw_images[i,domain]) as source_img:
                sizes[domain]=source_img.size
        if len(set(sizes.values()))!=1:
            raise RuntimeError(f'TEST {i+1}: resolucoes diferentes entre dominios: {sizes}')
        view_size=sizes['real']
        if original_size is None:
            original_size=view_size
        elif view_size!=original_size:
            raise RuntimeError(f'Resolucao diferente entre as vistas TEST: {view_size} != {original_size}')
        paths=[]
        for domain in DOMAINS:
            path=out/'images'/domain/f'eval_img_{i:04d}.png'
            img(raw_images[i,domain],path);paths.append(path)
        for domain,path in zip(DOMAINS,paths):
            if not path.is_file():raise RuntimeError('Imagem ausente '+str(path))
        index={'dataset_id':dataset_id,'image_index':i,
               'source_image':next(q['source_image'] for q in values['quality'] if int(q['image_index'])==i),
               'timestamp_or_index':i}
        for domain,path in zip(DOMAINS,paths):
            index[domain+'_image']=path.relative_to(out).as_posix()
            index[domain+'_sha256']=sha(path)
        for kind,oldmap in [('rgb',rgb_fig),('yolo',yolo_fig)]:
            target=out/f'figures/{kind}_triplets/eval_img_{i:04d}.jpg'
            old=oldmap[i]
            if old.is_file():
                shutil.copy2(old,target)
            elif kind=='rgb':
                triplet(paths,target,i,'TEST')
            else:
                raise FileNotFoundError('Figura YOLO deve existir; não refazer inferência: '+str(old))
            index[kind+'_triplet']=target.relative_to(out).as_posix()
        values['index'].append(index)
    # Integridade: vistas originais versus métricas e inferências.
    if len(values['yolo_image'])!=20 or len(values['detections'])<0:
        raise RuntimeError('Tabela YOLO por vista incompleta (esperado 20).')
    for summ in values['summary']:
        values['cost'].append({key:summ.get(key) for key in COST_COLUMNS})
    files={
        'final_experiment_summary.csv':(SUMMARY_COLUMNS,'summary'),
        'quality_by_image.csv':(QUALITY_COLUMNS,'quality'),
        'validation_per_view.csv':(VALIDATION_COLUMNS,'validation'),
        'yolo_overall.csv':(YOLO_COLUMNS,'yolo'),
        'yolo_by_image.csv':(YOLO_IMAGE_COLUMNS,'yolo_image'),
        'yolo_detections.csv':(DETECTION_COLUMNS,'detections'),
        'yolo_matches.csv':(MATCH_COLUMNS,'matches'),
        'pose_registration_quality.csv':(POSE_COLUMNS,'pose'),
        'image_index.csv':(INDEX_COLUMNS,'index'),
        'compute_cost.csv':(COST_COLUMNS,'cost'),
    }
    for filename,(header,key) in files.items():save_csv(out/'tables'/filename,header,values[key])
    # Gráficos compartilhados, com os mesmos nomes e eixos; dados ausentes indicados como N/D.
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    plots=[('test_psnr_db','PSNR TEST (dB)','test_psnr.png'),
           ('test_ssim','SSIM TEST','test_ssim.png'),
           ('test_lpips_alex','LPIPS Alex TEST (menor e melhor)','test_lpips.png'),
           ('training_minutes','Treinamento (minutos)','training_minutes.png'),
           ('f1_like','YOLO F1-like IoU 0.50','yolo_f1_like.png'),
           ('recall_like','YOLO Recall-like IoU 0.50','yolo_recall_like.png')]
    for key,title,filename in plots:
        fig,ax=plt.subplots(figsize=(7,4.2))
        scores=[num(x.get(key)) for x in values['summary']]
        ax.bar(['Splatfacto','Nerfacto'],[x if x is not None else 0. for x in scores])
        ax.set_title(dataset_id+' — '+title);ax.grid(axis='y',alpha=.2)
        for i,value in enumerate(scores):ax.text(i,(value or 0),f'{value:.3f}' if value is not None else 'N/D',ha='center',va='bottom')
        fig.tight_layout();fig.savefig(out/'figures/charts'/filename,dpi=150);plt.close(fig)
    # HTML único, com links relativos para as imagens presentes no ZIP.
    rows=''.join('<tr>'+''.join(f'<td>{html.escape(str(r.get(k,"")))}</td>' for k in SUMMARY_COLUMNS)+'</tr>' for r in values['summary'])
    hdr=''.join('<th>'+html.escape(k)+'</th>' for k in SUMMARY_COLUMNS)
    figtags=''.join(f'<h3>TEST {i+1:02d}</h3><img src="figures/rgb_triplets/eval_img_{i:04d}.jpg" alt="Tríade RGB TEST {i+1}"><img src="figures/yolo_triplets/eval_img_{i:04d}.jpg" alt="Tríade YOLO TEST {i+1}">' for i in range(10))
    (out/'ANALISE_FINAL.html').write_text('<!doctype html><html lang="pt-BR"><head><meta charset="utf-8"><title>TCC - '+html.escape(dataset_id)+'</title><style>body{max-width:1200px;margin:32px auto;font:15px/1.6 system-ui}table{border-collapse:collapse;display:block;overflow:auto}td,th{border:1px solid #bbb;padding:5px}img{max-width:100%;margin:8px 0}aside{background:#eee;padding:12px}</style></head><body><h1>Comparação Nerfacto × Splatfacto — '+html.escape(dataset_id)+'</h1><aside>95 TRAIN / 10 VAL / 10 TEST. YOLO na imagem real é pseudorreferência, não ground truth humano. Comparação entre datasets tem confundidores geométricos e de conteúdo. Tempos ns-render incluem sobrecarga.</aside><h2>Resumo</h2><table><tr>'+hdr+'</tr>'+rows+'</table><h2>Gráficos</h2>'+''.join(f'<img src="figures/charts/{p[2]}">' for p in plots)+'<h2>Comparações RGB e YOLO</h2>'+figtags+'</body></html>',encoding='utf-8')
    (out/'LEIA_ME.txt').write_text('TCC — ESQUEMA DE RESULTADOS '+SCHEMA+'\n'
        'Dataset: '+dataset_id+'\nTrês domínios: real, nerfacto, splatfacto\n'
        'images/: 30 PNG TEST com nomes idênticos nas três subpastas\n'
        'figures/: 10 tríades RGB, 10 YOLO, seis gráficos com nomes fixos\n'
        'tables/: mesmas tabelas e colunas nos três datasets\n'
        'provenance/: manifesto da execução original, metadados, hashes\n'
        'Não contém checkpoints, vídeos, PLY nem imagens de treino.\n'
        'Colunas não aplicáveis permanecem vazias, não são medidas fabricadas.\n'
        'TEST é usado exclusivamente para avaliação confirmatória; não retunar.\n',encoding='utf-8')
    metadata.update(schema=SCHEMA,split_counts={'train':95,'val':10,'test':10},
                    yolo_real_is_pseudoreference=True,
                    no_new_training=True,no_new_inference=True,
                    aggregation='mean of 10 images; YOLO micro counts',
                    resolution=list(original_size),sources={'native_manifest':str(manifest_source)})
    write(out/'provenance/source_run_metadata.json',metadata)
    file_records=[]
    for p in sorted(out.rglob('*')):
        if p.is_file():
            if p.is_symlink() or p.suffix.lower() in ('.ckpt','.ply','.pt','.mov','.mp4'):
                raise RuntimeError('Arquivo proibido no pacote: '+str(p))
            file_records.append({'path':p.relative_to(out).as_posix(),'sha256':sha(p),'bytes':p.stat().st_size})
    manifest={'experiment':'unified_TCC_result','schema':SCHEMA,'dataset_id':dataset_id,
             'created_utc':datetime.now(timezone.utc).isoformat(),
             'status':'completed','splits':{'train':95,'val':10,'test':10},
             'test_views':10,'rgb_images':30,
             'metrics_not_available':{
                  'nuscenes_scene0655':['COLMAP reprojection errors','matched sensor trajectories with other datasets'],
                  'estacionamento':['nuScenes LiDAR initialisation','official nuScenes poses'],
                  'uber':['nuScenes LiDAR initialisation','official nuScenes poses']
              }[dataset_id],
             'models':values['summary'],
             'limitations':['single sequence per scenario','correlated held-out views',
                 'not a complete closed-loop autonomous driving simulator',
                 'cross-dataset scores are not controlled experiments',
                 'YOLO on REAL is pseudoreference, not human ground truth'],
             'file_integrity_excluding_self':file_records}
    write(out/'final_experiment_manifest.json',manifest)
    for name in REQUIRED:
        if not (out/name).is_file():raise RuntimeError('Falta arquivo canônico: '+name)
    for domain in DOMAINS:
        imgs=list((out/'images'/domain).glob('eval_img_*.png'))
        if len(imgs)!=10:raise RuntimeError(f'{domain} possui {len(imgs)} imagens, não 10')
    archive=out.parent/(f'TCC_RESULTADOS_{dataset_id}_{out.name}.zip')
    if archive.exists():raise FileExistsError(archive)
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=5,allowZip64=True) as z:
        for p in sorted(out.rglob('*')):
            if p.is_file():z.write(p,p.relative_to(out).as_posix())
    with zipfile.ZipFile(archive) as z:
        corrupt=z.testzip()
        if corrupt:raise RuntimeError('ZIP corrompido em '+corrupt)
    return out,archive

def upload_single_zip(path,dataset_id,drive_root=None):
    """Exporta ZIP somente para diretório explicitamente indicado; verifica SHA."""
    path=Path(path)
    if not path.is_file():raise FileNotFoundError(path)
    export_dir = drive_root or os.environ.get('TCC_EXPORT_DIR', '').strip()
    if not export_dir:
        raise ValueError('Defina TCC_EXPORT_DIR explicitamente para habilitar exportação.')
    dest = Path(export_dir).expanduser()
    if not dest.is_absolute():
        raise ValueError('TCC_EXPORT_DIR deve ser um caminho absoluto.')
    dest.mkdir(parents=True, exist_ok=True)
    final=dest/path.name;partial=dest/(path.name+'.partial')
    if final.exists() or partial.exists():raise FileExistsError(final)
    shutil.copy2(path,partial)
    if sha(partial)!=sha(path):
        partial.unlink(missing_ok=True)
        raise RuntimeError('Cópia ao Drive falhou no SHA256')
    partial.replace(final)
    return final

'''
_UNIFIED_MODULE_FILE = Path('/content/tcc_unified_results_v1.py')
_UNIFIED_MODULE_FILE.write_text(_UNIFIED_MODULE_SOURCE, encoding='utf-8')
_spec = importlib.util.spec_from_file_location('tcc_unified_results_v1', _UNIFIED_MODULE_FILE)
_unified = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_unified)

H_UNIFIED_ROOT = Path('/content/tcc_resultados_padronizados') / ('estacionamento_' + H_STAMP)
H_UNIFIED_DIR, H_UNIFIED_ZIP = _unified.normalize(
    'video', H_RUN/'final', 'estacionamento', H_UNIFIED_ROOT,
    source_extra={'source_video':'private_input',
      'video_sha256':H_DONE.get('video_sha256'),
      'sampling_fps':H_SAMPLE_FPS,'pose_method':'COLMAP_TRAIN_only_PnP_VAL_TEST'}
)

# ------- EVIDÊNCIAS V4: SEM RENDER TRAIN; VAL + TEST / CUSTO / VIZINHOS -------
from pathlib import Path
import importlib.util
_TCC_EVIDENCE_SOURCE = r'''
"""Extensão V4: imagens reais de TRAIN/VAL/TEST e renderizações somente VAL/TEST.

Mantém 95 RGB reais TRAIN, 10 VAL e 10 TEST; gera 10 VAL por modelo usando
checkpoints já escolhidos na validação. As 10 TEST por modelo são copiadas
dos resultados originais, sem nova avaliação nem retreinamento.
Não renderiza TRAIN nem modifica transforms.json: mantém 95 TRAIN para
carregar sem conflito os embeddings de aparência do Nerfacto-Big.
"""
from pathlib import Path
from datetime import datetime, timezone
from PIL import Image, ImageDraw, ImageOps
import csv, hashlib, html, json, math, os, re, shutil, subprocess, time, zipfile

SPLITS = {'train':95,'val':10,'test':10}
DOMAINS = ('real','nerfacto','splatfacto')
FRAMES_FIELDS = ['dataset_id','split','split_index','chronological_index','source_image','source_type',
 'source_timestamp_us','nominal_time_seconds','real_png','nerfacto_png','splatfacto_png',
 'real_sha256','nerfacto_sha256','splatfacto_sha256','preceding_source_frame',
 'preceding_source_timestamp_us','preceding_gap_ms','following_source_frame',
 'following_source_timestamp_us','following_gap_ms','temporal_neighbor_scope',
 'nearest_train_before','nearest_train_before_gap_ms','nearest_train_after',
 'nearest_train_after_gap_ms','temporal_relation_to_train','render_provenance']
COST_FIELDS = ['dataset_id','method','training_minutes','training_wall_seconds',
 'train_gpu_peak_mib_sampled','train_gpu_peak_sample_interval_seconds',
 'checkpoint_size_mib','val_candidate_render_wall_seconds','test_render_wall_seconds',
 'train_final_render_wall_seconds','val_final_render_wall_seconds',
 'additional_final_renders_wall_seconds','additional_final_render_gpu_peak_mib_sampled',
 'render_total_seconds_including_overhead','test_render_seconds_per_view_including_overhead',
 'gpu_name','gpu_memory_measurement_note','render_timing_note','train_final_render_views','val_final_render_views','test_render_views']


def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(1024*1024),b''):h.update(block)
    return h.hexdigest()


def rd(path):return json.loads(Path(path).read_text(encoding='utf-8'))


def wr(path,object_):
    p=Path(path);p.parent.mkdir(parents=True,exist_ok=True)
    p.write_text(json.dumps(object_,indent=2,ensure_ascii=False,default=str,allow_nan=False)+'\n',encoding='utf-8')


def table(path,fields,rows):
    p=Path(path);p.parent.mkdir(parents=True,exist_ok=True)
    with p.open('w',newline='',encoding='utf-8') as f:
        writer=csv.DictWriter(f,fieldnames=fields,extrasaction='ignore')
        writer.writeheader();writer.writerows(rows)


def readtable(path):
    with Path(path).open('r',newline='',encoding='utf-8-sig') as f:return list(csv.DictReader(f))


def numeric(value):
    try:
        if value is None or value=='':return None
        n=float(value)
        return n if math.isfinite(n) else None
    except (ValueError,TypeError):return None


def pixel_png(source,destination):
    # Mantem resolucao NATIVA retificada: 791x444, 439x782, 800x450 etc.
    source=Path(source);destination=Path(destination)
    if not source.is_file():raise FileNotFoundError(source)
    destination.parent.mkdir(parents=True,exist_ok=True)
    with Image.open(source) as im:
        if im.width<2 or im.height<2:
            raise RuntimeError(f'{source}: dimensoes invalidas {im.size}')
        if im.mode=='RGB' and (im.format or '').upper()=='PNG':
            # Identidade pixel-a-pixel; evita transcoding sem necessidade.
            shutil.copy2(source,destination)
        else:
            im.convert('RGB').save(destination,format='PNG')
    return sha(destination)


def png_index(root,split,kind,stems):
    root=Path(root)/split/kind
    paths=list(root.rglob('*.png'))
    mapping={p.stem:p for p in paths}
    if len(paths)!=len(mapping) or set(mapping)!=set(stems):
        raise RuntimeError(f'{root}: faltam ou sobram PNGs; previsto={len(stems)}, obtido={len(paths)}')
    return mapping


def gpu_used_for_process(pid):
    try:
        p=subprocess.run(['nvidia-smi','--query-compute-apps=pid,used_gpu_memory',
                          '--format=csv,noheader,nounits'],capture_output=True,text=True,timeout=7)
        if p.returncode:return None
        readings=[]
        for line in p.stdout.splitlines():
            parts=[x.strip() for x in line.split(',')]
            if len(parts)<2:continue
            if parts[0].isdigit() and int(parts[0])==pid:
                value=numeric(parts[1].split()[0])
                if value is not None:readings.append(value)
        return max(readings) if readings else None
    except (FileNotFoundError,subprocess.TimeoutExpired):return None


def execute_render(command,log_path,timeout=7200):
    log_path=Path(log_path);log_path.parent.mkdir(parents=True,exist_ok=True)
    started=time.monotonic();peak=None;sample_count=0
    env=dict(os.environ)
    for key in ('PYTHONPATH','PYTHONHOME','PYTHONUSERBASE','PIP_TARGET'):
        env.pop(key,None)
    env['PYTHONNOUSERSITE']='1';env['PYTHONUNBUFFERED']='1'
    # Compatibilidade: Nerfstudio 1.1.5 usa torch.load sem weights_only explícito.
    # No PyTorch >= 2.6 o valor padrão mudou para True; alguns checkpoints
    # próprios da execução contêm escalares NumPy não permitidos nesse modo.
    # Aplicar apenas ao subprocesso ns-render com checkpoints confiáveis,
    # previamente validados pelos hashes do pipeline. NÃO altera os pesos.
    if env.get('TORCH_FORCE_WEIGHTS_ONLY_LOAD','').strip().lower() in ('1','true','yes','y'):
        raise RuntimeError(
            'TORCH_FORCE_WEIGHTS_ONLY_LOAD está ativado no ambiente: '
            'remova-o antes de renderizar checkpoints próprios do Nerfstudio 1.1.5.'
        )
    env['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD']='1'
    with log_path.open('w',encoding='utf-8') as log:
        log.write('CMD: '+' '.join(map(str,command))+'\n\n');log.flush()
        process=subprocess.Popen(list(map(str,command)),stdout=log,stderr=subprocess.STDOUT,env=env)
        try:
            while process.poll() is None:
                if time.monotonic()-started>timeout:
                    process.kill();process.wait()
                    raise TimeoutError(f'ns-render demorou mais que {timeout}s: {log_path}')
                sample=gpu_used_for_process(process.pid)
                if sample is not None:
                    peak=sample if peak is None else max(peak,sample)
                    sample_count+=1
                time.sleep(2)
            if process.returncode:
                raise RuntimeError(f'Falha do ns-render: {log_path}\n'+log_path.read_text(errors='replace')[-7000:])
        finally:
            if process.poll() is None:process.kill();process.wait()
    text=log_path.read_text(errors='replace')
    if not re.search('Done loading checkpoint from',text):
        raise RuntimeError('ns-render não confirmou carregamento de checkpoint: '+str(log_path))
    return {'wall_seconds':round(time.monotonic()-started,4),
            'gpu_peak_mib_sampled':peak,'gpu_memory_observations':sample_count,
            'gpu_memory_sampling_period_seconds':2,'log':str(log_path)}


def video_source_neighbor(run,stem,offset):
    m=re.fullmatch(r'frame_(\d+)',stem)
    if not m:return None
    n=int(m.group(1))+offset
    if n<1:return None
    source=Path(run)/'raw_frames'/f'frame_{n:06d}.png'
    return source if source.is_file() else None


def picture_neighbors(before,current,after,output,title):
    output=Path(output);output.parent.mkdir(parents=True,exist_ok=True)
    image=Image.new('RGB',(1200,260),(248,248,248))
    drawer=ImageDraw.Draw(image)
    for col,(src,label) in enumerate(((before,'ANTERIOR'),(current,'ATUAL'),(after,'POSTERIOR'))):
        drawer.text((col*400+6,5),label,fill=(17,28,43))
        if src is not None and Path(src).is_file():
            with Image.open(src) as im:
                im=ImageOps.contain(im.convert('RGB'),(400,225),Image.Resampling.LANCZOS)
                image.paste(im,(col*400+(400-im.width)//2,30+(225-im.height)//2))
        else:
            drawer.text((col*400+75,135),'INDISPONÍVEL',fill=(105,105,105))
    image.save(output,quality=88)


def extend(dataset_id,source_kind,run_root,dataset,unified_dir,unified_zip,
           selected_configs,selected_steps,ns_render,training_stats=None,
           extra_source_frames=None):
    """Atualiza diretório e ZIP canônico já normalizados; nunca treina/seleciona.

    source_kind = 'nuscenes' ou 'video', run_root contém outputs originais
    dataset tem transforms.json e as 115 imagens retificadas/normalizadas.
    """
    root=Path(run_root);dataset=Path(dataset);out=Path(unified_dir);zipfile_path=Path(unified_zip)
    if not all(x.exists() for x in (root,dataset,out,zipfile_path)):
        raise FileNotFoundError('Dados locais A–G/H/I ou ZIP canônico ausentes; é necessário manter a sessão.')
    manifest_path=out/'final_experiment_manifest.json'
    current=rd(manifest_path)
    if current.get('dataset_id')!=dataset_id or current.get('splits')!=SPLITS:
        raise RuntimeError('Arquivo canônico original não é do cenário/split esperado.')
    tf=rd(dataset/'transforms.json')
    frame_by_path={f['file_path']:f for f in tf['frames']}
    bysplit={}
    for sp,count in SPLITS.items():
        paths=tf[sp+'_filenames']
        if len(paths)!=count or len(set(paths))!=count:
            raise RuntimeError(f'Split {sp} não contém {count} frames únicos.')
        if not set(paths)<=set(frame_by_path):raise RuntimeError('Frames ausentes de transforms.json.')
        bysplit[sp]=sorted(paths,key=lambda p:(int(frame_by_path[p].get('timestamp',0)),p))
    if len({x for paths in bysplit.values() for x in paths})!=115:
        raise RuntimeError('Treino/validação/teste não disjuntos.')
    if source_kind not in ('nuscenes','video'):raise ValueError(source_kind)
    # Recupera FPS efetivo da execucao (NUNCA pressupor 4 FPS).
    video_fps=None
    if source_kind=='video':
        video_metadata=rd(root/'final'/'manifest.json')
        video_fps=float(video_metadata['fps_sampled'])
        if not math.isfinite(video_fps) or video_fps<=0:
            raise RuntimeError('FPS de amostragem invalido no manifesto do video')
    extra_source_frames={int(k):Path(v) for k,v in (extra_source_frames or {}).items()}
    for model in ('splatfacto','nerfacto'):
        config=Path(selected_configs[model]);step=int(selected_steps[model])
        if not config.is_file():raise FileNotFoundError('Config selecionado ausente: '+str(config))
        if step<0:raise ValueError(step)
    if not Path(ns_render).is_file():raise FileNotFoundError(ns_render)
    # Este manifesto é de 115 capturas, não das 40 imagens excluídas.
    canon_test=readtable(out/'tables/image_index.csv')
    if len(canon_test)!=10:raise RuntimeError('Manifesto canônico de TEST não tem dez linhas.')
    test_source_by_stem={str(r['source_image']):int(r['image_index']) for r in canon_test}
    if set(test_source_by_stem)!={Path(p).stem for p in bysplit['test']}:
        raise RuntimeError('Identidade TEST do pacote não corresponde a transforms.json.')

    # Só renderiza as 10 vistas VAL após seleção do checkpoint. As dez TEST
    # são copiadas dos resultados já avaliados, preservando o protocolo final.
    # O transforms original conserva 95 TRAIN, requisito para o embedding
    # de aparência [95,128] aprendido pelo Nerfacto-Big. Não editar TRAIN.
    source_transforms_sha=sha(dataset/'transforms.json')
    results={};base_render=out.parent/('_render_val_test_'+dataset_id+'_'+out.name)
    base_render.mkdir(parents=True,exist_ok=False)
    for method in ('splatfacto','nerfacto'):
        results[method]={}
        split='val'
        output=base_render/method/split
        output.mkdir(parents=True,exist_ok=False)
        paths=bysplit['val']
        command=[str(ns_render),'dataset','--load-config',str(selected_configs[method]),
          '--output-path',str(output),'--data',str(dataset),'--split','val',
          '--image-format','png','--rendered-output-names','rgb','gt-rgb']
        print(f'[EVIDÊNCIA V4] {dataset_id}/{method}/val: 10 vistas. '
              'TRAIN não será renderizado; TEST será reutilizado.',flush=True)
        results[method]['val']=execute_render(command,base_render/f'{method}_val.log')
        results[method]['val']['original_data_split']='val'
        results[method]['val']['render_split']='val'
        results[method]['val']['temporary_dataset']=False
        stems=[Path(p).stem for p in paths]
        rgb=png_index(output,'val','rgb',stems)
        gt=png_index(output,'val','gt-rgb',stems)
        # Referências de validação devem continuar idênticas às imagens originais.
        import numpy as np
        for rel in paths:
            stem=Path(rel).stem
            with Image.open(dataset/rel) as a, Image.open(gt[stem]) as b:
                ar=np.asarray(a.convert('RGB'),dtype=np.int16)
                br=np.asarray(b.convert('RGB'),dtype=np.int16)
                if (ar.ndim!=3 or ar.shape[2]!=3 or br.shape!=ar.shape
                        or int(np.max(np.abs(ar-br)))>1):
                    raise RuntimeError('Referencia VAL diferente da original: '+method+'/'+rel)
        results[method]['val']['rgb_map']={k:str(v) for k,v in rgb.items()}
    if sha(dataset/'transforms.json')!=source_transforms_sha:
        raise RuntimeError('O transforms.json mudou durante renderização de VAL.')
    # Evidências reais e geradas, sem alterar os arquivos TEST da etapa D/H/I.
    all_frames=[];chronological=[]
    for sp in ('train','val','test'):
        for i,rel in enumerate(bysplit[sp]):
            frame=frame_by_path[rel]
            raw_ts=frame.get('timestamp')
            if source_kind=='video':
                # Timestamp nominal: indice do frame dividido pelo FPS REAL do experimento.
                number=re.search(r'(\d+)$',Path(rel).stem)
                raw_ts=int(number.group(1)) if number else None
                nominal=(raw_ts-1)/video_fps if raw_ts is not None else None
                ordering=nominal if nominal is not None else i
                unit=f'video_sample_at_{video_fps:g}fps'
            else:
                raw_ts=int(raw_ts)
                nominal=raw_ts/1e6
                ordering=raw_ts/1e6
                unit='nuscenes_timestamp_us'
            chronological.append((ordering,rel,sp,i,raw_ts,nominal,unit))
    chronological.sort(key=lambda x:(x[0],x[1]))
    pos={p[1]:idx for idx,p in enumerate(chronological)}
    train_times=sorted([(t,rel) for t,rel,sp,_,_,_,_ in chronological if sp=='train'])
    # nuScenes preserva um grid de 155 amostras incluindo 40 guardas; vídeo preserva os frames brutos.
    raw_grid=[]
    if source_kind=='nuscenes':
        raw_grid=rd(dataset/'dataset_manifest.json').get('split',{}).get('raw_grid',[])
        raw_grid=sorted(raw_grid,key=lambda x:int(x['timestamp_us']))
    time_to_selected_rel={int(f.get('timestamp',-1)):rel for rel,f in frame_by_path.items()} if source_kind=='nuscenes' else {}
    for position,(order,rel,split,i,raw_ts,nominal,unit) in enumerate(chronological):
        stem=Path(rel).stem
        sources={'real':dataset/rel}
        for method in ('nerfacto','splatfacto'):
            if split=='test':
                original_index=test_source_by_stem[stem]
                sources[method]=out/f'images/{method}/eval_img_{original_index:04d}.png'
            elif split=='train':
                sources[method]=None  # Somente imagens reais para TRAIN.
            else:
                sources[method]=Path(results[method][split]['rgb_map'][stem])
        target_paths={};hashes={}
        for domain,src in sources.items():
            if src is None:
                target_paths[domain]=None;hashes[domain]=None
                continue
            dest=out/'images'/split/domain/f'frame_{i:04d}.png'
            hashes[domain]=pixel_png(src,dest)
            target_paths[domain]=dest.relative_to(out).as_posix()
        if split=='test':
            # As dez vistas usadas nas métricas TEST têm de permanecer pixel-idênticas.
            for method in ('nerfacto','splatfacto'):
                if sha(sources[method])!=hashes[method]:
                    with Image.open(sources[method]) as a,Image.open(out/target_paths[method]) as b:
                        import numpy as np
                        if not np.array_equal(np.asarray(a.convert('RGB')),np.asarray(b.convert('RGB'))):
                            raise RuntimeError('TEST PNG alterou pixels ao normalizar: '+stem)
        prev=chronological[position-1] if position else None
        foll=chronological[position+1] if position+1<len(chronological) else None
        scope='selected_115_frames_only'
        previous_source=dataset/prev[1] if prev else None
        following_source=dataset/foll[1] if foll else None
        before_ts=prev[4] if prev else None;after_ts=foll[4] if foll else None
        # Contexto de captura completo quando disponível.
        if source_kind=='video':
            raw_prev=video_source_neighbor(root,stem,-1)
            raw_next=video_source_neighbor(root,stem,+1)
            if raw_prev is not None:
                previous_source=raw_prev
                before_ts=int(re.search(r'(\d+)$',raw_prev.stem).group(1))
            if raw_next is not None:
                following_source=raw_next
                after_ts=int(re.search(r'(\d+)$',raw_next.stem).group(1))
            if raw_prev is not None and raw_next is not None:scope='all_extracted_video_frames'
        elif raw_grid:
            idx=next((n for n,x in enumerate(raw_grid) if int(x['timestamp_us'])==raw_ts),None)
            if idx is not None:
                if idx>0:
                    b=raw_grid[idx-1];before_ts=int(b['timestamp_us'])
                    selected_rel=time_to_selected_rel.get(before_ts)
                    previous_source=dataset/selected_rel if selected_rel else extra_source_frames.get(before_ts)
                    # Captura de guarda sem PNG do próprio A: usar arquivo nuScenes original, se disponível.
                if idx+1<len(raw_grid):
                    after_ts=int(raw_grid[idx+1]['timestamp_us'])
                    selected_rel=time_to_selected_rel.get(after_ts)
                    following_source=dataset/selected_rel if selected_rel else extra_source_frames.get(after_ts)
                scope='nuscenes_155_grid_timestamps_with_raw_neighbors' if extra_source_frames else 'nuscenes_155_grid_timestamps; neighbor_PNG_only_if_selected'
        # segundos no vídeo, microssegundos no nuScenes
        gap_before=((raw_ts-before_ts)*1000.0/video_fps if source_kind=='video' else (raw_ts-before_ts)/1000.) if before_ts is not None else None
        gap_after=((after_ts-raw_ts)*1000.0/video_fps if source_kind=='video' else (after_ts-raw_ts)/1000.) if after_ts is not None else None
        lower=[x for x in train_times if x[0]<order]
        upper=[x for x in train_times if x[0]>order]
        tprev=lower[-1] if lower else None;tnext=upper[0] if upper else None
        if split=='train':relation='training_observation'
        elif tprev and tnext:relation='temporal_interpolation_between_train_observations'
        elif tprev or tnext:relation='temporal_extrapolation_outside_train_span'
        else:relation='no_surrounding_train_observation'
        row={
          'dataset_id':dataset_id,'split':split,'split_index':i,'chronological_index':position,
          'source_image':rel,'source_type':frame.get('source_type','video_frame' if source_kind=='video' else None),
          'source_timestamp_us':raw_ts if source_kind=='nuscenes' else None,
          'nominal_time_seconds':nominal if source_kind=='video' else round(nominal-chronological[0][0],6),
          'real_png':target_paths['real'],'nerfacto_png':target_paths['nerfacto'],
          'splatfacto_png':target_paths['splatfacto'],
          'real_sha256':hashes['real'],'nerfacto_sha256':hashes['nerfacto'],'splatfacto_sha256':hashes['splatfacto'],
          'preceding_source_frame':str(before_ts) if before_ts is not None else None,
          'preceding_source_timestamp_us':before_ts if source_kind=='nuscenes' else None,
          'preceding_gap_ms':gap_before,'following_source_frame':str(after_ts) if after_ts is not None else None,
          'following_source_timestamp_us':after_ts if source_kind=='nuscenes' else None,
          'following_gap_ms':gap_after,'temporal_neighbor_scope':scope,
          'nearest_train_before':tprev[1] if tprev else None,
          'nearest_train_before_gap_ms':round((order-tprev[0])*1000,5) if tprev else None,
          'nearest_train_after':tnext[1] if tnext else None,
          'nearest_train_after_gap_ms':round((tnext[0]-order)*1000,5) if tnext else None,
          'temporal_relation_to_train':relation,
          'render_provenance':('original TEST render from final evaluation' if split=='test'
             else ('selected checkpoint VAL post-selection' if split=='val'
             else 'no TRAIN rendering requested'))
        }
        all_frames.append(row)
        figure=out/'figures/temporal_context'/split/f'frame_{i:04d}.jpg'
        if split in ('val','test'):
            picture_neighbors(previous_source,dataset/rel,following_source,figure,stem)
    assert len(all_frames)==115
    table(out/'tables/all_frames_manifest.csv',FRAMES_FIELDS,sorted(all_frames,key=lambda row:(row['split'],int(row['split_index']))))
    table(out/'tables/frame_temporal_context.csv',FRAMES_FIELDS,sorted(all_frames,key=lambda row:int(row['chronological_index'])))

    # Custos computacionais idênticos em todos os pacotes. Informação ausente NÃO vira zero.
    original_cost=readtable(out/'tables/compute_cost.csv')
    cost_by_method={r['method']:r for r in original_cost}
    summary=readtable(out/'tables/final_experiment_summary.csv')
    original_sum={r['method']:r for r in summary}
    native_manifest=rd(root/'final/manifest.json') if source_kind=='video' else None
    native_model=(native_manifest or {}).get('models',{})
    computed=[]
    for method in ('splatfacto','nerfacto'):
        old=cost_by_method.get(method,{})
        sm=original_sum[method]
        info=native_model.get(method,{}) if source_kind=='video' else {}
        training_min=numeric(old.get('training_minutes'))
        test_seconds=numeric(old.get('render_total_seconds_including_overhead'))
        trainstat=info.get('training_peak_gpu_memory_mib_observed') if source_kind=='video' else (training_stats or {}).get(method,{}).get('peak_gpu_memory_mib_sampled')
        trainvals=info.get('training_peak_gpu_memory_sampling_interval_seconds') if source_kind=='video' else (training_stats or {}).get(method,{}).get('peak_gpu_memory_sampling_interval_seconds')
        rec={
            'dataset_id':dataset_id,'method':method,'training_minutes':training_min,
            'training_wall_seconds':training_min*60 if training_min is not None else None,
            'train_gpu_peak_mib_sampled':trainstat,
            'train_gpu_peak_sample_interval_seconds':trainvals,
            'checkpoint_size_mib':numeric(sm.get('checkpoint_size_mib')),
            'val_candidate_render_wall_seconds':info.get('validation_candidate_render_wall_seconds'),
            'test_render_wall_seconds':test_seconds,
            'train_final_render_wall_seconds':None,
            'val_final_render_wall_seconds':results[method]['val']['wall_seconds'],
            'additional_final_renders_wall_seconds':round(results[method]['val']['wall_seconds'],5),
            'additional_final_render_gpu_peak_mib_sampled':max((v for v in
               (results[method]['val']['gpu_peak_mib_sampled'],)
                 if v is not None),default=None),
            'render_total_seconds_including_overhead':test_seconds,
            'test_render_seconds_per_view_including_overhead':test_seconds/10 if test_seconds is not None else None,
            'gpu_name':(native_manifest or {}).get('gpu') if source_kind=='video' else None,
            'gpu_memory_measurement_note':'nvidia-smi sampled process memory, 2s on extra renders; training GPU N/D if not recorded; never an exact peak',
            'render_timing_note':'TRAIN not rendered; VAL 10 views after checkpoint selection; TEST preserved from original evaluation. Times include startup and PNG writing, not pure FPS',
            'train_final_render_views':0,
            'val_final_render_views':10,
            'test_render_views':10,
        }
        computed.append(rec)
    table(out/'tables/compute_cost.csv',COST_FIELDS,computed)
    wr(out/'provenance/full_frame_render_experiment.json',{
      'dataset_id':dataset_id,'extension_version':'V4_val_test_only',
      'checkpoint_selection':{m:{'selected_step':int(selected_steps[m]),
          'config':str(selected_configs[m])} for m in selected_steps},
      'test_re_rendered':False,'test_heldout_unchanged':True,
      'train_rendered':False,'train_real_count':95,'validation_final_render_count_per_method':10,
      'validation_post_selection_only':True,'original_train_count_preserved':95,
      'original_transforms_sha256_before_and_after':
          [source_transforms_sha,sha(dataset/'transforms.json')],
      'render_profiling':results,'dataset_transforms_sha256':sha(dataset/'transforms.json'),
      'notes':['Nenhuma imagem TRAIN foi renderizada; as 95 observações reais permanecem arquivadas.',
               'VAL foi renderizada usando os checkpoints já selecionados sem nova seleção.',
               'TEST é reutilizado pixel a pixel, sem rerenderização nem reajuste.',
               f'Para video, tempo e posicao nominal na amostragem {video_fps:g} FPS, nao timestamp de hardware.' if source_kind=='video' else 'nuScenes: timestamps oficiais dos sensores.',
               'Para nuScenes, vizinhos físicos guardados podem não ter PNG exportado.',
               'Tempos de subprocesso não são FPS em tempo real.']})
    if source_kind=='video':
        (out/'provenance/video_sampling_and_splits.json').write_text(
            (root/'planned_split.json').read_text(encoding='utf-8'),encoding='utf-8')
    else:
        wr(out/'provenance/nuscenes_temporal_sampling.json',{
           'raw_grid':raw_grid,'grid_source':'A V7 dataset_manifest.split.raw_grid',
           'guard_count':sum(x.get('dataset_split')=='guard' for x in raw_grid)})
    # Novos gráficos focados em custo, ausentes são declarados como N/D.
    try:
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt
        for col,filename,title,unit in (
          ('training_minutes','training_cost_minutes.png','Treinamento (wall time)','min'),
          ('test_render_wall_seconds','test_process_time_seconds.png','TEST: processo completo ns-render','s'),
          ('additional_final_renders_wall_seconds','audit_renders_seconds.png','VAL (10): geração de evidências','s'),
          ('train_gpu_peak_mib_sampled','gpu_train_peak_sampled.png','VRAM observada no treinamento','MiB'),
          ('checkpoint_size_mib','checkpoint_size_mib.png','Tamanho do checkpoint selecionado','MiB')):
            values=[numeric(r.get(col)) for r in computed]
            fig,ax=plt.subplots(figsize=(7,4))
            bars=ax.bar(['Splatfacto','Nerfacto'],[v or 0.0 for v in values])
            ax.set_ylabel(unit);ax.set_title(title);ax.grid(axis='y',alpha=.2)
            for bar,value in zip(bars,values):
                ax.text(bar.get_x()+bar.get_width()/2,bar.get_height(),
                        f'{value:.2f}' if value is not None else 'N/D',ha='center',va='bottom')
            fig.tight_layout();fig.savefig(out/'figures/charts'/filename,dpi=145);plt.close(fig)
    except ImportError:
        print('AVISO: matplotlib ausente; métricas e PNGs ainda são válidos.')
    # Galeria contém 115 imagens reais; apenas VAL/TEST incluem as reconstruções.
    gallery_parts=["<!doctype html><html lang='pt-BR'><meta charset='utf-8'><title>TCC - evidências RGB</title><style>body{font:15px/1.5 system-ui;max-width:1400px;margin:30px auto;padding:12px}.trio{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:12px}img{max-width:100%}article{border-top:1px solid #ccc;padding:15px 0}summary{font-size:21px;cursor:pointer}</style><h1>Evidências RGB: todas as capturas reais</h1><p>115 imagens reais: 95 TRAIN, 10 VAL, 10 TEST. TRAIN contém somente originais. VAL (10) e TEST (10) têm renderizações por método. TEST foi reutilizado da avaliação final, sem rerenderização ou reajuste.</p>"]
    for split in ('train','val','test'):
        subset=sorted([r for r in all_frames if r['split']==split],key=lambda r:int(r['split_index']))
        gallery_parts.append('<details '+('open' if split=='test' else '')+'><summary>'+split.upper()+' — '+str(len(subset))+' capturas</summary>')
        for row in subset:
            i=int(row['split_index']);stem=html.escape(str(row['source_image']))
            gallery_parts.append('<article><h3>'+split.upper()+' '+str(i+1)+' — '+stem+'</h3><p>t = '+html.escape(str(row['nominal_time_seconds']))+' segundos</p><div class="trio">')
            for label,key in [('REAL','real_png'),('NERFACTO','nerfacto_png'),('SPLATFACTO','splatfacto_png')]:
                image_rel=row[key]
                if image_rel:
                    gallery_parts.append('<div><b>'+label+'</b><img loading="lazy" src="'+html.escape(image_rel,quote=True)+'"></div>')
                else:
                    gallery_parts.append('<div><b>'+label+'</b><p>Não renderizada (TRAIN preservado apenas como imagem real)</p></div>')
            gallery_parts.append('</div>')
            neighbor=f'figures/temporal_context/{split}/frame_{i:04d}.jpg'
            if (out/neighbor).exists():
                gallery_parts.append('<p>Captura anterior / atual / posterior:</p><img loading="lazy" src="'+neighbor+'">')
            gallery_parts.append('</article>')
        gallery_parts.append('</details>')
    gallery_parts.append('</html>')
    (out/'GALERIA_EVIDENCIAS_RGB.html').write_text(''.join(gallery_parts),encoding='utf-8')
    old_html=(out/'ANALISE_FINAL.html').read_text(encoding='utf-8')
    old_html=old_html.replace('<body>','<body><p><a href="GALERIA_EVIDENCIAS_RGB.html">Ver imagens reais e reconstruções VAL / TEST</a></p>',1)
    (out/'ANALISE_FINAL.html').write_text(old_html,encoding='utf-8')
    readme=out/'LEIA_ME.txt'
    readme.write_text(readme.read_text(encoding='utf-8')+'\nV4: 115 imagens reais (95 TRAIN / 10 VAL / 10 TEST). Nenhuma TRAIN renderizada. As 10 VAL e 10 TEST possuem reconstruções por método; TEST é cópia pixel-idêntica da avaliação original. Custos documentados em tables/compute_cost.csv; medições ausentes não são zero. Contexto temporal em tables/frame_temporal_context.csv; imagens originais excluídas pela guarda aparecem em metadados, se conhecidos. Galeria: GALERIA_EVIDENCIAS_RGB.html\n',encoding='utf-8')
    m=rd(manifest_path)
    m['evidence_version']='V4_95_10_10_val_test_only'
    m['evidence_counts']={'train_real':95,'val_real':10,'test_real':10,
        'train_reconstructed_per_method':0,
        'val_reconstructed_per_method':10,'test_reconstructed_per_method':10}
    m.pop('selected_train_indices',None)
    m.pop('selected_train_relpaths',None)
    m['test_rerendered']=False
    m['test_metric_protocol_unchanged']=True
    m['full_gallery']='GALERIA_EVIDENCIAS_RGB.html'
    m['temporal_evidence_csv']='tables/frame_temporal_context.csv'
    m['computational_cost_csv']='tables/compute_cost.csv'
    records=[]
    for p in sorted(out.rglob('*')):
        if p.is_file() and p!=manifest_path:
            if p.is_symlink() or p.suffix.lower() in ('.mov','.mp4','.ply','.ckpt','.pt'):
                raise RuntimeError('Arquivo proibido no ZIP final: '+str(p))
            records.append({'path':p.relative_to(out).as_posix(),'sha256':sha(p),'bytes':p.stat().st_size})
    m['file_integrity_excluding_self']=records
    wr(manifest_path,m)
    temporary=zipfile_path.with_suffix('.temporary.zip')
    with zipfile.ZipFile(temporary,'w',zipfile.ZIP_DEFLATED,compresslevel=4,allowZip64=True) as z:
        for p in sorted(out.rglob('*')):
            if p.is_file():z.write(p,p.relative_to(out).as_posix())
    with zipfile.ZipFile(temporary) as z:
        if z.testzip() is not None:raise RuntimeError('ZIP enriquecido corrompido')
        if len([x for x in z.namelist() if x.startswith('images/') and x.endswith('.png')])!=185:
            # 155 imagens organizadas (115 reais + 20 VAL + 20 TEST) + 30 TEST canônicos.
            raise RuntimeError('ZIP V4 incompleto: esperado 185 PNGs no total.')
    temporary.replace(zipfile_path)
    if sha(dataset/'transforms.json')!=source_transforms_sha:
        raise RuntimeError('Dataset original foi alterado pela extensão ilustrativa.')
    print('EVIDÊNCIAS VAL/TEST V4:',out,'| ZIP:',zipfile_path)
    print('TRAIN 95 reais / 0 renderizações; VAL 10/10/10; TEST 10/10/10; 155 imagens organizadas (+30 TEST canônicas).')
    return {'out':out,'zip':zipfile_path,'cost':computed,'frames':len(all_frames),'renders':results}

'''
_TCC_EVIDENCE_FILE = Path('/content/tcc_evidencias_val_test_v4.py')
_TCC_EVIDENCE_FILE.write_text(_TCC_EVIDENCE_SOURCE, encoding='utf-8')
_spec_evidence = importlib.util.spec_from_file_location('tcc_evidencias_val_test_v4',_TCC_EVIDENCE_FILE)
_evidencias = importlib.util.module_from_spec(_spec_evidence)
_spec_evidence.loader.exec_module(_evidencias)

_H_SELECTED = _evidencias.rd(H_RUN/'final/selection_before_test.json')
_H_CONFIGS = {name:H_RUN/'validation'/name/str(int(model['selected_step']))/'config.yml'
 for name,model in _H_SELECTED.items()}
H_EVIDENCE = _evidencias.extend(
 dataset_id='estacionamento',source_kind='video',run_root=H_RUN,
 dataset=H_RUN/'undistorted',unified_dir=H_UNIFIED_DIR,
 unified_zip=H_UNIFIED_ZIP,selected_configs=_H_CONFIGS,
 selected_steps={name:int(info['selected_step']) for name,info in _H_SELECTED.items()},
 ns_render=H_PYTHON.parent/'ns-render'
)
H_UNIFIED_DRIVE = (_unified.upload_single_zip(H_UNIFIED_ZIP,'estacionamento')
                   if os.environ.get('TCC_EXPORT_DIR', '').strip() else None)
H_FINAL_RESULTS_DIR = H_UNIFIED_DIR
H_FINAL_ZIP_PATH = H_UNIFIED_ZIP
print('H V6.5 — ESQUEMA COMUM:', _unified.SCHEMA)
print('Pasta final padronizada:', H_UNIFIED_DIR)
print('Exportação opcional do ZIP:', H_UNIFIED_DRIVE)


In [ ]:
# =============================================================================
# CÉLULA I V3.8 — Uber — 15.000 iterações para ambos os métodos | 95 TRAIN / 10 VAL / 10 TEST | COLMAP só TRAIN
# NeRF (nerfacto-big), Splatfacto, PSNR / SSIM / LPIPS e YOLO
# Executar após as células 0 -> SETUP -> 1 (Nerfstudio 1.1.5)
# COLMAP: treino-only + localização PnP e refinamento de pose em mapa FIXO.
# Mantém intermediários em /content; exportação do ZIP é opcional.
# =============================================================================
from pathlib import Path
from datetime import datetime, timezone
import os, subprocess, json, hashlib, shutil

# -----------------------------------------------------------------------------
# CONFIGURAÇÕES EDITÁVEIS
# -----------------------------------------------------------------------------
I_VIDEO = Path(os.environ['TCC_I_VIDEO']).expanduser() if os.environ.get('TCC_I_VIDEO') else None
I_DRIVE_RESULTS = Path(os.environ.get('TCC_EXPORT_DIR', '/content/tcc_results'))
I_SAMPLE_FPS = 8  # valor inicial; substituído automaticamente pela janela aprovada
I_SPLIT_MODE = 'ampliado_95_10_10'  # Experimento I V1 fixo; não altere no meio da rodada
I_YOLO_MODEL = 'yolov8x.pt'
I_PYTHON = Path(globals().get('TCC_PYTHON', '/content/tcc_env/bin/python'))
I_SEND_FINAL_TO_DRIVE = False  # exportação opcional do pacote canônico
I_RESUME_FROM = None  # Reconstrução e treinamento iniciados do zero.
I_REFINE_QUERY_POSES = True  # Apenas pose VAL/TEST no mapa TRAIN congelado
I_QUERY_MATCH_NEIGHBORS = 32  # Mais suporte visual para localização PnP; fixo antes do teste

# -----------------------------------------------------------------------------
# VALIDAR CAMINHO DO VÍDEO FORNECIDO PELO USUÁRIO
# -----------------------------------------------------------------------------
if not I_PYTHON.is_file():
    raise FileNotFoundError('Execute as células 0, SETUP e 1 primeiro: ' + str(I_PYTHON))
if not globals().get('ENVIRONMENT_VALIDATED', False):
    raise RuntimeError('Célula 1 não validou o ambiente. Execute 0, SETUP e 1.')
if I_SPLIT_MODE != 'ampliado_95_10_10':
    raise ValueError('A I V1 usa exclusivamente 95/10/10; mantenha I_SPLIT_MODE=ampliado_95_10_10.')
if not isinstance(I_REFINE_QUERY_POSES, bool):
    raise TypeError('I_REFINE_QUERY_POSES deve ser booleano.')
if not (8 <= I_QUERY_MATCH_NEIGHBORS <= 60):
    raise ValueError('I_QUERY_MATCH_NEIGHBORS fora da faixa permitida (8 a 60).')
if I_RESUME_FROM is not None:
    raise RuntimeError('O novo protocolo exige reconstrução nova. Deixe I_RESUME_FROM=None.')

# Arquivo de entrada fornecido explicitamente por quem executa.
if I_VIDEO is None:
    raise ValueError('Defina TCC_I_VIDEO em configuração privada antes de executar esta célula.')
if not I_VIDEO.is_file():
    raise FileNotFoundError('Vídeo de entrada não encontrado. Verifique TCC_I_VIDEO.')

I_STAMP = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
I_RUN = Path('/content/uber_experiments') / (I_SPLIT_MODE + '_' + I_STAMP)
I_RUN.mkdir(parents=True, exist_ok=False)
I_CONFIG = {
    'video': str(I_VIDEO.resolve()), 'run': str(I_RUN), 'python': str(I_PYTHON),
    'fps': I_SAMPLE_FPS, 'split_mode': I_SPLIT_MODE, 'yolo_model': I_YOLO_MODEL,
    'protocol_version': 'I_V3_8_windows_COLMAP_TRAIN_15k_each_95_10_10',
    'pose_refinement': I_REFINE_QUERY_POSES, 'query_train_neighbors': I_QUERY_MATCH_NEIGHBORS
}
(I_RUN / 'run_config.json').write_text(json.dumps(I_CONFIG, indent=2), encoding='utf-8')

I_WORKER_SOURCE = r'''
"""Pipeline Uber — câmera traseira voltada para a janela direita: Nerfstudio 1.1.5, execução pelo Python isolado do Colab."""
from pathlib import Path
from datetime import datetime, timezone
import argparse, csv, hashlib, html, json, math, os, re, shutil, subprocess, time, zipfile
import numpy as np
from PIL import Image, ImageDraw


def sha(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()


def save(p,obj):
    Path(p).parent.mkdir(parents=True,exist_ok=True)
    Path(p).write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str)+'\n')


def table(p,rows):
    if not rows: return
    with Path(p).open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)


RUN_SAMPLES = {}

def _gpu_process_vram_mib(process_pid):
    try:
        p = subprocess.run(['nvidia-smi','--query-compute-apps=pid,used_gpu_memory',
            '--format=csv,noheader,nounits'],capture_output=True,text=True,timeout=6)
        if p.returncode:return None
        readings=[]
        for line in p.stdout.splitlines():
            cols=[v.strip() for v in line.split(',')]
            if len(cols)<2:continue
            if cols[0].isdigit() and int(cols[0])==process_pid:
                try:readings.append(float(cols[1].split()[0]))
                except ValueError:pass
        return max(readings) if readings else None
    except (OSError,subprocess.TimeoutExpired):return None

def run(cmd,log,timeout=14400):
    cmd=list(map(str,cmd));print('Executando:', ' '.join(cmd),flush=True)
    start=time.monotonic();peak=None;observations=0
    with Path(log).open('w') as f:
        p=subprocess.Popen(cmd,stdout=f,stderr=subprocess.STDOUT)
        try:
            while p.poll() is None:
                if time.monotonic()-start>timeout: raise TimeoutError('Tempo excedido: '+str(log))
                vram=_gpu_process_vram_mib(p.pid)
                if vram is not None:
                    peak=vram if peak is None else max(peak,vram)
                    observations+=1
                time.sleep(2)
                if int(time.monotonic()-start)//30 != int(time.monotonic()-start-2)//30:
                    print(f'  {time.monotonic()-start:.0f}s; log local: {log}',flush=True)
            if p.returncode: raise RuntimeError(Path(log).read_text(errors='replace')[-5000:])
        finally:
            if p.poll() is None:
                p.terminate()
                try:p.wait(timeout=10)
                except subprocess.TimeoutExpired:p.kill();p.wait()
    seconds=time.monotonic()-start
    RUN_SAMPLES[str(Path(log))]={'wall_seconds':seconds,'gpu_peak_mib_sampled':peak,
        'gpu_memory_sample_count':observations,'gpu_memory_sampling_interval_seconds':2}
    return seconds

def pixel_hash(p):
    with Image.open(p) as im:
        im=im.convert('RGB');return hashlib.sha256(str(im.size).encode()+im.tobytes()).hexdigest()


def split_indices(n, mode):
    """Divisões temporais planejadas ANTES de qualquer reconstrução.

    comparavel_48_3_3: 48/3/3 (n>=80), guardas de um frame junto a holdouts.
    ampliado_95_10_10: grade histórica, exige exatamente os 95/10/10 se n=155.
    """
    if mode == 'comparavel_48_3_3':
        if n < 80:
            raise RuntimeError('Modo 48/3/3 precisa de pelo menos 80 frames.')
        anchors = sorted({int(round(x)) for x in np.linspace(8, n - 9, 6)})
        if len(anchors) != 6:
            raise RuntimeError('Não foi possível criar seis imagens reservadas.')
        val, test = anchors[::2], anchors[1::2]
        hold = set(val + test)
        guard = {j for i in hold for j in (i-1, i+1) if 0 <= j < n} - hold
        pool = [i for i in range(n) if i not in hold | guard]
        if len(pool) < 48:
            raise RuntimeError('Não restaram 48 imagens para treino.')
        train = [pool[int(round(x))] for x in np.linspace(0, len(pool)-1, 48)]
        if len(set(train)) != 48:
            raise RuntimeError('Divisão 48/3/3 produziu treino duplicado.')
    elif mode == 'ampliado_95_10_10':
        val = list(range(5, n-2, 15))
        test = list(range(10, n-2, 15))
        hold = set(val + test)
        guard = {j for i in hold for j in (i-1, i+1) if 0 <= j < n} - hold
        train = [i for i in range(n) if i not in hold | guard]
        if (len(train), len(val), len(test)) != (95, 10, 10):
            raise RuntimeError(
                'O modo ampliado foi definido para 155 frames (95/10/10). '
                f'Este vídeo extraiu {n} frames: {(len(train), len(val), len(test))}. '
                'Ajuste I_SAMPLE_FPS ou defina um novo protocolo antes da execução.'
            )
    else:
        raise ValueError('I_SPLIT_MODE não reconhecido: ' + mode)
    assert len(set(train) & (set(val) | set(test))) == 0
    return {'train': sorted(train), 'val': val, 'test': test, 'guard': sorted(guard)}



def psnr(a,b):
    mse=np.mean(((a.astype(np.float64)-b.astype(np.float64))/255.)**2)
    return float(-10*np.log10(max(mse,1e-15)))


def image_map(root,split,kind,expected):
    paths=list((Path(root)/split/kind).rglob('*.png'))
    found={p.stem:p for p in paths}
    want={Path(p).stem for p in expected}
    if len(paths)!=len(found) or set(found)!=want:
        raise RuntimeError(f'Renders incompletos: {root}/{split}/{kind}; esperado {len(want)}, obtido {len(found)}')
    return found


def render(config,step,split,dataset,out,nsrender):
    import yaml
    # Config confiável, gerado nesta execução pelo Nerfstudio.
    cfg=yaml.load(config.read_text(),Loader=yaml.Loader)
    cfg.load_step=int(step)
    assert cfg.pipeline.model.camera_optimizer.mode=='off'
    expected=cfg.get_checkpoint_dir()/f'step-{step:09d}.ckpt'
    assert expected.is_file(), expected
    out.mkdir(parents=True,exist_ok=False)
    selected=out/'config.yml';selected.write_text(yaml.dump(cfg))
    seconds=run([nsrender,'dataset','--load-config',selected,'--output-path',out/'renders',
                 '--data',dataset,'--split',split,'--image-format','png',
                 '--rendered-output-names','rgb','gt-rgb'],out/'render.log',3600)
    log=re.sub(r'\x1b\[[0-9;]*m','',(out/'render.log').read_text(errors='replace'))
    if not re.search(r'Done loading checkpoint from',log):
        raise RuntimeError('Render não confirmou carregamento do checkpoint.')
    # load_step e diretório são fixados no config; verificar número quando log não truncado.
    printed=re.findall(r'Done loading checkpoint from[^\n]*step-(\d+)\.ckpt',log)
    if printed and int(printed[-1])!=step:raise RuntimeError('Checkpoint de render divergente.')
    return out/'renders',seconds,expected


def init_train_points(dataset,tf,train_names,source_by_output,raw_dir,stats):
    from nerfstudio.data.utils.colmap_parsing_utils import read_images_binary,read_points3D_binary
    model=dataset/'sparse'
    if not (model/'images.bin').is_file():raise RuntimeError('Modelo COLMAP retificado não encontrado.')
    ims=read_images_binary(model/'images.bin');pts=read_points3D_binary(model/'points3D.bin')
    basename_train={Path(p).name for p in train_names}
    colors={}
    for iid,im in ims.items():
        if Path(im.name).name not in basename_train:continue
        # Coordenadas COLMAP referem-se às imagens copiadas sem redimensionamento.
        path=dataset/'images'/im.name
        if not path.is_file():raise FileNotFoundError(path)
        colors[iid]=np.asarray(Image.open(path).convert('RGB'))
    transform=np.asarray(tf.get('applied_transform',np.eye(4)[:3]),dtype=float)
    if transform.shape!=(3,4):raise RuntimeError('applied_transform inválida')
    rows=[]
    for p in pts.values():
        if not np.isfinite(p.xyz).all() or p.error>2.0:continue
        obs=[]
        for iid,idx in zip(p.image_ids,p.point2D_idxs):
            if iid not in colors:continue
            xy=np.rint(ims[iid].xys[idx]).astype(int);arr=colors[iid]
            if 0<=xy[0]<arr.shape[1] and 0<=xy[1]<arr.shape[0]:obs.append(arr[xy[1],xy[0]])
        if len(obs)<3:continue
        xyz=transform[:,:3]@p.xyz+transform[:,3]
        rgb=np.median(obs,axis=0).astype(np.uint8)
        rows.append((*xyz,*rgb))
    if len(rows)<500:raise RuntimeError(f'Somente {len(rows)} pontos confiáveis com >=3 observações TRAIN. Não iniciar treino.')
    if len(rows)>120000:
        idx=np.random.default_rng(42).choice(len(rows),120000,replace=False);rows=[rows[i] for i in sorted(idx)]
    ply=dataset/'train_supported_points.ply'
    with ply.open('w') as f:
        f.write('ply\nformat ascii 1.0\nelement vertex '+str(len(rows))+'\nproperty float x\nproperty float y\nproperty float z\nproperty uchar red\nproperty uchar green\nproperty uchar blue\nend_header\n')
        for r in rows:f.write('%.8f %.8f %.8f %d %d %d\n'%r)
    tf['ply_file_path']=ply.name
    stats.update(initial_points=len(rows),initial_points_sha256=sha(ply),point_color_source='TRAIN only',
                 point_geometry_source='TRAIN-only SfM, >=3 TRAIN observations, reprojection error <=2px')


def iou(a,b):
    xy=np.maximum(a[:2],b[:2]);zw=np.minimum(a[2:],b[2:]);inter=np.maximum(zw-xy,0).prod()
    area=lambda c:np.maximum(c[2:]-c[:2],0).prod()
    return float(inter/max(area(a)+area(b)-inter,1e-12))


def matches(real,pred,threshold):
    # Mesmo princípio de F: matching bipartido de cardinalidade máxima por classe.
    matrix=np.zeros((len(real),len(pred)))
    adjacency={}
    for i,a in enumerate(real):
        for j,b in enumerate(pred):
            if a['class_id']==b['class_id']:matrix[i,j]=iou(np.array(a['xyxy']),np.array(b['xyxy']))
        adjacency[i]=sorted([j for j in range(len(pred)) if matrix[i,j]>=threshold],key=lambda j:matrix[i,j],reverse=True)
    order=sorted(range(len(real)),key=lambda i:max((matrix[i,j] for j in adjacency[i]),default=0),reverse=True)
    assigned={}
    def augment(i,seen):
        for j in adjacency[i]:
            if j in seen:continue
            seen.add(j)
            if j not in assigned or augment(assigned[j],seen):assigned[j]=i;return True
        return False
    for i in order:augment(i,set())
    return [(i,j,float(matrix[i,j])) for j,i in sorted(assigned.items())]


def verify_input_gt(gt,input_path):
    a=np.asarray(Image.open(gt).convert('RGB')).astype(np.int16)
    b=np.asarray(Image.open(input_path).convert('RGB')).astype(np.int16)
    if a.shape!=b.shape or np.max(np.abs(a-b))>1:
        raise RuntimeError('GT renderizada diverge da referência retificada: '+str(input_path))



def refine_query_poses(root, original_model, train_model, query_splits, logs, stats):
    """Audita e opcionalmente refina somente as 6-DOF poses das consultas.

    Os pontos 3D / intrinsecos / poses TRAIN permanecem byte-identicos. A LM
    utiliza apenas 2D-3D das proprias consultas para estimar suas cameras,
    nunca a cor de referencia para otimizar o modelo. Nao usa PSNR de TEST.
    """
    import cv2
    from nerfstudio.data.utils.colmap_parsing_utils import (
        read_images_binary, read_points3D_binary, read_cameras_binary,
        qvec2rotmat, rotmat2qvec, write_images_binary
    )
    registered = read_images_binary(original_model / 'images.bin')
    training = read_images_binary(train_model / 'images.bin')
    points = read_points3D_binary(train_model / 'points3D.bin')
    cameras = read_cameras_binary(train_model / 'cameras.bin')
    image_by_name = {im.name: im for im in registered.values()}
    if any(name not in image_by_name for name in query_splits):
        raise RuntimeError('Faltam imagens reservadas no mapa para auditar poses.')
    # Evitar ajuste com matriz intrinseca incorreta.
    if len(cameras) != 1 or next(iter(cameras.values())).model != 'SIMPLE_RADIAL':
        raise RuntimeError('A I V1 espera o modelo SIMPLE_RADIAL de uma unica camera COLMAP.')
    camera = next(iter(cameras.values()))
    f, cx, cy, radial = map(float, camera.params)
    K = np.array([[f,0,cx],[0,f,cy],[0,0,1]], dtype=np.float64)
    D = np.array([radial,0.,0.,0.,0.], dtype=np.float64)
    centers=[]
    for im in training.values():
        R=qvec2rotmat(im.qvec)
        centers.append(-R.T@np.asarray(im.tvec))
    extent=float(np.linalg.norm(np.ptp(np.stack(centers),axis=0)))
    if extent<=1e-8:raise RuntimeError('Mapa de treino degenerado ao auditar poses.')
    rows=[]
    updated=dict(registered)
    min_correspondences=12
    threshold_px=6.0
    for name in sorted(query_splits):
        im=image_by_name[name]
        ids=np.asarray(im.point3D_ids)
        idx=np.array([j for j,pid in enumerate(ids) if int(pid) in points],dtype=int)
        if len(idx)<min_correspondences:
            raise RuntimeError(f'Pose {name} com somente {len(idx)} associacoes 2D-3D no mapa TRAIN.')
        X=np.asarray([points[int(ids[j])].xyz for j in idx],dtype=np.float64).reshape(-1,3)
        xy=np.asarray(im.xys[idx],dtype=np.float64).reshape(-1,2)
        R=qvec2rotmat(im.qvec)
        rvec,_=cv2.Rodrigues(R)
        tvec=np.asarray(im.tvec,dtype=np.float64).reshape(3,1)
        def errors(rr,tt):
            pts,_=cv2.projectPoints(X,rr,tt,K,D)
            return np.linalg.norm(pts.reshape(-1,2)-xy,axis=1)
        before=errors(rvec,tvec)
        inliers=before<=threshold_px
        n_inlier=int(inliers.sum())
        if n_inlier<min_correspondences:
            raise RuntimeError(
                f'Pose {name}: somente {n_inlier} correspondencias com erro <= {threshold_px} px. '
                'A localizacao nao tem suporte suficiente; nao avaliar TEST silenciosamente.'
            )
        center0=-R.T@tvec.reshape(3)
        after=before.copy()
        applied=False
        shift=0.0
        angle=0.0
        if stats['pose_refinement_requested'] and n_inlier>=min_correspondences:
            # Inliers predefinidos pela reprojecao da pose robusta obtida no COLMAP.
            # O filtro nao utiliza PSNR, SSIM ou YOLO.
            try:
                rr,tt=cv2.solvePnPRefineLM(X[inliers],xy[inliers],K,D,
                                            rvec.copy(),tvec.copy(),
                                            criteria=(cv2.TERM_CRITERIA_EPS+cv2.TERM_CRITERIA_COUNT,50,1e-8))
                proposed=errors(rr,tt)
                R1,_=cv2.Rodrigues(rr)
                center1=-R1.T@tt.reshape(3)
                shift=float(np.linalg.norm(center1-center0)/extent)
                relative_rot=R1@R.T
                angle=float(np.degrees(np.arccos(np.clip((np.trace(relative_rot)-1)/2,-1,1))))
                # Aceitacao predeterminada; protege contra saltos e degradacao.
                improved=(float(np.median(proposed[inliers]))<=float(np.median(before[inliers]))+1e-7)
                safe=bool(np.all(np.isfinite(proposed)) and shift<=0.10 and angle<=8.0)
                if improved and safe:
                    q=np.asarray(rotmat2qvec(R1),dtype=np.float64)
                    q/=np.linalg.norm(q)
                    updated[im.id]=im._replace(qvec=q,tvec=tt.reshape(3))
                    applied=True
                    after=proposed
            except cv2.error as exc:
                print(f'AVISO: refinamento LM nao aplicado a {name}: {exc}',flush=True)
        rows.append({
            'image':name, 'split':query_splits[name], 'matched_2d3d':len(idx), 'inliers_within_6px':n_inlier,
            'inlier_fraction':round(n_inlier/len(idx),6),
            'median_reprojection_before_px':float(np.median(before[inliers])),
            'median_reprojection_after_px':float(np.median(after[inliers])),
            'p90_reprojection_after_px':float(np.percentile(after[inliers],90)),
            'pose_only_lm_applied':applied,
            'pose_translation_change_over_train_extent':shift,
            'pose_rotation_change_degrees':angle,
        })
    # Copiar modelo: apenas images.bin pode mudar, camera.bin e points3D.bin
    # permanecem originais. Se nao houve melhora a localizacao original e mantida.
    refined_path=root/'sfm'/'registered_with_queries_pose_only'
    shutil.copytree(original_model,refined_path)
    write_images_binary(updated,refined_path/'images.bin')
    original_points=read_points3D_binary(refined_path/'points3D.bin')
    if set(points)!=set(original_points):
        raise RuntimeError('IDs dos pontos de treino mudaram durante a localizacao.')
    for pid,p in points.items():
        if np.max(np.abs(np.asarray(p.xyz)-np.asarray(original_points[pid].xyz)))>1e-12:
            raise RuntimeError('Coordenadas da nuvem TRAIN mudaram durante a localizacao.')
    if sha(refined_path/'cameras.bin')!=sha(train_model/'cameras.bin'):
        raise RuntimeError('Intrinsecos de treino foram modificados por engano.')
    for iid,orig in training.items():
        new=updated[iid]
        if np.max(np.abs(new.qvec-orig.qvec))>1e-12 or np.max(np.abs(new.tvec-orig.tvec))>1e-12:
            raise RuntimeError('Refinamento de consulta alterou uma pose TRAIN.')
    stats['query_pose_diagnostics']=rows
    stats['pose_refinement_applied_count']=sum(bool(r['pose_only_lm_applied']) for r in rows)
    stats['query_pose_min_inliers']=min(r['inliers_within_6px'] for r in rows)
    stats['query_pose_reference']='fixed TRAIN points / SIMPLE_RADIAL / per-query 2D-3D'
    print('Diagnostico de poses reservado:', [(r['image'],r['inliers_within_6px'],r['pose_only_lm_applied']) for r in rows],flush=True)
    return refined_path,rows


def colmap_inductive(root, originals, split, logs, stats):
    """Reconstrói somente TRAIN; VAL/TEST recebem pose via registro PnP.

    Sequência: extrator TRAIN -> matcher TRAIN -> mapper TRAIN ->
    extrator consultas -> pares CONSULTA/TRAIN -> image_registrator.
    Não roda bundle_adjuster nem triangulação após inserir consultas.
    """
    from nerfstudio.data.utils.colmap_parsing_utils import (
        read_images_binary, read_points3D_binary, read_cameras_binary
    )
    source = root / 'sfm_source'
    images_dir = source / 'images'
    images_dir.mkdir(parents=True)
    sfm = root / 'sfm'
    sfm.mkdir()
    database = sfm / 'database.db'
    active = sorted(split['train'] + split['val'] + split['test'])
    for i in active:
        shutil.copy2(originals[i], images_dir / originals[i].name)

    # O banco COLMAP contém inicialmente apenas as imagens de treinamento.
    train_files = [originals[i].name for i in split['train']]
    query_files = [originals[i].name for i in sorted(split['val'] + split['test'])]
    train_list = sfm / 'train.txt'
    query_list = sfm / 'query.txt'
    train_list.write_text('\n'.join(train_files) + '\n')
    query_list.write_text('\n'.join(query_files) + '\n')
    # Reutilizar EXATAMENTE o mapa de TRAIN aprovado no pré-teste.
    preflight=Path(stats['sfm_preflight_path'])
    if not (preflight/'database.db').is_file():
        raise RuntimeError('Banco COLMAP do pré-teste aprovado não encontrado.')
    if not list((preflight/'sparse').glob('*/images.bin')):
        raise RuntimeError('Modelo COLMAP do pré-teste aprovado não encontrado.')
    shutil.copy2(preflight/'database.db',database)
    out=sfm/'train_sparse'
    shutil.copytree(preflight/'sparse',out)
    candidates = []
    for p in sorted(out.glob('*/images.bin')):
        ims = read_images_binary(p)
        candidates.append((len(ims), p.parent))
    if not candidates:
        raise RuntimeError('COLMAP TRAIN não produziu reconstrução. Verifique logs e vídeo.')
    count, chosen = max(candidates, key=lambda x: x[0])
    stats['colmap_train_candidates'] = [
        {'model': p.name, 'train_registered': c} for c, p in candidates
    ]
    stats['colmap_train_selected'] = chosen.name
    if count != len(train_files):
        raise RuntimeError(
            f'COLMAP registrou {count}/{len(train_files)} imagens TRAIN. '
            'Para o protocolo comparável, não reduzirei silenciosamente o conjunto. '
            'Verifique o log colmap_train_mapper.log; aumente a sobreposição '
            'ou utilize o modo ampliado como experimento separado.'
        )
    train_images_before = read_images_binary(chosen / 'images.bin')
    train_points_before = read_points3D_binary(chosen / 'points3D.bin')
    cam_ids = sorted(read_cameras_binary(chosen / 'cameras.bin'))
    if len(cam_ids) != 1:
        raise RuntimeError('Esperava uma câmera intrínseca única para o mesmo vídeo.')
    camera_id = cam_ids[0]
    stats['train_point_count_colmap'] = len(train_points_before)
    stats['colmap_camera_id'] = int(camera_id)

    # Agora acrescentar imagens reservadas ao banco, sem recalcular o mapa.
    run(['colmap', 'feature_extractor',
         '--database_path', database, '--image_path', images_dir,
         '--image_list_path', query_list,
         '--ImageReader.existing_camera_id', str(camera_id),
         '--SiftExtraction.use_gpu', '0'], logs / 'colmap_query_features.log', 7200)
    pairs = []
    for qi in sorted(split['val'] + split['test']):
        # Pares SOMENTE consulta-treino. Não há pares consulta-consulta.
        closest = sorted(split['train'], key=lambda i: (abs(i-qi), i))[:stats['colmap_query_match_nearest_train_count']]
        for ti in closest:
            pairs.append(f'{originals[qi].name} {originals[ti].name}')
    pair_path = sfm / 'query_train_pairs.txt'
    pair_path.write_text('\n'.join(pairs) + '\n')
    run(['colmap', 'matches_importer', '--database_path', database,
         '--match_list_path', pair_path, '--match_type', 'pairs',
         '--SiftMatching.use_gpu', '0'], logs / 'colmap_query_matches.log', 14400)
    registered_path = sfm / 'registered_with_queries'
    registered_path.mkdir()
    run(['colmap', 'image_registrator', '--database_path', database,
         '--input_path', chosen, '--output_path', registered_path],
        logs / 'colmap_query_registration.log', 7200)

    train_images_after = read_images_binary(registered_path / 'images.bin')
    points_after = read_points3D_binary(registered_path / 'points3D.bin')
    expected_train = {im.name for im in train_images_before.values()}
    actual = {im.name for im in train_images_after.values()}
    missing = sorted(set(query_files) - actual)
    stats['train_registered'] = len(expected_train)
    stats['query_registered'] = len(set(query_files) & actual)
    stats['query_registration_missing'] = missing
    if not expected_train <= actual:
        raise RuntimeError('O registro de consultas perdeu imagens de treino.')
    if missing:
        raise RuntimeError(
            'Falha ao localizar todas as imagens reservadas no mapa de treino. '
            f'Faltantes ({len(missing)}): {missing}. '
            'NÃO vou substituir as imagens VAL/TEST nem fazer ajuste conjunto. '
            'Consulte os logs de consulta, aumente a cobertura do vídeo ou '
            'teste I_SPLIT_MODE="ampliado_95_10_10" como experimento separado.'
        )
    for iid, im in train_images_before.items():
        newer = train_images_after[iid]
        if (np.max(np.abs(np.asarray(im.qvec) - np.asarray(newer.qvec))) > 1e-10 or
                np.max(np.abs(np.asarray(im.tvec) - np.asarray(newer.tvec))) > 1e-10):
            raise RuntimeError('image_registrator alterou pose TRAIN; protocolo abortado.')
    if set(train_points_before) != set(points_after):
        raise RuntimeError('O conjunto de pontos SfM mudou após o registro das consultas.')
    for pid, p in train_points_before.items():
        if np.max(np.abs(np.asarray(p.xyz) - np.asarray(points_after[pid].xyz))) > 1e-10:
            raise RuntimeError('Geometria SfM mudou após o registro de consultas.')
    stats['training_geometry_frozen_after_query_registration'] = True
    stats['sfm_protocol'] = 'inductive: TRAIN-only SfM; query PnP + optional pose-only LM in fixed map; no joint BA'
    stats['sfm_uses_holdout_images_for_training_geometry'] = False
    stats['holdout_rgb_used_in_radiance_optimization'] = False

    # Refinamento estritamente de pose nas consultas: nenhum BA conjunto.
    query_splits={originals[i].name:'val' for i in split['val']}
    query_splits.update({originals[i].name:'test' for i in split['test']})
    final_model, pose_rows = refine_query_poses(
        root, registered_path, chosen, query_splits, logs, stats
    )
    table(root / 'final' / 'tables' / 'pose_registration_quality.csv', pose_rows)
    # A retificação usa as poses e intrínsecos estimados, SEM recalibrar geometria.
    undistorted = root / 'undistorted'
    run(['colmap', 'image_undistorter', '--image_path', images_dir,
         '--input_path', final_model, '--output_path', undistorted,
         '--output_type', 'COLMAP', '--max_image_size', '800'],
        logs / 'colmap_undistort.log', 3600)
    from nerfstudio.process_data.colmap_utils import colmap_to_json
    colmap_to_json(undistorted / 'sparse', undistorted)
    return undistorted


def normalize_fixed_from_train(dataset, tf, train_names, stats):
    """Define origem/escala usando SOMENTE as câmeras de TRAIN.

    Como orientação/centragem/autoscale do dataparser serão desligados,
    o mesmo sistema de coordenadas valerá em treino, validação e teste.
    """
    train_set = set(train_names)
    poses = np.array([np.asarray(f['transform_matrix'], dtype=np.float64)
                      for f in tf['frames'] if f['file_path'] in train_set])
    if len(poses) != len(train_names):
        raise RuntimeError('Poses TRAIN incompletas durante normalização.')
    centers = poses[:, :3, 3]
    centroid = centers.mean(axis=0)
    extent = float(np.max(np.abs(centers - centroid)))
    if extent <= 1e-5:
        raise RuntimeError('Geometria TRAIN degenerada sem translação útil.')
    scaling = 0.85 / extent
    for f in tf['frames']:
        mat = np.array(f['transform_matrix'], dtype=np.float64)
        mat[:3, 3] = (mat[:3, 3] - centroid) * scaling
        f['transform_matrix'] = mat.tolist()
    # Os pontos SfM serão transformados exatamente como as câmeras.
    a = np.eye(4)
    a[:3, :4] = np.asarray(tf.get('applied_transform', np.eye(4)[:3]), dtype=np.float64)
    n = np.eye(4)
    n[:3, :3] = np.eye(3) * scaling
    n[:3, 3] = -centroid * scaling
    tf['applied_transform'] = (n @ a)[:3].tolist()
    tf['orientation_override'] = 'none'
    stats['normalization'] = {
        'source': 'train_camera_centers_only',
        'center_saved_xyz': centroid.tolist(),
        'scale': scaling,
        'poses_max_abs_train_after': 0.85,
    }



def choose_colmap_window(root, video, split_mode, raw_dest, duration, cfg):
    """Testa janelas predefinidas com reconstrução COLMAP exclusivamente TRAIN.

    Critério previamente fixado: aceitar a PRIMEIRA janela que registre as
    95 câmeras TRAIN no MESMO modelo SfM. Nunca escolhe por métricas VAL/TEST.
    """
    from nerfstudio.data.utils.colmap_parsing_utils import read_images_binary

    candidates = [
        (8, 36.0), (8, 36.25), (8, 36.50), (8, 36.70),
        (10, 40.0), (12, 40.0),
        (16, 45.0), (20, 45.0),
    ]
    trials_root = root / 'window_trials'
    trials_root.mkdir(parents=True, exist_ok=False)
    records = []
    best = 0
    for fps, start_seconds in candidates:
        # Duração exigida para 155 amostras; não confundir com duração total do vídeo.
        window_seconds = 155.0 / fps
        record = {
            'fps': fps, 'start_seconds': start_seconds,
            'window_seconds_nominal': window_seconds,
            'registered_train': 0, 'status': 'pending',
        }
        if start_seconds + window_seconds > duration + 1e-6:
            record['status'] = 'skipped_outside_video'
            records.append(record)
            save(root / 'window_trials_results.json', records)
            print(f'Janela {fps} FPS / {start_seconds}s fora do vídeo; ignorada.',flush=True)
            continue

        tag = f'fps{fps}_start{start_seconds:05.2f}'.replace('.', 'p')
        trial = trials_root / tag
        trial.mkdir(parents=True, exist_ok=False)
        frames_dir = trial / 'frames'
        images_dir = trial / 'train_images'
        sparse_dir = trial / 'sparse'
        frames_dir.mkdir()
        images_dir.mkdir()
        sparse_dir.mkdir()
        database = trial / 'database.db'
        record['trial'] = str(trial)
        print(f'\n{"="*76}\nCOLMAP TRAIN: {fps} FPS a partir de {start_seconds:.2f}s '
              f'(155 frames / 95 TRAIN)\n{"="*76}', flush=True)
        try:
            run([
                'ffmpeg','-hide_banner','-loglevel','error',
                '-ss',str(start_seconds),'-i',str(video),
                '-map','0:v:0','-an',
                '-vf', f'fps={fps},scale=w=800:h=800:'
                       'force_original_aspect_ratio=decrease:force_divisible_by=2',
                '-frames:v','155',frames_dir/'frame_%06d.png'
            ],trial/'ffmpeg.log',600)
            frames = sorted(frames_dir.glob('frame_*.png'))
            if len(frames) != 155:
                raise RuntimeError(f'FFmpeg produziu {len(frames)} de 155 quadros')
            split = split_indices(155, split_mode)
            if tuple(len(split[k]) for k in ('train','val','test','guard')) != (95,10,10,40):
                raise RuntimeError('Split temporal diferente de 95/10/10 + 40 guardas')
            # Nenhum frame VAL, TEST ou guarda entra no SfM desta tentativa.
            for idx in split['train']:
                shutil.copy2(frames[idx],images_dir/frames[idx].name)
            run([
                'colmap','feature_extractor',
                '--database_path', database, '--image_path',images_dir,
                '--ImageReader.single_camera','1',
                '--ImageReader.camera_model','SIMPLE_RADIAL',
                '--SiftExtraction.use_gpu','0',
            ],trial/'features.log',7200)
            run([
                'colmap','exhaustive_matcher','--database_path',database,
                '--SiftMatching.use_gpu','0',
            ],trial/'matches.log',14400)
            run([
                'colmap','mapper','--database_path',database,
                '--image_path',images_dir,'--output_path',sparse_dir,
            ],trial/'mapper.log',14400)
            models=sorted(sparse_dir.glob('*/images.bin'))
            counts=[len(read_images_binary(p)) for p in models]
            registered=max(counts,default=0)
            best=max(best,registered)
            record.update(status='completed',registered_train=registered,
                          models_registered_counts=counts)
            print(f'COLMAP: {registered}/95 TRAIN na maior reconstrução',flush=True)
            if registered == 95:
                for frame in frames:
                    shutil.copy2(frame,raw_dest/frame.name)
                record['status']='selected'
                records.append(record)
                selection={
                    'fps':fps,'start_seconds':start_seconds,
                    'window_seconds_nominal':window_seconds,
                    'registered_train':95,
                    'preflight_path':str(trial),
                    'candidates_tested':records,
                    'criterion':'first predefined candidate with all 95 TRAIN registered in one COLMAP model',
                    'val_test_used_for_sfm_or_selection':False,
                    'candidate_search_uses_only_train_images':True,
                }
                save(root/'window_selection.json',selection)
                save(root/'window_trials_results.json',records)
                print('JANELA APROVADA:',fps,'FPS, início',start_seconds,'s',flush=True)
                return selection
        except KeyboardInterrupt:
            records.append({**record,'status':'interrupted'})
            save(root/'window_trials_results.json',records)
            raise
        except Exception as exc:
            record.update(status='failed',error=str(exc))
            print(f'COLMAP falhou em {tag}: {exc}',flush=True)
        records.append(record)
        save(root/'window_trials_results.json',records)
        # Limpar apenas quadros e cópias das tentativas descartadas, não os logs.
        shutil.rmtree(frames_dir,ignore_errors=True)
        shutil.rmtree(images_dir,ignore_errors=True)
        if database.is_file():database.unlink()
    raise RuntimeError(
        f'Nenhuma janela reconstruiu 95/95 TRAIN; melhor resultado: {best}/95. '
        f'Consulte {root/"window_trials_results.json"} e os logs em {trials_root}. '
        'Os modelos NÃO foram treinados.'
    )

def main():
    ap=argparse.ArgumentParser();ap.add_argument('--config',required=True);args=ap.parse_args()
    cfg=json.loads(Path(args.config).read_text());root=Path(cfg['run']);video=Path(cfg['video'])
    final=root/'final';final.mkdir();tables=final/'tables';tables.mkdir();figures=final/'figures';figures.mkdir()
    logs=root/'logs';logs.mkdir();raw=root/'raw_frames';raw.mkdir();dataset=root/'dataset'
    bin_dir=Path(cfg['python']).parent
    train=bin_dir/'ns-train';process=bin_dir/'ns-process-data';nsrender=bin_dir/'ns-render'
    import torch, importlib.metadata
    assert importlib.metadata.version('nerfstudio')=='1.1.5','Use o SETUP fornecido (Nerfstudio 1.1.5).'
    assert torch.cuda.is_available(),'GPU indisponível'
    if shutil.disk_usage(root).free<30*1024**3:raise RuntimeError('Necessários 30 GiB livres para intermediários locais.')
    for exe in (train,process,nsrender):assert exe.is_file(),exe
    for exe in ('ffmpeg','ffprobe','colmap'):assert shutil.which(exe), 'Execute SETUP: falta '+exe
    probe=json.loads(subprocess.check_output(['ffprobe','-v','error','-show_streams','-show_format','-of','json',str(video)],text=True))
    vs=next(s for s in probe['streams'] if s['codec_type']=='video')
    if vs.get('color_transfer') in ('smpte2084','arib-std-b67'):
        raise RuntimeError('Vídeo HDR detectado: converter para SDR com tone mapping antes desta célula.')
    duration=float(probe['format']['duration'])
    if abs(float(vs['width'])/float(vs['height'])-16/9)>0.03:
        raise RuntimeError('Uber deve ser vídeo horizontal 16:9 para preservar 800x450 sem cortes; grave em 1920x1080 SDR.')
    if not 38.75<=duration<=180:raise RuntimeError('Uber deve ter entre 38,75 e 180 segundos; para 4 FPS são necessários 155 quadros.')
    stats={'video_name':video.name,'video_sha256':sha(video),'duration_seconds':duration,
           'video_stream':{k:vs.get(k) for k in ('width','height','avg_frame_rate','pix_fmt','color_transfer')},
           'seed':42,'gpu':torch.cuda.get_device_name(0),'fps_sampled':cfg['fps'],
           'pose_refinement_requested':bool(cfg['pose_refinement']),
           'pose_optimization':'off','sfm_protocol':'train-only COLMAP; holdouts PnP localization',
           'test_pixels_used_in_radiance_optimization':False,'sfm_uses_holdout_images':False,
           'point_cloud_is_lidar':False,'storage_policy':'final_zip_only',
           'versions':{x:importlib.metadata.version(x) for x in ('nerfstudio','torch','gsplat','ultralytics')}}
    # A sessão nova usa mapas diferentes; proibir reaproveitamento de SfM conjunto.
    if cfg.get('resume_preprocessing'):
        raise RuntimeError('Não reutilizar SfM do protocolo anterior: I_RESUME_FROM=None.')
    window=choose_colmap_window(root,video,cfg['split_mode'],raw,duration,cfg)
    cfg['fps']=int(window['fps'])
    save(root/'run_config.json',cfg)
    stats['fps_sampled']=int(window['fps'])
    stats['sfm_preflight_path']=window['preflight_path']
    stats['window_search_candidates']=window['candidates_tested']
    stats['window_start_seconds']=float(window['start_seconds'])
    shutil.copy2(root/'window_selection.json',final/'window_selection.json')
    originals = sorted(raw.glob('*.png'))
    n = len(originals)
    if n < 60:
        raise RuntimeError(f'Apenas {n} frames extraídos; revise o FPS/amostragem.')
    split = split_indices(n, cfg['split_mode'])
    save(root / 'planned_split.json', split)
    stats['video_selection_window'] = f"155 frames at {window['fps']} FPS from {window['start_seconds']} seconds"
    stats['video_selection_policy'] = 'predefined candidate list; accept first with 95 TRAIN cameras in one SfM model; VAL/TEST not used for selection'
    stats['split_mode'] = cfg['split_mode']
    stats['protocol_version'] = cfg['protocol_version']
    stats['planned_split'] = {k: len(v) for k, v in split.items()}
    stats['colmap_query_match_nearest_train_count'] = int(cfg['query_train_neighbors'])
    print('Divisão planejada:', stats['planned_split'], flush=True)
    dataset = colmap_inductive(root, originals, split, logs, stats)
    tf = json.loads((dataset / 'transforms.json').read_text())
    frames = tf['frames']
    expected_names = {originals[i].name: i for i in split['train'] + split['val'] + split['test']}
    mapped = {}
    for frame in frames:
        name = Path(frame['file_path']).name
        if name not in expected_names:
            raise RuntimeError('Nome inesperado no transforms: ' + name)
        if not (dataset / frame['file_path']).is_file():
            raise RuntimeError('Imagem retificada ausente: ' + name)
        mapped[frame['file_path']] = expected_names[name]
    if len(mapped) != len(expected_names):
        raise RuntimeError('Faltam imagens após a retificação.')
    for sp in ('train', 'val', 'test'):
        names = sorted((p for p in mapped if mapped[p] in split[sp]), key=lambda p: mapped[p])
        if len(names) != len(split[sp]):
            raise RuntimeError(f'Split {sp} incompleto: {len(names)}/{len(split[sp])}')
        tf[sp + '_filenames'] = names
    # Sanidade de referência e checagem do gap temporal planejado.
    stats['splits'] = {sp: len(tf[sp + '_filenames']) for sp in ('train', 'val', 'test')}
    stats['minimum_train_holdout_gap_seconds'] = min(
        abs(mapped[a] - mapped[b]) / cfg['fps']
        for a in tf['train_filenames']
        for b in tf['val_filenames'] + tf['test_filenames']
    )
    stats['evaluation_rgb'] = 'COLMAP-undistorted SDR RGB PNG; one GT shared by both models'
    normalize_fixed_from_train(dataset, tf, tf['train_filenames'], stats)
    init_train_points(dataset, tf, tf['train_filenames'], mapped, raw, stats)
    save(dataset / 'transforms.json', tf)
    stats['transforms_sha256'] = sha(dataset / 'transforms.json')
    save(final / 'transforms.json', tf)
    image_rows=[]
    import cv2
    for s in ('train','val','test'):
        d=figures/'inputs'/s;d.mkdir(parents=True)
        for p in tf[s+'_filenames']:
            dest=d/Path(p).name;shutil.copy2(dataset/p,dest)
            arr=np.asarray(Image.open(dest).convert('RGB'))
            image_rows.append({'split':s,'frame_index':mapped[p],'nominal_time_seconds':mapped[p]/cfg['fps'],
                               'relative_path':dest.relative_to(final).as_posix(),'sha256':sha(dest),
                               'laplacian_variance':float(cv2.Laplacian(cv2.cvtColor(arr,cv2.COLOR_RGB2GRAY),cv2.CV_64F).var())})
    table(tables/'input_frames.csv',image_rows);print('Dataset pronto:',stats['splits'],flush=True)
    # Poses fixas para o primeiro baseline Uber — câmera traseira voltada para a janela direita. Não importar configs/dataset scene-0655.
    # Orçamento equiparado: 15.000 iterações por método.
    # Cada um mantém 5 (GS) ou 10 (NeRF) candidatos de validação, sem usar TEST.
    methods={'splatfacto':{'preset':'splatfacto','iterations':15000,'eval_steps':[4000,5000,6000,8000,14999]},
             'nerfacto':{'preset':'nerfacto-big','iterations':15000,'eval_steps':[1000,2000,3000,4000,5000,6000,8000,10000,12000,14999]}}
    for name, profile in methods.items():
        if profile['iterations'] != 15000 or profile['eval_steps'][-1] != 14999:
            raise RuntimeError(f'Agenda de 15.000 passos incorreta para {name}.')
        if len(set(profile['eval_steps'])) != len(profile['eval_steps']):
            raise RuntimeError(f'Checkpoints de VAL duplicados em {name}.')
    valrows=[];models={}
    for method,profile in methods.items():
        output=root/'models'/method
        command=[train,profile['preset'],'--data',dataset,'--output-dir',output,'--experiment-name','uber_'+cfg['split_mode']+'_'+method,
                 '--machine.seed','42','--max-num-iterations',profile['iterations'],'--steps-per-save','1000',
                 '--save-only-latest-checkpoint','False','--steps-per-eval-image','1000',
                 '--steps-per-eval-all-images','5000','--vis','tensorboard',
                 '--pipeline.model.camera-optimizer.mode','off']
        if method=='splatfacto':
            command+=['--pipeline.datamanager.cache-images','cpu','--pipeline.datamanager.cache-images-type','uint8',
                      '--pipeline.model.num-downscales','0','--pipeline.model.sh-degree','2',
                      '--pipeline.model.sh-degree-interval','500','--pipeline.model.densify-grad-thresh','0.001',
                      '--pipeline.model.stop-split-at','6000','--pipeline.model.stop-screen-size-at','750',
                      '--pipeline.model.cull-alpha-thresh','0.01','--pipeline.model.use-absgrad','False',
                      '--pipeline.model.use-bilateral-grid','False','--pipeline.model.rasterize-mode','antialiased',
                      '--pipeline.model.ssim-lambda','0.2']
        else:
            command+=['--pipeline.datamanager.train-num-rays-per-batch','8192',
                      '--pipeline.datamanager.eval-num-rays-per-batch','4096','--pipeline.model.implementation','tcnn']
        command+=['nerfstudio-data', '--downscale-factor', '1',
                     '--orientation-method', 'none', '--center-method', 'none',
                     '--auto-scale-poses', 'False', '--scene-scale', '1.5']
        seconds=run(command,logs/(method+'_train.log'),7200)
        configs=list(output.rglob('config.yml'))
        if len(configs)!=1:raise RuntimeError('Esperado um config novo por método.')
        config=configs[0];checkpoints={int(p.stem.split('-')[-1]):p for p in config.parent.glob('nerfstudio_models/step-*.ckpt')}
        if not set(profile['eval_steps'])<=set(checkpoints):raise RuntimeError('Treino incompleto; faltam checkpoints.')
        models[method]={'profile':profile,'training_seconds':seconds,'config':str(config),'command':list(map(str,command))}
        models[method]['training_peak_gpu_memory_mib_observed']=RUN_SAMPLES.get(str(logs/(method+'_train.log')),{}).get('gpu_peak_mib_sampled')
        models[method]['training_peak_gpu_memory_sampling_interval_seconds']=2
        for step in profile['eval_steps']:
            r,_,_=render(config,step,'val',dataset,root/'validation'/method/str(step),nsrender)
            gt=image_map(r,'val','gt-rgb',tf['val_filenames']);pred=image_map(r,'val','rgb',tf['val_filenames'])
            for stem in sorted(gt):
                a=np.asarray(Image.open(gt[stem]).convert('RGB'));b=np.asarray(Image.open(pred[stem]).convert('RGB'))
                assert a.shape==b.shape
                verify_input_gt(gt[stem],dataset/next(x for x in tf['val_filenames'] if Path(x).stem==stem))
                valrows.append({'method':method,'step':step,'image':stem,'psnr_db':psnr(a,b)})
            print(method,step,'VAL',np.mean([x['psnr_db'] for x in valrows if x['method']==method and x['step']==step]),flush=True)
        models[method]['validation_candidate_render_wall_seconds']=sum(
            RUN_SAMPLES.get(str(root/'validation'/method/str(s)/'render.log'),{}).get('wall_seconds',0)
            for s in profile['eval_steps'])
        means={s:float(np.mean([r['psnr_db'] for r in valrows if r['method']==method and r['step']==s])) for s in profile['eval_steps']}
        best=max(means,key=means.get);models[method].update(selected_step=best,val_psnr_db=means[best],validation_means=means)
    table(tables/'validation_per_view.csv',valrows)
    save(final/'selection_before_test.json',models)
    # Ambos os checkpoints já foram selecionados. Só agora avaliar TEST.
    from skimage.metrics import structural_similarity
    from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity
    lpips=LearnedPerceptualImagePatchSimilarity(net_type='alex',normalize=True).cpu().eval()
    metrics=[];renders={};real={};summary=[]
    for method in methods:
        selected=models[method];step=selected['selected_step']
        r,seconds,ckpt=render(Path(selected['config']),step,'test',dataset,root/'test'/method,nsrender)
        gt=image_map(r,'test','gt-rgb',tf['test_filenames']);pred=image_map(r,'test','rgb',tf['test_filenames'])
        renders[method]={};out=figures/'test'/method;out.mkdir(parents=True)
        for stem in sorted(gt):
            a=np.asarray(Image.open(gt[stem]).convert('RGB'));b=np.asarray(Image.open(pred[stem]).convert('RGB'))
            assert a.shape==b.shape
            input_path=dataset/next(x for x in tf['test_filenames'] if Path(x).stem==stem)
            verify_input_gt(gt[stem],input_path)
            if stem in real and pixel_hash(real[stem])!=pixel_hash(gt[stem]):raise RuntimeError('GT diferente entre métodos.')
            real_dir=figures/'test'/'real';real_dir.mkdir(parents=True,exist_ok=True)
            real[stem]=real_dir/(stem+'.png');shutil.copy2(gt[stem],real[stem])
            dest=out/(stem+'.png');shutil.copy2(pred[stem],dest);renders[method][stem]=dest
            to_tensor=lambda x:torch.from_numpy(x.copy()).permute(2,0,1).unsqueeze(0).float()/255.
            with torch.no_grad():distance=float(lpips(to_tensor(a),to_tensor(b)));lpips.reset()
            metrics.append({'method':method,'image':stem,'psnr_db':psnr(a,b),
                            'ssim':float(structural_similarity(a,b,channel_axis=-1,data_range=255,gaussian_weights=True,sigma=1.5,use_sample_covariance=False)),
                            'lpips_alex':distance})
        selected.update(checkpoint_sha256=sha(ckpt),checkpoint_size_mib=ckpt.stat().st_size/1024**2,render_process_seconds=seconds)
        selected['test_render_gpu_peak_mib_observed']=RUN_SAMPLES.get(str(root/'test'/method/'render.log'),{}).get('gpu_peak_mib_sampled')
        subset=[x for x in metrics if x['method']==method]
        summary.append({'method':method,'selected_step':step,'val_psnr_db':selected['val_psnr_db'],
                        'test_psnr_db':float(np.mean([x['psnr_db'] for x in subset])),
                        'test_ssim':float(np.mean([x['ssim'] for x in subset])),
                        'test_lpips_alex':float(np.mean([x['lpips_alex'] for x in subset])),
                        'training_minutes':selected['training_seconds']/60})
    del lpips
    table(tables/'test_per_view.csv',metrics);table(final/'summary.csv',summary)
    # YOLO idêntico ao protocolo anterior: classes COCO de trânsito, 1280, conf .25, NMS .70.
    from ultralytics import YOLO
    yolo=YOLO(cfg['yolo_model']);weights=Path(yolo.ckpt_path)
    stats['yolo']={'weights_sha256':sha(weights),'model':cfg['yolo_model'],'conf':0.25,'nms_iou':0.70,'imgsz':1280,
                   'classes':[0,1,2,3,5,7],'matching':'maximum cardinality per class','match_iou':[0.3,0.5,0.7],'reference':'YOLO real; not human GT'}
    det={};detrows=[];matchrows=[];byimage=[]
    triplets=figures/'rgb_triplets';triplets.mkdir();ytriplets=figures/'yolo_triplets';ytriplets.mkdir()
    for stem in sorted(real):
        rgb_panels=[];yolo_panels=[]
        for domain,path in [('real',real[stem]),('nerfacto',renders['nerfacto'][stem]),('splatfacto',renders['splatfacto'][stem])]:
            res=yolo.predict(str(path),imgsz=1280,conf=.25,iou=.70,classes=[0,1,2,3,5,7],max_det=300,device=0,verbose=False)[0]
            rows=[]
            for j,(box,score,cls) in enumerate(zip(res.boxes.xyxy.cpu().numpy(),res.boxes.conf.cpu().numpy(),res.boxes.cls.cpu().numpy())):
                d={'class_id':int(cls),'confidence':float(score),'xyxy':box.tolist()};rows.append(d)
                detrows.append({'image':stem,'domain':domain,'id':j,'class_id':int(cls),'confidence':float(score),
                                'x1':float(box[0]),'y1':float(box[1]),'x2':float(box[2]),'y2':float(box[3])})
            det[(stem,domain)]=rows
            rgb_panels.append(Image.open(path).convert('RGB'));yolo_panels.append(Image.fromarray(res.plot()[...,::-1]))
        for panels,target in ((rgb_panels,triplets),(yolo_panels,ytriplets)):
            w,h=panels[0].size;canvas=Image.new('RGB',(3*w,h+30),'white');draw=ImageDraw.Draw(canvas)
            for j,(im,title) in enumerate(zip(panels,['REAL','NERFACTO','SPLATFACTO'])):
                canvas.paste(im,(j*w,30));draw.text((j*w+8,7),title,fill='black')
            canvas.save(target/(stem+'.jpg'),quality=94)
        for method in methods:
            rr=det[(stem,'real')];pp=det[(stem,method)]
            for threshold in (.3,.5,.7):
                matched=matches(rr,pp,threshold)
                byimage.append({'image':stem,'method':method,'match_iou':threshold,'real_count':len(rr),'render_count':len(pp),'matched_count':len(matched)})
                for i,j,overlap in matched:matchrows.append({'image':stem,'method':method,'match_iou':threshold,'real_id':i,'render_id':j,'iou':overlap,'class_id':rr[i]['class_id']})
    semantic=[]
    for method in methods:
        for threshold in (.3,.5,.7):
            subset=[r for r in byimage if r['method']==method and r['match_iou']==threshold]
            nr=sum(x['real_count'] for x in subset);npred=sum(x['render_count'] for x in subset);nm=sum(x['matched_count'] for x in subset)
            semantic.append({'method':method,'match_iou':threshold,'real_count':nr,'render_count':npred,'matched_count':nm,
                             'recall_like':nm/nr if nr else None,'precision_like':nm/npred if npred else None,
                             'f1_like':2*nm/(nr+npred) if nr+npred else None})
    table(tables/'yolo_detections.csv',detrows);table(tables/'yolo_matches.csv',matchrows)
    table(tables/'yolo_by_image.csv',byimage);table(tables/'yolo_overall.csv',semantic)
    # Preservar validação selecionada como evidência visual.
    for method in methods:
        r=root/'validation'/method/str(models[method]['selected_step'])/'renders'
        out=figures/'validation'/method;out.mkdir(parents=True)
        for p in (r/'val'/'rgb').rglob('*.png'):shutil.copy2(p,out/p.name)
    stats['training_budget_steps_per_method']={'splatfacto':15000,'nerfacto':15000}
    stats['models']=models;stats['quality']=summary;stats['semantic']=semantic
    save(final/'manifest.json',stats)
    (final/'LEIA_ME.txt').write_text(
        'CÉLULA I V3.6 - 15000 iterações por método - protocolo ' + cfg['split_mode'] + '\n'
        'COLMAP: SfM apenas TRAIN; VAL/TEST por PnP e LM opcional apenas das poses, mapa congelado.\n'
        'Auditoria por vista: tables/pose_registration_quality.csv.\n'
        'CHECKPOINT: selecionado por PSNR médio de VAL antes da avaliação TEST.\n'
        'MÉTRICAS: PSNR/SSIM/LPIPS calculadas em TEST; detecções YOLO = pseudorreferência.\n'
        'OBS: estes resultados NÃO são diretamente comparáveis com nuScenes apenas pela igualdade do número de imagens.\n'
        'Não há checkpoints no ZIP final. Ver manifest.json para rastreabilidade.\n',
        encoding='utf-8'
    )
    def html_table(rows):
        if not rows:return '<p>Sem registros.</p>'
        cols=list(rows[0]);return '<table><tr>'+''.join('<th>'+html.escape(x)+'</th>' for x in cols)+'</tr>'+''.join('<tr>'+''.join('<td>'+html.escape(f'{r[c]:.5f}' if isinstance(r[c],float) else str(r[c]))+'</td>' for c in cols)+'</tr>' for r in rows)+'</table>'
    report="""<!doctype html><html lang="pt-BR"><meta charset="utf-8"><title>Uber — câmera traseira voltada para a janela direita: NeRF x 3DGS</title>
<style>body{font:16px/1.5 sans-serif;max-width:1200px;margin:30px auto;padding:15px}table{border-collapse:collapse;font-size:13px}td,th{border:1px solid #ccc;padding:6px}img{max-width:100%}</style>
<h1>Uber I V3.6 — NeRF e 3DGS: 15.000 iterações por método; 95/10/10</h1><p>Experimento separado de scene-0655. Poses estimadas por COLMAP.
O COLMAP reconstrói a geometria usando apenas imagens TRAIN. VAL e TEST são localizados posteriormente
por registro PnP e, quando seguro, otimização 6-DOF somente da câmera no mapa fixo, sem bundle adjustment conjunto.
A nuvem inicial usa pontos com >=3 observações TRAIN e cores
amostradas somente em TRAIN. Não é LiDAR. Guardas temporais reduzem proximidade imediata, mas não eliminam correlação.</p>
<p>Seleção de checkpoint por PSNR médio em VAL antes da avaliação TEST. PSNR/SSIM/LPIPS em RGB 8-bit,
na resolução vertical preservada. F1-like YOLO mede concordância com detecções em imagens reais, não mAP.</p><h2>Qualidade</h2>"""+html_table(summary)+'<h2>YOLO</h2>'+html_table([x for x in semantic if x['match_iou']==.5])
    report+='<h2>Auditoria geometrica VAL / TEST</h2>'+html_table(stats['query_pose_diagnostics'])
    for folder,label in [(triplets,'Comparações RGB'),(ytriplets,'Comparações YOLO')]:
        report+='<h2>'+label+'</h2>'
        for p in sorted(folder.glob('*.jpg')):report+='<p>'+html.escape(p.stem)+'</p><img loading="lazy" src="'+p.relative_to(final).as_posix()+'">'
    report+="""<h2>Limitações</h2><p>Um vídeo, uma execução, mesma trajetória. Mesmo com SfM TRAIN-only e PnP nos holdouts, as
poses de VAL/TEST dependem de correspondências visuais com o mapa de treino, e suas poses são refinadas
somente por reprojeção 2D-3D em geometria congelada. Reflexos e objetos móveis
podem dificultar reconstrução estática. Não comparar numericamente com nuScenes como se fosse o mesmo benchmark.
Os tempos de treino diferem entre presets; não é um experimento de orçamento computacional igual.</p>
<p>Referência de processamento: <a href="https://docs.nerf.studio/quickstart/custom_dataset.html">Nerfstudio: custom data</a>.</p></html>"""
    (final/'ANALISE_FINAL.html').write_text(report,encoding='utf-8')
    allowed={'.json','.csv','.html','.png','.jpg','.jpeg','.txt'}
    archive=root/'analise_iphone.zip'
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(final.rglob('*')):
            if p.is_symlink():raise RuntimeError('Link não permitido no pacote final.')
            if p.is_file():
                if p.suffix.lower() not in allowed:raise RuntimeError('Intermediário indevido na análise final: '+str(p))
                z.write(p,p.relative_to(final))
    with zipfile.ZipFile(archive) as z:assert z.testzip() is None
    save(root/'completed.json',{'archive':str(archive),'sha256':sha(archive),'summary':summary})
    print('CONCLUÍDO: somente análise pronta para exportar:',archive,flush=True)


if __name__=='__main__':main()

'''
I_WORKER = I_RUN / 'uber_worker.py'
I_WORKER.write_text(I_WORKER_SOURCE, encoding='utf-8')
compile(I_WORKER_SOURCE, str(I_WORKER), 'exec')

I_ENV = dict(globals().get('TCC_SUBPROCESS_ENV', os.environ))
for key in ('PYTHONHOME', 'PYTHONPATH', 'PYTHONUSERBASE', 'PIP_TARGET'):
    I_ENV.pop(key, None)
I_ENV.update({
    'PATH': str(I_PYTHON.parent) + ':/usr/local/cuda-12.8/bin:' + os.environ.get('PATH', ''),
    'PYTHONNOUSERSITE': '1', 'PYTHONUNBUFFERED': '1', 'QT_QPA_PLATFORM': 'offscreen',
    'CUDA_HOME': '/usr/local/cuda-12.8', 'CUDA_PATH': '/usr/local/cuda-12.8',
    'CUDACXX': '/usr/local/cuda-12.8/bin/nvcc', 'VIRTUAL_ENV': str(I_PYTHON.parent.parent)
})
if Path('/usr/local/cuda-12.8/lib64').is_dir():
    I_ENV['LD_LIBRARY_PATH'] = '/usr/local/cuda-12.8/lib64:' + I_ENV.get('LD_LIBRARY_PATH', '')
# Checkpoints carregados aqui são gerados localmente por esta própria execução.
# Compatibilidade com Nerfstudio 1.1.5 e torch >= 2.6 em ns-render.
if I_ENV.get('TORCH_FORCE_WEIGHTS_ONLY_LOAD', '').strip().lower() in ('1', 'true', 'yes', 'y'):
    raise RuntimeError('TORCH_FORCE_WEIGHTS_ONLY_LOAD impede carregar os checkpoints próprios do Nerfstudio 1.1.5.')
I_ENV['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'

print('Vídeo de entrada configurado para o cenário I.')
print('Protocolo I V3.8 (15.000 iterações por método):', I_SPLIT_MODE, '| refino pose-only:', I_REFINE_QUERY_POSES, '| vizinhos:', I_QUERY_MATCH_NEIGHBORS)
print('Execução local:', I_RUN)
print('Geometria COLMAP: somente TRAIN. VAL/TEST: PnP e refinamento pose-only (sem BA).')
I_PROC = subprocess.Popen(
    [str(I_PYTHON), str(I_WORKER), '--config', str(I_RUN / 'run_config.json')],
    cwd=I_RUN, env=I_ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
try:
    with (I_RUN / 'worker.log').open('w') as log:
        for line in I_PROC.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
    I_CODE = I_PROC.wait()
finally:
    if I_PROC.poll() is None:
        I_PROC.terminate()
        I_PROC.wait(timeout=15)
if I_CODE:
    raise RuntimeError('Pipeline interrompido. Consulte o log local: ' + str(I_RUN / 'worker.log'))
I_DONE = json.loads((I_RUN / 'completed.json').read_text())
I_ZIP = Path(I_DONE['archive'])

def i_sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

if i_sha(I_ZIP) != I_DONE['sha256']:
    raise RuntimeError('O ZIP produzido não passou na verificação SHA256.')
if I_SEND_FINAL_TO_DRIVE and os.environ.get('TCC_EXPORT_DIR', '').strip():
    I_DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
    I_DEST = I_DRIVE_RESULTS / ('analise_uber_' + I_SPLIT_MODE + '_' + I_STAMP + '.zip')
    I_TEMP = I_DEST.with_suffix('.zip.partial')
    if I_DEST.exists() or I_TEMP.exists():
        raise FileExistsError(I_DEST)
    try:
        shutil.copy2(I_ZIP, I_TEMP)
        if i_sha(I_TEMP) != I_DONE['sha256']:
            raise RuntimeError('Falha SHA256 ao copiar o ZIP para o Drive.')
        I_TEMP.replace(I_DEST)
    except Exception:
        I_TEMP.unlink(missing_ok=True)
        raise
    print('ZIP FINAL enviado ao Drive:', I_DEST)
print('\nMÉTRICAS DOS MODELOS:')
for row in I_DONE['summary']:
    print(row)
print('Abra ANALISE_FINAL.html dentro do ZIP para inspecionar as imagens.')


# ----------------- ESQUEMA ÚNICO DE RESULTADOS (V1) -----------------
# Biblioteca embutida: esta célula funciona independentemente de H e I.
from pathlib import Path
import importlib.util
_UNIFIED_MODULE_SOURCE = r'''
"""TCC — esquema canônico de resultados v1.
Biblioteca autocontida para uso pelo notebook no Colab. NÃO treina, não infere e
não usa resultados de TEST para alterar modelos. Só organiza e audita artefatos.
"""
from pathlib import Path
from datetime import datetime, timezone
import csv, hashlib, html, json, math, os, shutil, zipfile
from PIL import Image, ImageDraw

SCHEMA = 'TCC_UNIFIED_RESULTS_V1_95_10_10'
DOMAINS = ('real', 'nerfacto', 'splatfacto')
REQUIRED = (
    'ANALISE_FINAL.html', 'LEIA_ME.txt', 'final_experiment_manifest.json',
    'tables/final_experiment_summary.csv', 'tables/quality_by_image.csv',
    'tables/validation_per_view.csv', 'tables/yolo_overall.csv',
    'tables/yolo_by_image.csv', 'tables/yolo_detections.csv',
    'tables/yolo_matches.csv', 'tables/pose_registration_quality.csv',
    'tables/image_index.csv', 'tables/compute_cost.csv',
    'provenance/source_manifest.json', 'provenance/source_run_metadata.json',
)
SUMMARY_COLUMNS = ['dataset_id','method','selected_step','val_psnr_db','test_psnr_db',
 'test_ssim','test_lpips_alex','training_minutes','checkpoint_size_mib',
 'render_total_seconds_including_overhead','real_yolo_count','render_yolo_count',
 'matched_yolo_count','recall_like','precision_like','f1_like',
 'mean_matched_iou','detection_retention_ratio']
QUALITY_COLUMNS = ['dataset_id','image_index','source_image','method','psnr_db','ssim','lpips_alex']
VALIDATION_COLUMNS = ['dataset_id','method','step','image','psnr_db']
YOLO_COLUMNS = ['dataset_id','method','match_iou','real_count','render_count','matched_count',
 'recall_like','precision_like','f1_like','mean_matched_iou','detection_retention_ratio']
YOLO_IMAGE_COLUMNS = ['dataset_id','image_index','method','match_iou','real_count','render_count','matched_count',
 'recall_like','precision_like','f1_like']
DETECTION_COLUMNS = ['dataset_id','image_index','domain','det_index','class_id','confidence','x1','y1','x2','y2']
MATCH_COLUMNS = ['dataset_id','image_index','method','match_iou','real_det_index','render_det_index','class_id','iou']
POSE_COLUMNS = ['dataset_id','image_index','split','pose_source','pose_method','registered','inliers','reprojection_error_px','notes']
INDEX_COLUMNS = ['dataset_id','image_index','source_image','timestamp_or_index', 'real_sha256',
 'nerfacto_sha256','splatfacto_sha256','real_image','nerfacto_image','splatfacto_image',
 'rgb_triplet','yolo_triplet']
COST_COLUMNS = ['dataset_id','method','training_minutes','checkpoint_size_mib','render_total_seconds_including_overhead']

def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for part in iter(lambda:stream.read(1024*1024),b''):h.update(part)
    return h.hexdigest()

def read(path):return json.loads(Path(path).read_text(encoding='utf-8'))

def write(path,obj):
    p=Path(path);p.parent.mkdir(parents=True,exist_ok=True)
    p.write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str)+'\n',encoding='utf-8')

def read_csv(p):
    if not Path(p).is_file():return []
    with open(p,encoding='utf-8-sig',newline='') as stream:return list(csv.DictReader(stream))

def save_csv(p,fields,rows):
    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)
    with p.open('w',encoding='utf-8',newline='') as f:
        wr=csv.DictWriter(f,fieldnames=fields,extrasaction='ignore');wr.writeheader();wr.writerows(rows)

def num(x):
    try:
        v=float(x)
        return v if math.isfinite(v) else None
    except (TypeError,ValueError):return None

def eqfloat(a,b,tol=1e-4):
    if a is None or b is None:return a is None and b is None
    return abs(float(a)-float(b))<=tol

def img(src,target):
    # Preserva os pixels da imagem retificada pelo COLMAP, sem redimensionar.
    # O limite 800 no ffmpeg/COLMAP nao garante resolucao exata 800x450.
    src=Path(src);target=Path(target)
    if not src.is_file():raise FileNotFoundError('Imagem TEST ausente: '+str(src))
    with Image.open(src) as original:
        im=original.convert('RGB')
        if im.width < 2 or im.height < 2:
            raise RuntimeError('Imagem TEST com dimensoes invalidas: '+str(src))
        target.parent.mkdir(parents=True,exist_ok=True)
        im.save(target,format='PNG')

def triplet(files,out,i,title):
    # Três dominios precisam ter as MESMAS dimensoes para uma vista;
    # nao fabricar alinhamento redimensionando um dominio silenciosamente.
    with Image.open(files[0]) as first:
        w,h=first.size
    canvas=Image.new('RGB',(3*w,h+38),'white');draw=ImageDraw.Draw(canvas)
    for j,(domain,p) in enumerate(zip(DOMAINS,files)):
        with Image.open(p) as src:
            if src.size!=(w,h):
                raise RuntimeError(f'Triade TEST {i+1}: {domain} com {src.size}; esperado {(w,h)}')
            canvas.paste(src.convert('RGB'),(j*w,38))
        draw.text((j*w+14,11),f'{title} {i+1:02d} / {domain.upper()}',fill='#17283a')
    out.parent.mkdir(parents=True,exist_ok=True);canvas.save(out,format='JPEG',quality=93)

def normalize(source_kind,source_root,dataset_id,output_root,source_extra=None):
    """Normaliza final G (nuScenes) ou final H/I (capturas próprias).

    Retorna (diretorio_unificado, arquivo_zip); copia somente análise, nunca
    checkpoints, PLY, frames de TRAIN ou vídeo. Tabelas não aplicáveis são vazias,
    com cabeçalhos canônicos e declaração explícita de indisponibilidade.
    """
    if source_kind not in ('nuscenes','video') or dataset_id not in ('nuscenes_scene0655','estacionamento','uber'):
        raise ValueError('Origem ou dataset_id inesperado.')
    src=Path(source_root).resolve();out=Path(output_root)
    if not src.is_dir():raise FileNotFoundError(src)
    if out.exists():raise FileExistsError('Não sobrescrever análise anterior: '+str(out))
    out.mkdir(parents=True)
    for name in ['tables','figures/charts','figures/rgb_triplets','figures/yolo_triplets',
                 'images/real','images/nerfacto','images/splatfacto','provenance']:
        (out/name).mkdir(parents=True,exist_ok=True)
    extra=dict(source_extra or {})
    values={key:[] for key in ('summary','quality','validation','yolo','yolo_image',
                              'detections','matches','pose','index','cost')}
    raw_images={}
    rgb_fig={};yolo_fig={}
    manifest_source=src/('final_experiment_manifest.json' if source_kind=='nuscenes' else 'manifest.json')
    if not manifest_source.is_file():raise FileNotFoundError(manifest_source)
    native=read(manifest_source)
    shutil.copy2(manifest_source,out/'provenance/source_manifest.json')
    metadata={'original_source_kind':source_kind,'original_source_directory':str(src),
              'original_manifest_sha256':sha(manifest_source),
              'dataset_id':dataset_id,'notes':extra}
    if source_kind=='nuscenes':
        sm=read_csv(src/'tables/final_experiment_summary.csv')
        d=read(src/'provenance/D_final_report.json')
        for row in sm:
            method=row['method_key']
            values['summary'].append({'dataset_id':dataset_id, 'method':method,**{k:row.get(k) for k in SUMMARY_COLUMNS if k not in ('dataset_id','method')}})
        for imrow in d['per_view']:
            i=int(imrow['test_view'])-1
            for method in ('nerfacto','splatfacto'):
                values['quality'].append({'dataset_id':dataset_id,'image_index':i,'source_image':imrow['image'],
                    'method':method,'psnr_db':imrow[method+'_psnr_db'],'ssim':imrow[method+'_ssim'],
                    'lpips_alex':imrow[method+'_lpips_alex']})
        # Evidência VAL: caminhos provenientes desta mesma sessão B/C; sem suposições.
        csvs=extra.get('validation_csvs',{})
        for method in ('splatfacto','nerfacto'):
            valpath=Path(csvs.get(method,'/invalid_validation_csv'))
            rows=read_csv(valpath)
            if not rows:
                raise FileNotFoundError('CSV VAL da B/C não encontrado: '+str(valpath))
            for row in rows:
                values['validation'].append({'dataset_id':dataset_id,'method':method,
                    'step':row.get('step'),'image':row.get('image'),'psnr_db':row.get('psnr_db')})
        for row in read_csv(src/'tables/yolo_summary_overall.csv'):
            vals={'dataset_id':dataset_id,'method':row.get('render_domain'),'match_iou':.50}
            vals.update({k:row.get(k) for k in YOLO_COLUMNS if k not in vals});values['yolo'].append(vals)
        for row in read_csv(src/'tables/yolo_summary_by_image.csv'):
            if not eqfloat(num(row.get('match_iou')),0.50):
                continue
            vals={'dataset_id':dataset_id,'method':row.get('render_domain'),'image_index':row.get('image_index'),'match_iou':.50}
            vals.update({k:row.get(k) for k in YOLO_IMAGE_COLUMNS if k not in vals});values['yolo_image'].append(vals)
        for row in read_csv(src/'tables/yolo_detections_all.csv'):
            values['detections'].append({'dataset_id':dataset_id,**{k:row.get(k) for k in DETECTION_COLUMNS if k!='dataset_id'}})
        for row in read_csv(src/'tables/yolo_match_details.csv'):
            values['matches'].append({'dataset_id':dataset_id,'image_index':row.get('image_index'),
                 'method':row.get('render_domain'),'match_iou':.5,'real_det_index':row.get('real_det_index'),
                 'render_det_index':row.get('render_det_index'),'class_id':row.get('class_id'),'iou':row.get('iou')})
        tf=read(src/'provenance/transforms.json')
        fm={Path(p).stem:p for p in tf['test_filenames']}
        for i in range(10):
            for domain in DOMAINS:raw_images[i,domain]=src/f'images/{domain}/eval_img_{i:04d}.png'
            rgb_fig[i]=src/f'figures/rgb_triplets/RGB_TEST_{i:02d}.jpg'
            yolo_fig[i]=src/f'figures/yolo_triplets/YOLO_TEST_{i:02d}.jpg'
            values['pose'].append({'dataset_id':dataset_id,'image_index':i,'split':'test',
               'pose_source':'official_nuscenes_ego_pose_calibrated_sensor',
               'pose_method':'provided_extrinsics','registered':'not_applicable',
               'notes':'Sem COLMAP; sem reprojeção PnP estimada'})
        metadata['geometry']='nuScenes official sensor poses; TRAIN-only LiDAR for Splatfacto'
    else:
        sm=read_csv(src/'summary.csv')
        for row in sm:
            name=row['method']
            models=native['models'][name]
            values['summary'].append({
               'dataset_id':dataset_id,'method':name,'selected_step':row.get('selected_step'),
               'val_psnr_db':row.get('val_psnr_db'),'test_psnr_db':row.get('test_psnr_db'),
               'test_ssim':row.get('test_ssim'),'test_lpips_alex':row.get('test_lpips_alex'),
               'training_minutes':row.get('training_minutes'),
               'checkpoint_size_mib':models.get('checkpoint_size_mib'),
               'render_total_seconds_including_overhead':models.get('render_process_seconds')})
        for row in read_csv(src/'tables/test_per_view.csv'):
            values['quality'].append({'dataset_id':dataset_id,'image_index':-1,
                'source_image':row.get('image'),'method':row.get('method'),
                'psnr_db':row.get('psnr_db'),'ssim':row.get('ssim'),'lpips_alex':row.get('lpips_alex')})
        for row in read_csv(src/'tables/validation_per_view.csv'):
            values['validation'].append({'dataset_id':dataset_id,'method':row.get('method'),
                'step':row.get('step'),'image':row.get('image'),'psnr_db':row.get('psnr_db')})
        for row in read_csv(src/'tables/yolo_overall.csv'):
            if eqfloat(num(row.get('match_iou')),0.5):
                values['yolo'].append({'dataset_id':dataset_id,'method':row.get('method'),
                    **{k:row.get(k) for k in YOLO_COLUMNS if k not in ('dataset_id','method')}})
        for row in read_csv(src/'tables/yolo_by_image.csv'):
            if eqfloat(num(row.get('match_iou')),0.5):
                values['yolo_image'].append({'dataset_id':dataset_id,'method':row.get('method'),
                    'image_index':None,'source_image':row.get('image'),
                    **{k:row.get(k) for k in YOLO_IMAGE_COLUMNS if k not in ('dataset_id','method','image_index')}})
        for row in read_csv(src/'tables/yolo_detections.csv'):
            values['detections'].append({'dataset_id':dataset_id,'image_index':None,
                'source_image':row.get('image'),'domain':row.get('domain'), 'det_index':row.get('id'),
                **{k:row.get(k) for k in ('class_id','confidence','x1','y1','x2','y2')}})
        for row in read_csv(src/'tables/yolo_matches.csv'):
            if eqfloat(num(row.get('match_iou')),0.5):
                values['matches'].append({'dataset_id':dataset_id,'image_index':None,
                    'source_image':row.get('image'),'method':row.get('method'),'match_iou':row.get('match_iou'),
                    'real_det_index':row.get('real_id'),'render_det_index':row.get('render_id'),
                    'class_id':row.get('class_id'),'iou':row.get('iou')})
        posed=read_csv(src/'tables/pose_registration_quality.csv')
        metadata['pose_registration_original_records'] = posed  # Auditoria completa, sem perder colunas nativas.
        for row in posed:
            values['pose'].append({'dataset_id':dataset_id,**{k:row.get(k) for k in POSE_COLUMNS if k!='dataset_id'}})
        allstems=sorted(p.stem for p in (src/'figures/test/real').glob('*.png'))
        if len(allstems)!=10 or len(set(allstems))!=10:
            raise RuntimeError('H/I precisa conter 10 TEST em figures/test/real')
        index_for_stem={stem:i for i,stem in enumerate(allstems)}
        for i,stem in enumerate(allstems):
            for domain in DOMAINS:raw_images[i,domain]=src/'figures/test'/domain/(stem+'.png')
            rgb_fig[i]=src/'figures/rgb_triplets'/(stem+'.jpg')
            yolo_fig[i]=src/'figures/yolo_triplets'/(stem+'.jpg')
        for kind in ('quality','yolo_image','detections','matches'):
            for row in values[kind]:
                stem=row.get('source_image')
                if stem not in index_for_stem:raise RuntimeError(f'Imagem {stem} de {kind} fora do TEST')
                row['image_index']=index_for_stem[stem]
        # Reutilização das métricas reais; NÃO calcular métricas não medidas.
        sem={r['method']:r for r in values['yolo']}
        for row in values['summary']:
            y=sem.get(row['method'],{})
            for key in ('real_count','render_count','matched_count','recall_like','precision_like','f1_like',
                        'mean_matched_iou','detection_retention_ratio'):
                row[{'real_count':'real_yolo_count','render_count':'render_yolo_count',
                     'matched_count':'matched_yolo_count'}.get(key,key)]=y.get(key)
        metadata['geometry']='COLMAP TRAIN only; VAL/TEST PnP against frozen TRAIN model'
        metadata['source_video_hash']=native.get('video_sha256')
    if len(values['summary'])!=2 or set(r['method'] for r in values['summary'])!={'splatfacto','nerfacto'}:
        raise RuntimeError('Resumo deve conter exatamente os dois métodos.')
    if len(values['quality'])!=20 or len(values['yolo'])!=2:
        raise RuntimeError('Qualidade deve ter 20 linhas e resumo semântico duas linhas.')
    if len(values['validation'])!=150:
        raise RuntimeError(f'VAL incompleta: {len(values["validation"])} linhas, esperado 150 (5×10 + 10×10).')
    # Verificação independente das médias originais por vista em TEST.
    for summ in values['summary']:
        m=summ['method'];observed=[q for q in values['quality'] if q['method']==m]
        if len(observed)!=10 or len(set(int(r['image_index']) for r in observed))!=10:
            raise RuntimeError('10 imagens TEST não únicas para '+m)
        for sm_col,col in [('test_psnr_db','psnr_db'),('test_ssim','ssim'),('test_lpips_alex','lpips_alex')]:
            raw=[num(r[col]) for r in observed]
            if any(v is None for v in raw):raise RuntimeError('Métrica incompleta '+m+'/'+col)
            mean=sum(raw)/10
            if not eqfloat(mean,num(summ[sm_col]),tol=1e-4):
                raise RuntimeError(f'Média {m}/{col} não fecha: {mean}, manifesto {summ[sm_col]}')
    original_size=None
    for i in range(10):
        sizes={}
        for domain in DOMAINS:
            with Image.open(raw_images[i,domain]) as source_img:
                sizes[domain]=source_img.size
        if len(set(sizes.values()))!=1:
            raise RuntimeError(f'TEST {i+1}: resolucoes diferentes entre dominios: {sizes}')
        view_size=sizes['real']
        if original_size is None:
            original_size=view_size
        elif view_size!=original_size:
            raise RuntimeError(f'Resolucao diferente entre as vistas TEST: {view_size} != {original_size}')
        paths=[]
        for domain in DOMAINS:
            path=out/'images'/domain/f'eval_img_{i:04d}.png'
            img(raw_images[i,domain],path);paths.append(path)
        for domain,path in zip(DOMAINS,paths):
            if not path.is_file():raise RuntimeError('Imagem ausente '+str(path))
        index={'dataset_id':dataset_id,'image_index':i,
               'source_image':next(q['source_image'] for q in values['quality'] if int(q['image_index'])==i),
               'timestamp_or_index':i}
        for domain,path in zip(DOMAINS,paths):
            index[domain+'_image']=path.relative_to(out).as_posix()
            index[domain+'_sha256']=sha(path)
        for kind,oldmap in [('rgb',rgb_fig),('yolo',yolo_fig)]:
            target=out/f'figures/{kind}_triplets/eval_img_{i:04d}.jpg'
            old=oldmap[i]
            if old.is_file():
                shutil.copy2(old,target)
            elif kind=='rgb':
                triplet(paths,target,i,'TEST')
            else:
                raise FileNotFoundError('Figura YOLO deve existir; não refazer inferência: '+str(old))
            index[kind+'_triplet']=target.relative_to(out).as_posix()
        values['index'].append(index)
    # Integridade: vistas originais versus métricas e inferências.
    if len(values['yolo_image'])!=20 or len(values['detections'])<0:
        raise RuntimeError('Tabela YOLO por vista incompleta (esperado 20).')
    for summ in values['summary']:
        values['cost'].append({key:summ.get(key) for key in COST_COLUMNS})
    files={
        'final_experiment_summary.csv':(SUMMARY_COLUMNS,'summary'),
        'quality_by_image.csv':(QUALITY_COLUMNS,'quality'),
        'validation_per_view.csv':(VALIDATION_COLUMNS,'validation'),
        'yolo_overall.csv':(YOLO_COLUMNS,'yolo'),
        'yolo_by_image.csv':(YOLO_IMAGE_COLUMNS,'yolo_image'),
        'yolo_detections.csv':(DETECTION_COLUMNS,'detections'),
        'yolo_matches.csv':(MATCH_COLUMNS,'matches'),
        'pose_registration_quality.csv':(POSE_COLUMNS,'pose'),
        'image_index.csv':(INDEX_COLUMNS,'index'),
        'compute_cost.csv':(COST_COLUMNS,'cost'),
    }
    for filename,(header,key) in files.items():save_csv(out/'tables'/filename,header,values[key])
    # Gráficos compartilhados, com os mesmos nomes e eixos; dados ausentes indicados como N/D.
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    plots=[('test_psnr_db','PSNR TEST (dB)','test_psnr.png'),
           ('test_ssim','SSIM TEST','test_ssim.png'),
           ('test_lpips_alex','LPIPS Alex TEST (menor e melhor)','test_lpips.png'),
           ('training_minutes','Treinamento (minutos)','training_minutes.png'),
           ('f1_like','YOLO F1-like IoU 0.50','yolo_f1_like.png'),
           ('recall_like','YOLO Recall-like IoU 0.50','yolo_recall_like.png')]
    for key,title,filename in plots:
        fig,ax=plt.subplots(figsize=(7,4.2))
        scores=[num(x.get(key)) for x in values['summary']]
        ax.bar(['Splatfacto','Nerfacto'],[x if x is not None else 0. for x in scores])
        ax.set_title(dataset_id+' — '+title);ax.grid(axis='y',alpha=.2)
        for i,value in enumerate(scores):ax.text(i,(value or 0),f'{value:.3f}' if value is not None else 'N/D',ha='center',va='bottom')
        fig.tight_layout();fig.savefig(out/'figures/charts'/filename,dpi=150);plt.close(fig)
    # HTML único, com links relativos para as imagens presentes no ZIP.
    rows=''.join('<tr>'+''.join(f'<td>{html.escape(str(r.get(k,"")))}</td>' for k in SUMMARY_COLUMNS)+'</tr>' for r in values['summary'])
    hdr=''.join('<th>'+html.escape(k)+'</th>' for k in SUMMARY_COLUMNS)
    figtags=''.join(f'<h3>TEST {i+1:02d}</h3><img src="figures/rgb_triplets/eval_img_{i:04d}.jpg" alt="Tríade RGB TEST {i+1}"><img src="figures/yolo_triplets/eval_img_{i:04d}.jpg" alt="Tríade YOLO TEST {i+1}">' for i in range(10))
    (out/'ANALISE_FINAL.html').write_text('<!doctype html><html lang="pt-BR"><head><meta charset="utf-8"><title>TCC - '+html.escape(dataset_id)+'</title><style>body{max-width:1200px;margin:32px auto;font:15px/1.6 system-ui}table{border-collapse:collapse;display:block;overflow:auto}td,th{border:1px solid #bbb;padding:5px}img{max-width:100%;margin:8px 0}aside{background:#eee;padding:12px}</style></head><body><h1>Comparação Nerfacto × Splatfacto — '+html.escape(dataset_id)+'</h1><aside>95 TRAIN / 10 VAL / 10 TEST. YOLO na imagem real é pseudorreferência, não ground truth humano. Comparação entre datasets tem confundidores geométricos e de conteúdo. Tempos ns-render incluem sobrecarga.</aside><h2>Resumo</h2><table><tr>'+hdr+'</tr>'+rows+'</table><h2>Gráficos</h2>'+''.join(f'<img src="figures/charts/{p[2]}">' for p in plots)+'<h2>Comparações RGB e YOLO</h2>'+figtags+'</body></html>',encoding='utf-8')
    (out/'LEIA_ME.txt').write_text('TCC — ESQUEMA DE RESULTADOS '+SCHEMA+'\n'
        'Dataset: '+dataset_id+'\nTrês domínios: real, nerfacto, splatfacto\n'
        'images/: 30 PNG TEST com nomes idênticos nas três subpastas\n'
        'figures/: 10 tríades RGB, 10 YOLO, seis gráficos com nomes fixos\n'
        'tables/: mesmas tabelas e colunas nos três datasets\n'
        'provenance/: manifesto da execução original, metadados, hashes\n'
        'Não contém checkpoints, vídeos, PLY nem imagens de treino.\n'
        'Colunas não aplicáveis permanecem vazias, não são medidas fabricadas.\n'
        'TEST é usado exclusivamente para avaliação confirmatória; não retunar.\n',encoding='utf-8')
    metadata.update(schema=SCHEMA,split_counts={'train':95,'val':10,'test':10},
                    yolo_real_is_pseudoreference=True,
                    no_new_training=True,no_new_inference=True,
                    aggregation='mean of 10 images; YOLO micro counts',
                    resolution=list(original_size),sources={'native_manifest':str(manifest_source)})
    write(out/'provenance/source_run_metadata.json',metadata)
    file_records=[]
    for p in sorted(out.rglob('*')):
        if p.is_file():
            if p.is_symlink() or p.suffix.lower() in ('.ckpt','.ply','.pt','.mov','.mp4'):
                raise RuntimeError('Arquivo proibido no pacote: '+str(p))
            file_records.append({'path':p.relative_to(out).as_posix(),'sha256':sha(p),'bytes':p.stat().st_size})
    manifest={'experiment':'unified_TCC_result','schema':SCHEMA,'dataset_id':dataset_id,
             'created_utc':datetime.now(timezone.utc).isoformat(),
             'status':'completed','splits':{'train':95,'val':10,'test':10},
             'test_views':10,'rgb_images':30,
             'metrics_not_available':{
                  'nuscenes_scene0655':['COLMAP reprojection errors','matched sensor trajectories with other datasets'],
                  'estacionamento':['nuScenes LiDAR initialisation','official nuScenes poses'],
                  'uber':['nuScenes LiDAR initialisation','official nuScenes poses']
              }[dataset_id],
             'models':values['summary'],
             'limitations':['single sequence per scenario','correlated held-out views',
                 'not a complete closed-loop autonomous driving simulator',
                 'cross-dataset scores are not controlled experiments',
                 'YOLO on REAL is pseudoreference, not human ground truth'],
             'file_integrity_excluding_self':file_records}
    write(out/'final_experiment_manifest.json',manifest)
    for name in REQUIRED:
        if not (out/name).is_file():raise RuntimeError('Falta arquivo canônico: '+name)
    for domain in DOMAINS:
        imgs=list((out/'images'/domain).glob('eval_img_*.png'))
        if len(imgs)!=10:raise RuntimeError(f'{domain} possui {len(imgs)} imagens, não 10')
    archive=out.parent/(f'TCC_RESULTADOS_{dataset_id}_{out.name}.zip')
    if archive.exists():raise FileExistsError(archive)
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=5,allowZip64=True) as z:
        for p in sorted(out.rglob('*')):
            if p.is_file():z.write(p,p.relative_to(out).as_posix())
    with zipfile.ZipFile(archive) as z:
        corrupt=z.testzip()
        if corrupt:raise RuntimeError('ZIP corrompido em '+corrupt)
    return out,archive

def upload_single_zip(path,dataset_id,drive_root=None):
    """Exporta ZIP somente para diretório explicitamente indicado; verifica SHA."""
    path=Path(path)
    if not path.is_file():raise FileNotFoundError(path)
    export_dir = drive_root or os.environ.get('TCC_EXPORT_DIR', '').strip()
    if not export_dir:
        raise ValueError('Defina TCC_EXPORT_DIR explicitamente para habilitar exportação.')
    dest = Path(export_dir).expanduser()
    if not dest.is_absolute():
        raise ValueError('TCC_EXPORT_DIR deve ser um caminho absoluto.')
    dest.mkdir(parents=True, exist_ok=True)
    final=dest/path.name;partial=dest/(path.name+'.partial')
    if final.exists() or partial.exists():raise FileExistsError(final)
    shutil.copy2(path,partial)
    if sha(partial)!=sha(path):
        partial.unlink(missing_ok=True)
        raise RuntimeError('Cópia ao Drive falhou no SHA256')
    partial.replace(final)
    return final

'''
_UNIFIED_MODULE_FILE = Path('/content/tcc_unified_results_v1.py')
_UNIFIED_MODULE_FILE.write_text(_UNIFIED_MODULE_SOURCE, encoding='utf-8')
_spec = importlib.util.spec_from_file_location('tcc_unified_results_v1', _UNIFIED_MODULE_FILE)
_unified = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_unified)

I_SELECTED_WINDOW=json.loads((I_RUN/'window_selection.json').read_text(encoding='utf-8'))
I_SAMPLE_FPS=int(I_SELECTED_WINDOW['fps'])
I_UNIFIED_ROOT = Path('/content/tcc_resultados_padronizados') / ('uber_' + I_STAMP)
I_UNIFIED_DIR, I_UNIFIED_ZIP = _unified.normalize(
    'video', I_RUN/'final', 'uber', I_UNIFIED_ROOT,
    source_extra={'source_video':'private_input',
      'sampling_fps':I_SAMPLE_FPS,
      'window_rule': (f"155 frames at {I_SELECTED_WINDOW['fps']} FPS "
                      f"starting at {I_SELECTED_WINDOW['start_seconds']} seconds; "
                      "selected using COLMAP TRAIN-only"),
      'pose_method':'COLMAP_TRAIN_only_PnP_VAL_TEST',
      'camera_position':'back seat, facing forward, not a calibrated nuScenes front camera'}
)

# ------- EVIDÊNCIAS V4: SEM RENDER TRAIN; VAL + TEST / CUSTO / VIZINHOS -------
from pathlib import Path
import importlib.util
_TCC_EVIDENCE_SOURCE = r'''
"""Extensão V4: imagens reais de TRAIN/VAL/TEST e renderizações somente VAL/TEST.

Mantém 95 RGB reais TRAIN, 10 VAL e 10 TEST; gera 10 VAL por modelo usando
checkpoints já escolhidos na validação. As 10 TEST por modelo são copiadas
dos resultados originais, sem nova avaliação nem retreinamento.
Não renderiza TRAIN nem modifica transforms.json: mantém 95 TRAIN para
carregar sem conflito os embeddings de aparência do Nerfacto-Big.
"""
from pathlib import Path
from datetime import datetime, timezone
from PIL import Image, ImageDraw, ImageOps
import csv, hashlib, html, json, math, os, re, shutil, subprocess, time, zipfile

SPLITS = {'train':95,'val':10,'test':10}
DOMAINS = ('real','nerfacto','splatfacto')
FRAMES_FIELDS = ['dataset_id','split','split_index','chronological_index','source_image','source_type',
 'source_timestamp_us','nominal_time_seconds','real_png','nerfacto_png','splatfacto_png',
 'real_sha256','nerfacto_sha256','splatfacto_sha256','preceding_source_frame',
 'preceding_source_timestamp_us','preceding_gap_ms','following_source_frame',
 'following_source_timestamp_us','following_gap_ms','temporal_neighbor_scope',
 'nearest_train_before','nearest_train_before_gap_ms','nearest_train_after',
 'nearest_train_after_gap_ms','temporal_relation_to_train','render_provenance']
COST_FIELDS = ['dataset_id','method','training_minutes','training_wall_seconds',
 'train_gpu_peak_mib_sampled','train_gpu_peak_sample_interval_seconds',
 'checkpoint_size_mib','val_candidate_render_wall_seconds','test_render_wall_seconds',
 'train_final_render_wall_seconds','val_final_render_wall_seconds',
 'additional_final_renders_wall_seconds','additional_final_render_gpu_peak_mib_sampled',
 'render_total_seconds_including_overhead','test_render_seconds_per_view_including_overhead',
 'gpu_name','gpu_memory_measurement_note','render_timing_note','train_final_render_views','val_final_render_views','test_render_views']


def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(1024*1024),b''):h.update(block)
    return h.hexdigest()


def rd(path):return json.loads(Path(path).read_text(encoding='utf-8'))


def wr(path,object_):
    p=Path(path);p.parent.mkdir(parents=True,exist_ok=True)
    p.write_text(json.dumps(object_,indent=2,ensure_ascii=False,default=str,allow_nan=False)+'\n',encoding='utf-8')


def table(path,fields,rows):
    p=Path(path);p.parent.mkdir(parents=True,exist_ok=True)
    with p.open('w',newline='',encoding='utf-8') as f:
        writer=csv.DictWriter(f,fieldnames=fields,extrasaction='ignore')
        writer.writeheader();writer.writerows(rows)


def readtable(path):
    with Path(path).open('r',newline='',encoding='utf-8-sig') as f:return list(csv.DictReader(f))


def numeric(value):
    try:
        if value is None or value=='':return None
        n=float(value)
        return n if math.isfinite(n) else None
    except (ValueError,TypeError):return None


def pixel_png(source,destination):
    # Mantem resolucao NATIVA retificada: 791x444, 439x782, 800x450 etc.
    source=Path(source);destination=Path(destination)
    if not source.is_file():raise FileNotFoundError(source)
    destination.parent.mkdir(parents=True,exist_ok=True)
    with Image.open(source) as im:
        if im.width<2 or im.height<2:
            raise RuntimeError(f'{source}: dimensoes invalidas {im.size}')
        if im.mode=='RGB' and (im.format or '').upper()=='PNG':
            # Identidade pixel-a-pixel; evita transcoding sem necessidade.
            shutil.copy2(source,destination)
        else:
            im.convert('RGB').save(destination,format='PNG')
    return sha(destination)


def png_index(root,split,kind,stems):
    root=Path(root)/split/kind
    paths=list(root.rglob('*.png'))
    mapping={p.stem:p for p in paths}
    if len(paths)!=len(mapping) or set(mapping)!=set(stems):
        raise RuntimeError(f'{root}: faltam ou sobram PNGs; previsto={len(stems)}, obtido={len(paths)}')
    return mapping


def gpu_used_for_process(pid):
    try:
        p=subprocess.run(['nvidia-smi','--query-compute-apps=pid,used_gpu_memory',
                          '--format=csv,noheader,nounits'],capture_output=True,text=True,timeout=7)
        if p.returncode:return None
        readings=[]
        for line in p.stdout.splitlines():
            parts=[x.strip() for x in line.split(',')]
            if len(parts)<2:continue
            if parts[0].isdigit() and int(parts[0])==pid:
                value=numeric(parts[1].split()[0])
                if value is not None:readings.append(value)
        return max(readings) if readings else None
    except (FileNotFoundError,subprocess.TimeoutExpired):return None


def execute_render(command,log_path,timeout=7200):
    log_path=Path(log_path);log_path.parent.mkdir(parents=True,exist_ok=True)
    started=time.monotonic();peak=None;sample_count=0
    env=dict(os.environ)
    for key in ('PYTHONPATH','PYTHONHOME','PYTHONUSERBASE','PIP_TARGET'):
        env.pop(key,None)
    env['PYTHONNOUSERSITE']='1';env['PYTHONUNBUFFERED']='1'
    # Compatibilidade: Nerfstudio 1.1.5 usa torch.load sem weights_only explícito.
    # No PyTorch >= 2.6 o valor padrão mudou para True; alguns checkpoints
    # próprios da execução contêm escalares NumPy não permitidos nesse modo.
    # Aplicar apenas ao subprocesso ns-render com checkpoints confiáveis,
    # previamente validados pelos hashes do pipeline. NÃO altera os pesos.
    if env.get('TORCH_FORCE_WEIGHTS_ONLY_LOAD','').strip().lower() in ('1','true','yes','y'):
        raise RuntimeError(
            'TORCH_FORCE_WEIGHTS_ONLY_LOAD está ativado no ambiente: '
            'remova-o antes de renderizar checkpoints próprios do Nerfstudio 1.1.5.'
        )
    env['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD']='1'
    with log_path.open('w',encoding='utf-8') as log:
        log.write('CMD: '+' '.join(map(str,command))+'\n\n');log.flush()
        process=subprocess.Popen(list(map(str,command)),stdout=log,stderr=subprocess.STDOUT,env=env)
        try:
            while process.poll() is None:
                if time.monotonic()-started>timeout:
                    process.kill();process.wait()
                    raise TimeoutError(f'ns-render demorou mais que {timeout}s: {log_path}')
                sample=gpu_used_for_process(process.pid)
                if sample is not None:
                    peak=sample if peak is None else max(peak,sample)
                    sample_count+=1
                time.sleep(2)
            if process.returncode:
                raise RuntimeError(f'Falha do ns-render: {log_path}\n'+log_path.read_text(errors='replace')[-7000:])
        finally:
            if process.poll() is None:process.kill();process.wait()
    text=log_path.read_text(errors='replace')
    if not re.search('Done loading checkpoint from',text):
        raise RuntimeError('ns-render não confirmou carregamento de checkpoint: '+str(log_path))
    return {'wall_seconds':round(time.monotonic()-started,4),
            'gpu_peak_mib_sampled':peak,'gpu_memory_observations':sample_count,
            'gpu_memory_sampling_period_seconds':2,'log':str(log_path)}


def video_source_neighbor(run,stem,offset):
    m=re.fullmatch(r'frame_(\d+)',stem)
    if not m:return None
    n=int(m.group(1))+offset
    if n<1:return None
    source=Path(run)/'raw_frames'/f'frame_{n:06d}.png'
    return source if source.is_file() else None


def picture_neighbors(before,current,after,output,title):
    output=Path(output);output.parent.mkdir(parents=True,exist_ok=True)
    image=Image.new('RGB',(1200,260),(248,248,248))
    drawer=ImageDraw.Draw(image)
    for col,(src,label) in enumerate(((before,'ANTERIOR'),(current,'ATUAL'),(after,'POSTERIOR'))):
        drawer.text((col*400+6,5),label,fill=(17,28,43))
        if src is not None and Path(src).is_file():
            with Image.open(src) as im:
                im=ImageOps.contain(im.convert('RGB'),(400,225),Image.Resampling.LANCZOS)
                image.paste(im,(col*400+(400-im.width)//2,30+(225-im.height)//2))
        else:
            drawer.text((col*400+75,135),'INDISPONÍVEL',fill=(105,105,105))
    image.save(output,quality=88)


def extend(dataset_id,source_kind,run_root,dataset,unified_dir,unified_zip,
           selected_configs,selected_steps,ns_render,training_stats=None,
           extra_source_frames=None):
    """Atualiza diretório e ZIP canônico já normalizados; nunca treina/seleciona.

    source_kind = 'nuscenes' ou 'video', run_root contém outputs originais
    dataset tem transforms.json e as 115 imagens retificadas/normalizadas.
    """
    root=Path(run_root);dataset=Path(dataset);out=Path(unified_dir);zipfile_path=Path(unified_zip)
    if not all(x.exists() for x in (root,dataset,out,zipfile_path)):
        raise FileNotFoundError('Dados locais A–G/H/I ou ZIP canônico ausentes; é necessário manter a sessão.')
    manifest_path=out/'final_experiment_manifest.json'
    current=rd(manifest_path)
    if current.get('dataset_id')!=dataset_id or current.get('splits')!=SPLITS:
        raise RuntimeError('Arquivo canônico original não é do cenário/split esperado.')
    tf=rd(dataset/'transforms.json')
    frame_by_path={f['file_path']:f for f in tf['frames']}
    bysplit={}
    for sp,count in SPLITS.items():
        paths=tf[sp+'_filenames']
        if len(paths)!=count or len(set(paths))!=count:
            raise RuntimeError(f'Split {sp} não contém {count} frames únicos.')
        if not set(paths)<=set(frame_by_path):raise RuntimeError('Frames ausentes de transforms.json.')
        bysplit[sp]=sorted(paths,key=lambda p:(int(frame_by_path[p].get('timestamp',0)),p))
    if len({x for paths in bysplit.values() for x in paths})!=115:
        raise RuntimeError('Treino/validação/teste não disjuntos.')
    if source_kind not in ('nuscenes','video'):raise ValueError(source_kind)
    # Recupera FPS efetivo da execucao (NUNCA pressupor 4 FPS).
    video_fps=None
    if source_kind=='video':
        video_metadata=rd(root/'final'/'manifest.json')
        video_fps=float(video_metadata['fps_sampled'])
        if not math.isfinite(video_fps) or video_fps<=0:
            raise RuntimeError('FPS de amostragem invalido no manifesto do video')
    extra_source_frames={int(k):Path(v) for k,v in (extra_source_frames or {}).items()}
    for model in ('splatfacto','nerfacto'):
        config=Path(selected_configs[model]);step=int(selected_steps[model])
        if not config.is_file():raise FileNotFoundError('Config selecionado ausente: '+str(config))
        if step<0:raise ValueError(step)
    if not Path(ns_render).is_file():raise FileNotFoundError(ns_render)
    # Este manifesto é de 115 capturas, não das 40 imagens excluídas.
    canon_test=readtable(out/'tables/image_index.csv')
    if len(canon_test)!=10:raise RuntimeError('Manifesto canônico de TEST não tem dez linhas.')
    test_source_by_stem={str(r['source_image']):int(r['image_index']) for r in canon_test}
    if set(test_source_by_stem)!={Path(p).stem for p in bysplit['test']}:
        raise RuntimeError('Identidade TEST do pacote não corresponde a transforms.json.')

    # Só renderiza as 10 vistas VAL após seleção do checkpoint. As dez TEST
    # são copiadas dos resultados já avaliados, preservando o protocolo final.
    # O transforms original conserva 95 TRAIN, requisito para o embedding
    # de aparência [95,128] aprendido pelo Nerfacto-Big. Não editar TRAIN.
    source_transforms_sha=sha(dataset/'transforms.json')
    results={};base_render=out.parent/('_render_val_test_'+dataset_id+'_'+out.name)
    base_render.mkdir(parents=True,exist_ok=False)
    for method in ('splatfacto','nerfacto'):
        results[method]={}
        split='val'
        output=base_render/method/split
        output.mkdir(parents=True,exist_ok=False)
        paths=bysplit['val']
        command=[str(ns_render),'dataset','--load-config',str(selected_configs[method]),
          '--output-path',str(output),'--data',str(dataset),'--split','val',
          '--image-format','png','--rendered-output-names','rgb','gt-rgb']
        print(f'[EVIDÊNCIA V4] {dataset_id}/{method}/val: 10 vistas. '
              'TRAIN não será renderizado; TEST será reutilizado.',flush=True)
        results[method]['val']=execute_render(command,base_render/f'{method}_val.log')
        results[method]['val']['original_data_split']='val'
        results[method]['val']['render_split']='val'
        results[method]['val']['temporary_dataset']=False
        stems=[Path(p).stem for p in paths]
        rgb=png_index(output,'val','rgb',stems)
        gt=png_index(output,'val','gt-rgb',stems)
        # Referências de validação devem continuar idênticas às imagens originais.
        import numpy as np
        for rel in paths:
            stem=Path(rel).stem
            with Image.open(dataset/rel) as a, Image.open(gt[stem]) as b:
                ar=np.asarray(a.convert('RGB'),dtype=np.int16)
                br=np.asarray(b.convert('RGB'),dtype=np.int16)
                if (ar.ndim!=3 or ar.shape[2]!=3 or br.shape!=ar.shape
                        or int(np.max(np.abs(ar-br)))>1):
                    raise RuntimeError('Referencia VAL diferente da original: '+method+'/'+rel)
        results[method]['val']['rgb_map']={k:str(v) for k,v in rgb.items()}
    if sha(dataset/'transforms.json')!=source_transforms_sha:
        raise RuntimeError('O transforms.json mudou durante renderização de VAL.')
    # Evidências reais e geradas, sem alterar os arquivos TEST da etapa D/H/I.
    all_frames=[];chronological=[]
    for sp in ('train','val','test'):
        for i,rel in enumerate(bysplit[sp]):
            frame=frame_by_path[rel]
            raw_ts=frame.get('timestamp')
            if source_kind=='video':
                # Timestamp nominal: indice do frame dividido pelo FPS REAL do experimento.
                number=re.search(r'(\d+)$',Path(rel).stem)
                raw_ts=int(number.group(1)) if number else None
                nominal=(raw_ts-1)/video_fps if raw_ts is not None else None
                ordering=nominal if nominal is not None else i
                unit=f'video_sample_at_{video_fps:g}fps'
            else:
                raw_ts=int(raw_ts)
                nominal=raw_ts/1e6
                ordering=raw_ts/1e6
                unit='nuscenes_timestamp_us'
            chronological.append((ordering,rel,sp,i,raw_ts,nominal,unit))
    chronological.sort(key=lambda x:(x[0],x[1]))
    pos={p[1]:idx for idx,p in enumerate(chronological)}
    train_times=sorted([(t,rel) for t,rel,sp,_,_,_,_ in chronological if sp=='train'])
    # nuScenes preserva um grid de 155 amostras incluindo 40 guardas; vídeo preserva os frames brutos.
    raw_grid=[]
    if source_kind=='nuscenes':
        raw_grid=rd(dataset/'dataset_manifest.json').get('split',{}).get('raw_grid',[])
        raw_grid=sorted(raw_grid,key=lambda x:int(x['timestamp_us']))
    time_to_selected_rel={int(f.get('timestamp',-1)):rel for rel,f in frame_by_path.items()} if source_kind=='nuscenes' else {}
    for position,(order,rel,split,i,raw_ts,nominal,unit) in enumerate(chronological):
        stem=Path(rel).stem
        sources={'real':dataset/rel}
        for method in ('nerfacto','splatfacto'):
            if split=='test':
                original_index=test_source_by_stem[stem]
                sources[method]=out/f'images/{method}/eval_img_{original_index:04d}.png'
            elif split=='train':
                sources[method]=None  # Somente imagens reais para TRAIN.
            else:
                sources[method]=Path(results[method][split]['rgb_map'][stem])
        target_paths={};hashes={}
        for domain,src in sources.items():
            if src is None:
                target_paths[domain]=None;hashes[domain]=None
                continue
            dest=out/'images'/split/domain/f'frame_{i:04d}.png'
            hashes[domain]=pixel_png(src,dest)
            target_paths[domain]=dest.relative_to(out).as_posix()
        if split=='test':
            # As dez vistas usadas nas métricas TEST têm de permanecer pixel-idênticas.
            for method in ('nerfacto','splatfacto'):
                if sha(sources[method])!=hashes[method]:
                    with Image.open(sources[method]) as a,Image.open(out/target_paths[method]) as b:
                        import numpy as np
                        if not np.array_equal(np.asarray(a.convert('RGB')),np.asarray(b.convert('RGB'))):
                            raise RuntimeError('TEST PNG alterou pixels ao normalizar: '+stem)
        prev=chronological[position-1] if position else None
        foll=chronological[position+1] if position+1<len(chronological) else None
        scope='selected_115_frames_only'
        previous_source=dataset/prev[1] if prev else None
        following_source=dataset/foll[1] if foll else None
        before_ts=prev[4] if prev else None;after_ts=foll[4] if foll else None
        # Contexto de captura completo quando disponível.
        if source_kind=='video':
            raw_prev=video_source_neighbor(root,stem,-1)
            raw_next=video_source_neighbor(root,stem,+1)
            if raw_prev is not None:
                previous_source=raw_prev
                before_ts=int(re.search(r'(\d+)$',raw_prev.stem).group(1))
            if raw_next is not None:
                following_source=raw_next
                after_ts=int(re.search(r'(\d+)$',raw_next.stem).group(1))
            if raw_prev is not None and raw_next is not None:scope='all_extracted_video_frames'
        elif raw_grid:
            idx=next((n for n,x in enumerate(raw_grid) if int(x['timestamp_us'])==raw_ts),None)
            if idx is not None:
                if idx>0:
                    b=raw_grid[idx-1];before_ts=int(b['timestamp_us'])
                    selected_rel=time_to_selected_rel.get(before_ts)
                    previous_source=dataset/selected_rel if selected_rel else extra_source_frames.get(before_ts)
                    # Captura de guarda sem PNG do próprio A: usar arquivo nuScenes original, se disponível.
                if idx+1<len(raw_grid):
                    after_ts=int(raw_grid[idx+1]['timestamp_us'])
                    selected_rel=time_to_selected_rel.get(after_ts)
                    following_source=dataset/selected_rel if selected_rel else extra_source_frames.get(after_ts)
                scope='nuscenes_155_grid_timestamps_with_raw_neighbors' if extra_source_frames else 'nuscenes_155_grid_timestamps; neighbor_PNG_only_if_selected'
        # segundos no vídeo, microssegundos no nuScenes
        gap_before=((raw_ts-before_ts)*1000.0/video_fps if source_kind=='video' else (raw_ts-before_ts)/1000.) if before_ts is not None else None
        gap_after=((after_ts-raw_ts)*1000.0/video_fps if source_kind=='video' else (after_ts-raw_ts)/1000.) if after_ts is not None else None
        lower=[x for x in train_times if x[0]<order]
        upper=[x for x in train_times if x[0]>order]
        tprev=lower[-1] if lower else None;tnext=upper[0] if upper else None
        if split=='train':relation='training_observation'
        elif tprev and tnext:relation='temporal_interpolation_between_train_observations'
        elif tprev or tnext:relation='temporal_extrapolation_outside_train_span'
        else:relation='no_surrounding_train_observation'
        row={
          'dataset_id':dataset_id,'split':split,'split_index':i,'chronological_index':position,
          'source_image':rel,'source_type':frame.get('source_type','video_frame' if source_kind=='video' else None),
          'source_timestamp_us':raw_ts if source_kind=='nuscenes' else None,
          'nominal_time_seconds':nominal if source_kind=='video' else round(nominal-chronological[0][0],6),
          'real_png':target_paths['real'],'nerfacto_png':target_paths['nerfacto'],
          'splatfacto_png':target_paths['splatfacto'],
          'real_sha256':hashes['real'],'nerfacto_sha256':hashes['nerfacto'],'splatfacto_sha256':hashes['splatfacto'],
          'preceding_source_frame':str(before_ts) if before_ts is not None else None,
          'preceding_source_timestamp_us':before_ts if source_kind=='nuscenes' else None,
          'preceding_gap_ms':gap_before,'following_source_frame':str(after_ts) if after_ts is not None else None,
          'following_source_timestamp_us':after_ts if source_kind=='nuscenes' else None,
          'following_gap_ms':gap_after,'temporal_neighbor_scope':scope,
          'nearest_train_before':tprev[1] if tprev else None,
          'nearest_train_before_gap_ms':round((order-tprev[0])*1000,5) if tprev else None,
          'nearest_train_after':tnext[1] if tnext else None,
          'nearest_train_after_gap_ms':round((tnext[0]-order)*1000,5) if tnext else None,
          'temporal_relation_to_train':relation,
          'render_provenance':('original TEST render from final evaluation' if split=='test'
             else ('selected checkpoint VAL post-selection' if split=='val'
             else 'no TRAIN rendering requested'))
        }
        all_frames.append(row)
        figure=out/'figures/temporal_context'/split/f'frame_{i:04d}.jpg'
        if split in ('val','test'):
            picture_neighbors(previous_source,dataset/rel,following_source,figure,stem)
    assert len(all_frames)==115
    table(out/'tables/all_frames_manifest.csv',FRAMES_FIELDS,sorted(all_frames,key=lambda row:(row['split'],int(row['split_index']))))
    table(out/'tables/frame_temporal_context.csv',FRAMES_FIELDS,sorted(all_frames,key=lambda row:int(row['chronological_index'])))

    # Custos computacionais idênticos em todos os pacotes. Informação ausente NÃO vira zero.
    original_cost=readtable(out/'tables/compute_cost.csv')
    cost_by_method={r['method']:r for r in original_cost}
    summary=readtable(out/'tables/final_experiment_summary.csv')
    original_sum={r['method']:r for r in summary}
    native_manifest=rd(root/'final/manifest.json') if source_kind=='video' else None
    native_model=(native_manifest or {}).get('models',{})
    computed=[]
    for method in ('splatfacto','nerfacto'):
        old=cost_by_method.get(method,{})
        sm=original_sum[method]
        info=native_model.get(method,{}) if source_kind=='video' else {}
        training_min=numeric(old.get('training_minutes'))
        test_seconds=numeric(old.get('render_total_seconds_including_overhead'))
        trainstat=info.get('training_peak_gpu_memory_mib_observed') if source_kind=='video' else (training_stats or {}).get(method,{}).get('peak_gpu_memory_mib_sampled')
        trainvals=info.get('training_peak_gpu_memory_sampling_interval_seconds') if source_kind=='video' else (training_stats or {}).get(method,{}).get('peak_gpu_memory_sampling_interval_seconds')
        rec={
            'dataset_id':dataset_id,'method':method,'training_minutes':training_min,
            'training_wall_seconds':training_min*60 if training_min is not None else None,
            'train_gpu_peak_mib_sampled':trainstat,
            'train_gpu_peak_sample_interval_seconds':trainvals,
            'checkpoint_size_mib':numeric(sm.get('checkpoint_size_mib')),
            'val_candidate_render_wall_seconds':info.get('validation_candidate_render_wall_seconds'),
            'test_render_wall_seconds':test_seconds,
            'train_final_render_wall_seconds':None,
            'val_final_render_wall_seconds':results[method]['val']['wall_seconds'],
            'additional_final_renders_wall_seconds':round(results[method]['val']['wall_seconds'],5),
            'additional_final_render_gpu_peak_mib_sampled':max((v for v in
               (results[method]['val']['gpu_peak_mib_sampled'],)
                 if v is not None),default=None),
            'render_total_seconds_including_overhead':test_seconds,
            'test_render_seconds_per_view_including_overhead':test_seconds/10 if test_seconds is not None else None,
            'gpu_name':(native_manifest or {}).get('gpu') if source_kind=='video' else None,
            'gpu_memory_measurement_note':'nvidia-smi sampled process memory, 2s on extra renders; training GPU N/D if not recorded; never an exact peak',
            'render_timing_note':'TRAIN not rendered; VAL 10 views after checkpoint selection; TEST preserved from original evaluation. Times include startup and PNG writing, not pure FPS',
            'train_final_render_views':0,
            'val_final_render_views':10,
            'test_render_views':10,
        }
        computed.append(rec)
    table(out/'tables/compute_cost.csv',COST_FIELDS,computed)
    wr(out/'provenance/full_frame_render_experiment.json',{
      'dataset_id':dataset_id,'extension_version':'V4_val_test_only',
      'checkpoint_selection':{m:{'selected_step':int(selected_steps[m]),
          'config':str(selected_configs[m])} for m in selected_steps},
      'test_re_rendered':False,'test_heldout_unchanged':True,
      'train_rendered':False,'train_real_count':95,'validation_final_render_count_per_method':10,
      'validation_post_selection_only':True,'original_train_count_preserved':95,
      'original_transforms_sha256_before_and_after':
          [source_transforms_sha,sha(dataset/'transforms.json')],
      'render_profiling':results,'dataset_transforms_sha256':sha(dataset/'transforms.json'),
      'notes':['Nenhuma imagem TRAIN foi renderizada; as 95 observações reais permanecem arquivadas.',
               'VAL foi renderizada usando os checkpoints já selecionados sem nova seleção.',
               'TEST é reutilizado pixel a pixel, sem rerenderização nem reajuste.',
               f'Para video, tempo e posicao nominal na amostragem {video_fps:g} FPS, nao timestamp de hardware.' if source_kind=='video' else 'nuScenes: timestamps oficiais dos sensores.',
               'Para nuScenes, vizinhos físicos guardados podem não ter PNG exportado.',
               'Tempos de subprocesso não são FPS em tempo real.']})
    if source_kind=='video':
        (out/'provenance/video_sampling_and_splits.json').write_text(
            (root/'planned_split.json').read_text(encoding='utf-8'),encoding='utf-8')
    else:
        wr(out/'provenance/nuscenes_temporal_sampling.json',{
           'raw_grid':raw_grid,'grid_source':'A V7 dataset_manifest.split.raw_grid',
           'guard_count':sum(x.get('dataset_split')=='guard' for x in raw_grid)})
    # Novos gráficos focados em custo, ausentes são declarados como N/D.
    try:
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt
        for col,filename,title,unit in (
          ('training_minutes','training_cost_minutes.png','Treinamento (wall time)','min'),
          ('test_render_wall_seconds','test_process_time_seconds.png','TEST: processo completo ns-render','s'),
          ('additional_final_renders_wall_seconds','audit_renders_seconds.png','VAL (10): geração de evidências','s'),
          ('train_gpu_peak_mib_sampled','gpu_train_peak_sampled.png','VRAM observada no treinamento','MiB'),
          ('checkpoint_size_mib','checkpoint_size_mib.png','Tamanho do checkpoint selecionado','MiB')):
            values=[numeric(r.get(col)) for r in computed]
            fig,ax=plt.subplots(figsize=(7,4))
            bars=ax.bar(['Splatfacto','Nerfacto'],[v or 0.0 for v in values])
            ax.set_ylabel(unit);ax.set_title(title);ax.grid(axis='y',alpha=.2)
            for bar,value in zip(bars,values):
                ax.text(bar.get_x()+bar.get_width()/2,bar.get_height(),
                        f'{value:.2f}' if value is not None else 'N/D',ha='center',va='bottom')
            fig.tight_layout();fig.savefig(out/'figures/charts'/filename,dpi=145);plt.close(fig)
    except ImportError:
        print('AVISO: matplotlib ausente; métricas e PNGs ainda são válidos.')
    # Galeria contém 115 imagens reais; apenas VAL/TEST incluem as reconstruções.
    gallery_parts=["<!doctype html><html lang='pt-BR'><meta charset='utf-8'><title>TCC - evidências RGB</title><style>body{font:15px/1.5 system-ui;max-width:1400px;margin:30px auto;padding:12px}.trio{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:12px}img{max-width:100%}article{border-top:1px solid #ccc;padding:15px 0}summary{font-size:21px;cursor:pointer}</style><h1>Evidências RGB: todas as capturas reais</h1><p>115 imagens reais: 95 TRAIN, 10 VAL, 10 TEST. TRAIN contém somente originais. VAL (10) e TEST (10) têm renderizações por método. TEST foi reutilizado da avaliação final, sem rerenderização ou reajuste.</p>"]
    for split in ('train','val','test'):
        subset=sorted([r for r in all_frames if r['split']==split],key=lambda r:int(r['split_index']))
        gallery_parts.append('<details '+('open' if split=='test' else '')+'><summary>'+split.upper()+' — '+str(len(subset))+' capturas</summary>')
        for row in subset:
            i=int(row['split_index']);stem=html.escape(str(row['source_image']))
            gallery_parts.append('<article><h3>'+split.upper()+' '+str(i+1)+' — '+stem+'</h3><p>t = '+html.escape(str(row['nominal_time_seconds']))+' segundos</p><div class="trio">')
            for label,key in [('REAL','real_png'),('NERFACTO','nerfacto_png'),('SPLATFACTO','splatfacto_png')]:
                image_rel=row[key]
                if image_rel:
                    gallery_parts.append('<div><b>'+label+'</b><img loading="lazy" src="'+html.escape(image_rel,quote=True)+'"></div>')
                else:
                    gallery_parts.append('<div><b>'+label+'</b><p>Não renderizada (TRAIN preservado apenas como imagem real)</p></div>')
            gallery_parts.append('</div>')
            neighbor=f'figures/temporal_context/{split}/frame_{i:04d}.jpg'
            if (out/neighbor).exists():
                gallery_parts.append('<p>Captura anterior / atual / posterior:</p><img loading="lazy" src="'+neighbor+'">')
            gallery_parts.append('</article>')
        gallery_parts.append('</details>')
    gallery_parts.append('</html>')
    (out/'GALERIA_EVIDENCIAS_RGB.html').write_text(''.join(gallery_parts),encoding='utf-8')
    old_html=(out/'ANALISE_FINAL.html').read_text(encoding='utf-8')
    old_html=old_html.replace('<body>','<body><p><a href="GALERIA_EVIDENCIAS_RGB.html">Ver imagens reais e reconstruções VAL / TEST</a></p>',1)
    (out/'ANALISE_FINAL.html').write_text(old_html,encoding='utf-8')
    readme=out/'LEIA_ME.txt'
    readme.write_text(readme.read_text(encoding='utf-8')+'\nV4: 115 imagens reais (95 TRAIN / 10 VAL / 10 TEST). Nenhuma TRAIN renderizada. As 10 VAL e 10 TEST possuem reconstruções por método; TEST é cópia pixel-idêntica da avaliação original. Custos documentados em tables/compute_cost.csv; medições ausentes não são zero. Contexto temporal em tables/frame_temporal_context.csv; imagens originais excluídas pela guarda aparecem em metadados, se conhecidos. Galeria: GALERIA_EVIDENCIAS_RGB.html\n',encoding='utf-8')
    m=rd(manifest_path)
    m['evidence_version']='V4_95_10_10_val_test_only'
    m['evidence_counts']={'train_real':95,'val_real':10,'test_real':10,
        'train_reconstructed_per_method':0,
        'val_reconstructed_per_method':10,'test_reconstructed_per_method':10}
    m.pop('selected_train_indices',None)
    m.pop('selected_train_relpaths',None)
    m['test_rerendered']=False
    m['test_metric_protocol_unchanged']=True
    m['full_gallery']='GALERIA_EVIDENCIAS_RGB.html'
    m['temporal_evidence_csv']='tables/frame_temporal_context.csv'
    m['computational_cost_csv']='tables/compute_cost.csv'
    records=[]
    for p in sorted(out.rglob('*')):
        if p.is_file() and p!=manifest_path:
            if p.is_symlink() or p.suffix.lower() in ('.mov','.mp4','.ply','.ckpt','.pt'):
                raise RuntimeError('Arquivo proibido no ZIP final: '+str(p))
            records.append({'path':p.relative_to(out).as_posix(),'sha256':sha(p),'bytes':p.stat().st_size})
    m['file_integrity_excluding_self']=records
    wr(manifest_path,m)
    temporary=zipfile_path.with_suffix('.temporary.zip')
    with zipfile.ZipFile(temporary,'w',zipfile.ZIP_DEFLATED,compresslevel=4,allowZip64=True) as z:
        for p in sorted(out.rglob('*')):
            if p.is_file():z.write(p,p.relative_to(out).as_posix())
    with zipfile.ZipFile(temporary) as z:
        if z.testzip() is not None:raise RuntimeError('ZIP enriquecido corrompido')
        if len([x for x in z.namelist() if x.startswith('images/') and x.endswith('.png')])!=185:
            # 155 imagens organizadas (115 reais + 20 VAL + 20 TEST) + 30 TEST canônicos.
            raise RuntimeError('ZIP V4 incompleto: esperado 185 PNGs no total.')
    temporary.replace(zipfile_path)
    if sha(dataset/'transforms.json')!=source_transforms_sha:
        raise RuntimeError('Dataset original foi alterado pela extensão ilustrativa.')
    print('EVIDÊNCIAS VAL/TEST V4:',out,'| ZIP:',zipfile_path)
    print('TRAIN 95 reais / 0 renderizações; VAL 10/10/10; TEST 10/10/10; 155 imagens organizadas (+30 TEST canônicas).')
    return {'out':out,'zip':zipfile_path,'cost':computed,'frames':len(all_frames),'renders':results}

'''
_TCC_EVIDENCE_FILE = Path('/content/tcc_evidencias_val_test_v4.py')
_TCC_EVIDENCE_FILE.write_text(_TCC_EVIDENCE_SOURCE, encoding='utf-8')
_spec_evidence = importlib.util.spec_from_file_location('tcc_evidencias_val_test_v4',_TCC_EVIDENCE_FILE)
_evidencias = importlib.util.module_from_spec(_spec_evidence)
_spec_evidence.loader.exec_module(_evidencias)

_I_SELECTED = _evidencias.rd(I_RUN/'final/selection_before_test.json')
_I_CONFIGS = {name:I_RUN/'validation'/name/str(int(model['selected_step']))/'config.yml'
 for name,model in _I_SELECTED.items()}
I_EVIDENCE = _evidencias.extend(
 dataset_id='uber',source_kind='video',run_root=I_RUN,
 dataset=I_RUN/'undistorted',unified_dir=I_UNIFIED_DIR,
 unified_zip=I_UNIFIED_ZIP,selected_configs=_I_CONFIGS,
 selected_steps={name:int(info['selected_step']) for name,info in _I_SELECTED.items()},
 ns_render=I_PYTHON.parent/'ns-render'
)
I_UNIFIED_DRIVE = (_unified.upload_single_zip(I_UNIFIED_ZIP,'uber')
                   if os.environ.get('TCC_EXPORT_DIR', '').strip() else None)
I_FINAL_RESULTS_DIR = I_UNIFIED_DIR
I_FINAL_ZIP_PATH = I_UNIFIED_ZIP
print('I V3.8 — ESQUEMA COMUM:', _unified.SCHEMA)
print('Pasta final padronizada:', I_UNIFIED_DIR)
print('Exportação opcional do ZIP:', I_UNIFIED_DRIVE)
